# Choosing a train / validation / test split

**Goal:** decide how to split the Congress data (Congresses 108–118) for the two tasks in the proposal:

1. **Cosponsor prediction** (temporal link prediction): for a bill at time *t* (day 0 or day 7 after
   introduction), rank the eligible legislators who will cosponsor it in *(t, t+30]*.
2. **Bill advancement** (node classification): will a law-capable bill (HR, S, HJRES, SJRES) be
   reported out of committee within 180 days of introduction?

The default idea is a **by-Congress temporal split** (train on earlier Congresses, validate on the
next, test on the last). That split is only trustworthy if a set of conditions hold: similar label
rates, similar feature distributions, enough node overlap for the graph model, no leakage, fully
observed labels, and stable baseline scores across folds. Each section below tests a group of those
conditions and records a **check** (pass / fail / info). The last section collects every check and
compares the candidate splits.

**Run it:** JupyterLab, kernel with `pandas`, `pyarrow`, `numpy`, `scipy` (see
`notebooks/requirements.txt`). Charts use D3 v7, vendored at `notebooks/vendor/d3.v7.min.js` (no
internet needed). Run the cells top to bottom; charts appear as interactive SVGs (hover for values).

In [91]:
# --- Setup: imports, paths, constants -------------------------------------------------------------
import json
import os
import uuid
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def _find_root():
    """Repo root = first parent (from the working dir) that contains data/parsed/."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "parsed").is_dir():
            return p
    raise FileNotFoundError("Couldn't find the data/parsed/ folder; run the notebook from inside the repo.")


ROOT = _find_root()
PARSED = ROOT / "data" / "parsed"
VENDOR_D3 = ROOT / "notebooks" / "vendor" / "d3.v7.min.js"

CONGRESSES = list(range(108, 119))   # complete Congresses used for modeling
IN_PROGRESS = 119                    # still in session: labels censored, excluded from splits
LAW_TYPES = ["hr", "s", "hjres", "sjres"]
WINDOW_DAYS = 30                     # cosponsor prediction horizon
ADV_WINDOW_DAYS = 180                # advancement horizon
PARTY_CODE = {"Republican": "R", "Democratic": "D", "Independent": "I",
              "Independent Democrat": "ID", "Libertarian": "L"}
CHAMBER_OF_ORIGIN = {"House": "House of Representatives", "Senate": "Senate"}


def congress_start(c):
    """Congress c convenes on Jan 3 of 1789 + 2(c-1)."""
    return pd.Timestamp(year=1789 + 2 * (int(c) - 1), month=1, day=3)


def congress_end(c):
    return congress_start(int(c) + 1)


print(f"repo: {ROOT}\nparsed tables: {len(list(PARSED.glob('*.parquet')))}")

repo: /Users/alex_rivas/Documents/Stanford/CS224W/Final Project/cs224w-bill-cosponsor-prediction
parsed tables: 21


In [92]:
# --- Data loading ---------------------------------------------------------------------------------
@lru_cache(maxsize=None)
def load(table, columns=None):
    """Read data/parsed/<table>.parquet. `congress` becomes int; date-like object columns -> datetime."""
    cols = list(columns) if columns else None
    df = pd.read_parquet(PARSED / f"{table}.parquet", columns=cols)
    if "congress" in df.columns:
        df["congress"] = df["congress"].astype(int)
    for c in df.columns:
        if df[c].dtype == object and (c.endswith("_date") or c == "action_date"):
            df[c] = pd.to_datetime(df[c], errors="coerce")
        elif str(df[c].dtype).startswith("datetime64") and getattr(df[c].dt, "tz", None) is not None:
            df[c] = df[c].dt.tz_convert(None)
    return df


bills = load("bills").copy()
bills["law_capable"] = bills["bill_type"].isin(LAW_TYPES)
bills["chamber"] = bills["origin_chamber"].map(CHAMBER_OF_ORIGIN)
bills["day_of_congress"] = (bills["introduced_date"] - bills["congress"].map(congress_start)).dt.days
# Harmonised CRS policy area: categories absent from at least one complete Congress (taxonomy changes, e.g.
# a retired category) and missing labels fold into one bucket, so every section uses the same taxonomy.
_pa = bills[bills.law_capable & bills.congress.isin(CONGRESSES)].assign(pa=lambda d: d.policy_area.fillna("(missing)"))
_pa_tab = pd.crosstab(_pa.pa, _pa.congress)
PA_UNSTABLE = sorted(set(_pa_tab.index[(_pa_tab == 0).any(axis=1)]) | {"(missing)"})
bills["policy_area_harm"] = bills.policy_area.fillna("(missing)").where(
    lambda s: ~s.isin(PA_UNSTABLE), "(unstable/missing)")

# Data snapshot date = latest record modification (bills.update_date), normalised to the day. Used for record
# ages in the in-progress Congress (the only one observed before CRS and committees catch up).
SNAPSHOT_DATE = bills["update_date"].max().normalize()
SNAPSHOT_LATEST_INTRO = bills["introduced_date"].max()
# Policy area missing at prediction time (proxy): share of the in-progress Congress's law-capable bills without a
# CRS policy area, by age at SNAPSHOT_DATE. PA_MISS_T0 = the youngest bucket (<= 7 days), with its n.
PA_AGE_BUCKETS = [(0, 7, "0-7d"), (8, 30, "8-30d"), (31, 90, "31-90d"), (91, 180, "91-180d"), (181, 10**6, ">180d")]
_b_ip = bills[bills.law_capable & (bills.congress == IN_PROGRESS)]
_age_ip = (SNAPSHOT_DATE - _b_ip.introduced_date).dt.days
PA_MISSING_AT_T0 = pd.DataFrame([dict(age_bucket=lab, age_lo=lo, age_hi=min(hi, int(_age_ip.max())),
                                      n=int(m.sum()), n_missing=int(_b_ip.policy_area[m].isna().sum()),
                                      share_missing=float(_b_ip.policy_area[m].isna().mean()) if m.any() else np.nan)
                                 for lo, hi, lab in PA_AGE_BUCKETS for m in [_age_ip.between(lo, hi)]]
                                ).set_index("age_bucket")
PA_MISS_T0 = float(PA_MISSING_AT_T0.loc["0-7d", "share_missing"])
PA_MISS_T0_N = int(PA_MISSING_AT_T0.loc["0-7d", "n"])

members = load("members").copy()
members["party_code"] = members["party"].map(PARTY_CODE).fillna(members["party"])   # LATEST party (legacy)

# Member terms (year granularity) and party history: fill missing chambers and derive the party AS OF each
# Congress. As-of rule: the party-history interval containing Jan 1 of the Congress's second year
# (start_year <= first year and end_year >= second year); members not in office then (joined in year 2 or
# left in year 1) take the latest interval containing their term start. Fallback: latest party.
_terms = load("member_terms").copy()
_terms["y0"] = 1789 + 2 * (_terms.congress - 1)
_ch_fill = (_terms.sort_values("start_year").drop_duplicates(["bioguide_id", "congress"], keep="last")
            .set_index(["bioguide_id", "congress"])["chamber"])
_n_ch_missing = int(members.chamber.isna().sum())
members["chamber"] = members["chamber"].fillna(
    pd.Series(list(zip(members.bioguide_id, members.congress))).map(_ch_fill).set_axis(members.index))


def _party_asof_table():
    """(bioguide_id, congress) -> party code as of that Congress, from member_party_history."""
    ph = load("member_party_history", ("bioguide_id", "party_code", "start_year", "end_year"))
    t = (_terms.sort_values("start_year").drop_duplicates(["bioguide_id", "congress"])
         [["bioguide_id", "congress", "y0", "start_year"]].rename(columns={"start_year": "t0"}))
    h = t.merge(ph.rename(columns={"start_year": "h0", "end_year": "h1"}), on="bioguide_id")
    h1 = h.h1.fillna(9999)
    h["prio"] = np.select([(h.h0 <= h.y0) & (h1 >= h.y0 + 1), (h.h0 <= h.t0) & (h1 >= h.t0)], [2, 1], 0)
    h = h[h.prio > 0].sort_values(["prio", "h0"]).drop_duplicates(["bioguide_id", "congress"], keep="last")
    return h.set_index(["bioguide_id", "congress"])["party_code"]


_pasof = _party_asof_table()
members["party_code_asof"] = (pd.Series(list(zip(members.bioguide_id, members.congress))).map(_pasof)
                              .set_axis(members.index).fillna(members["party_code"]))
_n_switch = int((members.party_code_asof != members.party_code).sum())

print(f"bills: {len(bills):,} ({bills.law_capable.sum():,} law-capable); "
      f"members rows: {len(members):,}; congresses: {bills.congress.min()}-{bills.congress.max()}")
print(f"members: chamber filled from member_terms for {_n_ch_missing - int(members.chamber.isna().sum())}/"
      f"{_n_ch_missing} rows; party_code_asof differs from latest party_code in {_n_switch} member-Congress rows; "
      f"policy areas folded into '(unstable/missing)': {len(PA_UNSTABLE)}")
print(f"snapshot (max update_date): {SNAPSHOT_DATE.date()} (latest introduced_date {SNAPSHOT_LATEST_INTRO.date()}); "
      f"policy area missing among {IN_PROGRESS}th law-capable bills by age at snapshot: "
      + ", ".join(f"{r.Index} {r.share_missing:.0%} (n={r.n})" for r in PA_MISSING_AT_T0.itertuples())
      + f" -> PA_MISS_T0 = {PA_MISS_T0:.1%} (n={PA_MISS_T0_N}, small-sample proxy)")

bills: 173,437 (145,738 law-capable); members rows: 6,638; congresses: 108-119
members: chamber filled from member_terms for 75/75 rows; party_code_asof differs from latest party_code in 30 member-Congress rows; policy areas folded into '(unstable/missing)': 4
snapshot (max update_date): 2026-10-17 (latest introduced_date 2026-10-15); policy area missing among 119th law-capable bills by age at snapshot: 0-7d 86% (n=56), 8-30d 14% (n=585), 31-90d 1% (n=1123), 91-180d 0% (n=2160), >180d 0% (n=13023) -> PA_MISS_T0 = 85.7% (n=56, small-sample proxy)


In [93]:
# --- Shared task definitions (every section uses these, so labels are identical everywhere) -------
def build_cosponsor_events():
    """One row per active (non-withdrawn) cosponsorship with its lag from introduction (days)."""
    c = load("bill_cosponsors", ("bill_id", "bioguide_id", "party", "sponsorship_date", "is_original",
                                 "withdrawn_date", "congress"))
    c = c[c["withdrawn_date"].isna()].merge(
        bills[["bill_id", "introduced_date", "sponsor_bioguide_id", "sponsor_party", "chamber",
               "bill_type", "law_capable"]], on="bill_id", how="inner")
    c["lag_days"] = (c["sponsorship_date"] - c["introduced_date"]).dt.days
    return c


cosp = build_cosponsor_events()


def window_targets(t0, horizon=WINDOW_DAYS, law_capable_only=True, point_in_time=False, include_delegates=True):
    """Per bill: number of new cosponsors in (t0, t0+horizon] and the size of the eligible pool.

    Eligible = members of the bill's chamber, minus the sponsor and anyone who had already cosponsored by
    day t0. point_in_time=False (default, legacy): pool = everyone who served in that chamber that Congress.
    point_in_time=True: pool = members in office in the month of introduced_date + t0 (clipped into the bill's
    Congress); adds column `pool_month`. include_delegates=True (default; Task-1 pool): the pool comes from
    ROSTER_ASOF_ALL (voting members + Delegates / Resident Commissioners, who can cosponsor); False: the
    voting-only ROSTER_ASOF (pre-change behaviour). Bills whose window runs past the end of the Congress are
    flagged `censored` (exclude them from evaluation)."""
    b = bills[bills.law_capable] if law_capable_only else bills
    b = b[b.congress.isin(CONGRESSES + [IN_PROGRESS])][["bill_id", "congress", "chamber", "introduced_date"]].copy()
    pos = cosp[(cosp.lag_days > t0) & (cosp.lag_days <= t0 + horizon)].groupby("bill_id").size()
    seen = cosp[cosp.lag_days <= t0].groupby("bill_id").size()
    b["n_pos"] = b["bill_id"].map(pos).fillna(0).astype(int)
    b["n_seen"] = b["bill_id"].map(seen).fillna(0).astype(int)
    if point_in_time:
        ref = (b["introduced_date"] + pd.Timedelta(days=t0)).clip(
            upper=b["congress"].map(congress_end) - pd.Timedelta(days=ROSTER_REF_DAY + 5))
        b["pool_month"] = ref.dt.to_period("M").dt.to_timestamp()
        pool = (ROSTER_ASOF_ALL if include_delegates else ROSTER_ASOF).groupby(["month", "congress", "chamber"])["bioguide_id"].size()
        b["pool"] = pool.reindex(pd.MultiIndex.from_arrays([b.pool_month, b.congress, b.chamber])).to_numpy(float)
        b["eligible"] = (b["pool"] - 1 - b["n_seen"]).clip(lower=0)
    else:
        pool = members.dropna(subset=["chamber"]).groupby(["congress", "chamber"])["bioguide_id"].nunique()
        b["pool"] = [pool.get((c, ch), np.nan) for c, ch in zip(b.congress, b.chamber)]
        b["eligible"] = b["pool"] - 1 - b["n_seen"]
    b["censored"] = b["introduced_date"] + pd.Timedelta(days=t0 + horizon) > b["congress"].map(congress_end)
    return b


def adv_cutoff(c):
    """Last introduction date whose 180-day advancement window ends inside Congress c (later = censored)."""
    return congress_end(c) - pd.Timedelta(days=ADV_WINDOW_DAYS)


def build_advancement_labels(window=ADV_WINDOW_DAYS):
    """Per law-capable bill: reported out of committee within `window` days; `censored` if the window
    runs past the end of the Congress (unpassed bills expire at adjournment).

    Alternative label (label_alt): reported OR passed the chamber of origin within `window` days, which also
    counts bills that advance without a committee report (suspension / unanimous consent).
    `passed_origin_date` = first 'Passed/agreed to in <origin chamber>' action. (Named passed_origin_date, not
    passed_date, so it does not collide with section A's local `passed_date` merge.)"""
    a = load("bill_actions", ("bill_id", "action_date", "action_type", "text", "congress"))
    rep = a[a.action_type.isin(["Committee", "Calendars"])
            & a.text.str.contains(r"\bReported\b", regex=True, na=False)
            & ~a.text.str.contains("Ordered to be", na=False)]
    first_rep = rep.groupby("bill_id")["action_date"].min()
    pas = a[a.text.str.match(r"Passed/agreed to in (House|Senate)", na=False)]
    pas = pas.assign(chamber=pas.text.str.extract(r"in (House|Senate)", expand=False).map(CHAMBER_OF_ORIGIN))
    first_pass = pas.groupby(["bill_id", "chamber"])["action_date"].min()
    b = bills[bills.law_capable][["bill_id", "congress", "chamber", "introduced_date", "sponsor_party",
                                  "policy_area", "policy_area_harm"]].copy()
    b["reported_date"] = b["bill_id"].map(first_rep)
    days = (b["reported_date"] - b["introduced_date"]).dt.days
    b["label"] = days.between(0, window).astype(int)
    b["censored"] = b["introduced_date"] + pd.Timedelta(days=window) > b["congress"].map(congress_end)
    b["passed_origin_date"] = first_pass.reindex(pd.MultiIndex.from_arrays([b.bill_id, b.chamber])).to_numpy()
    pdays = (b["passed_origin_date"] - b["introduced_date"]).dt.days
    b["label_alt"] = ((b["label"] == 1) | pdays.between(0, window)).astype(int)
    return b


# --- Who is in office when, and which party they caucus with -----------------------------------------
NONVOTING_TYPES = ("Delegate", "Resident Commissioner")   # member_terms.member_type of non-voting members


@lru_cache(maxsize=None)
def _activity_span():
    """First / last dated activity (sponsorship or cosponsorship) per (congress, bioguide_id)."""
    act = pd.concat([cosp[["congress", "bioguide_id", "sponsorship_date"]].set_axis(["congress", "bioguide_id", "d"], axis=1),
                     bills[["congress", "sponsor_bioguide_id", "introduced_date"]].set_axis(["congress", "bioguide_id", "d"], axis=1)])
    return act.dropna().groupby(["congress", "bioguide_id"])["d"].agg(first="min", last="max")


def in_office_at(date, voting_only=True, slack_days=30):
    """Members in office on `date`: DataFrame (bioguide_id, congress, chamber, state_code, district).

    member_terms has year granularity, so a term of the Congress containing `date` counts if
    start_year <= year(date) <= end_year. Then, using activity dates (first / last sponsorship or
    cosponsorship in that Congress; activity is not chamber-specific):
      * a member with terms in both chambers keeps the earlier one (the switch happens later that year);
      * seats with more members than they hold (1 per House district, 2 per Senate state) keep the members
        whose activity span is closest to `date` (ties: the later arrival);
      * other terms starting or ending mid-Congress in year(date) need activity within +-slack_days of date.
    Approximate by construction (year-granular terms). voting_only drops Delegates / Resident Commissioners."""
    date = pd.Timestamp(date)
    y = date.year
    c = 1 + (y - 1789) // 2 - (1 if (y % 2 == 1 and date < pd.Timestamp(year=y, month=1, day=3)) else 0)
    t = _terms[(_terms.congress == c) & (_terms.start_year <= y) & (_terms.end_year.fillna(9999) >= y)]
    if voting_only:
        t = t[~t.member_type.isin(NONVOTING_TYPES)]
    t = t.sort_values("start_year", kind="stable").drop_duplicates("bioguide_id").copy()
    t["seat"] = t.state_code + "-" + np.where(t.chamber == "Senate", "S", t.district.fillna(0).astype(int).astype(str))
    span = _activity_span().reindex(pd.MultiIndex.from_arrays([t.congress, t.bioguide_id]))
    first, last = span["first"].set_axis(t.index), span["last"].set_axis(t.index)
    t["dist"] = (pd.concat([(first - date).dt.days, (date - last).dt.days], axis=1).max(axis=1)
                 .clip(lower=0).fillna(1e9))
    t["first"] = first
    cap = np.where(t.chamber == "Senate", 2, 1)
    over = t.groupby(["chamber", "seat"])["bioguide_id"].transform("size").to_numpy() > cap
    t["rank"] = (t.sort_values(["dist", "first"], ascending=[True, False]).groupby(["chamber", "seat"]).cumcount()
                 .reindex(t.index))
    edge = (((t.start_year == y) & (y != t.y0)) | ((t.end_year == y) & (y != t.y0 + 2))).to_numpy()
    keep = np.where(over, t["rank"].to_numpy() < cap, ~edge | (t["dist"].to_numpy() <= slack_days))
    return t.loc[keep, ["bioguide_id", "congress", "chamber", "state_code", "district"]]


def congress_midpoint(c):
    """Reference date for point-in-time rosters: one year after the Congress convenes."""
    return congress_start(c) + pd.Timedelta(days=365)


# Caucus: D/R members caucus with their as-of party. Others (I, ID, L) are assigned from the data: the party
# whose bills they cosponsor more often in that Congress (share of D vs R sponsors > 0.5); exact ties -> 'other'.
CAUCUS_MIN_SHARE = 0.5
_dshare = (cosp[cosp.sponsor_party.isin(["D", "R"])].assign(isd=lambda d: d.sponsor_party == "D")
           .groupby(["bioguide_id", "congress"])["isd"].mean())
_mk = pd.MultiIndex.from_arrays([members.bioguide_id, members.congress])
members["caucus"] = np.where(members.party_code_asof.isin(["D", "R"]), members.party_code_asof,
                             np.select([_dshare.reindex(_mk).to_numpy() > CAUCUS_MIN_SHARE,
                                        _dshare.reindex(_mk).to_numpy() < 1 - CAUCUS_MIN_SHARE], ["D", "R"], "other"))

# Point-in-time roster: members in office on day ROSTER_REF_DAY of each month (in_office_at), from the start of
# the first complete Congress to the snapshot. One row per (month, member).
#   ROSTER_ASOF_ALL: voting members + Delegates / Resident Commissioners (column `voting` = False for them). This is
#     the Task-1 eligible pool (delegates can cosponsor; they cast most positives outside the voting-only pool, C17)
#     and the default of window_targets(point_in_time=True) and roster_asof(date).
#   ROSTER_ASOF: voting members only (unchanged; use for seat / majority counts, which exclude delegates).
# roster_asof(date, include_delegates=True) returns the rows of the month containing `date`.
ROSTER_REF_DAY = 15
_ROSTER_COLS = ["month", "congress", "chamber", "bioguide_id", "caucus", "party_code_asof", "state_code", "district"]


def _build_roster_asof():
    """ROSTER_ASOF_ALL (all members, with `voting`). Voting rows come from in_office_at(voting_only=True) so the
    voting-only roster is exactly the pre-change ROSTER_ASOF; delegates are added from voting_only=False."""
    months = pd.date_range(congress_start(CONGRESSES[0]).to_period("M").to_timestamp(), SNAPSHOT_DATE, freq="MS")
    cau = members.drop_duplicates(["bioguide_id", "congress"]).set_index(["bioguide_id", "congress"])
    nonvoting = set(_terms.loc[_terms.member_type.isin(NONVOTING_TYPES), "bioguide_id"])
    parts = []
    for m in months:
        d = m + pd.Timedelta(days=ROSTER_REF_DAY - 1)
        v = in_office_at(d).assign(voting=True)
        a = in_office_at(d, voting_only=False)
        dl = a[a.bioguide_id.isin(nonvoting) & ~a.bioguide_id.isin(v.bioguide_id)].assign(voting=False)
        parts += [v.assign(month=m), dl.assign(month=m)]
    r = pd.concat(parts, ignore_index=True)
    k = pd.MultiIndex.from_arrays([r.bioguide_id, r.congress])
    r["caucus"] = k.map(cau["caucus"]).fillna("other")
    r["party_code_asof"] = k.map(cau["party_code_asof"])
    return r[_ROSTER_COLS + ["voting"]].sort_values(["month", "voting"], ascending=[True, False], kind="stable").reset_index(drop=True)


ROSTER_ASOF_ALL = _build_roster_asof()
ROSTER_ASOF = ROSTER_ASOF_ALL.loc[ROSTER_ASOF_ALL.voting, _ROSTER_COLS].reset_index(drop=True)


def roster_asof(date, include_delegates=True):
    """Members in office in the month containing `date`: rows of ROSTER_ASOF_ALL (default, Task-1 pool, with
    `voting`) or of the voting-only ROSTER_ASOF (include_delegates=False)."""
    m = pd.Timestamp(date).to_period("M").to_timestamp()
    r = ROSTER_ASOF_ALL if include_delegates else ROSTER_ASOF
    return r[r.month == m]

# Senate ties (50-50 after caucusing) are broken by the Vice President, whose party is not in the parsed
# data (committee_history has no chairs). Documented constant: VP party = President's party.
VP_PARTY = {108: "R", 109: "R", 110: "R", 111: "D", 112: "D", 113: "D", 114: "D", 115: "R", 116: "R",
            117: "D", 118: "D", 119: "R"}


@lru_cache(maxsize=None)
def _majority_party_table():
    """Majority party per (congress, chamber), point-in-time at the Congress midpoint.

    Counts voting members in office at congress_midpoint(c) (in_office_at; delegates excluded), each with
    the party AS OF that Congress (members.party_code_asof) and independents assigned to their caucus
    (members.caucus, data-derived). Ties are broken by the Vice President's party (Senate only, VP_PARTY),
    cross-checked against the data: the party sponsoring most committee-reported bills in that chamber.
    Columns: congress, chamber, majority, seats_D, seats_R, seats_other, n_voting, tie_broken_by."""
    rows = []
    cau = members.drop_duplicates(["bioguide_id", "congress"]).set_index(["bioguide_id", "congress"])["caucus"]
    for c in sorted(members.congress.unique()):
        io = in_office_at(congress_midpoint(c))
        io = io[io.congress == c]
        io = io.assign(caucus=pd.MultiIndex.from_arrays([io.bioguide_id, io.congress]).map(cau)).fillna({"caucus": "other"})
        for ch, g in io.groupby("chamber"):
            n = g.caucus.value_counts()
            d, r = int(n.get("D", 0)), int(n.get("R", 0))
            row = dict(congress=int(c), chamber=ch, seats_D=d, seats_R=r, seats_other=int(len(g) - d - r),
                       n_voting=int(len(g)))
            if d != r:
                row.update(majority="D" if d > r else "R", tie_broken_by="none (seat count)")
            else:
                vp = VP_PARTY.get(int(c)) if ch == "Senate" else None
                rep = adv[(adv.congress == c) & (adv.chamber == ch) & (adv.label == 1) & adv.sponsor_party.isin(["D", "R"])]
                dat = rep.sponsor_party.value_counts(normalize=True)
                dat_p = dat.idxmax() if len(dat) else None
                chk = (f"reported-bill sponsors {dat_p} {dat.max():.0%} "
                       f"({'agrees' if dat_p == vp else 'DISAGREES'})" if dat_p else "no reported bills")
                row.update(majority=vp or dat_p, tie_broken_by=(f"VP party ({chk})" if vp else f"data: {chk}"))
            rows.append(row)
    cols = ["congress", "chamber", "majority", "seats_D", "seats_R", "seats_other", "n_voting", "tie_broken_by"]
    return pd.DataFrame(rows)[cols]


def majority_party():
    """Majority party per (congress, chamber); see _majority_party_table for the derivation. Returns a copy
    with columns congress, chamber, majority, seats_D, seats_R, seats_other, n_voting, tie_broken_by."""
    return _majority_party_table().copy()


adv = build_advancement_labels()
MAJORITY = majority_party()
print(f"cosponsorship events: {len(cosp):,}; advancement labels: {len(adv):,} "
      f"({adv[~adv.censored].label.mean():.1%} positive among uncensored; alternative label_alt = reported OR "
      f"passed origin chamber within {ADV_WINDOW_DAYS} d: {adv[~adv.censored].label_alt.mean():.1%})")
_rs = ROSTER_ASOF[ROSTER_ASOF.congress.isin(CONGRESSES)].groupby(["month", "chamber"]).size().groupby("chamber").agg(["min", "median", "max"])
print(f"ROSTER_ASOF: {ROSTER_ASOF.month.nunique()} months, voting members in office per month (complete Congresses): "
      + "; ".join(f"{ch[:5]} min {r['min']:.0f} / median {r['median']:.0f} / max {r['max']:.0f}" for ch, r in _rs.iterrows()))
_rd = ROSTER_ASOF_ALL[~ROSTER_ASOF_ALL.voting & ROSTER_ASOF_ALL.congress.isin(CONGRESSES)].groupby("month").size()
print(f"ROSTER_ASOF_ALL (Task-1 pool, default of window_targets(point_in_time=True) / roster_asof()): adds delegates and "
      f"resident commissioners, {_rd.min()}-{_rd.max()} per month (median {_rd.median():.0f}); ROSTER_ASOF stays voting-only")
_ind = members[~members.party_code_asof.isin(["D", "R"]) & members.congress.isin(CONGRESSES + [IN_PROGRESS])].copy()
_ind["dsh"] = _dshare.reindex(pd.MultiIndex.from_arrays([_ind.bioguide_id, _ind.congress])).to_numpy()
_indg = _ind.groupby(["name", "party_code_asof", "caucus"]).agg(c0=("congress", "min"), c1=("congress", "max"),
                                                                 s0=("dsh", "min"), s1=("dsh", "max")).reset_index()
print("Non-D/R members -> data-derived caucus (share of their cosponsorships on D-sponsored bills; near 0.5 = weak "
      "evidence): " + "; ".join(f"{r.name.split(',')[0]} {r.party_code_asof} {r.c0}" + (f"-{r.c1}" if r.c1 > r.c0 else "")
                                + f" ({r.s0:.2f}" + (f"-{r.s1:.2f}" if r.c1 > r.c0 else "") + f")->{r.caucus}"
                                for r in _indg.itertuples()))
print("Majority per chamber (point-in-time at mid-Congress, caucus-adjusted; ties -> tie_broken_by):")
print(MAJORITY.assign(chamber=MAJORITY.chamber.str[:5]).pivot(index="chamber", columns="congress",
      values="majority").to_string())
_ties = MAJORITY[MAJORITY.tie_broken_by != "none (seat count)"]
print("ties: " + ("; ".join(f"{r.congress} {r.chamber}: {r.seats_D}-{r.seats_R} -> {r.majority} by {r.tie_broken_by}"
                            for r in _ties.itertuples()) or "none"))

# Shared party-regime labels (one definition for A15b, C7/C8, D26): REGIME[c] = '<House majority>H/<Senate majority>S',
# e.g. 'RH/RS', from MAJORITY (majority_party()). REGIME_HS[c] = (house, senate); regime_match(a, b) = number of
# chambers (0-2) whose majority party is the same in Congresses a and b.
_REG_HS = MAJORITY.pivot(index="congress", columns="chamber", values="majority")
REGIME_HS = {int(c): (r["House of Representatives"], r["Senate"]) for c, r in _REG_HS.iterrows()}
REGIME = {c: f"{h}H/{s}S" for c, (h, s) in REGIME_HS.items()}


def regime_match(a, b):
    """Number of chambers (0, 1, 2) with the same majority party in Congresses a and b (from REGIME_HS)."""
    return int(sum(x == y for x, y in zip(REGIME_HS[int(a)], REGIME_HS[int(b)])))


_same_reg = [c for c in CONGRESSES if REGIME.get(c) == REGIME.get(IN_PROGRESS)]
print("REGIME (House/Senate majority): " + ", ".join(f"{c} {r}" for c, r in REGIME.items())
      + f" | complete Congresses with {IN_PROGRESS}'s regime ({REGIME.get(IN_PROGRESS)}): {_same_reg}")

cosponsorship events: 2,102,904; advancement labels: 145,738 (5.4% positive among uncensored; alternative label_alt = reported OR passed origin chamber within 180 d: 7.9%)
ROSTER_ASOF: 286 months, voting members in office per month (complete Congresses): House min 429 / median 435 / max 435; Senat min 99 / median 100 / max 100
ROSTER_ASOF_ALL (Task-1 pool, default of window_targets(point_in_time=True) / roster_asof()): adds delegates and resident commissioners, 5-6 per month (median 6); ROSTER_ASOF stays voting-only
Non-D/R members -> data-derived caucus (share of their cosponsorships on D-sponsored bills; near 0.5 = weak evidence): Amash I 116 (0.54)->D; Jeffords I 108-109 (0.51-0.60)->D; King I 113-119 (0.52-0.73)->D; Lieberman ID 110-112 (0.71-0.78)->D; Sanders I 108-119 (0.71-0.95)->D; Sinema I 118 (0.55)->D
Majority per chamber (point-in-time at mid-Congress, caucus-adjusted; ties -> tie_broken_by):
congress 108 109 110 111 112 113 114 115 116 117 118 119
chamber                  

In [94]:
# --- Candidate splits (single source of truth: every section evaluates exactly these pools) -----------
# Each split expands into evaluation units via iter_eval_units(split): one dict per (fold, role) with
# the Congresses to fit on and the Congress (or window) evaluated. Tuning units fit on `train`; test
# units refit on train + val. Definitions follow builder D's backtest.
FOLD_C = list(range(112, CONGRESSES[-1] + 1))      # rolling / sliding folds: test c, val c-1
SLIDE_K = 3                                        # sliding window length (Congresses)
S4_CUTS = (0.6, 0.8)                               # within-Congress: train <60% | val 60-80% | test >=80%
S5_FRACS = (0.70, 0.15, 0.15)                      # random bill split (leaky reference)


def _rng_label(cs):
    cs = sorted(cs)
    return f"{cs[0]}-{cs[-1]}" if len(cs) > 1 else str(cs[0])


SPLIT_DEFS = {
    "S1a": dict(name="Expanding by Congress (test 117)", kind="expanding",
                train=list(range(108, 116)), val=[116], test=[117],
                desc="train 108-115 | val 116 | test 117 (test refit on 108-116)"),
    "S1b": dict(name="Expanding by Congress (test 118)", kind="expanding",
                train=list(range(108, 117)), val=[117], test=[118],
                desc="train 108-116 | val 117 | test 118 (test refit on 108-117)"),
    "S2": dict(name=f"Sliding window k={SLIDE_K}", kind="sliding", k=SLIDE_K, folds=FOLD_C,
               desc=f"for test c: train c-{SLIDE_K + 1}..c-2 | val c-1 | test c (refit on c-{SLIDE_K}..c-1)"),
    "S3": dict(name="Rolling origin, expanding", kind="rolling", folds=FOLD_C,
               desc="for test c: train 108..c-2 | val c-1 | test c (refit on 108..c-1); "
                    f"c in {FOLD_C[0]}-{FOLD_C[-1]}"),
    "S4": dict(name="Within-Congress temporal", kind="within", folds=[116, 117, 118], cuts=S4_CUTS,
               desc="prior Congresses + bills of c introduced before its 60% date quantile | "
                    "val 60-80% | test last 20% (only events dated before the cutoff are used)"),
    "S5": dict(name="Random bill split (leaky reference)", kind="random", fracs=S5_FRACS,
               desc="law-capable bills of 108-118 shuffled 70/15/15; ignores time entirely"),
}


def iter_eval_units(split=None):
    """Yield evaluation units of one split (key of SPLIT_DEFS) or of all splits (split=None).

    Each unit: dict(split, fold, role in {'val','test'}, train (list of whole Congresses to fit on),
    eval_congress (int, or None for S5), train_label, within). `within` is None except for S4, where it is
    (c, train_q, eval_lo_q, eval_hi_q): also fit on bills of Congress c introduced before its train_q date
    quantile and evaluate bills of c between the eval_lo_q and eval_hi_q quantiles. S5 units carry
    `random=(fracs, role)` and train=all CONGRESSES (the rows are split at random, not by Congress)."""
    keys = list(SPLIT_DEFS) if split is None else [split]
    for s in keys:
        d = SPLIT_DEFS[s]
        k = d["kind"]
        if k == "expanding":
            v, t = d["val"][0], d["test"][0]
            units = [("val", s, list(d["train"]), v), ("test", s, list(d["train"]) + [v], t)]
        elif k in ("rolling", "sliding"):
            units = []
            for c in d["folds"]:
                tv = list(range(CONGRESSES[0], c - 1)) if k == "rolling" else list(range(c - 1 - d["k"], c - 1))
                tt = list(range(CONGRESSES[0], c)) if k == "rolling" else list(range(c - d["k"], c))
                units += [("val", f"c={c}", tv, c - 1), ("test", f"c={c}", tt, c)]
        elif k == "within":
            q1, q2 = d["cuts"]
            for c in d["folds"]:
                prior = list(range(CONGRESSES[0], c))
                yield dict(split=s, fold=f"c={c}", role="val", train=prior, eval_congress=c,
                           train_label=f"{_rng_label(prior)} + {c}<{q1:.0%}", within=(c, q1, q1, q2))
                yield dict(split=s, fold=f"c={c}", role="test", train=prior, eval_congress=c,
                           train_label=f"{_rng_label(prior)} + {c}<{q2:.0%}", within=(c, q2, q2, 1.0))
            continue
        elif k == "random":
            for role, lab in (("val", f"random {d['fracs'][0]:.0%}"),
                              ("test", f"random {d['fracs'][0] + d['fracs'][1]:.0%}")):
                yield dict(split=s, fold="random", role=role, train=list(CONGRESSES), eval_congress=None,
                           train_label=lab, within=None, random=(d["fracs"], role))
            continue
        else:
            raise ValueError(f"unknown split kind {k}")
        for role, fold, tr, ev in units:
            yield dict(split=s, fold=fold, role=role, train=tr, eval_congress=ev, train_label=_rng_label(tr),
                       within=None)


print(pd.DataFrame([(k, v["name"], v["desc"], sum(1 for _ in iter_eval_units(k))) for k, v in SPLIT_DEFS.items()],
                   columns=["split", "name", "definition", "eval units"]).to_string(index=False))

# Protocol variants (section D): same Congress pools as a base split, different protocol. They are candidate
# splits for the decision section (SPLIT_RESULTS has rows for them) but are NOT yielded by iter_eval_units(),
# whose consumers only know the five base kinds. "+stream": the eval Congress's own events dated
# <= introduced_date + t0 of each scored bill are added to the history (temporal-GNN deployment).
SPLIT_VARIANTS = {
    "S1a+stream": dict(base="S1a", stream=True, desc="S1a + eval-Congress events dated <= introduced_date + t0"),
    "S1b+stream": dict(base="S1b", stream=True, desc="S1b + eval-Congress events dated <= introduced_date + t0"),
    "S3+stream": dict(base="S3", stream=True, desc="S3 rolling folds + eval-Congress events dated <= introduced_date + t0"),
    "S1pool": dict(base="S1a", stream=False, train=list(range(108, 116)), val=[116], test=[117, 118],
                   desc="train 108-115 | val 116 | test 117+118 pooled (refit on 108-116 once)"),
}
ALL_SPLITS = list(SPLIT_DEFS) + list(SPLIT_VARIANTS)   # every candidate a check's by_split may refer to
BY_CONGRESS_KINDS = ("expanding", "sliding", "rolling")


def split_kind(split):
    """Kind of a split or variant ('expanding', 'sliding', 'rolling', 'within', 'random')."""
    return SPLIT_DEFS[SPLIT_VARIANTS[split]["base"] if split in SPLIT_VARIANTS else split]["kind"]


def split_eval_congresses(split, roles=("val", "test")):
    """Congresses evaluated (in the given roles) by a split or variant. S4: its folds; S5: all CONGRESSES."""
    if split in SPLIT_VARIANTS and "test" in SPLIT_VARIANTS[split]:
        v = SPLIT_VARIANTS[split]
        return sorted(set(v["val"] * ("val" in roles) + v["test"] * ("test" in roles)))
    base = SPLIT_VARIANTS[split]["base"] if split in SPLIT_VARIANTS else split
    if SPLIT_DEFS[base]["kind"] == "random":
        return list(CONGRESSES)
    return sorted({u["eval_congress"] for u in iter_eval_units(base) if u["role"] in roles})


# Shared S5 draw (bill_id -> 'train'/'val'/'test'): law-capable bills of CONGRESSES in `bills` order, seed 5,
# cut at S5_FRACS. Identical to section D's original local draw, so C and D use the same random split.
_s5b = bills.loc[bills.law_capable & bills.congress.isin(CONGRESSES), "bill_id"].to_numpy()
S5_ROLE = pd.Series(pd.cut(np.random.default_rng(5).random(len(_s5b)),
                           [0, S5_FRACS[0], S5_FRACS[0] + S5_FRACS[1], 1.0001],
                           labels=["train", "val", "test"]).astype(str), index=_s5b, name="s5_role")

# Shared S4 cut dates: S4_CUT_DATES[c][q] = introduction-date quantile q of Congress c's law-capable bills, floored
# to the day (q = 1.0 -> day after the Congress ends). Rule: fit on bills introduced < cut[q1];
# val = cut[q1] <= introduced < cut[q2]; test = introduced >= cut[q2] (and < cut[1.0]).
_EPOCH = np.datetime64("1970-01-01", "D")


def _s4_cut(c, q):
    if q >= 1.0:
        return congress_end(c) + pd.Timedelta(days=1)
    d = bills.loc[bills.law_capable & (bills.congress == c), "introduced_date"].dropna()
    day = int(np.quantile(d.to_numpy().astype("datetime64[D]").astype(np.int64), q))
    return pd.Timestamp(_EPOCH + np.timedelta64(day, "D"))


S4_CUT_DATES = {c: {q: _s4_cut(c, q) for q in (*S4_CUTS, 1.0)} for c in SPLIT_DEFS["S4"]["folds"]}
print("variants (SPLIT_VARIANTS, not in iter_eval_units): " + "; ".join(f"{k}: {v['desc']}" for k, v in SPLIT_VARIANTS.items()))
print(f"S5_ROLE: {S5_ROLE.value_counts().to_dict()} | S4_CUT_DATES: " + "; ".join(
    f"{c}: {d[S4_CUTS[0]].date()} / {d[S4_CUTS[1]].date()}" for c, d in S4_CUT_DATES.items()))

split                                name                                                                                                                                        definition  eval units
  S1a    Expanding by Congress (test 117)                                                                                        train 108-115 | val 116 | test 117 (test refit on 108-116)           2
  S1b    Expanding by Congress (test 118)                                                                                        train 108-116 | val 117 | test 118 (test refit on 108-117)           2
   S2                  Sliding window k=3                                                                                 for test c: train c-4..c-2 | val c-1 | test c (refit on c-3..c-1)          14
   S3           Rolling origin, expanding                                                                   for test c: train 108..c-2 | val c-1 | test c (refit on 108..c-1); c in 112-118          14


In [95]:
# --- Drift statistics -----------------------------------------------------------------------------
def psi(expected, actual, bins=10):
    """Population Stability Index of `actual` vs `expected` (numeric). <0.1 stable, 0.1-0.25 moderate, >0.25 shift."""
    e, a = np.asarray(expected, float), np.asarray(actual, float)
    e, a = e[~np.isnan(e)], a[~np.isnan(a)]
    edges = np.unique(np.quantile(e, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    pe = np.histogram(e, edges)[0] / len(e) + 1e-6
    pa = np.histogram(a, edges)[0] / len(a) + 1e-6
    return float(np.sum((pa - pe) * np.log(pa / pe)))


def psi_categorical(expected, actual):
    """PSI for categorical values (same thresholds as psi)."""
    pe = pd.Series(expected).value_counts(normalize=True)
    pa = pd.Series(actual).value_counts(normalize=True)
    idx = pe.index.union(pa.index)
    pe, pa = pe.reindex(idx, fill_value=0) + 1e-6, pa.reindex(idx, fill_value=0) + 1e-6
    return float(np.sum((pa - pe) * np.log(pa / pe)))


def jsd(p, q):
    """Jensen-Shannon divergence (base 2, in [0, 1]) between two count/probability vectors."""
    p, q = np.asarray(p, float), np.asarray(q, float)
    p, q = p / p.sum(), q / q.sum()
    m = (p + q) / 2
    kl = lambda x, y: np.sum(np.where(x > 0, x * np.log2(np.where(x > 0, x, 1) / np.where(y > 0, y, 1)), 0))
    return float((kl(p, m) + kl(q, m)) / 2)


def ks(a, b):
    """Two-sample Kolmogorov-Smirnov statistic and p-value."""
    r = stats.ks_2samp(np.asarray(a, float), np.asarray(b, float))
    return float(r.statistic), float(r.pvalue)


def smd(a, b):
    """Standardized mean difference (|d| < 0.1 negligible, > 0.2 notable)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    s = np.sqrt((np.nanvar(a) + np.nanvar(b)) / 2)
    return float((np.nanmean(b) - np.nanmean(a)) / s) if s > 0 else 0.0


def psi_level(v):
    return "stable" if v < 0.1 else "moderate" if v < 0.25 else "shift"


# --- Shared "different from train noise" rule ------------------------------------------------------------
# Every section judges a per-Congress quantity against the SAME fixed reference set REL_REF (the S1a train
# Congresses 108-115), whatever split or pool it is looking at, so verdicts are comparable across sections.
REL_REF = list(SPLIT_DEFS["S1a"]["train"])
REL_ALPHA = 0.05


def rel_to_train(values, train=None, side="two", log=False, alpha=REL_ALPHA, cands=None):
    """Judge per-Congress values against the spread of a fixed reference set of train Congresses.

    values: Series indexed by Congress. For a distance (PSI, JSD, KS, AUROC drop), give each reference Congress
    its leave-one-out distance to the other reference Congresses and each candidate its distance to the
    reference pool. train: reference Congresses (default REL_REF; keep the default unless you have a reason).
    side: 'two' | 'upper' (only large values are bad, e.g. distances) | 'lower'. log: judge log(values)
    (rates, counts, ratios; values <= 0 become NaN). cands: Congresses to judge (default: every other Congress
    in `values`).

    Returns (tab, info). tab has one row per Congress (reference rows judged by nested leave-one-out against the
    other reference Congresses, role='ref'; candidates role='cand') with columns
      value, z (vs reference mean / SD, comparable to the old z > 2 rules), t_pi (t statistic of the
      prediction interval for one new Congress: (x - mean) / (sd * sqrt(1 + 1/n))), pi_lo / pi_hi (interval on
      the original scale), fail (outside the (1 - alpha) t prediction interval, df = n - 1), trend_pred,
      trend_z (studentised residual from a linear trend over the reference Congresses: separates a steady
      trend from a break; df = n - 2), trend_fail, in_range (inside the reference min-max).
    info: n, mean, sd, tcrit, alpha, side, log, ref, fa_count / fa_rate (reference Congresses that fail the
      rule against the other n - 1 = per-Congress false-alarm rate of the rule), k (number of candidates),
      fwer_loo = 1 - (1 - fa_rate)^k (chance that at least one of k unchanged candidates fails, from the
      nested LOO), fwer_nominal = 1 - (1 - alpha)^k, rule (text for record_check threshold)."""
    train = list(REL_REF if train is None else train)
    v = pd.Series(values, dtype=float)
    f = (lambda s: np.log(s.where(s > 0))) if log else (lambda s: s)
    inv = np.exp if log else (lambda s: s)
    fv = f(v)
    ref = fv.reindex(train).dropna()
    cands = [c for c in v.index if c not in train] if cands is None else list(cands)
    n = len(ref)
    q = 1 - alpha / 2 if side == "two" else 1 - alpha

    def judge(x, r, cx):
        m, s, k = r.mean(), r.std(ddof=1), len(r)
        tc = stats.t.ppf(q, k - 1)
        half = tc * s * np.sqrt(1 + 1 / k)
        with np.errstate(divide="ignore", invalid="ignore"):
            t = (x - m) / (s * np.sqrt(1 + 1 / k))
            z = (x - m) / s
            xs = np.asarray(r.index, float)
            sxx = ((xs - xs.mean()) ** 2).sum()
            b1 = ((xs - xs.mean()) * (r.values - r.mean())).sum() / sxx
            pred = r.mean() + b1 * (cx - xs.mean())
            sres = np.sqrt(((r.values - (r.mean() + b1 * (xs - xs.mean()))) ** 2).sum() / (k - 2))
            tz = (x - pred) / (sres * np.sqrt(1 + 1 / k + (cx - xs.mean()) ** 2 / sxx))
        tct = stats.t.ppf(q, k - 2)
        flag = {"two": lambda a, c: np.abs(a) > c, "upper": lambda a, c: a > c, "lower": lambda a, c: a < -c}[side]
        return dict(z=z, t_pi=t, pi_lo=float(inv(m - half)), pi_hi=float(inv(m + half)),
                    fail=bool(flag(t, tc)) if np.isfinite(x) else None, trend_pred=float(inv(pred)),
                    trend_z=float(tz), trend_fail=bool(flag(tz, tct)) if np.isfinite(x) else None)

    rows = []
    for c in ref.index:
        rows.append(dict(congress=c, role="ref", value=v[c], **judge(ref[c], ref.drop(c), float(c))))
    for c in cands:
        x = fv.get(c, np.nan)
        rows.append(dict(congress=c, role="cand", value=v.get(c, np.nan), **judge(x, ref, float(c))))
    tab = pd.DataFrame(rows).set_index("congress")
    inr = tab.value.between(v.reindex(ref.index).min(), v.reindex(ref.index).max())
    tab["in_range"] = inr.astype(object).where(tab.role == "cand", None)   # None for reference rows
    fa = tab[tab.role == "ref"]
    fa_count = int((fa.fail == True).sum())   # noqa: E712 (None-safe)
    fa_rate = fa_count / n if n else np.nan
    k = int((tab.role == "cand").sum())
    tail = {"two": "two-sided", "upper": "upper tail", "lower": "lower tail"}[side]
    info = dict(n=n, mean=float(inv(ref.mean())), sd=float(ref.std(ddof=1)), tcrit=float(stats.t.ppf(q, n - 1)),
                alpha=alpha, side=side, log=log, ref=_rng_label(ref.index), fa_count=fa_count, fa_rate=fa_rate,
                fa_ids=[int(c) for c in fa.index[fa.fail == True]], k=k,   # noqa: E712
                fwer_loo=float(1 - (1 - fa_rate) ** k) if n else np.nan, fwer_nominal=float(1 - (1 - alpha) ** k),
                rule=f"outside {1 - alpha:.0%} t prediction interval of ref {_rng_label(ref.index)} "
                     f"({tail}{', log scale' if log else ''}; n={n})")
    return tab, info


def rel_detail(tab, info, fmt=".3f"):
    """One-line summary of rel_to_train output for record_check(detail=...)."""
    cand = tab[tab.role == "cand"]
    return (f"ref {info['ref']} PI [{cand.pi_lo.iloc[0] if len(cand) else np.nan:{fmt}}, "
            f"{cand.pi_hi.iloc[0] if len(cand) else np.nan:{fmt}}] ({info['rule']}); "
            + ", ".join(f"{c}={r.value:{fmt}} (t={r.t_pi:+.1f}, trend_z={r.trend_z:+.1f}{', FAIL' if r.fail else ''})"
                        for c, r in cand.iterrows())
            + f"; nested-LOO false alarms {info['fa_count']}/{info['n']} (FWER over {info['k']} candidates "
              f"~{info['fwer_loo']:.0%}, nominal {info['fwer_nominal']:.0%})")


REL_SPLIT_MODES = ("cands", "excess", "any")
REL_SPLIT_MODE = "cands"   # default mode of rel_by_split for rel_to_train tabs (frames with a `role` column)


def _rel_verdicts(tab, col):
    """Per-Congress verdicts (bool, None dropped) and per-Congress false-alarm rates from a rel_to_train tab."""
    s = tab[col] if isinstance(tab, pd.DataFrame) else pd.Series(tab)
    s = s[[x is not None and not (isinstance(x, float) and np.isnan(x)) for x in s]]
    role = tab["role"] if isinstance(tab, pd.DataFrame) and "role" in tab.columns else None
    return s, role


def rel_split_table(tab, col="fail", splits=None, roles=("val", "test"), alpha=REL_ALPHA):
    """Per-split breakdown of per-Congress verdicts (rel_to_train's tab, or any frame / Series by Congress).

    One row per split: n_eval (Congresses judged), n_fail, n_cand / n_cand_fail (candidate Congresses only,
    role 'cand'; for inputs without `role` every Congress counts as a candidate), expected = sum over the judged
    Congresses of their per-Congress false-alarm rate, binom_p = P(X >= n_fail) for X ~ Poisson-binomial with those
    rates, and the three verdicts: pass_any (no judged Congress fails; legacy), pass_cands (no candidate Congress
    fails; None if the split evaluates no candidate), pass_excess (binom_p >= 0.05).
    Per-Congress false-alarm rate = max(nested-LOO fa_rate of the tab (share of reference rows failing `col`),
    alpha): the same empirical rate applies to candidates (judged against the reference) and to reference
    Congresses (judged by nested LOO); floored at the nominal alpha because n = 8 cannot show a rate of 0.
    For inputs without reference rows (plain Series of threshold verdicts) the rate is alpha."""
    s, role = _rel_verdicts(tab, col)
    splits = [k for k in ALL_SPLITS if split_kind(k) in BY_CONGRESS_KINDS] if splits is None else splits
    if role is not None and (role == "ref").any():
        ref_v = [bool(x) for x in s[s.index.isin(role.index[role == "ref"])]]
        fa = float(np.mean(ref_v)) if ref_v else alpha
    else:
        fa = alpha
    rate = max(fa, alpha)
    rows = []
    for k in splits:
        ev = [c for c in split_eval_congresses(k, roles) if c in s.index]
        fails = [bool(s[c]) for c in ev]
        is_c = [True if role is None else role.get(c) == "cand" for c in ev]
        n_fail, n_c = int(sum(fails)), int(sum(is_c))
        n_cf = int(sum(f for f, ic in zip(fails, is_c) if ic))
        pmf = np.array([1.0])
        for _ in ev:   # Poisson-binomial (identical rates -> binomial), exact by convolution
            pmf = np.convolve(pmf, [1 - rate, rate])
        p = float(pmf[n_fail:].sum()) if ev else np.nan
        rows.append(dict(split=k, n_eval=len(ev), n_fail=n_fail, n_cand=n_c, n_cand_fail=n_cf,
                         fa_rate=rate, expected=rate * len(ev), binom_p=p,
                         pass_any=(not any(fails)) if ev else None,
                         pass_cands=(n_cf == 0) if n_c else None,
                         pass_excess=bool(p >= 0.05) if ev else None))
    return pd.DataFrame(rows).set_index("split")


def rel_by_split(tab, col="fail", splits=None, roles=("val", "test"), mode=None):
    """Map per-Congress verdicts (rel_to_train's tab, or any frame / Series indexed by Congress) onto candidate
    splits for record_check(by_split=...). Values: True (holds) / False (fails) / None (not judged).

    mode (see rel_split_table for the numbers behind each):
      'cands'  (default for rel_to_train tabs, i.e. inputs with a `role` column): a split fails if any CANDIDATE
               Congress (role 'cand', 116-118) it evaluates fails; reference Congresses (109-115, evaluated only by
               S2/S3 folds) are not judged, because their nested-LOO verdict uses a reference that contains later
               Congresses and would make 7-fold splits fail more by construction. A split that evaluates no
               candidate gets None.
      'excess' : fails if the number of failing evaluated Congresses (reference + candidates) exceeds chance:
               Poisson-binomial p < 0.05 given per-Congress false-alarm rates. Low power for splits that evaluate
               only 2 Congresses (S1a / S1b need both to fail), so use it as a cross-check.
      'any'    (legacy; default for plain Series / frames without `role`, e.g. deterministic threshold verdicts such
               as coverage < 95%): fails if any evaluated Congress fails.
    Default splits: by-Congress splits and their variants (S4 / S5 reuse Congresses for training, so a
    per-Congress verdict does not apply; give their entries explicitly if your check covers them)."""
    has_role = isinstance(tab, pd.DataFrame) and "role" in tab.columns
    mode = (REL_SPLIT_MODE if has_role else "any") if mode is None else mode
    if mode not in REL_SPLIT_MODES:
        raise ValueError(f"mode must be one of {REL_SPLIT_MODES}")
    t = rel_split_table(tab, col=col, splits=splits, roles=roles)
    v = t[f"pass_{mode}"].astype(object)
    return {k: (None if v[k] is None or (isinstance(v[k], float) and np.isnan(v[k])) else bool(v[k])) for k in t.index}

In [96]:
# --- Check registry: every section records its conditions here; the final section summarizes -------
CHECKS = []
# Owner section per shared quantity: when several sections test the same quantity, record_check(...,
# quantity=key) lets checks_summary() count it once (the owner's verdict; disagreements are flagged).
QUANTITY_OWNERS = {
    # A: labels / targets and covariates
    "label_rate_t1": "A", "label_rate_t2": "A", "cosp_density": "A", "orig_share": "A", "majority_or": "A",
    "covariate_psi": "A", "drift_gradual": "A", "label_validity_t2": "A", "label_rate_t2_alt": "A",
    "label_count_t2": "A", "label_rate_t2_chamber": "A", "majority_or_regime": "A",
    "majority_or_regime_effect": "A", "same_party_share": "A", "bill_volume": "A", "topic_mix_jsd": "A",
    "joint_covariate_shift": "A", "model_calibration_t2": "A", "model_auroc_transfer_t2": "A", "pool_distance": "A",
    "deployment_proxy": "A", "drift_type": "A", "cand_fail_count": "A",
    "concept_drift_t1": "A", "model_auroc_transfer_t1": "A", "regime_similarity": "A",
    # B: graph structure
    "node_overlap": "B", "cold_start": "B", "degree_dist": "B", "graph_structure": "B", "sponsor_overlap": "B",
    "node_identity_switch": "B",   # party / chamber switchers (B11; D25 tests the same quantity from the model side)
    "train_eval_input_shift": "B",  # B12 (construction requirement: train vs eval graph inputs built alike)
    # C: temporal structure, censoring, leakage
    "subject_coverage": "C", "annotation_coverage": "C", "lag_dist": "C", "party_regime": "C", "censoring": "C",
    "label_completeness": "C", "task2_t0": "C",
    # leakage, one quantity per leakage path (the broad "leakage" key is kept only for backward compatibility:
    # it hid B's leakage FAILs behind C's INFO; tag new checks with a per-path key)
    "leakage_snapshot_fields": "C", "leakage_related_links": "B", "leakage_committee_edges": "B",
    "leakage_full_graph": "B", "leakage": "C",
    # D: backtest
    "score_stability": "D", "val_test_gap": "D", "learned_model_selection": "D", "embedding_staleness": "D",
    "regime_weighting": "D",
}

# Check kinds. The decision section RANKS splits on 'validity' checks only; 'drift' checks choose remedies.
#   validity    : does the split give an honest, usable estimate? (leakage, val predicts test, enough eval Congresses,
#                 score stability). A FAIL here is a reason to prefer another split.
#   drift       : does the evaluation / deployment target differ from train? Every forward split fails these by
#                 design (that is the split exposing drift); a FAIL selects remedies (recency / regime weighting,
#                 recalibration, per-Congress reporting), not a different split.
#   requirement : construction rule every split must follow (same as requirement=True).
CHECK_KINDS = ("validity", "drift", "requirement")
# Default kind for checks that pass kind=None, by quantity (clear-cut cases only; record_check(kind=...) overrides,
# and `kind_source` says which was used). Checks with neither stay kind=None ('unclassified' in the scores).
QUANTITY_KIND = {
    **{q: "drift" for q in (
        "label_rate_t1", "label_rate_t2", "cosp_density", "orig_share", "majority_or", "covariate_psi",
        "drift_gradual", "label_rate_t2_alt", "label_count_t2", "label_rate_t2_chamber", "majority_or_regime",
        "majority_or_regime_effect", "same_party_share", "bill_volume", "topic_mix_jsd", "joint_covariate_shift",
        "model_calibration_t2", "model_auroc_transfer_t2", "model_auroc_transfer_t1", "concept_drift_t1",
        "pool_distance", "deployment_proxy", "drift_type", "cand_fail_count", "regime_similarity", "degree_dist",
        "graph_structure", "lag_dist", "party_regime", "subject_coverage", "annotation_coverage")},
    **{q: "validity" for q in (
        "leakage_snapshot_fields", "leakage_related_links", "leakage_committee_edges", "leakage_full_graph",
        "leakage", "score_stability", "val_test_gap", "learned_model_selection", "censoring", "label_completeness",
        "cold_start")},
    # sponsor / node-identity turnover: the eval graph differs from train (choose remedies, not a split)
    **{q: "drift" for q in ("sponsor_overlap", "node_identity_switch")},
    "train_eval_input_shift": "requirement",
}


def _py(v):
    """numpy scalars -> plain Python (keeps by_split values JSON-friendly)."""
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (np.integer, np.floating)):
        return v.item()
    return v


def record_check(section, check_id, condition, passed, metric=None, threshold=None, detail="", quantity=None,
                 by_split=None, requirement=False, kind=None, na_reason=None, t0_task2=None):
    """passed: True / False, or None for informational checks. Re-running a cell replaces its check.
    quantity: optional key (see QUANTITY_OWNERS) naming the measured quantity, so checks of the same
    quantity in different sections are deduplicated by checks_summary().
    by_split: optional dict split -> verdict for checks whose outcome depends on the split. Keys are names in
    ALL_SPLITS (SPLIT_DEFS + SPLIT_VARIANTS); values True (holds for that split) / False (fails) / None (not
    judged), or a number (the split's metric; then give `passed` per the threshold). Give EVERY split in
    ALL_SPLITS a key: splits you do not judge get an explicit None plus a reason in `na_reason` (omitted splits are
    stored in `by_split_omitted`, counted as not judged, and flagged on the printed line). rel_by_split() builds it
    from per-Congress verdicts. Leave by_split=None for split-independent checks.
    na_reason: why splits with a None verdict are not judged: one string for all of them, or dict split -> str.
    kind: 'validity' | 'drift' | 'requirement' (CHECK_KINDS). Default: 'requirement' if requirement=True, else
    QUANTITY_KIND[quantity] if listed, else None (unclassified). Only 'validity' checks rank splits.
    requirement=True (or kind='requirement'): a construction rule every split must follow (e.g. "filter related
    links to t0"), not a property one split satisfies better than another. Listed by checks_requirements(),
    deduplicated separately, excluded from checks_by_split(), never counted as split failures.
    t0_task2: for Task-2 checks, the prediction time (days after introduction) the check assumes, e.g. 0 / 7
    (no policy area) or 30 (policy area available; bills reported by t0 excluded). None = not a Task-2 check."""
    global CHECKS
    if kind is not None and kind not in CHECK_KINDS:
        raise ValueError(f"record_check {check_id}: kind must be one of {CHECK_KINDS} or None, got {kind!r}")
    requirement = bool(requirement) or kind == "requirement"
    if not requirement and kind is None and QUANTITY_KIND.get(quantity) == "requirement":
        requirement, kind = True, "requirement"   # quantity default: keep the requirement flag consistent
        kind_source = "quantity default"
    elif requirement:
        kind_source = "explicit" if kind == "requirement" or kind is None else "requirement=True overrides " + kind
        kind = "requirement"
    elif kind is not None:
        kind_source = "explicit"
    elif quantity in QUANTITY_KIND:
        kind, kind_source = QUANTITY_KIND[quantity], "quantity default"
    else:
        kind_source = "unset"
    omitted, no_reason = [], []
    if by_split is not None:
        by_split = {str(k): _py(v) for k, v in dict(by_split).items()}
        unknown = [k for k in by_split if k not in ALL_SPLITS]
        if unknown:
            print(f"  (record_check {check_id}: unknown split keys {unknown}; known: {ALL_SPLITS})")
        omitted = [k for k in ALL_SPLITS if k not in by_split]
        if isinstance(na_reason, dict):
            na_reason = {str(k): str(v) for k, v in na_reason.items()}
            no_reason = [k for k, v in by_split.items() if v is None and k not in na_reason]
        elif not na_reason:
            no_reason = [k for k, v in by_split.items() if v is None]
    CHECKS = [c for c in CHECKS if c["id"] != check_id]
    CHECKS.append(dict(section=section, id=check_id, condition=condition, passed=passed,
                       metric=metric, threshold=threshold, detail=detail, quantity=quantity, by_split=by_split,
                       requirement=requirement, kind=kind, kind_source=kind_source, na_reason=na_reason,
                       by_split_omitted=omitted, t0_task2=t0_task2))
    mark = {True: "PASS", False: "FAIL", None: "INFO"}[passed] + ("/REQ" if requirement else "")
    m = "" if metric is None else f" | metric={metric:.4g}" if isinstance(metric, (int, float, np.floating)) else f" | {metric}"
    bs = ""
    if by_split:
        bad = [k for k, v in by_split.items() if v is False]
        bs = f" [fails for: {', '.join(bad)}]" if bad else ""
        if omitted or no_reason:
            bs += f" [not judged, no reason: {', '.join(omitted + no_reason)}]"
    print(f"[{mark}] {check_id}: {condition}{m}" + (f" (threshold {threshold})" if threshold is not None else "") + bs)


_CHECK_COLS = ["section", "id", "condition", "passed", "metric", "threshold", "detail", "quantity", "by_split",
               "requirement", "kind", "kind_source", "na_reason", "by_split_omitted", "t0_task2"]


def checks_summary(dedup=True, requirements=True, kinds=None):
    """DataFrame of recorded checks. dedup=True keeps one row per `quantity` (checks without a quantity are
    kept as-is; requirement and non-requirement checks of one quantity are kept apart): the owner section's
    check if present (QUANTITY_OWNERS), else the first recorded. Added columns: `dup_ids` (other checks of that
    quantity), `hidden_fail_ids` (hidden checks that FAIL), and `conflict` = True when the hidden checks'
    pass/fail verdicts disagree with each other or with the kept check, including a kept INFO (None) hiding a
    FAIL. When conflict is True, read hidden_fail_ids before counting the row as a pass / info.
    requirements: True (all rows), False (drop requirement=True rows: use this to count split failures),
    'only' (just the construction requirements; same as checks_requirements()).
    kinds: None (all) or an iterable of kinds to keep after dedup ('validity', 'drift', 'requirement'; use None
    inside the iterable for unclassified checks).
    `by_split` holds the per-split dict given to record_check (None if split-independent); see
    checks_by_split() for a long table and checks_split_scores() for per-split n_judged / n_fail."""
    df = pd.DataFrame(CHECKS, columns=_CHECK_COLS)
    df["requirement"] = df.requirement.fillna(False).astype(bool)
    if requirements == "only":
        df = df[df.requirement]
    elif requirements is False:
        df = df[~df.requirement]
    if dedup and not df.empty:
        df["_key"] = (df.quantity.where(df.quantity.notna(), "id:" + df.id)
                      + np.where(df.requirement & df.quantity.notna(), "|req", ""))
        df["_own"] = (df.section == df.quantity.map(QUANTITY_OWNERS)).astype(int)
        out = df.sort_values("_own", ascending=False, kind="stable").drop_duplicates("_key")
        kept = dict(zip(out._key, out.id))
        dup, hid, conf = {}, {}, {}
        for k, g in df.groupby("_key", sort=False):
            others = g[g.id != kept[k]]
            own_v = g.loc[g.id == kept[k], "passed"].iloc[0]
            dup[k] = ", ".join(others.id)
            hid[k] = ", ".join(others.id[others.passed.map(lambda x: x is False or x is np.False_)])
            vs = {x for x in g.passed if x is not None}
            conf[k] = len(vs) > 1 or (own_v is None and bool(hid[k]))
        out["dup_ids"] = out._key.map(dup)
        out["hidden_fail_ids"] = out._key.map(hid)
        out["conflict"] = out._key.map(conf)
        df = out.sort_index().drop(columns=["_key", "_own"])
    if kinds is not None:
        ks = set(kinds)
        df = df[df.kind.map(lambda k: (k if isinstance(k, str) else None) in ks)]
    return df


def checks_requirements():
    """Construction requirements (record_check(..., requirement=True)), one row per check (not deduplicated)."""
    return checks_summary(dedup=False, requirements="only")


def _na_reason_for(reason, split):
    if isinstance(reason, dict):
        return reason.get(split)
    return reason or None


def checks_by_split(dedup=True, kinds=None, include_omitted=False):
    """Long table of per-split verdicts: one row per (check, split) from by_split. Columns section, id,
    quantity, kind, passed (overall), split, value (as recorded), split_passed (True/False/None; None when the
    value is a metric or not judged), judged (split_passed is True/False), na_reason (why not judged, if given;
    '(omitted)' for splits missing from by_split). Checks without by_split and requirement checks are omitted
    (they apply to every split alike). include_omitted=True adds a row (judged=False) for every split in ALL_SPLITS
    missing from a check's by_split, so per-split counts never treat 'no entry' as 'not failing'."""
    cols = ["section", "id", "quantity", "kind", "passed", "split", "value", "split_passed", "judged", "na_reason"]
    rows = []
    for r in checks_summary(dedup=dedup, requirements=False, kinds=kinds).itertuples():
        bsd = r.by_split or {}
        for k, v in bsd.items():
            sp = v if isinstance(v, bool) or v is None else None
            rows.append(dict(section=r.section, id=r.id, quantity=r.quantity, kind=r.kind, passed=r.passed, split=k,
                             value=v, split_passed=sp, judged=isinstance(sp, bool),
                             na_reason=None if isinstance(sp, bool) else _na_reason_for(r.na_reason, k)))
        if include_omitted and bsd:
            for k in ALL_SPLITS:
                if k not in bsd:
                    rows.append(dict(section=r.section, id=r.id, quantity=r.quantity, kind=r.kind, passed=r.passed,
                                     split=k, value=None, split_passed=None, judged=False, na_reason="(omitted)"))
    return pd.DataFrame(rows, columns=cols)


def checks_split_scores(kinds=("validity",), dedup=True):
    """Per-split scoreboard over split-dependent checks of the given kinds (default: validity only, the ones that
    should rank splits). One row per split in ALL_SPLITS: n_checks (split-dependent checks of these kinds),
    n_judged (True/False verdicts for this split), n_fail, n_pass, n_na_reason (None with a reason),
    n_not_judged (None without a reason, metric-only values, or omitted from by_split), fail_rate = n_fail /
    n_judged. Compare splits on n_fail AND n_judged: a split that was not judged has not passed.
    kinds=None scores every kind; include None in the iterable for unclassified checks.
    Prints a warning listing split-dependent checks with kind=None (unless None is among `kinds`), so
    unclassified checks are not dropped from the ranking without notice."""
    if kinds is not None and None not in set(kinds):
        un = checks_summary(dedup=dedup, requirements=False, kinds=(None,))
        un = un[un.by_split.map(lambda d: isinstance(d, dict) and len(d) > 0)]
        if len(un):
            print(f"  WARNING checks_split_scores: {len(un)} split-dependent check(s) have kind=None and are NOT "
                  f"scored: {', '.join(un.id)}. Classify them (record_check(kind=...) or QUANTITY_KIND) or pass "
                  f"kinds including None to score them.")
    lb = checks_by_split(dedup=dedup, kinds=kinds, include_omitted=True)
    rows = []
    for s in ALL_SPLITS:
        g = lb[lb.split == s]
        nj = int(g.judged.sum())
        nf = int((g.split_passed == False).sum())  # noqa: E712 (object column)
        nr = int((~g.judged & g.na_reason.notna() & (g.na_reason != "(omitted)")).sum())
        rows.append(dict(split=s, n_checks=int(g.id.nunique()), n_judged=nj, n_fail=nf, n_pass=nj - nf,
                         n_na_reason=nr, n_not_judged=int(len(g) - nj - nr),
                         fail_rate=(nf / nj if nj else np.nan),
                         fail_ids=", ".join(g.id[g.split_passed == False])))  # noqa: E712
    return pd.DataFrame(rows).set_index("split")

In [97]:
# --- D3 charts ------------------------------------------------------------------------------------
# Charts are SVGs drawn by D3 v7 in the notebook output. `show_d3` embeds the data as JSON and runs a
# JS function body with (el, data, d3, opts) in scope. D3 itself is injected once per session from the
# vendored file (fallback: CDN). Palette: reference categorical palette (fixed order, never cycled).
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
PARTY_COLORS = {"D": "#2a78d6", "R": "#e34948", "I": "#7a7a7a"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

_HARNESS = os.environ.get("NB_HARNESS_OUT")   # set only by the offline test harness
_D3_EMITTED = False
HARNESS_CHARTS = []

# Shared JS: tooltip + axis styling, available to every chart as `viz` inside the function body.
_VIZ_JS = r"""
const viz = {
  ink: '%INK%', ink2: '%INK2%', grid: '%GRID%', palette: %PALETTE%,
  tooltip(el) {
    const t = d3.select(el).style('position', 'relative').append('div')
      .style('position', 'absolute').style('pointer-events', 'none').style('background', '#fff')
      .style('border', '1px solid #c9c8c2').style('border-radius', '4px').style('padding', '4px 8px')
      .style('font', '12px sans-serif').style('color', '%INK%').style('opacity', 0);
    return {
      show(html, event) {
        const [x, y] = d3.pointer(event, el);
        t.html(html).style('left', (x + 12) + 'px').style('top', (y - 10) + 'px').style('opacity', 1);
      },
      hide() { t.style('opacity', 0); },
    };
  },
  axis(g) {
    g.selectAll('path,line').attr('stroke', '%INK2%');
    g.selectAll('text').attr('fill', '%INK2%').style('font', '11px sans-serif');
    return g;
  },
  title(svg, text, sub) {
    svg.append('text').attr('x', 0).attr('y', 14).attr('fill', '%INK%')
      .style('font', '600 13px sans-serif').text(text);
    if (sub) svg.append('text').attr('x', 0).attr('y', 30).attr('fill', '%INK2%')
      .style('font', '11px sans-serif').text(sub);
  },
  legend(svg, items, x, y) {
    const g = svg.append('g').attr('transform', `translate(${x},${y})`);
    let dx = 0;
    items.forEach(([label, color]) => {
      const it = g.append('g').attr('transform', `translate(${dx},0)`);
      it.append('rect').attr('width', 10).attr('height', 10).attr('rx', 2).attr('fill', color);
      it.append('text').attr('x', 14).attr('y', 9).attr('fill', '%INK2%').style('font', '11px sans-serif').text(label);
      dx += 24 + label.length * 6.5;
    });
    return g;
  },
};
""".replace("%INK%", INK).replace("%INK2%", INK_2).replace("%GRID%", GRID).replace("%PALETTE%", json.dumps(PALETTE))


def _to_jsonable(data):
    if isinstance(data, pd.DataFrame):
        df = data.copy()
        for c in df.columns:
            if str(df[c].dtype).startswith("datetime"):
                df[c] = df[c].dt.strftime("%Y-%m-%d")
        return json.loads(df.to_json(orient="records"))
    return json.loads(json.dumps(data, default=lambda o: o.item() if hasattr(o, "item") else str(o)))


def show_d3(js_body, data, height=320, width=720, opts=None, name=None):
    """Render a D3 chart. `js_body` is JS run with (el, data, d3, opts, viz) in scope; `el` is the
    container div. Draw into d3.select(el).append('svg')... and keep text in ink colors."""
    global _D3_EMITTED
    cid = "d3_" + uuid.uuid4().hex[:10]
    payload = dict(data=_to_jsonable(data), opts=dict(opts or {}, width=width, height=height))
    func = f"function(el, data, d3, opts) {{\n{_VIZ_JS}\n{js_body}\n}}"
    if _HARNESS is not None:
        HARNESS_CHARTS.append(dict(id=cid, name=name or cid, func=func, payload=payload))
        print(f"<chart {name or cid}: {len(payload['data']) if isinstance(payload['data'], list) else 1} records>")
        return
    from IPython.display import HTML, display
    d3_tag = ""
    if not _D3_EMITTED and VENDOR_D3.exists():
        d3_tag = f"<script>{VENDOR_D3.read_text()}</script>"
        _D3_EMITTED = True
    html = f"""{d3_tag}<div id="{cid}" style="width:{width}px;min-height:{height}px"></div>
<script>(function() {{
  const payload = {json.dumps(payload)};
  const draw = {func};
  function go() {{ const el = document.getElementById("{cid}"); el.innerHTML = '';
                   try {{ draw(el, payload.data, window.d3, payload.opts); }}
                   catch (e) {{ el.innerHTML = '<pre style="color:#c00">' + e + '</pre>'; }} }}
  if (window.d3) go();
  else {{ const s = document.createElement('script'); s.src = 'https://cdn.jsdelivr.net/npm/d3@7';
         s.onload = go; document.head.appendChild(s); }}
}})();</script>"""
    display(HTML(html))


def reset_d3():
    """Re-inject D3 on the next chart (use after reloading the browser page mid-session)."""
    global _D3_EMITTED
    _D3_EMITTED = False


# --- Ready-made charts (use these when they fit; write custom js_body with show_d3 otherwise) -------
_LINE_JS = r"""
const {width, height, x, y, series, title, subtitle, yfmt, ylabel, xlabel, yzero} = opts;
const m = {top: subtitle ? 44 : 30, right: 120, bottom: 36, left: 56};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title || '', subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const keys = series ? Array.from(new Set(data.map(d => d[series]))) : ['all'];
const color = d3.scaleOrdinal(keys, viz.palette);
const xs = d3.scaleLinear(d3.extent(data, d => +d[x]), [0, W]);
const yext = d3.extent(data, d => +d[y]);
const ys = d3.scaleLinear([yzero === false ? yext[0] : Math.min(0, yext[0]), yext[1]], [H, 0]).nice();
g.append('g').call(d3.axisLeft(ys).ticks(5).tickSize(-W).tickFormat(d3.format(yfmt || ',~g')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(Math.min(12, data.length)).tickFormat(d3.format('d'))).call(viz.axis);
if (ylabel) g.append('text').attr('x', -m.left + 4).attr('y', -8).attr('fill', viz.ink2).style('font', '11px sans-serif').text(ylabel);
if (xlabel) g.append('text').attr('x', W).attr('y', H + 32).attr('text-anchor', 'end').attr('fill', viz.ink2).style('font', '11px sans-serif').text(xlabel);
const tip = viz.tooltip(el);
keys.forEach(k => {
  const rows = data.filter(d => !series || d[series] === k).sort((a, b) => a[x] - b[x]);
  g.append('path').datum(rows).attr('fill', 'none').attr('stroke', color(k)).attr('stroke-width', 2)
    .attr('d', d3.line().defined(d => d[y] !== null && !isNaN(d[y])).x(d => xs(+d[x])).y(d => ys(+d[y])));
  g.selectAll(null).data(rows.filter(d => d[y] !== null)).join('circle').attr('cx', d => xs(+d[x])).attr('cy', d => ys(+d[y]))
    .attr('r', 4).attr('fill', color(k)).attr('stroke', '#fff').attr('stroke-width', 1.5)
    .on('mousemove', (e, d) => tip.show(`<b>${series ? k + ' · ' : ''}${d[x]}</b><br>${d3.format(yfmt || ',~g')(d[y])}`, e))
    .on('mouseleave', () => tip.hide());
  if (series) {
    const last = rows[rows.length - 1];
    if (last) g.append('text').attr('x', xs(+last[x]) + 8).attr('y', ys(+last[y]) + 4).attr('fill', viz.ink2)
      .style('font', '11px sans-serif').text(k);
  }
});
if (series && keys.length > 1) viz.legend(svg, keys.map(k => [String(k), color(k)]), m.left, height - 10);
"""


def line_chart(df, x, y, series=None, title="", subtitle="", yfmt=",~g", ylabel="", xlabel="",
               yzero=True, height=300, width=720, name=None):
    """Lines over x (usually congress), one per `series` value. Hover points for values."""
    cols = [x, y] + ([series] if series else [])
    d = df[cols].copy()
    if series:
        d[series] = d[series].astype(str)
    show_d3(_LINE_JS, d, height, width, dict(x=x, y=y, series=series, title=title, subtitle=subtitle,
                                             yfmt=yfmt, ylabel=ylabel, xlabel=xlabel, yzero=yzero), name=name)


_BAR_JS = r"""
const {width, height, x, y, series, title, subtitle, yfmt, ylabel} = opts;
const m = {top: subtitle ? 44 : 30, right: 16, bottom: series ? 52 : 36, left: 56};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title || '', subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const xsVals = Array.from(new Set(data.map(d => String(d[x]))));
const keys = series ? Array.from(new Set(data.map(d => String(d[series])))) : ['all'];
const color = d3.scaleOrdinal(keys, viz.palette);
const x0 = d3.scaleBand(xsVals, [0, W]).paddingInner(0.2).paddingOuter(0.1);
const x1 = d3.scaleBand(keys, [0, x0.bandwidth()]).padding(0.08);
const ylo = Math.min(0, d3.min(data, d => +d[y]) || 0), yhi = Math.max(0, d3.max(data, d => +d[y]) || 0);
const ys = d3.scaleLinear([ylo, yhi > ylo ? yhi : ylo + 1], [H, 0]).nice();
g.append('g').call(d3.axisLeft(ys).ticks(5).tickSize(-W).tickFormat(d3.format(yfmt || ',~g')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(x0)).call(viz.axis);
if (ylo < 0) g.append('line').attr('x1', 0).attr('x2', W).attr('y1', ys(0)).attr('y2', ys(0)).attr('stroke', viz.ink2);
if (ylabel) g.append('text').attr('x', -m.left + 4).attr('y', -8).attr('fill', viz.ink2).style('font', '11px sans-serif').text(ylabel);
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('rect')
  .attr('x', d => x0(String(d[x])) + (series ? x1(String(d[series])) : 0))
  .attr('width', series ? x1.bandwidth() : x0.bandwidth())
  .attr('y', d => ys(Math.max(0, +d[y]))).attr('height', d => Math.abs(ys(0) - ys(+d[y])))
  .attr('rx', 2).attr('fill', d => color(series ? String(d[series]) : 'all'))
  .on('mousemove', (e, d) => tip.show(`<b>${d[x]}${series ? ' · ' + d[series] : ''}</b><br>${d3.format(yfmt || ',~g')(d[y])}`, e))
  .on('mouseleave', () => tip.hide());
if (series && keys.length > 1) viz.legend(svg, keys.map(k => [k, color(k)]), m.left, height - 10);
"""


def bar_chart(df, x, y, series=None, title="", subtitle="", yfmt=",~g", ylabel="", height=300,
              width=720, name=None):
    """Vertical bars (grouped by `series` if given). Hover bars for values."""
    cols = [x, y] + ([series] if series else [])
    show_d3(_BAR_JS, df[cols], height, width, dict(x=x, y=y, series=series, title=title,
                                                   subtitle=subtitle, yfmt=yfmt, ylabel=ylabel), name=name)


_HEAT_JS = r"""
const {width, height, x, y, v, title, subtitle, fmt, domain, diverging} = opts;
const m = {top: subtitle ? 48 : 34, right: 70, bottom: 40, left: 130};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title || '', subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const xv = Array.from(new Set(data.map(d => String(d[x])))), yv = Array.from(new Set(data.map(d => String(d[y]))));
const xs = d3.scaleBand(xv, [0, W]).padding(0.04), ys = d3.scaleBand(yv, [0, H]).padding(0.04);
const ext = domain || d3.extent(data, d => +d[v]);
const color = diverging
  ? d3.scaleDiverging([ext[0], 0, ext[1]], t => d3.interpolateRgbBasis(['#2a78d6', '#f2f1ee', '#e34948'])(t))
  : d3.scaleSequential(ext, d3.interpolateRgb('#eef4fc', '#0f3f7a'));
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('rect').attr('x', d => xs(String(d[x]))).attr('y', d => ys(String(d[y])))
  .attr('width', xs.bandwidth()).attr('height', ys.bandwidth()).attr('rx', 2)
  .attr('fill', d => d[v] === null ? '#f7f7f5' : color(+d[v]))
  .on('mousemove', (e, d) => tip.show(`<b>${d[y]} · ${d[x]}</b><br>${d[v] === null ? 'n/a' : d3.format(fmt || '.3f')(d[v])}`, e))
  .on('mouseleave', () => tip.hide());
if (xv.length * yv.length <= 200) g.selectAll(null).data(data.filter(d => d[v] !== null)).join('text')
  .attr('x', d => xs(String(d[x])) + xs.bandwidth() / 2).attr('y', d => ys(String(d[y])) + ys.bandwidth() / 2 + 4)
  .attr('text-anchor', 'middle').style('font', '10px sans-serif')
  .attr('fill', d => { const c = d3.hsl(color(+d[v])); return c.l < 0.55 ? '#fff' : viz.ink; })
  .text(d => d3.format(fmt || '.3f')(d[v]));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs)).call(viz.axis);
g.append('g').call(d3.axisLeft(ys)).call(viz.axis);
"""


def heatmap(df, x, y, value, title="", subtitle="", fmt=".3f", domain=None, diverging=False,
            height=360, width=720, name=None):
    """Matrix of cells colored by `value` (sequential blue, or diverging blue-gray-red around 0)."""
    d = df[[x, y, value]].copy()
    d[value] = d[value].astype(float).where(d[value].notna(), None)
    show_d3(_HEAT_JS, d, height, width, dict(x=x, y=y, v=value, title=title, subtitle=subtitle, fmt=fmt,
                                             domain=domain, diverging=diverging), name=name)


print("setup complete: load(), cosp, adv (label, label_alt), adv_cutoff(), window_targets(point_in_time=), "
      "in_office_at(), ROSTER_ASOF / roster_asof(), majority_party(), REGIME / REGIME_HS / regime_match(), SNAPSHOT_DATE, PA_MISSING_AT_T0 / PA_MISS_T0, "
      "SPLIT_DEFS / SPLIT_VARIANTS / iter_eval_units(), S5_ROLE, S4_CUT_DATES, drift stats, rel_to_train() / "
      "rel_detail() / rel_by_split(mode=) / rel_split_table(), record_check(by_split=, kind=, na_reason=, t0_task2=) / "
      "checks_summary(kinds=) / checks_requirements() / checks_by_split() / checks_split_scores(), ROSTER_ASOF_ALL, "
      "show_d3() / line_chart() / bar_chart() / heatmap()")

setup complete: load(), cosp, adv (label, label_alt), adv_cutoff(), window_targets(point_in_time=), in_office_at(), ROSTER_ASOF / roster_asof(), majority_party(), REGIME / REGIME_HS / regime_match(), SNAPSHOT_DATE, PA_MISSING_AT_T0 / PA_MISS_T0, SPLIT_DEFS / SPLIT_VARIANTS / iter_eval_units(), S5_ROLE, S4_CUT_DATES, drift stats, rel_to_train() / rel_detail() / rel_by_split(mode=) / rel_split_table(), record_check(by_split=, kind=, na_reason=, t0_task2=) / checks_summary(kinds=) / checks_requirements() / checks_by_split() / checks_split_scores(), ROSTER_ASOF_ALL, show_d3() / line_chart() / bar_chart() / heatmap()


## A. Label / target and covariate stationarity

Every forward split (train on earlier data, evaluate on later data: S1a/S1b/S2/S3 by Congress, and S4
within the last Congresses) evaluates Congresses that come **after** its training data. The model will
also be deployed on a later Congress (119). So any shift found here is **a difference between history and
the deployment target**, not a defect of a forward split: the forward split is the protocol that measures
it honestly, and the random split S5 is the one that hides it. A drift FAIL below therefore selects a
**remedy** (recency-weighted training, recalibration on the val Congress, density-normalised metrics,
per-Congress reporting), not a different split. All A checks are recorded as `kind='drift'`; every
forward split, including S4 (whose test units are the late parts of 116-118), gets the same verdict from
the candidate Congresses it evaluates, and S5 is left unjudged (None), which is not the same as passing.
Four kinds of shift are tested:

- **Label shift**: the positive rate P(y) changes (e.g. fewer bills get reported). Threshold-based
  metrics, calibration and AUPRC all move with the base rate even when the model is unchanged.
- **Label-definition drift**: the label stops measuring the same thing (e.g. bills advance without a
  committee report more often), so a base-rate change is partly an artifact.
- **Concept shift**: the relationship P(y | x) changes. This is the shift that makes a trained model
  wrong. It can be *regime-conditional* (it depends on which party controls the chamber), in which case
  a regime feature fixes it.
- **Covariate shift**: the inputs P(x) change, marginally or jointly.

**Train pool and candidates.** The reference pool is split S1a's train set (108-115, `REL_REF`). The
candidate val/test Congresses are 116, 117 and 118 (S1a: val 116, test 117; S1b: val 117, test 118).

**Verdict rule (shared `rel_to_train`).** A candidate **fails** a check part when it falls outside the
95% **t prediction interval** for one new Congress built from the 8 reference Congresses
(mean +/- t(7, 0.975) * sd * sqrt(1 + 1/8), about 2.5 SD; log scale for rates and counts; one-sided for
distances such as PSI or AUROC, where a reference Congress's value is its leave-one-out distance to the
other 7). Three extra readings guard against over-calling shift:

- **Multiplicity.** Each check has several parts (features, chambers, t0 values) times 3 candidates. A
  check is recorded as **FAIL only if at least one test survives a Holm correction** within that check.
  The family holds one test per (part x candidate) cell plus one **joint test per part** that asks
  whether the 3 candidates as a group sit off the reference mean (two-sample t of the candidate mean vs
  the 8 reference Congresses; it catches a drop that recurs in every candidate but is too small to
  survive in any single cell). The raw per-part t-PI failures are still printed and kept in `detail`,
  next to the family-wise nested-LOO false-alarm rate. `by_split`: a split fails if a candidate Congress
  it evaluates has a Holm-surviving cell, or a raw cell failure confirmed by a Holm-surviving joint test
  (same direction).
- **Trend vs break.** Each failing level quantity gets one of three labels: **break** (also outside the
  prediction interval of a linear trend fitted on 108-115), **consistent with trend** (inside the trend
  interval AND the 108-115 slope is itself significant), or **undetermined** (inside the trend interval
  but no significant slope: with n = 8 and 1-3 Congresses of extrapolation, the trend test simply has
  too little power to separate trend from level shift). Only the first two map to a remedy.
- **Deployment proxy (A15).** The in-progress 119th Congress is where the model would be used; A15 checks
  which Congresses look most like it, and A15b whether similarity follows **recency** or **party regime**
  (same House / Senate majorities), with and without the sponsor-party features that encode party
  control by construction.

Fixed bands (e.g. 0.8-1.25x) are kept as a secondary column. Congress 119 is excluded from every split.

In [98]:
# Section A shared settings: reference pool, candidates, verdict helpers (wrapping rel_to_train), a bill feature
# table (complete Congresses + 119 for A15) and a small numpy logistic regression (domain classifier, Task-2 model).
A_TRAIN = list(REL_REF)                                # 108-115 (= SPLIT_DEFS['S1a']['train'])
A_CANDS = [c for c in CONGRESSES if c > A_TRAIN[-1]]   # 116-118
A_ALPHA = REL_ALPHA                                    # per-part level of the t prediction interval
A_RATIO_BAND = (0.8, 1.25)                             # secondary fixed band (+/-20%, symmetric in log space)
A_PP_BAND = 0.05                                       # secondary fixed band for shares (+/-5 pp)
A_CH_SHORT = {"House of Representatives": "House", "Senate": "Senate"}
A_ZTAB, A_REFTAB, A_FAM = [], [], []                   # candidate parts, reference nested-LOO parts, per-check families
A_IMPL = {"trend": "predictable drift: add Congress-index/time features, sliding window (S2) or rolling refit (S3)",
          "break": ("level change beyond the train trend: recency-weighted training (refit on train+val) and recalibrate on "
                    "the val Congress; section D's D26 backtests regime weighting on top of recency (keep regime as a "
                    "feature, not a weighting scheme, unless D26 shows a gain)"),
          "undetermined": "no remedy implied: the trend test has too little power to call it trend or break"}
A_IMPL_SHORT = {"trend": "time features / S2 / S3 refit", "break": "recency-weighted training + recalibrate on val",
                "undetermined": "(none: low power)"}
# How the A drift verdicts map onto splits: every forward split is judged from the candidate Congresses it evaluates.
A_S4_NOTE = ("S4 is judged from its folds 116-118 (its test units are the late part of each, scored by a model fitted on "
             "earlier Congresses plus the early part of the same Congress, so the Congress-level shift applies, partly "
             "absorbed by the in-Congress training bills)")
A_NA = {"S5": ("random bill split: train and test mix every Congress, so it hides the history->deployment shift; "
               "not judged, which is NOT passing")}


def a_d(series, nd=3):
    """Series -> plain {int key: rounded float} dict for readable detail text."""
    return {(int(k) if isinstance(k, (int, np.integer)) else k): round(float(v), nd) for k, v in series.items()}


def a_cohen_h(p1, p2):
    """Cohen's h effect size for two proportions (0.2 small, 0.5 medium)."""
    return float(2 * np.arcsin(np.sqrt(p2)) - 2 * np.arcsin(np.sqrt(p1)))


def a_two_prop(pos1, n1, pos2, n2):
    """Chi-square test (2x2, no continuity correction) for equal proportions -> p-value."""
    tab = np.array([[pos1, n1 - pos1], [pos2, n2 - pos2]], float)
    return float(stats.chi2_contingency(tab, correction=False)[1])


def a_p(t, df, side):
    """p-value of a t statistic for the given tail."""
    if t is None or not np.isfinite(t) or df < 1:
        return np.nan
    return float({"two": 2 * stats.t.sf(abs(t), df), "upper": stats.t.sf(t, df), "lower": stats.t.cdf(t, df)}[side])


def a_rel(values, log=False, side="two", band=None, train=None):
    """rel_to_train for the candidates 116-118, plus p-values, a trend/break label and the secondary fixed band.
    Returns (tab with ref + cand rows, info)."""
    tab, info = rel_to_train(values, train=train, side=side, log=log, cands=A_CANDS)
    n, ref = info["n"], tab[tab.role == "ref"]
    cand = tab.role == "cand"
    tab["p"] = [a_p(r.t_pi, n - 1 if r.role == "cand" else n - 2, side) for r in tab.itertuples()]
    tab["trend_p"] = [a_p(r.trend_z, n - 2 if r.role == "cand" else n - 3, side) for r in tab.itertuples()]
    # Significance of the 108-115 linear slope itself (log scale when log=True): needed to call a failure a "trend".
    fr = (np.log(ref.value.where(ref.value > 0)) if log else ref.value).astype(float).dropna()
    slope_p = float(stats.linregress(np.asarray(fr.index, float), fr.values).pvalue) if len(fr) > 2 else np.nan
    trend_ok = bool(slope_p < A_ALPHA)
    tab["kind"] = np.where(tab.fail.eq(True),
                           np.where(tab.trend_fail.eq(True), "break", "trend" if trend_ok else "undetermined"),
                           np.where(tab.trend_fail.eq(True), "off-trend", ""))
    info.update(slope_p=slope_p,lo=ref.value.min(), hi=ref.value.max(), band=band,
                band_train_fail=None if band is None else int((~ref.value.between(*band)).sum()),
                band_cand_fail=None if band is None else [int(c) for c, v in tab.value[cand].items()
                                                          if not band[0] <= v <= band[1]])
    return tab, info


def a_holm(p, alpha=None):
    """Holm step-down survivors (bool array) for a vector of p-values (NaN -> 1)."""
    alpha = A_ALPHA if alpha is None else alpha
    p = np.nan_to_num(np.asarray(p, float), nan=1.0)
    order = np.argsort(p)
    ok = p[order] <= alpha / (len(p) - np.arange(len(p)))
    k = len(p) if ok.all() else int(np.argmin(ok))
    out = np.zeros(len(p), bool)
    out[order[:k]] = True
    return out


def a_split_from(cand_fail):
    """by_split from per-candidate verdicts {congress: fails?}: a forward split fails if a candidate it evaluates fails
    (reference Congresses are not judged; same as rel_by_split's default 'cands' mode). S4 is judged from its folds
    (116-118, the same candidates); S5 gets None (see A_NA)."""
    t = pd.DataFrame({"fail": pd.Series(cand_fail, dtype=object), "role": "cand"})
    bs = rel_by_split(t, col="fail")
    s4 = [c for c in SPLIT_DEFS["S4"]["folds"] if c in cand_fail and cand_fail[c] is not None]
    bs["S4"] = (not any(bool(cand_fail[c]) for c in s4)) if s4 else None
    bs["S5"] = None
    return bs


def a_split_metric(per_congress, roles=("test",)):
    """by_split of a per-Congress metric: mean over the Congresses each split evaluates in `roles` (S4: its folds;
    S5: None)."""
    out = {}
    for k in ALL_SPLITS:
        if split_kind(k) == "random":
            out[k] = None
            continue
        cs = SPLIT_DEFS["S4"]["folds"] if split_kind(k) == "within" else split_eval_congresses(k, roles=roles)
        v = [per_congress[c] for c in cs if c in per_congress]
        out[k] = round(float(np.mean(v)), 4) if v else None
    return out


def a_joint(cand_t, n, side):
    """Joint test that the k candidates as a group are off the reference mean. With t_i = (x_i - m) / (s sqrt(1 + 1/n)),
    the two-sample t of the candidate mean vs the n reference values is mean(t_i) * sqrt(1 + 1/n) / sqrt(1/k + 1/n),
    t-distributed with n - 1 df under H0 (it accounts for the reference mean the candidates share) -> (t, p)."""
    ct = np.asarray(cand_t, float)
    ct = ct[np.isfinite(ct)]
    if len(ct) < 2 or n < 3:
        return np.nan, np.nan
    tj = float(ct.mean() * np.sqrt(1 + 1 / n) / np.sqrt(1 / len(ct) + 1 / n))
    return tj, a_p(tj, n - 1, side)


def a_record(cid, cond, parts, why, quantity=None, fmt=".3f", band_label="", level=True):
    """Print one line per part, the family-wise false-alarm rate, and record a check that FAILs if any test survives a
    Holm correction within the check. Family = (part x candidate) cells + one joint test per part (candidates as a
    group vs the reference mean, a_joint). Raw per-part t-PI failures stay in `detail`. parts: list of (label, tab,
    info) from a_rel. level=False for distances (LOO PSI / JSD / domain AUROC): no trend/break label."""
    lines, worst, wscore, refrows, crow, jrow = [], 0.0, -np.inf, [], [], []
    for lab, t, i in parts:
        name = f"{cid} {lab}".strip()
        cand, ref = t[t.role == "cand"], t[t.role == "ref"]
        for c, r in cand.iterrows():
            crow.append(dict(check=cid, part=name, lab=lab, congress=int(c), t=float(r.t_pi), p=r.p,
                             fail=bool(r.fail), trend_z=float(r.trend_z), trend_p=r.trend_p,
                             trend_fail=bool(r.trend_fail), kind=r.kind, level=level, fa_rate=i["fa_rate"], n=i["n"]))
            sc = {"two": abs(r.t_pi), "upper": r.t_pi, "lower": -r.t_pi}[i["side"]]   # worst in the failing direction
            if np.isfinite(r.t_pi) and sc > wscore:
                worst, wscore = float(r.t_pi), sc
        tj, pj = a_joint(cand.t_pi.values, i["n"], i["side"])
        jrow.append(dict(part=name, lab=lab, t=tj, p=pj, side=i["side"]))
        for c, r in ref.iterrows():
            refrows.append(dict(check=cid, part=name, congress=int(c), fail=bool(r.fail),
                                trend_fail=bool(r.trend_fail), kind=r.kind, level=level))
        lo, hi = cand.pi_lo.iloc[0], cand.pi_hi.iloc[0]
        print(f"  {name}: ref {i['lo']:{fmt}}..{i['hi']:{fmt}}, 95% PI [{lo:{fmt}}, {hi:{fmt}}] (LOO false alarms "
              f"{i['fa_count']}/{i['n']}) | " + " | ".join(
                  f"{c}: {r.value:{fmt}} t={r.t_pi:+.1f} tr={r.trend_z:+.1f}{(' FAIL-' + r.kind if level else ' FAIL') if r.fail else ''}"
                  for c, r in cand.iterrows()) + f" | joint t={tj:+.1f} p={pj:.2g}")
        band = (f"; secondary fixed band {band_label} fails {i['band_train_fail']}/{i['n']} reference Congresses and "
                f"candidates {i['band_cand_fail']}") if i["band"] is not None else ""
        lines.append(f"[{lab or 'all'}] " + rel_detail(t, i, fmt) + f"; joint candidate t={tj:+.2f} (p={pj:.2g})" + band)
    cr, jr = pd.DataFrame(crow), pd.DataFrame(jrow)
    hs = a_holm(np.r_[cr.p.values, jr.p.values])   # one Holm family: cells + joint tests
    cr["holm"] = hs[:len(cr)] & cr.fail.values      # Holm survivor (and outside the PI)
    jr["holm"] = hs[len(cr):]
    # A Holm-surviving joint test confirms the recurring shift: candidates whose own cell fails the raw t-PI in the same
    # direction then count as failing too (the joint test supplies the multiplicity control their cells lacked).
    cr["jholm"] = False
    for r in jr[jr.holm].itertuples():
        cr.loc[(cr.part == r.part) & cr.fail & (np.sign(cr.t) == np.sign(r.t)), "jholm"] = True
    cr["sig"] = cr.holm | cr.jholm
    A_ZTAB.extend(cr.drop(columns="lab").to_dict("records"))
    A_REFTAB.extend(refrows)
    rf = pd.DataFrame(refrows)
    anyf = rf.groupby("congress").fail.any()
    fam = dict(check=cid, parts=len(parts), n_ref=len(anyf), fwer_loo=float(anyf.mean()),
               fwer_nominal=float(1 - (1 - A_ALPHA) ** len(parts)), fwer_ids=[int(c) for c in anyf.index[anyf]],
               joint_survivors=[r.part for r in jr[jr.holm].itertuples()])
    A_FAM.append(fam)
    lab_of = lambda r: f"{r.lab + ' ' if r.lab else ''}{r.congress}" + (f" ({r.kind})" if level else "")
    raw = [lab_of(r) for r in cr[cr.fail].itertuples()]
    surv = [lab_of(r) for r in cr[cr.holm].itertuples()]
    jsurv = [f"{r.lab or 'all'} joint (t={r.t:+.1f})" for r in jr[jr.holm].itertuples()]
    jraw = [f"{r.lab or 'all'} joint (t={r.t:+.1f})" for r in jr[jr.p < A_ALPHA].itertuples()]
    print(f"  {cid}: raw t-PI failures {raw or 'none'}; Holm survivors (decide the verdict): cells {surv or 'none'}, "
          f"joint {jsurv or 'none'}" + (f"; family-wise nested-LOO false-alarm rate {fam['fwer_loo']:.0%} over "
                                        f"{len(parts)} parts" if len(parts) > 1 else ""))
    record_check("A", cid, cond, not (surv or jsurv), metric=float(worst),
                 threshold=parts[0][2]["rule"] + f"; FAIL only if a test survives Holm over {len(cr)} cells + {len(jr)} joint tests",
                 detail=f"VERDICT RULE: FAIL iff >= 1 test survives a Holm correction over the {len(cr)} (part x candidate) "
                        f"cells and {len(jr)} joint-candidate tests (one per part: candidate mean vs reference mean, "
                        f"two-sample t, df n-1) of this check at {A_ALPHA:.0%}. Holm survivors: cells {surv or 'none'}; "
                        f"joint {jsurv or 'none'}. Raw per-part t-PI failures (secondary, not multiplicity-corrected): "
                        f"{raw or 'none'}; raw joint p < {A_ALPHA}: {jraw or 'none'}. "
                        + "; ".join(lines) + f". Family-wise nested-LOO false-alarm rate over {len(parts)} part(s): "
                        f"{fam['fwer_loo']:.0%} (nominal {fam['fwer_nominal']:.0%}). Labels: break = outside the 108-115 "
                        f"linear-trend PI; trend = inside it and the 108-115 slope is significant; undetermined = "
                        f"neither. KIND drift: a FAIL means the later Congresses (and so the deployment target) differ "
                        f"from 108-115; the forward split measures this; apply the remedy, do not switch splits. by_split: a "
                        f"split fails if a candidate it evaluates has a Holm-surviving cell, or a raw cell failure in the "
                        f"direction of a Holm-surviving joint test; {A_S4_NOTE}; S5 not judged. " + why,
                 quantity=quantity, by_split=a_split_from(cr.groupby("congress").sig.any().to_dict()),
                 kind="drift", na_reason=A_NA)


# Bill feature table (law-capable bills, complete Congresses + the in-progress one for A15). Row i of A_X = row i
# of fb_all; fb = complete Congresses only (index kept, so fb.index indexes A_X).
_cau = members.drop_duplicates(["bioguide_id", "congress"]).set_index(["bioguide_id", "congress"])["caucus"]
A_ALLC = CONGRESSES + [IN_PROGRESS]
fb_all = bills[bills.law_capable & bills.congress.isin(A_ALLC)][
    ["bill_id", "congress", "chamber", "bill_type", "policy_area", "policy_area_harm", "sponsor_party",
     "sponsor_bioguide_id", "day_of_congress"]].reset_index(drop=True)
fb_all["ch"] = fb_all.chamber.map(A_CH_SHORT)
fb_all = fb_all.merge(MAJORITY.assign(ch=MAJORITY.chamber.map(A_CH_SHORT))[["congress", "ch", "majority"]],
                      on=["congress", "ch"], how="left")
# Sponsor bloc = caucus (independents with the party they caucus with), falling back to the party code.
fb_all["sponsor_caucus"] = pd.Series(pd.MultiIndex.from_arrays([fb_all.sponsor_bioguide_id, fb_all.congress]).map(_cau),
                                     index=fb_all.index).fillna(fb_all.sponsor_party)
fb_all["sponsor_status"] = np.where(fb_all.sponsor_caucus == fb_all.majority, "majority", "minority")
cl_all = cosp[cosp.law_capable & cosp.congress.isin(A_ALLC)]
fb_all["n_original"] = fb_all.bill_id.map(cl_all[cl_all.is_original].groupby("bill_id").size()).fillna(0)
fb_all["day_bin"] = np.minimum(fb_all.day_of_congress.clip(lower=0) // 91, 7).astype(int).astype(str)
A_CAT = ["ch", "bill_type", "sponsor_caucus", "sponsor_status", "policy_area_harm", "day_bin"]
for f in A_CAT:   # categorical design columns as plain strings (missing -> its own level)
    fb_all[f] = fb_all[f].fillna("(missing)").astype(str)
fb = fb_all[fb_all.congress.isin(CONGRESSES)]
cl = cl_all[cl_all.congress.isin(CONGRESSES)]
A_NOCAU = fb[fb.sponsor_caucus.eq("(missing)")].groupby("congress").size()
A_LEVELS = {f: fb[f].value_counts().index.tolist() for f in A_CAT}   # first (most frequent) = reference level
A_XNAMES = ["(intercept)"] + [f"{f}={l}" for f in A_CAT for l in A_LEVELS[f][1:]] + ["log1p(n_original)"]
A_X = np.column_stack([np.ones(len(fb_all))] + [(fb_all[f].values == l).astype(float)
                                                 for f in A_CAT for l in A_LEVELS[f][1:]] + [np.log1p(fb_all.n_original.values)])
A_ROW = pd.Series(fb_all.index.values, index=fb_all.bill_id)
# Policy area is assigned by CRS weeks after introduction (PA_MISS_T0 of fresh bills lack it), so it is NOT
# available at t0. A_NOPA = design columns without policy area (the t0-safe feature set).
A_NOPA = np.array([not n.startswith("policy_area_harm=") for n in A_XNAMES])
# Sponsor-party columns encode party control by construction (sponsor_status = caucus == majority; the share of bills
# sponsored by each caucus moves with the majority). A15/A15b also use t0-safe sets without them.
A_FSETS = {
    "no party": A_NOPA & np.array([not n.startswith(("sponsor_caucus=", "sponsor_status=")) for n in A_XNAMES]),
    "majority only": A_NOPA & np.array([not n.startswith("sponsor_caucus=") for n in A_XNAMES]),
    "with party": A_NOPA,
}



def a_logit(X, y, w=None, l2=1.0, iters=40):
    """L2-penalised logistic regression by Newton/IRLS (intercept unpenalised) -> coefficients."""
    w = np.ones(len(y)) if w is None else w
    pen = np.full(X.shape[1], float(l2))
    pen[0] = 0.0
    b = np.zeros(X.shape[1])
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(X @ b, -30, 30)))
        H = (X * (w * p * (1 - p))[:, None]).T @ X + np.diag(pen)
        step = np.linalg.solve(H, X.T @ (w * (y - p)) - pen * b)
        b += step
        if np.abs(step).max() < 1e-7:
            break
    return b


def a_auc(y, s):
    """AUROC from ranks (Mann-Whitney)."""
    y = np.asarray(y).astype(bool)
    r = stats.rankdata(s)
    n1, n0 = y.sum(), (~y).sum()
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


adv_u = adv[~adv.censored & adv.congress.isin(CONGRESSES)].copy()
adv_u["ch"] = adv_u.chamber.map(A_CH_SHORT)
adv_u = adv_u.merge(fb[["bill_id", "sponsor_caucus", "sponsor_status", "majority"]], on="bill_id", how="left")
print(f"train pool {A_TRAIN[0]}-{A_TRAIN[-1]} | candidates {A_CANDS} | uncensored advancement labels: {len(adv_u):,} | "
      f"bill features: {A_X.shape[1]} columns over {len(fb):,} law-capable bills | verdict: outside the {1 - A_ALPHA:.0%} t prediction interval of {A_TRAIN[0]}-{A_TRAIN[-1]}")
print(f"majority party of {IN_PROGRESS} (for A15): " + "; ".join(f"{r.chamber}: {r.majority}" for r in MAJORITY[MAJORITY.congress == IN_PROGRESS].itertuples())
      + f" | bills whose sponsor has no caucus/party (treated as minority; own level in the classifiers): {A_NOCAU.to_dict()}")

train pool 108-115 | candidates [116, 117, 118] | uncensored advancement labels: 110,802 | bill features: 47 columns over 128,791 law-capable bills | verdict: outside the 95% t prediction interval of 108-115
majority party of 119 (for A15): House of Representatives: R; Senate: R | bills whose sponsor has no caucus/party (treated as minority; own level in the classifiers): {108: 1, 109: 1, 110: 2, 111: 1, 117: 11}


### A1. Advancement base rate (Task 2) per Congress, and what drives its change
**Why it matters:** if the share of bills reported within 180 days differs a lot between train and the
held-out Congress, a model trained on 108-115 is mis-calibrated there, and AUPRC / F1 on the test
Congress cannot be compared with validation scores. **How to read it:** `rate` per Congress; `ratio` =
rate / pooled-train rate, `h` = Cohen's h and `p` = chi-square p (train rows: vs the other train
Congresses). With about 10k bills almost every gap is "significant", so the verdict uses the t prediction
interval of the train Congresses (`t` = standardised distance, `tr` = residual vs the 108-115 trend). The decomposition splits each candidate's log rate change into a **volume part**
(more bills introduced) and a **count part** (fewer bills reported), both relative to the mean train
Congress.

In [99]:
# A1: uncensored advancement rate per Congress; t prediction interval vs train Congresses (log scale); volume/count decomposition.
a1 = adv_u.groupby("congress").agg(n=("label", "size"), n_reported=("label", "sum"), rate=("label", "mean"))
a1["censored_share"] = adv[adv.congress.isin(CONGRESSES)].groupby("congress")["censored"].mean()
tr = adv_u[adv_u.congress.isin(A_TRAIN)]
p_tr = tr.label.mean()
_ref = {c: (tr[tr.congress != c] if c in A_TRAIN else tr) for c in CONGRESSES}
a1["ratio"] = [a1.loc[c, "rate"] / _ref[c].label.mean() for c in a1.index]
a1["h"] = [a_cohen_h(_ref[c].label.mean(), a1.loc[c, "rate"]) for c in a1.index]
a1["p"] = [a_two_prop(_ref[c].label.sum(), len(_ref[c]), a1.loc[c, "n_reported"], a1.loc[c, "n"]) for c in a1.index]
print(f"pooled train rate = {p_tr:.4f} (n={len(tr):,})")
print(a1.round(4).to_string())
t1, i1 = a_rel(a1.rate, log=True, band=(p_tr * A_RATIO_BAND[0], p_tr * A_RATIO_BAND[1]))
a_record("A1", "Advancement base rate of each candidate within the spread of train Congresses", [("", t1, i1)],
         "Rates compared on the log scale. FAIL implies: report rank metrics (AUROC) next to AUPRC, recalibrate or "
         "re-threshold on the validation Congress, and compare val/test scores only after base-rate normalisation "
         "(e.g. AUPRC lift = AUPRC / base rate). See A1b/A1c for how much of the drop is a label artifact.",
         quantity="label_rate_t2", fmt=".4f", band_label="0.8-1.25x pooled train")

# Decomposition: log(rate_c / rate_ref) = log(n_reported_c / mean_train) - log(n_c / mean_train).
ref_n, ref_k = a1.loc[A_TRAIN, "n"].mean(), a1.loc[A_TRAIN, "n_reported"].mean()
dec = pd.DataFrame({"n_ratio": a1.n / ref_n, "n_reported_ratio": a1.n_reported / ref_k})
dec["rate_ratio"] = dec.n_reported_ratio / dec.n_ratio
dec["log_change"] = np.log(dec.rate_ratio)
dec["volume_part"] = -np.log(dec.n_ratio)
dec["count_part"] = np.log(dec.n_reported_ratio)
dec["volume_share"] = (dec.volume_part / dec.log_change).where(dec.log_change < 0)
print("decomposition vs the mean train Congress (log_change = volume_part + count_part):")
print(dec.loc[A_CANDS].round(3).to_string())
tk, ik = a_rel(a1.n_reported, log=True)
a_record("A1d", "Number of reported bills (numerator of the Task-2 rate) within the spread of train Congresses",
         [("n_reported", tk, ik)],
         f"Volume share of the log rate drop: {a_d(dec.loc[A_CANDS, 'volume_share'], 2)} (1 = drop entirely from "
         f"more bills; < 1 = committees also reported fewer bills). FAIL here means the rate drop is not only a "
         f"volume effect: the reporting process itself changed (or the label lost coverage, see A1b).",
         quantity="label_count_t2", fmt=".0f")

pooled train rate = 0.0650 (n=72,139)
              n  n_reported   rate  censored_share  ratio       h      p
congress                                                                
108        7533         606 0.0804          0.1262 1.2732  0.0670 0.0000
109        9494         524 0.0552          0.1125 0.8303 -0.0472 0.0000
110        9784         761 0.0778          0.1283 1.2350  0.0579 0.0000
111        9382         444 0.0473          0.1289 0.6998 -0.0876 0.0000
112        9592         414 0.0432          0.0961 0.6316 -0.1104 0.0000
113        7727         447 0.0578          0.1493 0.8786 -0.0332 0.0071
114        8948         653 0.0730          0.1247 1.1429  0.0362 0.0011
115        9679         839 0.0867          0.1520 1.4066  0.0960 0.0000
116       11866         491 0.0414          0.1725 0.6367 -0.1058 0.0000
117       12953         364 0.0281          0.1502 0.4324 -0.1787 0.0000
118       13844         611 0.0441          0.1643 0.6791 -0.0922 0.0000
  A1: ref 0.0

### A1b-A1c. Does the Task-2 label measure the same thing in every Congress?
**Why it matters:** "reported out of committee within 180 days" misses bills that advance without a
committee report: suspension of the rules in the House, unanimous consent in the Senate. If that route
grows in later Congresses, part of the base-rate drop is a **label-definition artifact**, and a model
would be penalised for correctly ranking bills that did advance. **How to read it:** among uncensored
bills that passed their chamber of origin within 180 days, the share *not* reported within 180 days
(`unrep_share`; `never_rep` = no Reported action at all). A1c recomputes the base rate with an
alternative label (reported **or** passed the origin chamber within 180 days) and checks whether the
candidates' drop shrinks under it.

In [100]:
# A1b/A1c: share of chamber-passed bills without a report; alternative label (reported OR passed origin chamber).
_act = load("bill_actions", ("bill_id", "action_date", "text"))
_pass = _act[_act.text.str.match(r"Passed/agreed to in (House|Senate)", na=False)]
_pass = (_pass.assign(ch=_pass.text.str.extract(r"in (House|Senate)", expand=False))
         .groupby(["bill_id", "ch"])["action_date"].min().rename("passed_date").reset_index())
lv = adv_u.merge(_pass, on=["bill_id", "ch"], how="left")
lv["passed180"] = (lv.passed_date - lv.introduced_date).dt.days.between(0, ADV_WINDOW_DAYS)
lv["alt"] = ((lv.label == 1) | lv.passed180).astype(int)
ps = lv[lv.passed180]
vt = ps.assign(unrep=ps.label == 0, never=ps.reported_date.isna())
val_ch = vt.groupby(["congress", "ch"]).agg(n_passed=("unrep", "size"), unrep_share=("unrep", "mean")).unstack()
val_ch.columns = [f"{a}_{b}" for a, b in val_ch.columns]
val = vt.groupby("congress").agg(n_passed=("unrep", "size"), unrep_share=("unrep", "mean"), never_rep=("never", "mean"))
val = val.join(val_ch.drop(columns=[c for c in val_ch.columns if c.startswith("n_passed")]))
val["alt_rate"] = lv.groupby("congress")["alt"].mean()
val["label_rate"] = a1.rate
print(val.round(3).to_string())
tb, ib = a_rel(val.unrep_share, side="upper")
a_record("A1b", "Share of chamber-passed bills that the Task-2 label misses (no report within 180 d) not above train spread",
         [("", tb, ib)],
         "Pass routes without a committee report (suspension, unanimous consent) are invisible to the label. FAIL "
         "means the label covers less real advancement in that Congress; use the alternative label (A1c) as a "
         "robustness target or add 'passed chamber' to the positive class.", quantity="label_validity_t2")
ta, ia = a_rel(val.alt_rate, log=True)
p_alt = lv[lv.congress.isin(A_TRAIN)].alt.mean()
shr = pd.DataFrame({"log_ratio_label": np.log(val.label_rate / p_tr), "log_ratio_alt": np.log(val.alt_rate / p_alt)}).loc[A_CANDS]
shr["drop_shrinks"] = shr.log_ratio_alt.abs() < shr.log_ratio_label.abs()
print("candidate log ratio vs pooled train, Task-2 label vs alternative label:")
print(shr.round(3).to_string())
a_record("A1c", "Alternative advancement label (reported OR passed origin chamber in 180 d): base rate within train spread",
         [("", ta, ia)],
         f"The candidates' gap shrinks under the alternative label for {int(shr.drop_shrinks.sum())}/{len(shr)} candidates "
         f"(log ratios {a_d(shr.log_ratio_label, 2)} -> {a_d(shr.log_ratio_alt, 2)}). If the alternative "
         f"label passes where A1 fails, the A1 drop is partly a label-definition artifact rather than a population change.",
         quantity="label_rate_t2_alt", fmt=".4f")

          n_passed  unrep_share  never_rep  unrep_share_House  unrep_share_Senate  alt_rate  label_rate
congress                                                                                               
108            645       0.4030     0.3710             0.4530              0.3150    0.1150      0.0800
109            584       0.5260     0.4850             0.5460              0.4840    0.0880      0.0550
110            787       0.4170     0.3710             0.4000              0.4930    0.1110      0.0780
111            593       0.5130     0.4640             0.4940              0.6040    0.0800      0.0470
112            353       0.4250     0.3800             0.4010              0.5060    0.0590      0.0430
113            411       0.4040     0.3940             0.4040              0.4050    0.0790      0.0580
114            608       0.3470     0.3190             0.3430              0.3640    0.0970      0.0730
115            786       0.3910     0.3520             0.3790   

### A2. Advancement base rate by chamber
**Why it matters:** House and Senate bills have different reporting rates, so a pooled rate can hide
opposite movements. A by-Congress split needs each chamber to be roughly stable, or the model needs
chamber as a feature (and per-chamber evaluation). **How to read it:** the chart shows the rate per
chamber, the pooled rate and the pooled rate under A1c's alternative label; the check applies the t
prediction-interval rule within each chamber.

In [101]:
# A2: advancement rate per Congress x chamber; t PI per chamber vs the same chamber's train Congresses.
a2 = adv_u.groupby(["congress", "ch"])["label"].mean().unstack()
print(a2.round(4).T.to_string())
parts2 = []
for ch in a2.columns:
    pc = adv_u[adv_u.congress.isin(A_TRAIN) & (adv_u.ch == ch)].label.mean()
    parts2.append((ch,) + a_rel(a2[ch], log=True, band=(pc * A_RATIO_BAND[0], pc * A_RATIO_BAND[1])))
chart = pd.concat([a2.stack().rename("rate").reset_index().rename(columns={"ch": "series"}),
                   a1.rate.reset_index().assign(series="Both (Task-2 label)"),
                   val.alt_rate.rename("rate").reset_index().assign(series="Both (reported or passed)")])
line_chart(chart, "congress", "rate", series="series", title="Task-2 positive rate per Congress",
           subtitle=f"Uncensored law-capable bills; train {A_TRAIN[0]}-{A_TRAIN[-1]}, candidates {A_CANDS[0]}-{A_CANDS[-1]}; "
                    "last line = alternative label", yfmt=".1%", ylabel="positive share", xlabel="Congress",
           name="A_adv_rate_by_chamber")
a_record("A2", "Per-chamber advancement rate of each candidate within the same chamber's train spread", parts2,
         "If one chamber drifts more than the other, evaluate per chamber and include chamber as a feature.",
         quantity="label_rate_t2_chamber", fmt=".4f", band_label="0.8-1.25x same-chamber train")

congress    108    109    110    111    112    113    114    115    116    117    118
ch                                                                                   
House    0.0695 0.0499 0.0722 0.0538 0.0459 0.0576 0.0744 0.0905 0.0351 0.0257 0.0429
Senate   0.1005 0.0636 0.0888 0.0368 0.0381 0.0584 0.0705 0.0792 0.0525 0.0326 0.0465


  A2 House: ref 0.0459..0.0905, 95% PI [0.0352, 0.1120] (LOO false alarms 0/8) | 116: 0.0351 t=-2.4 tr=-2.5 FAIL-break | 117: 0.0257 t=-3.6 tr=-3.5 FAIL-break | 118: 0.0429 t=-1.6 tr=-1.8 | joint t=-4.0 p=0.0055
  A2 Senate: ref 0.0368..0.1005, 95% PI [0.0252, 0.1594] (LOO false alarms 0/8) | 116: 0.0525 t=-0.5 tr=-0.1 | 117: 0.0326 t=-1.7 tr=-1.0 | 118: 0.0465 t=-0.8 tr=-0.2 | joint t=-1.6 p=0.16
  A2: raw t-PI failures ['House 116 (break)', 'House 117 (break)']; Holm survivors (decide the verdict): cells none, joint ['House joint (t=-4.0)']; family-wise nested-LOO false-alarm rate 0% over 2 parts
[FAIL] A2: Per-chamber advancement rate of each candidate within the same chamber's train spread | metric=-3.645 (threshold outside 95% t prediction interval of ref 108-115 (two-sided, log scale; n=8); FAIL only if a test survives Holm over 6 cells + 2 joint tests) [fails for: S1a, S1b, S2, S3, S1a+stream, S1b+stream, S3+stream, S1pool, S4]


### A3. Cosponsor-task labels (Task 1) at t0 = 0 and t0 = 7
**Why it matters:** ranking metrics (Hits@k, MRR, AUPRC) depend on how many positives each bill has
and on the positive density among eligible legislators. If later Congresses have fewer
post-introduction cosponsors, the test task is harder and scores drop for reasons unrelated to the
model. **How to read it:** per Congress and t0: share of bills with at least one positive, mean
positives per bill, positives / eligible (density) and censored share. The eligible pool is the
**point-in-time** roster (members in office, delegates included, in the month of introduction + t0), the
same pool section D ranks; `density_legacy` uses the full-Congress roster for comparison. A3 checks
density, A3b the share of bills with any positive.

In [102]:
# A3: window_targets(t0, point_in_time=True) per Congress (uncensored bills): >=1 positive share, mean positives,
# density over the point-in-time pool (ROSTER_ASOF_ALL, delegates included; same pool as section D); legacy pool shown.
a3_rows = []
for t0 in (0, 7):
    wt = window_targets(t0, point_in_time=True)
    wt = wt[wt.congress.isin(CONGRESSES)]
    u = wt[~wt.censored & wt.eligible.gt(0)]
    g = u.groupby("congress").agg(bills=("n_pos", "size"), any_pos=("n_pos", lambda s: (s > 0).mean()),
                                  mean_pos=("n_pos", "mean"), pos=("n_pos", "sum"), elig=("eligible", "sum"))
    g["density"] = g.pos / g.elig
    wl = window_targets(t0)   # legacy full-Congress pool (secondary)
    wl = wl[wl.congress.isin(CONGRESSES) & ~wl.censored & wl.eligible.gt(0)].groupby("congress")[["n_pos", "eligible"]].sum()
    g["density_legacy"] = wl.n_pos / wl.eligible
    g["censored_share"] = wt.groupby("congress")["censored"].mean()
    g["t0"] = t0
    a3_rows.append(g.reset_index())
a3 = pd.concat(a3_rows, ignore_index=True)
print(a3.pivot(index="congress", columns="t0", values=["any_pos", "mean_pos", "density", "density_legacy", "censored_share"]).round(4).to_string())
line_chart(a3.assign(t0=lambda d: "t0=" + d.t0.astype(str)), "congress", "density", series="t0",
           title="Cosponsor-task positive density (new cosponsors / eligible legislators)",
           subtitle="Positives = cosponsors joining in (t0, t0+30]; point-in-time pool (incl. delegates); uncensored law-capable bills",
           yfmt=".2%", ylabel="density", xlabel="Congress", name="A_cosp_density")
p3, p3b = [], []
for t0, g in a3.groupby("t0"):
    s = g.set_index("congress")
    m = s.loc[A_TRAIN, "pos"].sum() / s.loc[A_TRAIN, "elig"].sum()
    p3.append((f"t0={t0}",) + a_rel(s.density, log=True, band=(m * A_RATIO_BAND[0], m * A_RATIO_BAND[1])))
    p3b.append((f"t0={t0}",) + a_rel(s.any_pos, log=True))
a_record("A3", "Cosponsor positive density (t0=0 and t0=7) of each candidate within train spread", p3,
         "Hits@k/MRR/AUPRC move with positive density: when this fails, val/test scores are not comparable to "
         "train-era scores. Use density-normalised metrics (lift over random) and compare models only within the "
         "same held-out Congress. Section C should reference this check rather than re-test it. Denominator = "
         "point-in-time eligible pool (ROSTER_ASOF_ALL in the month of introduction + t0, delegates included), the pool "
         "section D ranks; legacy full-Congress-pool density is printed as density_legacy (secondary).",
         quantity="cosp_density", fmt=".4f", band_label="0.8-1.25x pooled train")
a_record("A3b", "Share of bills with at least one new cosponsor (t0=0 and t0=7) within train spread", p3b,
         "Bills with no positive contribute nothing to ranking metrics; a falling share shrinks the effective test set.",
         quantity="label_rate_t1", fmt=".3f")

         any_pos        mean_pos        density        density_legacy        censored_share       
t0             0      7        0      7       0      7              0      7              0      7
congress                                                                                          
108       0.3417 0.3092   2.6315 2.2929  0.0084 0.0074         0.0084 0.0073         0.0017 0.0017
109       0.2966 0.2610   2.3410 1.9425  0.0077 0.0064         0.0076 0.0064         0.0143 0.0143
110       0.3498 0.3086   2.7635 2.3032  0.0086 0.0072         0.0084 0.0070         0.0052 0.0052
111       0.3426 0.2980   2.4237 1.9709  0.0079 0.0065         0.0077 0.0062         0.0123 0.0199
112       0.3091 0.2723   2.1881 1.7133  0.0069 0.0054         0.0068 0.0053         0.0150 0.0186
113       0.3868 0.3348   2.9735 2.3704  0.0093 0.0074         0.0091 0.0073         0.0142 0.0203
114       0.3789 0.3221   2.5330 1.8993  0.0080 0.0060         0.0079 0.0060         0.0157 0.0213
115       

  A3 t0=0: ref 0.0069..0.0093, 95% PI [0.0062, 0.0102] (LOO false alarms 0/8) | 116: 0.0084 t=+0.6 tr=+0.8 | 117: 0.0065 t=-2.0 tr=-1.0 | 118: 0.0047 t=-5.2 tr=-3.2 FAIL-break | joint t=-3.5 p=0.011
  A3 t0=7: ref 0.0054..0.0074, 95% PI [0.0048, 0.0087] (LOO false alarms 0/8) | 116: 0.0065 t=+0.1 tr=+1.0 | 117: 0.0049 t=-2.2 tr=-0.9 | 118: 0.0035 t=-4.9 tr=-2.8 FAIL-break | joint t=-3.6 p=0.0086
  A3: raw t-PI failures ['t0=0 118 (break)', 't0=7 118 (break)']; Holm survivors (decide the verdict): cells ['t0=0 118 (break)', 't0=7 118 (break)'], joint none; family-wise nested-LOO false-alarm rate 0% over 2 parts
[FAIL] A3: Cosponsor positive density (t0=0 and t0=7) of each candidate within train spread | metric=-5.204 (threshold outside 95% t prediction interval of ref 108-115 (two-sided, log scale; n=8); FAIL only if a test survives Holm over 6 cells + 2 joint tests) [fails for: S1b, S2, S3, S1b+stream, S3+stream, S1pool, S4]
  A3b t0=0: ref 0.297..0.387, 95% PI [0.273, 0.436] (LOO fals

### A4. Original-cosponsor share (what is left to predict after day 0)
**Why it matters:** Task 1 only counts cosponsors who join *after* t0. Cosponsors listed at
introduction are inputs (t0 = 7) or simply absent from the target. If the share of cosponsorships that
are original rises, the target shrinks in later Congresses and the day-0 graph is denser: both the label
and the input distribution move. **How to read it:** share of active cosponsorships (law-capable
bills) that are original, and the shares joining in days 1-30 and after day 30.

In [103]:
# A4: share of cosponsorships that are original (day-0) per Congress; t PI vs train Congresses.
a4 = cl.groupby("congress")["is_original"].mean().rename("original_share").to_frame()
a4["post_intro_le30"] = cl.assign(w=cl.lag_days.between(1, WINDOW_DAYS)).groupby("congress")["w"].mean()
a4["post_intro_gt30"] = cl.assign(w=cl.lag_days > WINDOW_DAYS).groupby("congress")["w"].mean()
tr4 = cl[cl.congress.isin(A_TRAIN)].is_original.mean()
print(a4.round(3).T.to_string())
t4, i4 = a_rel(a4.original_share, band=(tr4 - A_PP_BAND, tr4 + A_PP_BAND))
a_record("A4", "Original-cosponsor share of each candidate within train spread", [("", t4, i4)],
         "A rising original share means fewer post-introduction positives and a denser day-0 graph; implies a "
         "sliding window (recent Congresses) or explicit Congress-level time features. Section C should reference "
         "this check rather than re-test it.", quantity="orig_share", band_label="+/-5 pp of pooled train")

congress           108    109    110    111    112    113    114    115    116    117    118
original_share  0.3890 0.3650 0.3300 0.3410 0.3310 0.3300 0.3590 0.3940 0.3830 0.4600 0.4660
post_intro_le30 0.1890 0.2030 0.2130 0.2070 0.2050 0.2200 0.1960 0.1820 0.2040 0.1720 0.1410
post_intro_gt30 0.4220 0.4320 0.4560 0.4520 0.4640 0.4500 0.4450 0.4240 0.4140 0.3680 0.3930
  A4: ref 0.330..0.394, 95% PI [0.289, 0.420] (LOO false alarms 0/8) | 116: 0.383 t=+1.0 tr=+0.8 | 117: 0.460 t=+3.8 tr=+2.8 FAIL-break | 118: 0.466 t=+4.0 tr=+2.7 FAIL-break | joint t=+4.6 p=0.0024
  A4: raw t-PI failures ['117 (break)', '118 (break)']; Holm survivors (decide the verdict): cells ['117 (break)', '118 (break)'], joint ['all joint (t=+4.6)']
[FAIL] A4: Original-cosponsor share of each candidate within train spread | metric=4.03 (threshold outside 95% t prediction interval of ref 108-115 (two-sided; n=8); FAIL only if a test survives Holm over 3 cells + 1 joint tests) [fails for: S1a, S1b, S2, S3, S1a+strea

### A5. Concept stability: majority vs minority sponsor advancement, overall and by regime
**Why it matters:** majority-party sponsorship is one of the strongest Task-2 signals. If its effect
(odds ratio, OR) changes between train and the held-out Congress, the learned relationship is wrong
there. The effect may depend on **which party controls the chamber**. If candidates match train
Congresses with the same controlling party, the remedy is a regime feature or interaction, not a
different split. **How to read it:** A5 compares the pooled-chamber OR per Congress with train
Congresses. A5b compares each candidate chamber-Congress with the train chamber-Congresses that have
the same chamber and controlling party (random-effects z: sampling variance plus the between-Congress
variance seen in train). A5c tests in train whether D-control and R-control ORs differ. Majority from
`majority_party()`; sponsors are assigned to their caucus.

In [104]:
# A5: majority-sponsor OR per Congress (pooled) and per chamber x controlling party; regime-matched z-scores.
def a_cnt(df):
    """2x2 counts (majority pos, majority neg, minority pos, minority neg) + 0.5 continuity."""
    m, n = df[df.sponsor_status == "majority"].label, df[df.sponsor_status == "minority"].label
    return np.array([m.sum(), (1 - m).sum(), n.sum(), (1 - n).sum()], float) + 0.5


def a_lor(k):
    return float(np.log(k[0] * k[3] / (k[1] * k[2]))), float((1 / k).sum())


r5 = adv_u.groupby(["congress", "sponsor_status"])["label"].mean().unstack().add_prefix("rate_")
r5["OR"] = [np.exp(a_lor(a_cnt(g))[0]) for _, g in adv_u.groupby("congress")]
print(r5.round(3).T.to_string())
t5, i5 = a_rel(r5.OR, log=True)
a_record("A5", "Majority-sponsor odds ratio (both chambers pooled) of each candidate within train spread", [("", t5, i5)],
         "Pooled over chambers and regimes; see A5b for the regime-matched comparison.", quantity="majority_or", fmt=".2f")

cells = pd.DataFrame([dict(congress=c, ch=ch, majority=g.majority.iloc[0], k=a_cnt(g)) for (c, ch), g in
                      adv_u.groupby(["congress", "ch"])])
cells[["l", "v"]] = [a_lor(k) for k in cells.k]
cells["regime"] = cells.ch + "-" + cells.majority
ctr = cells[cells.congress.isin(A_TRAIN)]


def a_pool(df):
    return a_lor(np.sum(np.stack(df.k.values), axis=0) - 0.5 * (len(df) - 1))


def a_tau2(key):
    """Between-Congress excess variance of train cell log ORs around their `key` group's pooled log OR."""
    res = [(r.l - a_pool(ctr[ctr[key] == getattr(r, key)])[0]) ** 2 - r.v for r in ctr.itertuples()]
    return max(0.0, float(np.mean(res)))


tau_reg, tau_ch = a_tau2("regime"), a_tau2("ch")
rows = []
for r in cells.itertuples():
    same = ctr[(ctr.regime == r.regime) & (ctr.congress != r.congress)]
    chm = ctr[(ctr.ch == r.ch) & (ctr.congress != r.congress)]
    lr, vr = a_pool(same) if len(same) else (np.nan, np.nan)
    lc, vc = a_pool(chm)
    rows.append(dict(congress=r.congress, ch=r.ch, regime=r.regime, OR=np.exp(r.l), n_ref=len(same),
                     OR_regime_ref=np.exp(lr), z_regime=(r.l - lr) / np.sqrt(r.v + vr + tau_reg),
                     OR_chamber_ref=np.exp(lc), z_chamber=(r.l - lc) / np.sqrt(r.v + vc + tau_ch)))
a5b = pd.DataFrame(rows)
print(f"between-Congress tau^2 of log OR: within regime {tau_reg:.3f}, within chamber (regime ignored) {tau_ch:.3f}; "
      "candidate cells below (train cells are in the chart)")
print(a5b[a5b.congress.isin(A_CANDS)].round(2).to_string(index=False))
A_ZC5 = float(stats.norm.ppf(1 - A_ALPHA / 2))   # random-effects z is approximately normal
a5b["p"] = 2 * stats.norm.sf(a5b.z_regime.abs())
a5b["fail"] = a5b.z_regime.abs() > A_ZC5
a5b["holm"] = False
_ic = a5b.congress.isin(A_CANDS)
a5b.loc[_ic, "holm"] = a_holm(a5b.loc[_ic, "p"].values) & a5b.loc[_ic, "fail"].values   # Holm over the 6 candidate cells
cz = a5b[_ic]
rz = a5b[a5b.congress.isin(A_TRAIN)]
for r in a5b.itertuples():   # one part per chamber, so per-Congress failure counts line up with the other checks
    if r.congress in A_CANDS:
        A_ZTAB.append(dict(check="A5b", part=f"A5b {r.ch}", congress=int(r.congress), t=float(r.z_regime), p=float(r.p),
                           fail=bool(r.fail), trend_z=np.nan, trend_p=np.nan, trend_fail=False, kind="", level=False,
                           fa_rate=float(rz.fail.mean()), n=len(rz), holm=bool(r.holm), jholm=False,
                           sig=bool(r.holm)))
    elif r.congress in A_TRAIN:
        A_REFTAB.append(dict(check="A5b", part=f"A5b {r.ch}", congress=int(r.congress), fail=bool(r.fail),
                             trend_fail=False, kind="", level=False))
any5 = rz.groupby("congress").fail.any()
A_FAM.append(dict(check="A5b", parts=2, n_ref=len(any5), fwer_loo=float(any5.mean()),
                  fwer_nominal=float(1 - (1 - A_ALPHA) ** 2), fwer_ids=[int(c) for c in any5.index[any5]]))
fails5 = [f"{r.regime} {r.congress}" for r in cz.itertuples() if r.fail]
surv5 = [f"{r.regime} {r.congress}" for r in cz.itertuples() if r.holm]
record_check("A", "A5b", "Each candidate chamber-Congress's majority-sponsor OR matches train Congresses of the same chamber x controlling party",
             not surv5, metric=float(cz.z_regime.abs().max()),
             threshold=f"|z_regime| > {A_ZC5:.2f} (random-effects, 95% two-sided) AND survives Holm over {len(cz)} candidate cells",
             detail=(f"VERDICT RULE: FAIL iff a candidate cell survives Holm. Holm survivors: {surv5 or 'none'}; raw failures "
                     f"{fails5 or 'none'}. z_regime per candidate cell {dict(zip(cz.regime + ' ' + cz.congress.astype(str), cz.z_regime.round(2)))}; "
                     f"mean |z| regime-matched {cz.z_regime.abs().mean():.2f} vs chamber-only {cz.z_chamber.abs().mean():.2f}. "
                     f"Nested-LOO false alarms among {len(rz)} train cells: {int(rz.fail.sum())}; train Congresses with ANY "
                     f"failing cell {int(any5.sum())}/{len(any5)}. If regime matching passes while "
                     f"A5 (pooled) fails, the shift is regime-conditional and already present in train: add a chamber x "
                     f"controlling-party feature / majority x party interaction rather than calling it concept drift. "
                     f"House-D has only {len(ctr[ctr.regime == 'House-D'])} train cells, so that reference is thin. No joint "
                     f"candidate test here: each cell has its own regime-matched reference. {A_S4_NOTE}; S5 not judged."),
             quantity="majority_or_regime", by_split=a_split_from(cz.groupby("congress").holm.any().to_dict()),
             kind="drift", na_reason=A_NA)

# A5c: in train, does the OR differ by controlling party within a chamber? (cell means, random-effects SE)
rows = []
for ch, g in ctr.groupby("ch"):
    d, r = g[g.majority == "D"], g[g.majority == "R"]
    se = np.sqrt(((d.v + tau_reg).sum() / len(d) ** 2) + ((r.v + tau_reg).sum() / len(r) ** 2))
    rows.append(dict(ch=ch, n_D=len(d), OR_Dcontrol=np.exp(d.l.mean()), n_R=len(r), OR_Rcontrol=np.exp(r.l.mean()),
                     diff_logOR=d.l.mean() - r.l.mean(), z=(d.l.mean() - r.l.mean()) / se))
a5c = pd.DataFrame(rows)
print(a5c.round(2).to_string(index=False))
record_check("A", "A5c", "Majority-sponsor effect differs by controlling party within train (regime interaction needed?)", None,
             metric=float(a5c.diff_logOR.abs().max()), threshold=f"|z| > {A_ZC5:.2f} => model needs a regime interaction",
             detail=(f"log OR difference D-control minus R-control per chamber {dict(zip(a5c.ch, a5c.diff_logOR.round(2)))}, "
                     f"z {dict(zip(a5c.ch, a5c.z.round(2)))}. Every regime of the candidates occurs in train "
                     f"({sorted(set(cz.regime))} vs train {sorted(set(ctr.regime))}), so a by-Congress split can learn it "
                     f"if the regime is a feature."), quantity="majority_or_regime_effect")
show_d3(r"""
const {width, height, title, subtitle, split, colors} = opts;
const m = {top: 48, right: 20, bottom: 48, left: 56};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title, subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const xs = d3.scaleLinear([d3.min(data, d => d.congress) - 0.6, d3.max(data, d => d.congress) + 0.6], [0, W]);
const ys = d3.scaleLog([d3.min(data, d => d.lo) * 0.9, d3.max(data, d => d.hi) * 1.1], [H, 0]);
g.append('g').call(d3.axisLeft(ys).ticks(6, '~g').tickSize(-W)).call(a => a.selectAll('line').attr('stroke', viz.grid))
  .call(a => a.select('.domain').remove()).call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(12).tickFormat(d3.format('d'))).call(viz.axis);
g.append('line').attr('x1', xs(split)).attr('x2', xs(split)).attr('y1', 0).attr('y2', H).attr('stroke', viz.ink2).attr('stroke-dasharray', '4,3');
g.append('text').attr('x', xs(split) + 4).attr('y', 10).attr('fill', viz.ink2).style('font', '11px sans-serif').text('candidates');
g.append('text').attr('x', -m.left + 4).attr('y', -8).attr('fill', viz.ink2).style('font', '11px sans-serif').text('odds ratio (log scale)');
const off = d => d.ch === 'House' ? -0.15 : 0.15;
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('line').attr('x1', d => xs(d.congress + off(d))).attr('x2', d => xs(d.congress + off(d)))
  .attr('y1', d => ys(d.lo)).attr('y2', d => ys(d.hi)).attr('stroke', d => colors[d.majority]).attr('stroke-width', 1.5).attr('opacity', 0.6);
g.selectAll(null).data(data).join('path')
  .attr('d', d => d3.symbol(d.ch === 'House' ? d3.symbolCircle : d3.symbolSquare, 64)())
  .attr('transform', d => `translate(${xs(d.congress + off(d))},${ys(d.OR)})`)
  .attr('fill', d => colors[d.majority]).attr('stroke', '#fff').attr('stroke-width', 1)
  .on('mousemove', (e, d) => tip.show(`<b>${d.ch} ${d.congress}</b> (${d.majority} control)<br>OR ${d.OR.toFixed(2)} [${d.lo.toFixed(2)}, ${d.hi.toFixed(2)}]`, e))
  .on('mouseleave', () => tip.hide());
viz.legend(svg, [['D controls chamber', colors.D], ['R controls chamber', colors.R]], m.left, height - 10);
svg.append('text').attr('x', width - 20).attr('y', height - 1).attr('text-anchor', 'end').attr('fill', viz.ink2)
  .style('font', '11px sans-serif').text('circle = House, square = Senate; bars = 95% CI');
""", cells.assign(OR=np.exp(cells.l), lo=np.exp(cells.l - 1.96 * np.sqrt(cells.v)), hi=np.exp(cells.l + 1.96 * np.sqrt(cells.v)))
    [["congress", "ch", "majority", "OR", "lo", "hi"]], height=340, width=720,
    opts=dict(title="Majority-sponsor odds ratio of being reported, per chamber and Congress",
              subtitle="Colour = party controlling the chamber (majority_party()); compare candidates with same-colour train points",
              split=A_TRAIN[-1] + 0.5, colors=PARTY_COLORS), name="A_majority_or_regime")

congress          108    109    110    111    112    113    114    115    116    117    118
sponsor_status                                                                             
rate_majority  0.1200 0.0810 0.0950 0.0550 0.0600 0.0840 0.1050 0.1260 0.0500 0.0320 0.0590
rate_minority  0.0380 0.0230 0.0470 0.0290 0.0180 0.0250 0.0360 0.0440 0.0290 0.0220 0.0250
OR             3.4690 3.7180 2.1130 1.9560 3.5110 3.6010 3.1180 3.1240 1.7240 1.4350 2.4750
  A5: ref 1.96..3.72, 95% PI [1.61, 5.60] (LOO false alarms 0/8) | 116: 1.72 t=-2.1 tr=-1.7 | 117: 1.44 t=-2.8 tr=-2.1 FAIL-undetermined | 118: 2.48 t=-0.7 tr=-0.6 | joint t=-2.9 p=0.022
  A5: raw t-PI failures ['117 (undetermined)']; Holm survivors (decide the verdict): cells none, joint none
[PASS] A5: Majority-sponsor odds ratio (both chambers pooled) of each candidate within train spread | metric=-2.791 (threshold outside 95% t prediction interval of ref 108-115 (two-sided, log scale; n=8); FAIL only if a test survives Holm over 3

### A6. Concept stability: partisanship of new cosponsors (P(y | x) for Task 1)
**Why it matters:** a link-prediction model will learn "same party as the sponsor" as a strong prior.
If the share of new cosponsors from the sponsor's party drifts (polarisation), the learned prior is off
in the held-out Congress. **How to read it:** among cosponsors joining in days 1-30 of law-capable bills
with a D/R sponsor, the share from the same party and from the other major party.

In [105]:
# A6: same-party and cross-party share of new (lag 1..30 days) cosponsors per Congress; t PI vs train.
nc = cl[cl.lag_days.between(1, WINDOW_DAYS) & cl.sponsor_party.isin(["D", "R"])]
nc = nc.assign(same=nc.party == nc.sponsor_party, cross=nc.party.isin(["D", "R"]) & (nc.party != nc.sponsor_party))
a6 = nc.groupby("congress")[["same", "cross"]].mean()
a6["n_new_cosp"] = nc.groupby("congress").size()
tr6 = nc[nc.congress.isin(A_TRAIN)].same.mean()
print(a6.round(3).T.to_string())
t6, i6 = a_rel(a6.same, band=(tr6 - A_PP_BAND, tr6 + A_PP_BAND))
a_record("A6", "Same-party share of new cosponsors of each candidate within train spread", [("", t6, i6)],
         "Party homophily is the main prior a cosponsor model learns; a swing beyond train spread changes which "
         "candidates rank on top (recalibrate the party prior on recent Congresses).", quantity="same_party_share",
         band_label="+/-5 pp of pooled train")

congress           108         109         110         111         112         113         114         115         116         117         118
same            0.7120      0.7280      0.7880      0.8250      0.8300      0.8080      0.8160      0.7900      0.8320      0.8600      0.8320
cross           0.2820      0.2650      0.2070      0.1710      0.1670      0.1900      0.1810      0.2080      0.1650      0.1380      0.1660
n_new_cosp 22,400.0000 24,633.0000 30,804.0000 25,863.0000 22,928.0000 26,534.0000 25,492.0000 25,272.0000 36,895.0000 30,200.0000 23,736.0000
  A6: ref 0.712..0.830, 95% PI [0.676, 0.898] (LOO false alarms 0/8) | 116: 0.832 t=+1.0 tr=-0.2 | 117: 0.860 t=+1.6 tr=+0.1 | 118: 0.832 t=+0.9 tr=-0.7 | joint t=+1.8 p=0.11
  A6: raw t-PI failures none; Holm survivors (decide the verdict): cells none, joint none
[PASS] A6: Same-party share of new cosponsors of each candidate within train spread | metric=1.557 (threshold outside 95% t prediction interval of ref 108-115 (two

### A7-A8. Marginal covariate drift (PSI) and bill volume
**Why it matters:** even with stable labels, a held-out Congress with a different mix of policy areas,
bill types, chambers, sponsor status, original-cosponsor counts or introduction timing tests the model
on a differently weighted population. **How to read it:** PSI of each candidate vs pooled 108-115
bills; a train Congress's value is its PSI vs the other 7 train Congresses. The t prediction-interval rule
decides, on the **log scale** (PSI is bounded at 0 and right-skewed, so a raw-scale interval has a
negative lower bound and over-calls large values; section B judges PSI the same way) (the
usual 0.1 / 0.25 PSI bands are shown in the chart subtitle only). The chart shows PSI divided by the
largest train LOO PSI, so bars above 1 mean "more drift than any train Congress". `policy_area_harm`
folds categories absent from some Congress (taxonomy changes) into one bucket (`PA_UNSTABLE`).

In [106]:
# A7: PSI per feature (candidates vs train pool; train Congresses LOO), t PI vs train; A8: bill volume.
A7_FEATS = {"policy_area_harm": "cat", "bill_type": "cat", "ch": "cat", "sponsor_status": "cat",
            "n_original": "num", "day_of_congress": "num"}


def a_psi(feat, ref, cur):
    return psi(ref[feat], cur[feat]) if A7_FEATS[feat] == "num" else psi_categorical(ref[feat], cur[feat])


trb = fb[fb.congress.isin(A_TRAIN)]
p7, bars = [], []
for f in A7_FEATS:
    s = pd.Series({c: a_psi(f, trb[trb.congress != c], trb[trb.congress == c]) for c in A_TRAIN}
                  | {c: a_psi(f, trb, fb[fb.congress == c]) for c in A_CANDS})
    p7.append((f,) + a_rel(s, side="upper", log=True))   # PSI >= 0 and right-skewed: judge on the log scale (as B3)
    bars += [dict(feature=f, congress=str(c), ratio=s[c] / s[A_TRAIN].max()) for c in A_CANDS]
raw = {c: round(psi_categorical(trb.policy_area.fillna("(missing)"), fb[fb.congress == c].policy_area.fillna("(missing)")), 3)
       for c in A_CANDS}
print(f"raw (unharmonised) policy_area PSI vs train, inflated by taxonomy changes: {raw}; folded: {PA_UNSTABLE}")
a_record("A7", "Marginal covariate drift (PSI of 6 bill features) of each candidate within train LOO spread", p7,
         "One part per feature, log-scale PI. FAIL on a feature means the candidate's mix differs more than any typical train "
         "Congress differs from the rest: reweight training bills toward recent Congresses, or use a sliding window.",
         quantity="covariate_psi", level=False)
bar_chart(pd.DataFrame(bars), "feature", "ratio", series="congress",
          title="Covariate drift of each candidate relative to train noise (PSI / max train LOO PSI)",
          subtitle="> 1 = the candidate drifts more than any train Congress does from the other train Congresses",
          yfmt=".2f", ylabel="PSI ratio", name="A_psi_features")
vol = fb.groupby("congress").size()
t8, i8 = a_rel(vol, log=True, band=(vol[A_TRAIN].mean() * A_RATIO_BAND[0], vol[A_TRAIN].mean() * A_RATIO_BAND[1]))
a_record("A8", "Law-capable bill volume of each candidate within train spread", [("", t8, i8)],
         "More bills per Congress means more bill nodes per legislator (higher degree) and lower per-bill base "
         "rates; normalise degree/count features per Congress. Section B's degree checks cover the graph side.",
         quantity="bill_volume", fmt=".0f", band_label="0.8-1.25x mean train Congress")

raw (unharmonised) policy_area PSI vs train, inflated by taxonomy changes: {116: 0.436, 117: 0.357, 118: 0.382}; folded: ['(missing)', 'Commemorations', 'Private Legislation', 'Social Sciences and History']
  A7 policy_area_harm: ref 0.035..0.272, 95% PI [0.030, 0.661] (LOO false alarms 0/8) | 116: 0.244 t=+0.7 tr=-0.1 | 117: 0.238 t=+0.6 tr=-0.3 | 118: 0.288 t=+0.9 tr=-0.2 | joint t=+1.1 p=0.14
  A7 bill_type: ref 0.000..0.007, 95% PI [0.000, 0.045] (LOO false alarms 0/8) | 116: 0.002 t=+0.2 tr=+0.4 | 117: 0.002 t=+0.3 tr=+0.5 | 118: 0.002 t=+0.4 tr=+0.5 | joint t=+0.5 p=0.33
  A7 ch: ref 0.000..0.006, 95% PI [0.000, 0.127] (LOO false alarms 0/8) | 116: 0.000 t=-0.5 tr=-0.6 | 117: 0.000 t=-4.5 tr=-3.5 | 118: 0.000 t=-0.1 tr=-0.4 | joint t=-2.7 p=0.98
  A7 sponsor_status: ref 0.003..0.096, 95% PI [0.001, 0.163] (LOO false alarms 1/8) | 116: 0.000 t=-2.3 tr=-1.8 | 117: 0.000 t=-2.3 tr=-1.7 | 118: 0.002 t=-1.3 tr=-0.9 | joint t=-3.1 p=0.99
  A7 n_original: ref 0.003..0.072, 95% PI [0.002

  A8: ref 8621..11414, 95% PI [8008, 13213] (LOO false alarms 1/8) | 116: 14340 t=+3.1 tr=+2.1 FAIL-undetermined | 117: 15242 t=+3.7 tr=+2.3 FAIL-undetermined | 118: 16565 t=+4.5 tr=+2.6 FAIL-break | joint t=+5.9 p=0.00058
  A8: raw t-PI failures ['116 (undetermined)', '117 (undetermined)', '118 (break)']; Holm survivors (decide the verdict): cells ['116 (undetermined)', '117 (undetermined)', '118 (break)'], joint ['all joint (t=+5.9)']
[FAIL] A8: Law-capable bill volume of each candidate within train spread | metric=4.5 (threshold outside 95% t prediction interval of ref 108-115 (two-sided, log scale; n=8); FAIL only if a test survives Holm over 3 cells + 1 joint tests) [fails for: S1a, S1b, S2, S3, S1a+stream, S1b+stream, S3+stream, S1pool, S4]


### A9-A10. Congress x Congress drift matrix (policy-area mix)
**Why it matters:** the heatmap shows which Congresses look alike. If distance grows with the time
gap, drift is gradual: the most recent Congresses are the best training data and an expanding window
dilutes them. **How to read it:** JSD (base 2, 0 = identical) of the harmonised policy-area
distribution. A9 compares each candidate's mean JSD to the train Congresses with each train Congress's
mean JSD to the other train Congresses (same rule as PSI, so A7 and A9 agree in method).

In [107]:
# A9: pairwise JSD of policy-area mix; candidate-to-train vs within-train (LOO) mean distances. A10: gradualness.
pa = fb.groupby(["congress", "policy_area_harm"]).size().unstack(fill_value=0)
J = pd.DataFrame([[jsd(pa.loc[i], pa.loc[j]) for j in pa.index] for i in pa.index], index=pa.index, columns=pa.index)
hm = J.stack().rename("jsd").reset_index(level=1).rename(columns={"congress": "vs"}).reset_index()
hm.columns = ["Congress", "vs", "jsd"]
heatmap(hm, "vs", "Congress", "jsd", title="Policy-area mix divergence between Congresses (JSD)",
        subtitle="Law-capable bills, harmonised policy areas; 0 = identical mix. Darker = more different", fmt=".3f",
        height=420, width=640, name="A_jsd_heatmap")
mj = pd.Series({c: J.loc[c, [t for t in A_TRAIN if t != c]].mean() for c in A_TRAIN}
               | {c: J.loc[c, A_TRAIN].mean() for c in A_CANDS})
t9, i9 = a_rel(mj, side="upper")
gap = np.array([abs(i - j) for i in J.index for j in J.columns])
rho = stats.spearmanr(gap[gap > 0], J.values.ravel()[gap > 0]).statistic
nearest = {c: int(J.loc[c].drop(c).idxmin()) for c in A_CANDS}
a_record("A9", "Each candidate's mean policy-area JSD to train within the train LOO spread", [("", t9, i9)],
         f"Nearest Congress of each candidate: {nearest}.", quantity="topic_mix_jsd", fmt=".4f", level=False)
record_check("A", "A10", "Drift is gradual in time (Spearman of Congress gap vs policy-area JSD)", None,
             metric=float(rho), threshold="rho > 0.5 => gradual drift",
             detail=(f"rho={rho:.2f}. Strongly positive rho means distant Congresses differ most, so recent Congresses "
                     f"are better training data (favours sliding window S2 / rolling origin S3)."), quantity="drift_gradual")

  A9: ref 0.0311..0.0661, 95% PI [0.0244, 0.0697] (LOO false alarms 1/8) | 116: 0.0492 t=+0.2 tr=-0.1 | 117: 0.0506 t=+0.3 tr=-0.1 | 118: 0.0573 t=+0.9 tr=+0.3 | joint t=+0.7 p=0.25
  A9: raw t-PI failures none; Holm survivors (decide the verdict): cells none, joint none
[PASS] A9: Each candidate's mean policy-area JSD to train within the train LOO spread | metric=0.8612 (threshold outside 95% t prediction interval of ref 108-115 (upper tail; n=8); FAIL only if a test survives Holm over 3 cells + 1 joint tests)
[INFO] A10: Drift is gradual in time (Spearman of Congress gap vs policy-area JSD) | metric=0.6822 (threshold rho > 0.5 => gradual drift)


### A11. Joint covariate shift: can a classifier tell a candidate's bills from train bills?
**Why it matters:** each marginal PSI can look fine while the joint distribution shifts (e.g. the same
policy areas, but sponsored by a different party). A **domain classifier** (logistic regression on the
bill features: chamber, type, sponsor caucus and majority status, harmonised policy area,
introduction-day bin, log original cosponsors) is trained to separate train-pool bills from one
Congress's bills; its 2-fold cross-validated AUROC is 0.5 when the two are indistinguishable. Policy area
is **not** known at t0 for most fresh bills (`PA_MISS_T0`), so a second part drops it (the t0-safe
feature set). **How to read it:** candidates are compared with train Congresses classified against the
other 7 (LOO). The top coefficients say which features drive the separation.

In [108]:
# A11: domain-classifier AUROC (pool vs one Congress), with and without policy area; candidates vs train LOO.
_dom_cache = {}


def a_dom_rows(ip, ic, cols=None, seed=0, ratio=2):
    """2-fold CV AUROC (and full-data coefficients) of a balanced logistic classifier: A_X rows ip (label 0,
    subsampled to ratio x len(ic)) vs rows ic (label 1). cols: boolean column mask of A_X (None = all)."""
    rng = np.random.default_rng(seed)
    ip = rng.choice(ip, min(len(ip), ratio * len(ic)), replace=False)
    idx = np.concatenate([ip, ic])
    X = A_X[idx] if cols is None else A_X[idx][:, cols]
    y = np.r_[np.zeros(len(ip)), np.ones(len(ic))]
    w = np.where(y == 1, 0.5 / y.mean(), 0.5 / (1 - y.mean()))
    fold = rng.integers(0, 2, len(idx))
    s = np.empty(len(idx))
    for k in (0, 1):
        b = a_logit(X[fold != k], y[fold != k], w[fold != k])
        s[fold == k] = X[fold == k] @ b
    return a_auc(y, s), a_logit(X, y, w)


def a_domain(pool, c, cols=None, max_day=None):
    """Cached domain AUROC: bills of `pool` Congresses vs Congress c (optionally only bills introduced by day
    max_day of their Congress)."""
    key = (tuple(sorted(pool)), c, None if cols is None else tuple(cols), max_day)
    if key not in _dom_cache:
        ok = np.ones(len(fb_all), bool) if max_day is None else (fb_all.day_of_congress <= max_day).values
        cg = fb_all.congress.values
        _dom_cache[key] = a_dom_rows(np.flatnonzero(np.isin(cg, pool) & ok), np.flatnonzero((cg == c) & ok), cols, seed=c)
    return _dom_cache[key]


dom = pd.DataFrame({tag: {c: a_domain([t for t in A_TRAIN if t != c] if c in A_TRAIN else A_TRAIN, c, cols)[0]
                          for c in CONGRESSES} for tag, cols in [("all", None), ("noPA", A_NOPA)]})
print("domain AUROC (train: vs other train Congresses; candidates: vs 108-115):")
print(dom.T.round(3).to_string())
_tops = []
for c in A_CANDS:
    b = pd.Series(a_domain(A_TRAIN, c)[1][1:], index=A_XNAMES[1:])
    top = b.reindex(b.abs().sort_values(ascending=False).index[:2])
    _tops.append(f"{c}: " + ", ".join(f"{k} {v:+.2f}" for k, v in top.items()))
print("top separating features (log-odds of being the candidate): " + " | ".join(_tops))
a_record("A11", "Joint covariate shift (domain-classifier AUROC vs train pool) of each candidate within train LOO spread",
         [("all",) + a_rel(dom["all"], side="upper"), ("noPA",) + a_rel(dom["noPA"], side="upper")],
         f"Proxy for multivariate P(x) shift on bill features (graph features not included). Part 'all' includes policy "
         f"area, which is missing for {PA_MISS_T0:.0%} of bills <= 7 days old (PA_MISS_T0, n={PA_MISS_T0_N}; small-sample "
         f"proxy), so only 'noPA' is the t0-safe view. FAIL: the candidate occupies regions of feature space that train "
         f"covers thinly (the deployment Congress will too): prefer recency-weighted training (A14) or importance "
         f"weighting by the domain classifier's odds, and recalibrate on val. The features include sponsor caucus and "
         f"majority status, whose mix moves with party control (A15b), so part of this separation is the regime "
         f"itself; regime is better used as a feature than as a weighting scheme (section D's D26 backtests regime weighting).",
         quantity="joint_covariate_shift", level=False)

domain AUROC (train: vs other train Congresses; candidates: vs 108-115):
        108    109    110    111    112    113    114    115    116    117    118
all  0.6380 0.6580 0.6150 0.6690 0.6050 0.5860 0.6330 0.6510 0.6290 0.6300 0.6460
noPA 0.6040 0.6260 0.6030 0.6680 0.5930 0.5420 0.5960 0.6050 0.5870 0.5840 0.5910
top separating features (log-odds of being the candidate): 116: policy_area_harm=Foreign Trade and International Finance -2.19, policy_area_harm=(unstable/missing) -1.64 | 117: sponsor_caucus=(missing) +3.10, policy_area_harm=Foreign Trade and International Finance -1.89 | 118: policy_area_harm=(unstable/missing) -2.71, policy_area_harm=Foreign Trade and International Finance -1.78
  A11 all: ref 0.586..0.669, 95% PI [0.575, 0.689] (LOO false alarms 0/8) | 116: 0.629 t=-0.1 tr=+0.2 | 117: 0.630 t=-0.1 tr=+0.3 | 118: 0.646 t=+0.5 tr=+0.6 | joint t=+0.2 p=0.44
  A11 noPA: ref 0.542..0.668, 95% PI [0.534, 0.676] (LOO false alarms 1/8) | 116: 0.587 t=-0.5 tr=+0.1 | 117: 0.584 

### A12-A13. Calibration and concept stability of a fitted Task-2 model
**Why it matters:** marginal ORs are a partial view. A logistic model using the same bill features as A11
is fitted on earlier Congresses and scored on a later one. The main comparison is **forward-chained**
(one step ahead, like split S3): Congress c is scored by a model fitted on 108..c-1, for c = 110-115
(train reference) and 116-118 (candidates). A leave-one-out variant (train Congress scored by a model
fitted on the other 7, which is interpolation and so easier) is shown for context. **How to read it:**
`OE` = observed / mean predicted rate (calibration-in-the-large; this mostly tracks A1's base rate).
`slope` = calibration slope: 1 means relative risks transfer, < 1 means the model is over-confident (the
feature effects weakened). `coef_shift` = majority-status log-odds effect refitted on Congress c alone,
minus the effect in the fitted model. A12 checks slope and coefficient shift; A13 checks AUROC with and
without policy area (`auroc_noPA` = t0-safe features only, since policy area is mostly missing at t0).

In [109]:
# A12/A13: logistic Task-2 model, forward-chained (fit 108..c-1) and LOO; O/E, calibration slope, AUROC, coef shift.
ia_rows = adv_u.bill_id.map(A_ROW).values
Xa, ya, ca = A_X[ia_rows], adv_u.label.values.astype(float), adv_u.congress.values
j_min = A_XNAMES.index("sponsor_status=minority")
_bc = {c: a_logit(Xa[ca == c], ya[ca == c]) for c in CONGRESSES}   # per-Congress refit (for coef_shift)


def a_transfer(fit_on, c, cols=None):
    X = Xa if cols is None else Xa[:, cols]
    b = a_logit(X[np.isin(ca, fit_on)], ya[np.isin(ca, fit_on)])
    m = ca == c
    eta = X[m] @ b
    if cols is not None:
        return dict(auroc=a_auc(ya[m], eta))
    cal = a_logit(np.column_stack([np.ones(m.sum()), eta]), ya[m], l2=0.0)
    return dict(OE=ya[m].mean() / (1 / (1 + np.exp(-eta))).mean(), slope=cal[1], auroc=a_auc(ya[m], eta),
                coef_shift=-_bc[c][j_min] + b[j_min])


A_FWD = [c for c in A_TRAIN if c >= CONGRESSES[0] + 2]   # need >= 2 earlier Congresses to fit on
fwd = pd.DataFrame({c: a_transfer(list(range(CONGRESSES[0], c)), c) for c in A_FWD + A_CANDS}).T
fwd["auroc_noPA"] = pd.Series({c: a_transfer(list(range(CONGRESSES[0], c)), c, A_NOPA)["auroc"] for c in A_FWD + A_CANDS})
loo = pd.DataFrame({c: a_transfer([t for t in A_TRAIN if t != c] if c in A_TRAIN else A_TRAIN, c) for c in CONGRESSES}).T
print(fwd.join(loo[["slope", "auroc"]].add_suffix("_loo"), how="outer").dropna(subset=["auroc"]).round(3).to_string())
a_record("A12", "Fitted Task-2 model transfers forward: calibration slope and majority-coefficient shift within train spread",
         [("slope",) + a_rel(fwd.slope, train=A_FWD), ("coef_shift",) + a_rel(fwd.coef_shift, train=A_FWD)],
         f"Forward-chained reference {A_FWD[0]}-{A_FWD[-1]} (fit 108..c-1; n={len(A_FWD)}, so the PI is wide). Model "
         f"includes policy area (not known at t0 for most bills). O/E {a_d(fwd.loc[A_CANDS, 'OE'], 2)} vs "
         f"train {fwd.loc[A_FWD, 'OE'].min():.2f}-{fwd.loc[A_FWD, 'OE'].max():.2f}: an O/E shift alone is fixed by "
         f"recalibrating the intercept on val; a slope below train spread means feature effects weakened (concept "
         f"drift): train on recent Congresses, add regime interactions (A5b/A5c), and recalibrate slope on val. LOO "
         f"slopes (interpolation) for context: {a_d(loo.slope, 2)}.", quantity="model_calibration_t2")
a_record("A13", "Fitted Task-2 model AUROC (forward-chained) on each candidate not below the train spread",
         [("auroc",) + a_rel(fwd.auroc, side="lower", train=A_FWD),
          ("auroc_noPA",) + a_rel(fwd.auroc_noPA, side="lower", train=A_FWD)],
         f"Ranking quality of a simple feature model one Congress ahead (baseline backtests are in section D). Part "
         f"'auroc' uses policy area (optimistic at t0), 'auroc_noPA' only t0-safe features. LOO AUROC for context: "
         f"{a_d(loo.auroc, 3)}. FAIL means the held-out Congress is harder to rank than past one-step-ahead "
         f"Congresses were: expect lower test scores than validation-era scores, and do not read a val-to-test drop as "
         f"overfitting.", quantity="model_auroc_transfer_t2")

        OE  slope  auroc  coef_shift  auroc_noPA  slope_loo  auroc_loo
110 1.4610 0.9450 0.7450     -0.7870      0.6200     1.0500     0.7520
111 0.7000 0.9100 0.7490     -0.7360      0.6310     1.0670     0.7560
112 0.7880 0.9660 0.7810      0.2360      0.6590     1.0510     0.7840
113 0.8840 0.9940 0.7560      0.0810      0.6810     1.0700     0.7610
114 1.0940 0.7230 0.7120     -0.4670      0.6630     0.7450     0.7180
115 1.2750 0.8290 0.7270     -0.4500      0.6520     0.8290     0.7270
116 0.6160 0.6750 0.6660     -0.2890      0.5620     0.6750     0.6660
117 0.4680 0.7910 0.6740     -0.6950      0.5580     0.7560     0.6780
118 0.7260 0.6210 0.6640     -0.1360      0.5990     0.5910     0.6610
  A12 slope: ref 0.723..0.994, 95% PI [0.613, 1.176] (LOO false alarms 1/6) | 116: 0.675 t=-2.0 tr=-0.9 | 117: 0.791 t=-0.9 tr=+0.3 | 118: 0.621 t=-2.5 tr=-0.6 | joint t=-2.8 p=0.039
  A12 coef_shift: ref -0.787..0.236, 95% PI [-1.527, 0.819] (LOO false alarms 0/6) | 116: -0.289 t=+0.1 tr=

### A18. Concept stability for Task 1: do pair-feature effects on cosponsoring stay the same?
**Why it matters:** A6 looks at one marginal share; A12/A13 test P(y | x) only for Task 2. A link
predictor for Task 1 learns how much each (bill, legislator) pair feature raises the chance of
cosponsoring: past cosponsoring of this sponsor's bills, reciprocity (the sponsor cosponsored the
legislator's bills), the legislator's general activity, same caucus, same state. If those effects moved,
model selection on the val Congress can still work (ranking survives) while calibrated scores and the
learned weights drift. **How to read it:** for each Congress, a logistic model on sampled (bill,
eligible legislator) pairs at t0 = 7 (point-in-time pool incl. delegates; history counted in the 730
days before introduction + 7 d). A18 judges each coefficient of 116-118 against 109-115 (108 is dropped
from the reference because its look-back window starts before the data). A18b scores each Congress
with a model fitted on all earlier Congresses (forward-chained, like A13): AUROC over pairs and the
calibration slope.

In [110]:
# A18: per-Congress logistic model on (bill, eligible legislator) pairs at t0=7 with 5 time-aware pair features.
A18_T0, A18_LOOK, A18_NB, A18_NEG = 7, 730, 1500, 0.1   # t0, look-back days, bills sampled per Congress, negative keep rate
A18_REF = [c for c in A_TRAIN if c > CONGRESSES[0]]    # 109-115 (108's look-back is truncated at the data start)
A18_F = ["log1p(hist s->l)", "log1p(recip l->s)", "log1p(activity l)", "same_caucus", "same_state"]
_ids = pd.Index(pd.unique(pd.concat([cosp.bioguide_id, cosp.sponsor_bioguide_id, ROSTER_ASOF_ALL.bioguide_id]).dropna()))
_NL, _BIG = len(_ids), 100_000
_ev = cosp.dropna(subset=["sponsor_bioguide_id"])
_eday = ((_ev.sponsorship_date.values.astype("datetime64[D]") - _EPOCH).astype(np.int64))
_ls, _ss = _ids.get_indexer(_ev.bioguide_id), _ids.get_indexer(_ev.sponsor_bioguide_id)
_K_SL = np.sort((_ss.astype(np.int64) * _NL + _ls) * _BIG + _eday)   # (sponsor, cosponsor) events by day
_K_L = np.sort(_ls.astype(np.int64) * _BIG + _eday)                  # cosponsor activity by day


def a_count(sorted_keys, key, day):
    """Number of events with this key dated in [day - A18_LOOK, day)."""
    return (np.searchsorted(sorted_keys, key * _BIG + day, "left")
            - np.searchsorted(sorted_keys, key * _BIG + day - A18_LOOK, "left")).astype(float)


_wt7 = window_targets(A18_T0, point_in_time=True)
_wt7 = _wt7[_wt7.congress.isin(CONGRESSES) & ~_wt7.censored & _wt7.eligible.gt(0)].merge(
    bills[["bill_id", "sponsor_bioguide_id", "sponsor_state"]], on="bill_id").merge(fb[["bill_id", "sponsor_caucus"]], on="bill_id")
_pos = cosp[(cosp.lag_days > A18_T0) & (cosp.lag_days <= A18_T0 + WINDOW_DAYS)][["bill_id", "bioguide_id"]].assign(y=1.0)
_seen = cosp[cosp.lag_days <= A18_T0][["bill_id", "bioguide_id"]].assign(seen=True)
_rost = ROSTER_ASOF_ALL[["month", "congress", "chamber", "bioguide_id", "caucus", "state_code"]].rename(columns={"month": "pool_month"})
_rng18 = np.random.default_rng(18)
P18 = []
for c in CONGRESSES:
    b = _wt7[_wt7.congress == c]
    b = b.iloc[_rng18.choice(len(b), min(A18_NB, len(b)), replace=False)]
    pr = b.merge(_rost, on=["pool_month", "congress", "chamber"])
    pr = pr[pr.bioguide_id != pr.sponsor_bioguide_id].merge(_seen, on=["bill_id", "bioguide_id"], how="left")
    pr = pr[pr.seen.isna()].merge(_pos, on=["bill_id", "bioguide_id"], how="left")
    pr["y"] = pr.y.fillna(0.0)
    pr = pr[(pr.y == 1) | (_rng18.random(len(pr)) < A18_NEG)].copy()   # all positives, A18_NEG of negatives (weighted back)
    pr["w"] = np.where(pr.y == 1, 1.0, 1 / A18_NEG)
    day = ((pr.introduced_date.values.astype("datetime64[D]") - _EPOCH).astype(np.int64)) + A18_T0
    sc, lc = _ids.get_indexer(pr.sponsor_bioguide_id).astype(np.int64), _ids.get_indexer(pr.bioguide_id).astype(np.int64)
    pr["f0"] = np.log1p(a_count(_K_SL, sc * _NL + lc, day))
    pr["f1"] = np.log1p(a_count(_K_SL, lc * _NL + sc, day))
    pr["f2"] = np.log1p(a_count(_K_L, lc, day))
    pr["f3"] = (pr.caucus == pr.sponsor_caucus).astype(float)
    pr["f4"] = (pr.state_code == pr.sponsor_state).astype(float)
    P18.append(pr[["congress", "y", "w", "f0", "f1", "f2", "f3", "f4"]])
P18 = pd.concat(P18, ignore_index=True)
X18 = np.column_stack([np.ones(len(P18))] + [P18[f"f{k}"].values for k in range(len(A18_F))])
y18, w18, c18 = P18.y.values, P18.w.values, P18.congress.values
coef18 = pd.DataFrame({c: a_logit(X18[c18 == c], y18[c18 == c], w18[c18 == c])[1:] for c in CONGRESSES}, index=A18_F).T
print(f"pairs per Congress after negative sampling (positives all, negatives x{A18_NEG}): "
      f"{P18.groupby('congress').size().min():,}-{P18.groupby('congress').size().max():,}; positives "
      f"{int(P18.y.sum()):,}. Coefficients (log-odds per unit; per Congress fit):")
print(coef18.T.round(3).to_string())
a_record("A18", "Task-1 pair-feature effects (logistic coefficients, t0=7) of each candidate within the 109-115 spread",
         [(f,) + a_rel(coef18[f], train=A18_REF) for f in A18_F],
         f"Model: logistic on {A18_NB} sampled uncensored bills per Congress x all eligible point-in-time legislators "
         f"(ROSTER_ASOF_ALL, delegates included; sponsor and day-0..7 cosponsors removed), label = cosponsors in "
         f"({A18_T0}, {A18_T0 + WINDOW_DAYS}] days; history features counted over the {A18_LOOK} days before "
         f"introduction + {A18_T0} d (point in time). Reference 109-115 (n={len(A18_REF)}). FAIL on a coefficient = "
         f"the learned weight of that feature is stale for the candidate: refit / fine-tune on recent Congresses "
         f"(S2/S3 refit, or streaming) and recalibrate scores on val. A 1-hop/2-hop GNN learns similar signals, so "
         f"this is a proxy for the learned model's concept drift.", quantity="concept_drift_t1")

# A18b: forward-chained transfer (fit 108..c-1, score c): pair AUROC and calibration slope.
A18_FWD = [c for c in A_TRAIN if c >= CONGRESSES[0] + 2]
rows = {}
for c in A18_FWD + A_CANDS:
    fit = np.isin(c18, list(range(CONGRESSES[0], c)))
    bf = a_logit(X18[fit], y18[fit], w18[fit])
    m = c18 == c
    eta = X18[m] @ bf
    cal = a_logit(np.column_stack([np.ones(m.sum()), eta]), y18[m], w18[m], l2=0.0)
    rows[c] = dict(auroc=a_auc(y18[m], eta), slope=cal[1])
f18 = pd.DataFrame(rows).T
print("forward-chained (fit 108..c-1): " + " | ".join(f"{c}: AUROC {r.auroc:.3f}, slope {r.slope:.2f}" for c, r in f18.iterrows()))
a_record("A18b", "Task-1 pair model fitted on earlier Congresses: AUROC and calibration slope on each candidate within forward-chained spread",
         [("auroc",) + a_rel(f18.auroc, side="lower", train=A18_FWD), ("slope",) + a_rel(f18.slope, train=A18_FWD)],
         f"Forward-chained reference {A18_FWD[0]}-{A18_FWD[-1]} (n={len(A18_FWD)}). AUROC over sampled pairs (negative "
         f"sampling leaves AUROC unbiased). A pass on AUROC with a fail on A18 coefficients = ranking transfers, weights / "
         f"calibration do not: selection on val is fine, but recalibrate. A fail on AUROC = the held-out Congress is "
         f"harder to rank than past one-step-ahead Congresses.", quantity="model_auroc_transfer_t1")
line_chart(coef18.rename_axis("congress").reset_index().melt("congress", var_name="series", value_name="coef"),
           "congress", "coef", series="series", title="Task-1 pair-feature effects per Congress (logistic coefficients, t0=7)",
           subtitle=f"Fit per Congress on sampled (bill, eligible legislator) pairs; reference 109-115, candidates 116-118",
           yfmt=".2f", ylabel="log-odds coefficient", xlabel="Congress", yzero=False, name="A_task1_coefs")

pairs per Congress after negative sampling (positives all, negatives x0.1): 47,290-52,115; positives 30,703. Coefficients (log-odds per unit; per Congress fit):
                      108    109     110    111     112     113     114     115     116     117     118
log1p(hist s->l)   0.4580 0.5630  0.5670 0.5990  0.5070  0.6510  0.7230  0.5850  0.4160  0.5360  0.5590
log1p(recip l->s) -0.0580 0.0270 -0.1060 0.0440  0.1240 -0.0100 -0.1990 -0.0700 -0.0750 -0.0780 -0.0690
log1p(activity l) -0.0930 0.4440  0.2070 0.0710 -0.0150  0.0390  0.1190  0.1490  0.1550  0.1930  0.1180
same_caucus        0.8870 0.6720  1.1170 0.8170  1.1480  1.0650  1.0470  0.9280  1.3150  1.1860  1.3620
same_state         0.8400 0.6930  0.7420 0.5210  0.8640  0.7020  0.7510  0.8970  0.6320  0.3700  0.8230
  A18 log1p(hist s->l): ref 0.507..0.723, 95% PI [0.417, 0.781] (LOO false alarms 1/7) | 116: 0.416 t=-2.5 tr=-2.8 FAIL-break | 117: 0.536 t=-0.9 tr=-1.5 | 118: 0.559 t=-0.5 tr=-1.3 | joint t=-2.0 p=0.093
  A18 log1

### A14. Which candidate split's training pool is closest to the Congress it evaluates?
**Why it matters:** this compares the shared candidate splits (`SPLIT_DEFS`: S1a/S1b expanding, S2
sliding k=`SLIDE_K`, S3 rolling-origin expanding) on the distribution distance between each evaluation
unit's training Congresses and its evaluated Congress. A smaller distance means the evaluated Congress
is better represented by that split's training data. **How to read it:** mean distance per split and
role over its evaluation units (`iter_eval_units`). The chart shows domain-classifier AUROC per test
fold for S2 and S3 (S1a/S1b units are a subset of S3's units). This is a proxy: closer distributions
do not guarantee better scores (section D's backtest decides), and sliding pools have fewer examples.

In [111]:
# A14: distance between each eval unit's train Congresses and its eval Congress, for S1a/S1b/S2/S3.
d3t = window_targets(7, point_in_time=True)   # same point-in-time pool as A3 and section D
d3t = d3t[~d3t.censored & d3t.eligible.gt(0) & d3t.congress.isin(CONGRESSES)]
dens = d3t.groupby("congress")[["n_pos", "eligible"]].sum()
advk = adv_u.groupby("congress")["label"].agg(["sum", "size"])
rows = []
for s in ["S1a", "S1b", "S2", "S3"]:
    for u in iter_eval_units(s):
        pool, c = u["train"], u["eval_congress"]
        rows.append(dict(split=s, fold=u["fold"], role=u["role"], eval_congress=c, train=u["train_label"],
                         domain_auc=a_domain(pool, c)[0],
                         adv_logratio=abs(np.log((advk.loc[c, "sum"] / advk.loc[c, "size"]) /
                                                 (advk.loc[pool, "sum"].sum() / advk.loc[pool, "size"].sum()))),
                         dens_logratio=abs(np.log((dens.loc[c, "n_pos"] / dens.loc[c, "eligible"]) /
                                                  (dens.loc[pool, "n_pos"].sum() / dens.loc[pool, "eligible"].sum()))),
                         topic_jsd=jsd(pa.loc[pool].sum(), pa.loc[c])))
a14 = pd.DataFrame(rows)
A14_M = ["domain_auc", "adv_logratio", "dens_logratio", "topic_jsd"]
print(a14.groupby(["split", "role"])[A14_M].mean().round(4).to_string())
cmp = a14[a14.split.isin(["S2", "S3"]) & (a14.role == "test")].pivot(index="eval_congress", columns="split", values=A14_M)
wins = {m: int((cmp[m]["S2"] < cmp[m]["S3"]).sum()) for m in A14_M}
line_chart(a14[a14.split.isin(["S2", "S3"]) & (a14.role == "test")], "eval_congress", "domain_auc", series="split",
           title="How distinguishable is each test Congress from its training pool? (domain AUROC)",
           subtitle=f"S2 = sliding k={SLIDE_K} (train c-{SLIDE_K}..c-1), S3 = expanding (108..c-1); 0.5 = indistinguishable",
           yfmt=".3f", ylabel="domain AUROC", xlabel="test Congress", yzero=False, name="A_domain_auc_by_split")
share14 = float(sum(wins.values()) / (len(cmp) * len(wins)))
_d14 = a14[a14.role == "test"].groupby("split").domain_auc.mean()
by14 = {k: round(float(_d14[SPLIT_VARIANTS[k]["base"] if k in SPLIT_VARIANTS else k]), 4) for k in ALL_SPLITS
        if k in _d14.index or (k in SPLIT_VARIANTS and SPLIT_VARIANTS[k]["base"] in _d14.index and "test" not in SPLIT_VARIANTS[k])}
_pv = SPLIT_VARIANTS["S1pool"]   # S1pool: test refit on train + val, scored on each pooled test Congress
by14["S1pool"] = round(float(np.mean([a_domain(_pv["train"] + _pv["val"], c)[0] for c in _pv["test"]])), 4)
by14.update(S4=None, S5=None)
record_check("A", "A14", f"Sliding (S2, k={SLIDE_K}) training pools are closer to their test Congress than expanding (S3) pools",
             None, metric=share14, threshold="> 0.5 favours sliding window",
             detail=(f"S2 closer than S3 in {wins} of {len(cmp)} test folds per metric (domain AUROC, |log| advancement-rate "
                     f"ratio, |log| cosponsor-density ratio t0=7, topic JSD). by_split = mean test-unit domain AUROC (lower = "
                     f"closer; '+stream' variants share their base split's pools). Proxy only: distribution distance, not model performance; a {SLIDE_K}-Congress pool has "
                     f"fewer examples (section D's backtest decides)."),
             quantity="pool_distance", by_split=by14, kind="drift",
             na_reason={"S4": "within-Congress pool (prior Congresses + early part of c): a whole-Congress domain AUROC "
                              "does not apply", **A_NA})

            domain_auc  adv_logratio  dens_logratio  topic_jsd
split role                                                    
S1a   test      0.6107        0.7857         0.2781     0.0312
      val       0.6292        0.4514         0.0117     0.0391
S1b   test      0.6244        0.2588         0.5906     0.0316
      val       0.6107        0.7857         0.2781     0.0312
S2    test      0.6216        0.3846         0.1818     0.0237
      val       0.6346        0.4264         0.1229     0.0255
S3    test      0.6314        0.3538         0.2140     0.0401
      val       0.6371        0.3738         0.1411     0.0382


[INFO] A14: Sliding (S2, k=3) training pools are closer to their test Congress than expanding (S3) pools | metric=0.6429 (threshold > 0.5 favours sliding window)


### A15. Does the test Congress look like the deployment target (the in-progress 119th)?
**Why it matters:** a test score is only a good forecast of deployed performance if the test Congress
resembles the Congress the model will be used on. 119 is in progress (excluded from every split), but its
bills introduced so far can be compared with earlier Congresses. **How to read it:** to make the
comparison fair, every Congress is restricted to bills introduced by the same day of its Congress as the
latest 119th bill. The domain classifier uses t0-safe features only (no policy area, which young 119th
bills mostly lack). The **primary** feature set also drops sponsor caucus and majority status: the share
of bills sponsored by each party moves with the majority (table below), so those features make
same-majority Congresses look alike by construction. The `with party` column keeps them for comparison.
`119 vs c` = AUROC separating 119's bills from Congress c's (lower = more alike); `c vs c-1` shows how far
consecutive Congresses usually are. PSIs per feature compare 119 with 116-118 and with 108-117. INFO only:
it tells you which candidate test Congress (117 for S1a, 118 for S1b) is the better deployment proxy.

In [112]:
# A15: 119 (same day-of-Congress window) vs each complete Congress: domain AUROC (t0-safe, with / without sponsor-party
# features), sponsor-party mix per Congress, and A7-feature PSIs.
A15_DAY = int(fb_all.loc[fb_all.congress == IN_PROGRESS, "day_of_congress"].max())
fw = fb_all[fb_all.day_of_congress.between(0, A15_DAY)]
print(f"window: bills introduced in days 0-{A15_DAY} of each Congress; {IN_PROGRESS} has {int((fw.congress == IN_PROGRESS).sum()):,} "
      f"law-capable bills; policy area present for {fw[fw.congress == IN_PROGRESS].policy_area.notna().mean():.1%} of them "
      f"(118 same window: {fw[fw.congress == 118].policy_area.notna().mean():.1%})")
mix15 = fw.groupby("congress").agg(D_sponsored=("sponsor_caucus", lambda s: (s == "D").mean()),
                                   majority_sponsored=("sponsor_status", lambda s: (s == "majority").mean()))
mix15["regime"] = pd.Series(REGIME)
print("sponsor-party mix (same window): the D-sponsored share follows the majority, so caucus / majority-status features "
      "encode the regime")
print(mix15.T.to_string())
A15_P = "no party"   # primary feature set (see markdown)
d15 = pd.DataFrame({f"119 vs c ({fs})": {c: a_domain([c], IN_PROGRESS, A_FSETS[fs], A15_DAY)[0] for c in CONGRESSES}
                    for fs in (A15_P, "with party")})
d15[f"c vs c-1 ({A15_P})"] = pd.Series({c: a_domain([c - 1], c, A_FSETS[A15_P], A15_DAY)[0] for c in CONGRESSES[1:]})
_pool15 = [c for c in CONGRESSES if c < CONGRESSES[-1]]
d15_pool = {fs: a_domain(_pool15, IN_PROGRESS, A_FSETS[fs], A15_DAY)[0] for fs in (A15_P, "with party")}
_cons = d15[f"c vs c-1 ({A15_P})"].dropna()
print(d15.T.round(3).to_string())
print(f"119 vs pooled {CONGRESSES[0]}-{CONGRESSES[-2]}: " + ", ".join(f"{k} {v:.3f}" for k, v in d15_pool.items())
      + f"; consecutive-pair reference ({A15_P}): median {_cons.median():.3f}, max {_cons.max():.3f}")
r15 = []
for f in A7_FEATS:
    cons = [a_psi(f, fw[fw.congress == c - 1], fw[fw.congress == c]) for c in CONGRESSES[1:]]
    r15.append(dict(feature=f, consecutive_median=np.median(cons), consecutive_max=np.max(cons),
                    **{f"119_vs_{c}": a_psi(f, fw[fw.congress == c], fw[fw.congress == IN_PROGRESS]) for c in A_CANDS},
                    **{f"119_vs_{CONGRESSES[0]}-{CONGRESSES[-2]}": a_psi(f, fw[fw.congress.isin(CONGRESSES[:-1])],
                                                                         fw[fw.congress == IN_PROGRESS])}))
r15 = pd.DataFrame(r15).set_index("feature")
print("PSI of each feature, 119 vs ... (same day window; policy_area_harm inflated by young bills without a policy area):")
print(r15.round(3).to_string())
line_chart(d15.rename_axis("congress").reset_index().melt("congress", var_name="series", value_name="auroc").dropna(),
           "congress", "auroc", series="series",
           title=f"How distinguishable is the in-progress {IN_PROGRESS}th Congress from each earlier Congress?",
           subtitle=f"Domain AUROC on t0-safe bill features, bills from days 0-{A15_DAY}; 0.5 = indistinguishable; "
                    "'no party' drops sponsor caucus / majority status", yfmt=".3f", ylabel="domain AUROC",
           xlabel="Congress", yzero=False, name="A_deploy_119")
_p15 = d15[f"119 vs c ({A15_P})"].to_dict()
_p15w = d15["119 vs c (with party)"].to_dict()
by15 = a_split_metric(_p15)
best15 = min(A_CANDS, key=lambda c: _p15[c])
n_over = int((r15[f"119_vs_{CONGRESSES[-1]}"] > r15.consecutive_max).sum())
_rk = lambda d: ", ".join(f"{c}: {d[c]:.3f}" for c in sorted(d, key=d.get))
record_check("A", "A15", f"Candidate test Congresses resemble the deployment target ({IN_PROGRESS}, in progress)", None,
             metric=float(_p15[CONGRESSES[-1]]), threshold="lower domain AUROC vs 119 = better deployment proxy (INFO)",
             detail=(f"Domain AUROC of 119 vs each complete Congress (t0-safe features WITHOUT sponsor caucus / majority "
                     f"status, days 0-{A15_DAY}), most similar first: {_rk(_p15)}. With the sponsor-party features (partly "
                     f"measures party control by construction: the D-sponsored share is "
                     f"{a_d(mix15.D_sponsored, 3)}): {_rk(_p15w)}. Candidates ({A15_P}): "
                     + ", ".join(f"{c}: {_p15[c]:.3f}" for c in A_CANDS) + f"; vs pooled {CONGRESSES[0]}-{CONGRESSES[-2]} "
                     f"{d15_pool[A15_P]:.3f}; consecutive-Congress reference median {_cons.median():.3f} (max "
                     f"{_cons.max():.3f}). Closest candidate: {best15}; closest overall: {min(_p15, key=_p15.get)} "
                     f"({A15_P}) vs {min(_p15w, key=_p15w.get)} (with party). Features whose 119-vs-118 PSI exceeds the "
                     f"largest consecutive-pair PSI: {n_over}/{len(r15)}. by_split = mean {A15_P} domain AUROC of the split's "
                     f"test Congresses vs 119 (lower = test set closer to deployment; S4 = its folds 116-118; S5 not judged). "
                     f"Proxy: 119 is partial (bills introduced by day {A15_DAY}; later-session bills and outcomes unseen)."),
             quantity="deployment_proxy", by_split=by15, kind="drift", na_reason=A_NA)

window: bills introduced in days 0-650 of each Congress; 119 has 16,947 law-capable bills; policy area present for 99.1% of them (118 same window: 99.9%)
sponsor-party mix (same window): the D-sponsored share follows the majority, so caucus / majority-status features encode the regime
congress              108    109    110    111    112    113    114    115    116    117    118    119
D_sponsored        0.4817 0.4436 0.6434 0.7066 0.5683 0.5542 0.4675 0.4896 0.6431 0.5894 0.4837 0.4833
majority_sponsored 0.5181 0.5564 0.6434 0.7066 0.6030 0.5566 0.5325 0.5104 0.5906 0.5894 0.5622 0.5167
regime              RH/RS  RH/RS  DH/DS  DH/DS  RH/DS  RH/DS  RH/RS  RH/RS  DH/RS  DH/DS  RH/DS  RH/RS
                         108    109    110    111    112    113    114    115    116    117    118
119 vs c (no party)   0.5750 0.6080 0.5660 0.5880 0.6210 0.5490 0.5530 0.5390 0.5410 0.5590 0.5480
119 vs c (with party) 0.5750 0.6080 0.6490 0.7130 0.6860 0.6430 0.5520 0.5400 0.6980 0.6000 0.5610
c vs 

[INFO] A15: Candidate test Congresses resemble the deployment target (119, in progress) | metric=0.5477 (threshold lower domain AUROC vs 119 = better deployment proxy (INFO))


### A15b. Does similarity to 119 follow recency or party regime, once party-encoding features are removed?
**Why it matters:** if Congresses with the **same House / Senate majorities** look alike in substance,
regime is a useful feature; if similarity follows **recency**, recent data matter most. A classifier that
sees sponsor caucus and majority status will find same-regime Congresses alike **by construction** (the
sponsor-party mix moves with the majority), so the test is run three times: `no party` (chamber, bill
type, day bin, original cosponsors), `majority only` (adds the caucus-agnostic 'sponsor in majority'
flag) and `with party` (adds sponsor caucus; the earlier version). **How to read it:** every pair of
Congresses (108-119, same day window) gets a domain AUROC (heatmap = `no party`). A regression over pairs,
AUROC ~ gap + [gap = 1] + same_regime, separates time distance (the dummy absorbs the consecutive pairs,
several of which share a regime; the count is printed) from sharing the regime; its one-sided p-value comes from permuting the
regime labels across Congresses. Regimes = shared `REGIME` from `majority_party()`. Whatever the result,
section D's D26 is the outcome-based test of regime weighting; this is a bill-feature proxy.

In [113]:
# A15b: pairwise domain AUROC (days 0..A15_DAY) for 108-119 under 3 t0-safe feature sets;
# AUROC ~ 1 + gap + [gap=1] + same_regime, permutation p of the same_regime coefficient.
A15_C = CONGRESSES + [IN_PROGRESS]
A_REGIME = pd.Series({c: REGIME[c] for c in A15_C})
pairs15 = pd.DataFrame([dict(a=i, b=j, gap=j - i) for k, i in enumerate(A15_C) for j in A15_C[k + 1:]])
for fs, cols in A_FSETS.items():
    pairs15[fs] = [a_domain([r.a], r.b, cols, A15_DAY)[0] for r in pairs15.itertuples()]
A15_NPERM = 2000
_X15 = np.column_stack([np.ones(len(pairs15)), pairs15.gap.values, (pairs15.gap.values == 1).astype(float)])


def a_regime_fit(reg, y):
    """OLS of pair AUROC on [1, gap, gap==1, same_regime] -> coefficients (same_regime last)."""
    same = (reg.reindex(pairs15.a).values == reg.reindex(pairs15.b).values).astype(float)
    return np.linalg.lstsq(np.column_stack([_X15, same]), y, rcond=None)[0]


_rng15, _rng15g = np.random.default_rng(15), np.random.default_rng(151)
_perms = [pd.Series(_rng15.permutation(A_REGIME.values), index=A_REGIME.index) for _ in range(A15_NPERM)]
_ia, _ib = (np.searchsorted(A15_C, pairs15[k].values) for k in ("a", "b"))
_same15 = (A_REGIME.values[_ia] == A_REGIME.values[_ib]).astype(float)
_Xgap = []
for _ in range(A15_NPERM):   # design matrices with permuted time positions (regimes stay with their Congress)
    _pos = _rng15g.permutation(len(A15_C)).astype(float)
    _g = np.abs(_pos[_ib] - _pos[_ia])
    _Xgap.append(np.column_stack([np.ones(len(_g)), _g, (_g == 1).astype(float), _same15]))
same119 = [c for c in CONGRESSES if A_REGIME[c] == A_REGIME[IN_PROGRESS]]
oth119 = [c for c in CONGRESSES if c not in same119]
reg15 = {}
for fs in A_FSETS:
    y = pairs15[fs].values
    b = a_regime_fit(A_REGIME, y)
    perm = np.array([a_regime_fit(pr, y)[-1] for pr in _perms])
    p119 = {c: a_domain([c], IN_PROGRESS, A_FSETS[fs], A15_DAY)[0] for c in CONGRESSES}
    # Recency test: permute the Congresses' time positions (regimes stay attached to their Congress) -> gap coefficient.
    gperm = [np.linalg.lstsq(Xg, y, rcond=None)[0][1] for Xg in _Xgap]
    reg15[fs] = dict(b_same=b[-1], p_perm=float((np.sum(perm <= b[-1]) + 1) / (A15_NPERM + 1)), b_gap=b[1],
                     p_gap=float((np.sum(np.array(gperm) >= b[1]) + 1) / (A15_NPERM + 1)), b_gap1=b[2],
                     auroc_119_same=float(np.mean([p119[c] for c in same119])),
                     auroc_119_other=float(np.mean([p119[c] for c in oth119])),
                     closest_to_119=int(min(p119, key=p119.get)))
reg15 = pd.DataFrame(reg15).T.astype({"closest_to_119": int})
print(f"pairs regression AUROC ~ 1 + gap + [gap=1] + same_regime (n={len(pairs15)} pairs; {A15_NPERM} permutations): "
      f"p_perm = one-sided p that sharing the regime LOWERS the AUROC (regime labels permuted); p_gap = one-sided p that "
      f"distance in time RAISES it (Congress time positions permuted); consecutive pairs sharing a regime: "
      f"{int(_same15[pairs15.gap.values == 1].sum())}/{int((pairs15.gap == 1).sum())}")
print(reg15.to_string(float_format=lambda v: f"{v:.4f}"))
hm15 = pd.concat([pairs15.rename(columns={"a": "x", "b": "y"}), pairs15.rename(columns={"a": "y", "b": "x"})])
hm15 = hm15.assign(x=hm15.x.astype(str) + " " + hm15.x.map(A_REGIME), y=hm15.y.astype(str) + " " + hm15.y.map(A_REGIME))
heatmap(hm15, "x", "y", A15_P, title="How distinguishable are two Congresses' bills? (pairwise domain AUROC)",
        subtitle=f"t0-safe features without sponsor caucus / majority status, bills from days 0-{A15_DAY}; "
                 "labels = Congress and House/Senate majority; 0.5 = alike", fmt=".2f", height=520, width=680,
        name="A_pairwise_regime_heatmap")
_r = reg15.loc[A15_P]
_rw = reg15.loc["with party"]
reg_np = bool(_r.b_same < 0 and _r.p_perm < A_ALPHA)
cand_match = [c for c in A_CANDS if A_REGIME[c] == A_REGIME[IN_PROGRESS]]
record_check("A", "A15b", f"Similarity between Congresses follows party regime after removing sponsor-party features "
                          f"(deployment target {IN_PROGRESS})", None,
             metric=float(_r.b_same),
             threshold=f"no-party same-regime coefficient < 0 with permutation p < {A_ALPHA} => regime is a real similarity axis (INFO)",
             detail=(f"Pairs regression AUROC ~ 1 + gap + [gap=1] + same_regime over {len(pairs15)} pairs of 108-119 "
                     f"(days 0-{A15_DAY}, t0-safe). same_regime coefficient (one-sided permutation p, {A15_NPERM} regime "
                     f"permutations): " + "; ".join(f"{fs}: {r.b_same:+.4f} (p={r.p_perm:.3f}; gap {r.b_gap:+.4f}/Congress p={r.p_gap:.3f}, "
                                                   f"[gap=1] {r.b_gap1:+.4f}; 119 vs same-regime {r.auroc_119_same:.3f} vs "
                                                   f"other {r.auroc_119_other:.3f}; closest {int(r.closest_to_119)})"
                                                   for fs, r in reg15.iterrows())
                     + f". The 'with party' version partly measures party control BY CONSTRUCTION: sponsor_status = "
                       f"(sponsor caucus == majority) and the D-sponsored share moves with the majority "
                       f"({a_d(mix15.D_sponsored, 3)}), so a classifier separating sponsor-party mixes finds "
                       f"same-regime Congresses alike. The no-party coefficient is the substantive test: "
                       + ("regime remains a similarity axis without party features"
                          if reg_np else "the regime effect does NOT survive without party features, so 'bills of 119 "
                                         "resemble same-regime Congresses' is not supported beyond the sponsor-party mix")
                     + f" (with party: {_rw.b_same:+.4f}, p={_rw.p_perm:.3f}). Candidate test Congresses sharing 119's "
                       f"regime: {cand_match or 'none'}. Remedy: use regime / majority status as a FEATURE (it is known at "
                       f"t0 and the model can learn regime-conditional effects, A5b/A5c); do not weight training Congresses "
                       f"by regime unless section D's D26 shows a gain over recency weighting. Proxy: bill-feature "
                       f"similarity only (no outcomes), 12 Congresses, regimes confounded with era."),
             quantity="regime_similarity", kind="drift")

pairs regression AUROC ~ 1 + gap + [gap=1] + same_regime (n=66 pairs; 2000 permutations): p_perm = one-sided p that sharing the regime LOWERS the AUROC (regime labels permuted); p_gap = one-sided p that distance in time RAISES it (Congress time positions permuted); consecutive pairs sharing a regime: 4/11
               b_same  p_perm   b_gap  p_gap  b_gap1  auroc_119_same  auroc_119_other  closest_to_119
no party      -0.0035  0.3013  0.0034 0.0555 -0.0026          0.5688           0.5674             115
majority only -0.0127  0.0980  0.0023 0.1644 -0.0090          0.5685           0.5838             115
with party    -0.0619  0.0020 -0.0004 0.5527 -0.0156          0.5685           0.6501             115


[INFO] A15b: Similarity between Congresses follows party regime after removing sponsor-party features (deployment target 119) | metric=-0.003493 (threshold no-party same-regime coefficient < 0 with permutation p < 0.05 => regime is a real similarity axis (INFO))


### Section A summary: multiplicity, trend vs break, and what survives
**Why it matters:** section A runs dozens of check parts per candidate. At a 5% level, about one part in
twenty fails by chance even if nothing changed, and the parts are correlated. Every check above is
already decided by its **Holm survivors** (cells plus joint candidate tests). A17 then asks whether a
candidate fails more than chance predicts. Because several checks track the same underlying quantity
(A1 / A1c / A1d / A2 all see the Task-2 rate drop), A17 counts **clusters of checks** (one per underlying
quantity family), each with a false-alarm rate of 1 - 0.95^m for m member checks, and compares the count
with a Poisson-binomial null. A drift FAIL here means "the later Congresses, and so the deployment
target, differ from history; the forward split measures it; apply the remedy", **not** "this split is
invalid". The trend-vs-break table labels every failing level part and the remedy each implies. The
heatmap shows the standardised distance `t` of every check part.

In [114]:
# Section A summary: per-candidate failing clusters vs chance (A17), raw part counts, trend vs break (A16); t heatmap;
# judged drift checks per split (S4 and the by-Congress splits must be judged on the same checks).
sa = pd.DataFrame([c for c in CHECKS if c["section"] == "A"])
zt = pd.DataFrame(A_ZTAB).drop_duplicates(["part", "congress"], keep="last").reset_index(drop=True)
rt = pd.DataFrame(A_REFTAB).drop_duplicates(["part", "congress"], keep="last")
ref_share = rt.groupby("congress").fail.mean()
A_CLUSTERS = {"Task-2 base rate": ["A1", "A1c", "A1d", "A2"], "Task-2 label validity": ["A1b"],
              "Task-2 concept / model": ["A5", "A5b", "A12", "A13"], "Task-1 label": ["A3", "A3b", "A4"],
              "Task-1 concept / model": ["A6", "A18", "A18b"], "Covariates": ["A7", "A8", "A9", "A11"]}
_cl_of = {k: cl for cl, ks in A_CLUSTERS.items() for k in ks}
_unc = sorted(set(zt.check) - set(_cl_of))
for k in _unc:   # any check not listed forms its own cluster
    A_CLUSTERS[k] = [k]
    _cl_of[k] = k
zt["cluster"] = zt.check.map(_cl_of)
cl_rate = {cl: 1 - (1 - A_ALPHA) ** sum(k in set(zt.check) for k in ks) for cl, ks in A_CLUSTERS.items()}
cl_rate = {cl: r for cl, r in cl_rate.items() if r > 0}


def a_pbinom_sf(k, rates):
    """P(X >= k) for X = sum of independent Bernoulli(rates) (Poisson-binomial, exact by convolution)."""
    pmf = np.array([1.0])
    for r in rates:
        pmf = np.convolve(pmf, [1 - r, r])
    return float(pmf[k:].sum())


n_checks = zt.check.nunique()
mult = []
for c in A_CANDS:
    g = zt[zt.congress == c]
    kc = g.groupby("check").sig.any()
    kcl = g.groupby("cluster").sig.any()
    k, n = int(g.fail.sum()), len(g)
    mult.append(dict(congress=c, parts=n, raw_failing=k, ref_share_max=float(ref_share.max()),
                     checks_failing=f"{int(kc.sum())}/{n_checks}",
                     check_binom_p=float(stats.binom.sf(int(kc.sum()) - 1, n_checks, A_ALPHA)),
                     clusters_failing=f"{int(kcl.sum())}/{len(cl_rate)}",
                     expected=round(sum(cl_rate.values()), 2), cluster_p=a_pbinom_sf(int(kcl.sum()), list(cl_rate.values())),
                     failing_clusters=", ".join(kcl.index[kcl]) or "none"))
mult = pd.DataFrame(mult).set_index("congress")
print(f"Section A: {(sa.passed == True).sum()} pass, {(sa.passed == False).sum()} fail, {sa.passed.isna().sum()} info "   # noqa: E712
      f"(verdicts = Holm survivors within each check, cells + joint tests). Per candidate: raw failing parts, failing checks "
      f"(binomial, correlated: optimistic) and failing clusters (Poisson-binomial; decides A17):")
print(mult.round(3).to_string())
A_CAND_FAILS = {c: zt[(zt.congress == c) & zt.fail].part.tolist() for c in A_CANDS}
A_SURVIVE = {c: zt[(zt.congress == c) & zt.sig].part.tolist() for c in A_CANDS}
print("Holm survivors (cell or joint): " + " | ".join(f"{c}: {', '.join(A_SURVIVE[c]) or 'none'}" for c in A_CANDS))
excess = {c: bool(mult.loc[c, "cluster_p"] < A_ALPHA) for c in A_CANDS}
record_check("A", "A17", "Each candidate's number of failing A check clusters (Holm survivors) is consistent with chance",
             not any(excess.values()), metric=float(max(int(v.split("/")[0]) for v in mult.clusters_failing)),
             threshold=f"FAIL if Poisson-binomial p < {A_ALPHA} for the number of failing clusters ({len(cl_rate)} "
                       f"clusters, false-alarm rate 1 - 0.95^m each)",
             detail=("; ".join(f"{c}: {r.clusters_failing} clusters failing ({r.failing_clusters}; expected {r.expected}, "
                               f"p={r.cluster_p:.2g}); {r.checks_failing} checks with a survivor (binomial p="
                               f"{r.check_binom_p:.2g}, treats correlated checks as independent); raw failing parts "
                               f"{int(r.raw_failing)}/{int(r.parts)}" for c, r in mult.iterrows())
                     + f". Clusters (one per underlying quantity family): "
                     + "; ".join(f"{cl}: {', '.join(ks)}" for cl, ks in A_CLUSTERS.items())
                     + f". Per-cluster null rate 1 - 0.95^m assumes independent members; positive correlation inside a "
                       f"cluster makes the true rate lower, so the cluster p is conservative. Reference Congresses' raw "
                       f"nested-LOO failing share: mean {ref_share.mean():.0%}, max {ref_share.max():.0%}. KIND drift: a "
                       f"flagged candidate means the post-115 Congresses (and the deployment target) differ from 108-115 "
                       f"beyond chance; every forward split measures that honestly. Remedy: recency-weighted training "
                       f"(refit on train+val), recalibration / re-thresholding on the val Congress, density-normalised "
                       f"metrics, per-Congress reporting; regime as a feature (A15b), regime weighting only if D26 shows a "
                       f"gain. {A_S4_NOTE}; S5 not judged (hides the shift)."),
             quantity="cand_fail_count", by_split=a_split_from(excess), kind="drift", na_reason=A_NA)

# Trend vs break for failing level quantities (distances are excluded: their LOO values have no time trend).
tb = zt[zt.level & zt.fail][["part", "congress", "t", "trend_z", "kind", "sig"]].copy()
tb["implies"] = tb.kind.map(A_IMPL_SHORT)
print("\nTrend vs break (failing level parts; t / tr = distance from the stationary / linear-trend PI; * = Holm survivor "
      "(cell or joint); 'trend' needs a significant 108-115 slope):")
for part, g in tb.groupby("part", sort=False):
    print(f"  {part}: " + "; ".join(f"{r.congress} {r.kind}{'*' if r.sig else ''} (t={r.t:+.1f}, tr={r.trend_z:+.1f})"
                                   for r in g.itertuples()) + " -> " + " / ".join(f"{k}: {A_IMPL_SHORT[k]}" if g.kind.nunique() > 1 else A_IMPL_SHORT[k] for k in g.kind.unique()))
if not len(tb):
    print("  none")
off = zt[zt.level & ~zt.fail & zt.trend_fail]
if len(off):
    print("Off-trend but inside the stationary PI (the train trend predicted a change that did not happen): "
          + ", ".join(f"{r.part} {r.congress} (tr={r.trend_z:+.1f})" for r in off.itertuples()))
cnt = tb.kind.value_counts()
brk_c = tb[(tb.kind == "break") & tb.sig].groupby("congress").size()
_lst = lambda k: ", ".join(f"{r.part} {r.congress}" for r in tb[tb.kind == k].itertuples()) or "none"
by16 = {k: int(sum(brk_c.get(c, 0) for c in (SPLIT_DEFS["S4"]["folds"] if split_kind(k) == "within"
                                              else split_eval_congresses(k)) if c in A_CANDS))
        for k in ALL_SPLITS if split_kind(k) != "random"}
by16["S5"] = None
record_check("A", "A16", "Candidate drift is a predictable trend rather than a structural break (failing level quantities)", None,
             metric=float(cnt.get("break", 0) / max(len(tb), 1)), threshold="share of failing level parts that are breaks (INFO)",
             detail=(f"{cnt.get('break', 0)} break, {cnt.get('trend', 0)} consistent with trend, {cnt.get('undetermined', 0)} "
                     f"undetermined among {len(tb)} failing level parts (raw t-PI failures). Breaks: {_lst('break')}. "
                     f"Consistent with trend: {_lst('trend')}. Undetermined: {_lst('undetermined')}. Break => {A_IMPL['break']}. "
                     f"Trend => {A_IMPL['trend']}. Undetermined => {A_IMPL['undetermined']}. A15b: once sponsor-party "
                     f"features are removed, the regime similarity effect is {reg15.loc[A15_P, 'b_same']:+.4f} (p="
                     f"{reg15.loc[A15_P, 'p_perm']:.3f}); section D's D26 is the outcome-based test of regime weighting. "
                     f"by_split = number of Holm-surviving (cell or joint) break parts among the split's evaluated CANDIDATE "
                     f"Congresses (S4: its folds 116-118; S5 not judged; reference Congresses not counted, so 7-fold splits are "
                     f"not penalised by construction). These breaks are differences between history and the deployment "
                     f"target, not defects of the split that reveals them. The trend PI widens with extrapolation distance, "
                     f"so breaks are hardest to detect for 118."),
             quantity="drift_type", by_split=by16, kind="drift", na_reason=A_NA)

# Judged drift checks per split (section A): S4 and the by-Congress splits must be judged on the same checks.
_lb = checks_by_split(dedup=False, include_omitted=True)
_lb = _lb[_lb.section == "A"]
jcount = _lb.groupby("split").apply(lambda g: pd.Series(dict(
    judged=int(g.judged.sum()), failing=int((g.split_passed == False).sum()),   # noqa: E712
    metric_only=int((~g.judged & g.value.notna()).sum())))).reindex(ALL_SPLITS)
print("\nSection A split-dependent checks per split (judged = True/False verdicts; A14/A15/A16 carry numeric by_split metrics instead):")
print(jcount[["judged", "failing"]].T.to_string())
_fwd = [k for k in ALL_SPLITS if split_kind(k) in BY_CONGRESS_KINDS + ("within",)]
print(f"forward splits judged on the same number of A drift checks: {jcount.loc[_fwd, 'judged'].nunique() == 1} "
      f"({ {k: int(v) for k, v in jcount.loc[_fwd, 'judged'].items()} }); S5 judged on {int(jcount.loc['S5', 'judged'])} (random split hides the shift)")

tmax = float(np.ceil(np.nanmax(np.abs(zt.t.values))))
heatmap(zt.assign(congress=zt.congress.astype(str), t=zt.t.round(2)), "congress", "part", "t",
        title="Section A: standardised distance t of each candidate from the 8 train Congresses",
        subtitle=f"t = (x - mean) / (sd * sqrt(1 + 1/n)); fails beyond the t({len(A_TRAIN) - 1}) 95% point "
                 f"(one-sided for distances); red = above train, blue = below", fmt="+.1f",
        domain=[-tmax, tmax], diverging=True, height=40 + 18 * zt.part.nunique() + 60, width=560, name="A_t_summary")

Section A: 13 pass, 6 fail, 5 info (verdicts = Holm survivors within each check, cells + joint tests). Per candidate: raw failing parts, failing checks (binomial, correlated: optimistic) and failing clusters (Poisson-binomial; decides A17):
          parts  raw_failing  ref_share_max checks_failing  check_binom_p clusters_failing  expected  cluster_p                                                                           failing_clusters
congress                                                                                                                                                                                                  
116          36            6         0.1330           4/19         0.0130              4/6    0.8900     0.0050                Covariates, Task-2 base rate, Task-2 concept / model, Task-2 label validity
117          36            9         0.1330           5/19         0.0020              5/6    0.8900     0.0000  Covariates, Task-1 label, Task-2 base

## B. Graph and node conditions

A GNN does not only see a table of bills: it sees **nodes** (legislators, bills, committees,
subjects, policy areas) and **edges** (cosponsorship, sponsorship, committee referral, subject tags,
related-bill links). A by-Congress split is only trustworthy for a graph model if
(1) the nodes the model must score at test time are known, either from training or from history
inside the test Congress that is available before the prediction time t0 (otherwise learned node
embeddings are useless and the model must be *inductive*), (2) the graph's degree structure and
density look the same across Congresses (message passing aggregates over neighbours, so a shift in
degree changes every representation), (3) the relation schema and vocabularies (committee codes,
subject terms, policy areas) are stable, so a "committee" or "subject" node means the same thing in
train and test, and (4) no edge carries information from after t0 (related-bill links to later
bills, committee edges that record the label, cosponsor edges dated after t0).

**Evaluation pools.** Every pool here comes from the shared `SPLIT_DEFS` / `iter_eval_units()` (the
same units section D backtests): S1a (train 108-115, val 116, test 117), S1b (train 108-116, val 117,
test 118), S2 (sliding, k = `SLIDE_K`) and S3 (rolling, expanding) folds, plus the `S1pool` variant
(val 116, test 117 + 118). Many units share the same (train Congresses, evaluated Congress) pair, so
metrics are computed once per distinct **pool** (ids P01, P02, ...; the mapping to units is printed
once below, later tables are keyed by pool id). Tables show the pools that evaluate 116-118; the
per-split lines summarise all folds. The `+stream` variants (eval-Congress events dated <= intro + t0
allowed) and S4 (within-Congress: prior Congresses + events before the cut date) are evaluated where
the protocol changes the answer (cold start, sponsor history). S5 (random) ignores time and is not
evaluated. Congress 108 is the first in the data, so "never seen before" is **left-censored** there.

**Noise rule (no pool-size bias).** "Differs from train" verdicts use ONE fixed reference: the
leave-one-out spread of the S1a train Congresses 108-115 (`REL_REF`, shared `rel_to_train`), whatever
pool is evaluated. A short S2 pool is not judged against its own 3 Congresses (the max of 3 values is
exceeded by chance about 1/(3+1) = 25% of the time). Because a split has many evaluation units, a split
is judged on the **excess** of failures over the number expected by chance (binomial test: observed
failures vs alpha x tests), not on "any unit fails".

In [115]:
# Shared helpers: evaluation pools from SPLIT_DEFS (+ S1pool), per-split summaries, noise-excess test, as-of edge counts.
B_ALL = CONGRESSES  # complete Congresses; 119 shown only where noted
B_EVAL = [116, 117, 118]
B_SPLITS = [s for s in ("S1a", "S1b", "S2", "S3") if s in SPLIT_DEFS]
_bu = [dict(split=u["split"], fold=u["fold"], role=u["role"], c=int(u["eval_congress"]),
            train=tuple(u["train"]), train_label=u["train_label"])
       for s in B_SPLITS for u in iter_eval_units(s)]
if "S1pool" in SPLIT_VARIANTS:   # protocol variant with its own pools: val 116 on 108-115, test 117 + 118 on 108-116
    _v = SPLIT_VARIANTS["S1pool"]
    _bu.append(dict(split="S1pool", fold="pool", role="val", c=_v["val"][0], train=tuple(_v["train"]),
                    train_label=_rng_label(_v["train"])))
    _bu += [dict(split="S1pool", fold="pool", role="test", c=c, train=tuple(_v["train"] + _v["val"]),
                 train_label=_rng_label(_v["train"] + _v["val"])) for c in _v["test"]]
B_UNITS = pd.DataFrame(_bu)
B_UNIT_SPLITS = list(dict.fromkeys(B_UNITS.split))
B_STREAM = {k: v["base"] for k, v in SPLIT_VARIANTS.items() if v.get("stream")}   # e.g. 'S1a+stream' -> 'S1a'
B_UNITS["tag"] = B_UNITS.split + " " + B_UNITS.role
B_POOLS = (B_UNITS.groupby(["train_label", "c"], sort=False)
           .agg(train=("train", "first"), used_by=("tag", lambda t: ", ".join(dict.fromkeys(t)))).reset_index())
B_POOLS["k"] = B_POOLS.train.map(len)
B_POOLS = B_POOLS.sort_values(["c", "k"], ascending=[True, False]).reset_index(drop=True)
B_POOLS["pool"] = [f"P{i + 1:02d}" for i in range(len(B_POOLS))]
B_UNITS = B_UNITS.merge(B_POOLS[["train_label", "c", "pool"]], on=["train_label", "c"])


def b_eval_pools(fn):
    """Apply fn(train_congress_list, eval_congress) -> dict to every distinct pool."""
    return pd.DataFrame([dict(pool=p.pool, train=p.train_label, eval=p.c, **fn(list(p.train), p.c))
                         for p in B_POOLS.itertuples()])


def b_show(df, cols, extra_key=()):
    """Print the pools that evaluate 116-118 (compact; pool ids -> units mapping printed once at the top)."""
    d = df[df["eval"].isin(B_EVAL)]
    print(d[["pool", "train", "eval", *extra_key, *cols]].round(3).to_string(index=False))


def b_by_split(df, col, agg="max", splits=None):
    """Per split: aggregate of `col` over all of the split's evaluation units."""
    m = B_UNITS.merge(df[["pool", col]], on="pool")
    return m.groupby("split", sort=False)[col].agg(agg).reindex(splits or B_UNIT_SPLITS)


def b_fmt_split(s, fmt=".1%"):
    return ", ".join(f"{k} {v:{fmt}}" for k, v in s.items() if pd.notna(v))


def b_split_dict(base, stream=None, extra=None):
    """by_split dict: base splits from `base`; '+stream' variants from `stream` (keyed by base split) or a
    copy of the base value when the protocol does not change the answer; `extra` adds e.g. S4."""
    out = {k: v for k, v in dict(base).items() if pd.notna(v)}
    src = dict(base) if stream is None else dict(stream)
    out.update({v: src[b] for v, b in B_STREAM.items() if b in src and pd.notna(src[b])})
    out.update(extra or {})
    return out


B_NA_DEFAULT = {
    "S5": "random bill split: ignores time and Congress boundaries (leaky reference); cross-Congress graph conditions do not apply",
    "S4": "within-Congress split: train and eval share one Congress's nodes / vocabulary; not evaluated by this check",
}


def b_bs(by_split, reasons=None):
    """record_check kwargs: by_split with a key for every split in ALL_SPLITS (None = not judged) and the
    na_reason for each None (defaults in B_NA_DEFAULT, overridden per split by `reasons`)."""
    bs = {k: (None if by_split.get(k) is None or (isinstance(by_split.get(k), float) and np.isnan(by_split.get(k)))
              else by_split.get(k)) for k in ALL_SPLITS}
    r = {**B_NA_DEFAULT, **(reasons or {})}
    return dict(by_split=bs, na_reason={k: r.get(k, "not evaluated by this check") for k, v in bs.items() if v is None})


def b_record(*args, by_split=None, na_reason=None, **kw):
    """record_check with every split keyed: splits missing from by_split get None plus a default na_reason."""
    if by_split is not None and na_reason is None:
        kw.update(b_bs(by_split))
    else:
        kw.update(by_split=by_split, na_reason=na_reason)
    record_check(*args, **kw)


def b_excess(df, fail_cols, alpha=REL_ALPHA, splits=None):
    """Noise-calibrated split verdict. df: one row per pool with boolean fail columns (each test has false-alarm
    probability alpha under 'no drift'). Per split: distinct pools, tests, observed failures, expected
    failures, binomial P(X >= observed); the split passes unless that probability < 0.05. Tests in one pool
    are positively correlated, so the binomial p is optimistic about excess (a conservative-to-fail proxy)."""
    rows = []
    for s in (splits or B_UNIT_SPLITS):
        d = df[df.pool.isin(B_UNITS.loc[B_UNITS.split == s, "pool"])]
        n = len(d) * len(fail_cols)
        obs = int(d[fail_cols].to_numpy().sum())
        p = float(stats.binom.sf(obs - 1, n, alpha)) if n else np.nan
        rows.append(dict(split=s, pools=len(d), tests=n, fails=obs, expected=alpha * n, p_excess=p, ok=bool(p >= 0.05)))
    return pd.DataFrame(rows).set_index("split")


def b_s4_pool(df, col):
    """S4 upper bound for vocabulary checks: value of the (108..c-1 -> c) pool, max over S4 folds (S4 also sees
    the early part of c, so its true unseen share can only be lower)."""
    if "S4" not in SPLIT_DEFS:
        return None
    lab = {c: _rng_label(range(CONGRESSES[0], c)) for c in SPLIT_DEFS["S4"]["folds"]}
    v = df[[lab.get(e) == t for t, e in zip(df.train, df["eval"])]][col]
    return float(v.max()) if len(v) else None


def b_num(v):
    """Compact number format for transposed tables: integers without decimals, shares with 3 decimals."""
    return "" if pd.isna(v) else f"{v:,.0f}" if float(v).is_integer() and abs(v) >= 2 else f"{v:.3f}"


def b_gini(x):
    """Gini coefficient of a non-negative vector (0 = equal, 1 = all mass on one node)."""
    x = np.sort(np.asarray(x, float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return np.nan
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def b_top_share(x, q=0.10):
    """Share of total degree held by the top q fraction of nodes."""
    x = np.sort(np.asarray(x, float))[::-1]
    k = max(1, int(np.ceil(q * len(x))))
    return float(x[:k].sum() / x.sum()) if x.sum() > 0 else np.nan


mem = members[members.congress.isin(B_ALL)].dropna(subset=["chamber"])[["bioguide_id", "congress", "chamber", "party_code_asof"]]
cosp_c = cosp[cosp.congress.isin(B_ALL)]

# Dated edges per legislator and Congress (cosponsorships + sponsored bills of any type) for as-of counts.
_bact = pd.concat([cosp_c[["congress", "bioguide_id", "sponsorship_date"]].set_axis(["congress", "bioguide_id", "d"], axis=1),
                   bills.loc[bills.congress.isin(B_ALL), ["congress", "sponsor_bioguide_id", "introduced_date"]]
                   .set_axis(["congress", "bioguide_id", "d"], axis=1)]).dropna()
_bact["d"] = _bact["d"].astype("datetime64[ns]")
B_DAILY = _bact.groupby(["congress", "bioguide_id", "d"]).size().rename("n").reset_index()
B_DAILY["n_cum"] = B_DAILY.groupby(["congress", "bioguide_id"])["n"].cumsum()
B_DAILY = B_DAILY.sort_values("d", kind="stable")
B_DEG_FINAL = B_DAILY.groupby(["congress", "bioguide_id"])["n_cum"].max()
B_ACTIVE = _bact.groupby("congress")["bioguide_id"].agg(lambda s: set(s))   # legislators with >= 1 edge


def b_active_in(pool):
    """Legislators with at least one edge (sponsor or cosponsor) in any Congress of `pool`."""
    return set().union(*[B_ACTIVE[c] for c in pool if c in B_ACTIVE.index])


def b_count_asof(congress, ids, when):
    """Edges (sponsorships + cosponsorships) of legislator ids[i] in congress[i] dated <= when[i]."""
    left = pd.DataFrame({"congress": np.asarray(congress, int), "bioguide_id": pd.Series(ids).fillna("").to_numpy(),
                         "d": pd.to_datetime(pd.Series(when)).to_numpy().astype("datetime64[ns]"),
                         "_i": np.arange(len(ids))}).sort_values("d", kind="stable")
    m = pd.merge_asof(left, B_DAILY[["congress", "bioguide_id", "d", "n_cum"]], on="d",
                      by=["congress", "bioguide_id"], direction="backward")
    return m.sort_values("_i")["n_cum"].fillna(0).to_numpy()


# Task-1 positive edges per t0: cosponsorships with lag in (t0, t0+30] on uncensored law-capable bills.
B_WT = {t0: window_targets(t0) for t0 in (0, 7)}
B_POS = {}
for t0, wt in B_WT.items():
    pe = cosp_c[cosp_c.law_capable & (cosp_c.lag_days > t0) & (cosp_c.lag_days <= t0 + WINDOW_DAYS)
                & cosp_c.bill_id.isin(wt.loc[~wt.censored, "bill_id"])].copy()
    pe["n_hist"] = b_count_asof(pe.congress, pe.bioguide_id, pe.introduced_date + pd.Timedelta(days=t0))
    B_POS[t0] = pe
pos_edges = B_POS[0]

# S4 units (within-Congress): prior Congresses + events of c dated before the unit's lower cut date.
B_S4 = [dict(role=u["role"], c=u["within"][0], prior=list(u["train"]),
             lo=S4_CUT_DATES[u["within"][0]][u["within"][2]], hi=S4_CUT_DATES[u["within"][0]][u["within"][3]])
        for u in iter_eval_units("S4")] if "S4" in SPLIT_DEFS else []

print(f"members (complete Congresses): {len(mem):,} rows, {mem.bioguide_id.nunique():,} legislators; "
      f"active cosponsorships: {len(cosp_c):,}; Task-1 positive edges t0=0: {len(B_POS[0]):,}, t0=7: {len(B_POS[7]):,}")
print(f"{len(B_POOLS)} distinct evaluation pools ({len(B_UNITS)} units of {', '.join(B_UNIT_SPLITS)}); "
      "pool ids used in every table below:")
print(B_POOLS[["pool", "train_label", "c", "used_by"]].rename(columns={"train_label": "train", "c": "eval"})
      .to_string(index=False))

members (complete Congresses): 6,083 rows, 1,369 legislators; active cosponsorships: 1,920,006; Task-1 positive edges t0=0: 294,865, t0=7: 232,856
16 distinct evaluation pools (35 units of S1a, S1b, S2, S3, S1pool); pool ids used in every table below:
pool   train  eval                                         used_by
 P01 108-110   111                                  S2 val, S3 val
 P02 108-111   112                                 S3 test, S3 val
 P03 109-111   112                                 S2 test, S2 val
 P04 108-112   113                                 S3 test, S3 val
 P05 110-112   113                                 S2 test, S2 val
 P06 108-113   114                                 S3 test, S3 val
 P07 111-113   114                                 S2 test, S2 val
 P08 108-114   115                                 S3 test, S3 val
 P09 112-114   115                                 S2 test, S2 val
 P10 108-115   116            S1a val, S3 test, S3 val, S1pool val
 P11 113-11

### B1. Legislator node overlap with train (transductive view)

**Why it matters.** Task 1 ranks legislators for a bill. A transductive model (a learned embedding
per legislator, e.g. an embedding table or a GNN with ID features) can only score legislators it saw
in training. A brand-new member falls back to an untrained vector. **This is a model and protocol
requirement, not a split-ranking criterion**: every election replaces a sizeable share of members
(first table), so every by-Congress split has a similar cold-start share (see the per-split line). Whether it hurts
depends on whether the model may use history inside the evaluated Congress, which B1b measures.
**How to read it.** The first table is per Congress against *all* earlier Congresses. The second
table evaluates the candidate pools. "New to chamber" counts legislators who served before, but only
in the other chamber.

In [116]:
# Cold-start rates per Congress (vs all earlier) and for the candidate pools; member retention heatmap.
first_seen = mem.groupby("bioguide_id")["congress"].min()
first_seen_ch = mem.groupby(["bioguide_id", "chamber"])["congress"].min()
rows = []
for c in B_ALL[1:]:
    m = mem[mem.congress == c].drop_duplicates(["bioguide_id", "chamber"])
    new = m.bioguide_id.map(first_seen) == c
    new_ch = np.array([first_seen_ch.get((b, ch), c) == c for b, ch in zip(m.bioguide_id, m.chamber)]) & ~new.values
    act = cosp_c[cosp_c.congress == c].bioguide_id.drop_duplicates()
    pe = pos_edges[pos_edges.congress == c]
    rows.append(dict(congress=c, members=m.bioguide_id.nunique(), new_members=float(new.mean()),
                     new_to_chamber=float(new_ch.mean()),
                     new_active_cosponsors=float((act.map(first_seen) == c).mean()),
                     new_pos_edges=float((pe.bioguide_id.map(first_seen) == c).mean())))
b1 = pd.DataFrame(rows)
print("One-step-ahead cold start (legislator never a member in any earlier Congress in the data):")
print(b1.set_index("congress").T.to_string(float_format=b_num))

mem_u = mem.drop_duplicates(["bioguide_id", "congress"])
act_u = cosp_c.drop_duplicates(["bioguide_id", "congress"])


def _b1(pool, c):
    seen = set(mem.loc[mem.congress.isin(pool), "bioguide_id"])
    return dict(members=float((~mem_u.loc[mem_u.congress == c, "bioguide_id"].isin(seen)).mean()),
                active_cosponsors=float((~act_u.loc[act_u.congress == c, "bioguide_id"].isin(seen)).mean()),
                pos_edges=float((~pos_edges.loc[pos_edges.congress == c, "bioguide_id"].isin(seen)).mean()))


b1p = b_eval_pools(_b1)
print("\nShare UNSEEN as a member in the train Congresses, by pool:")
b_show(b1p, ["members", "active_cosponsors", "pos_edges"])
b1_split = b_by_split(b1p, "pos_edges")
print("max share of positive edges from unseen legislators over all units: " + b_fmt_split(b1_split))

line_chart(b1.melt(id_vars="congress", value_vars=["new_members", "new_active_cosponsors", "new_pos_edges", "new_to_chamber"],
                   var_name="measure", value_name="share"),
           "congress", "share", series="measure", yfmt=".0%",
           title="Legislator cold start per Congress (never seen in any earlier Congress)",
           subtitle="Share of members / active cosponsors / Task-1 positive edges involving a new legislator; 108 omitted (left-censored)",
           ylabel="share new", name="B_cold_start_line")

# Retention heatmap: share of Congress j's members who were members in Congress i (i <= j).
sets = {c: set(mem.loc[mem.congress == c, "bioguide_id"]) for c in B_ALL}
heat = pd.DataFrame([dict(eval_congress=str(j), seen_in=str(i), share=len(sets[j] & sets[i]) / len(sets[j]) if i <= j else None)
                     for i in B_ALL for j in B_ALL])
heatmap(heat, "eval_congress", "seen_in", "share", fmt=".2f", domain=[0, 1], height=420,
        title="Member overlap: share of column Congress's members who also served in row Congress",
        subtitle="Overlap decays with distance: a train pool far in the past covers fewer test legislators",
        name="B_member_overlap_heatmap")

w1 = float(b1_split.max())
b_record("B", "B1", "Share of val/test Task-1 positive edges from legislators unseen in train (transductive-only model)",
             None, metric=round(w1, 4), threshold="info: model/protocol requirement",
             quantity="node_overlap", **b_bs({k: round(float(v), 4) for k, v in b1_split.items()},
                                           {v: "same train-cold share as the base split; the +stream warm-up is B1b"
                                            for v in B_STREAM}),
             detail=(f"Max over by-Congress units of positive edges whose cosponsor was not a member in the train Congresses: "
                     f"{b_fmt_split(b1_split)}. One-step-ahead new-edge share is {b1.new_pos_edges.min():.1%}-"
                     f"{b1.new_pos_edges.max():.1%} in every Congress (train-to-train included): elections, not the split, drive "
                     "it, so it does not rank the by-Congress splits. A purely transductive model with no within-Congress history "
                     "loses this slice. Requirement: inductive legislator features (party, state, chamber, committees) OR "
                     "message passing over eval-Congress edges dated <= t0 (B1b shows this warms up almost all of them)."))

One-step-ahead cold start (legislator never a member in any earlier Congress in the data):
congress                109   110   111   112   113   114   115   116   117   118
members                 546   554   560   552   554   547   561   550   557   554
new_members           0.095 0.130 0.150 0.196 0.173 0.128 0.132 0.182 0.140 0.164
new_to_chamber        0.009 0.007 0.007 0.011 0.014 0.013 0.005 0.009 0.005 0.009
new_active_cosponsors 0.095 0.130 0.150 0.193 0.173 0.128 0.133 0.182 0.139 0.165
new_pos_edges         0.083 0.119 0.121 0.204 0.172 0.115 0.116 0.180 0.127 0.181

Share UNSEEN as a member in the train Congresses, by pool:
pool   train  eval  members  active_cosponsors  pos_edges
 P10 108-115   116   0.1820             0.1820     0.1800
 P11 113-115   116   0.1840             0.1840     0.1840
 P12 108-116   117   0.1400             0.1390     0.1270
 P13 114-116   117   0.1400             0.1390     0.1270
 P14 108-117   118   0.1640             0.1650     0.1810
 P15 108-

[INFO] B1: Share of val/test Task-1 positive edges from legislators unseen in train (transductive-only model) | metric=0.3145 (threshold info: model/protocol requirement)


### B1b / B1c. Within-Congress warm-up before t0 (streaming view of cold start) and neighbourhood depth

**Why it matters.** In the realistic streaming setup the model may use every edge of the evaluated
Congress dated **on or before** the prediction time (bill introduction + t0): earlier cosponsorships
and sponsorships by the same legislator. A new member who already cosponsored other bills before t0
is no longer an empty node. But one or two edges are not a neighbourhood: message passing over a node
with 1-2 neighbours gives a representation dominated by those few bills. B1b counts a positive as
warm after >= 1 in-Congress edge; **B1c asks for >= 5** (the low end of typical GraphSAGE-style
neighbour-sampling fan-outs) and also shows >= 10. The verdicts are per protocol: base splits forbid
eval-Congress history (their cold share is B1's train-cold share), `+stream` variants allow it up to
intro + t0, S4 allows events of the Congress dated before its cut date.
**How to read it.** One row per (pool, t0). `cold_train` = positives whose cosponsor has no edge in
the train Congresses; `cold_lt1/5/10` = of all positives, cold in train AND fewer than 1 / 5 / 10
in-Congress edges by intro + t0 (`cold_lt1` is B1b's streaming-cold share); `new_ge1/5/10` = of the
train-cold positives, share with >= 1 / 5 / 10 in-Congress edges by t0; `all_ge5` = of ALL positives,
share whose cosponsor has >= 5 in-Congress edges by t0; `spon_cold` = evaluated bills whose sponsor
has no train edge and no other in-Congress edge by t0.

In [117]:
# Streaming cold start and in-Congress degree at intro + t0 for every pool; S4 units use events before the cut date.
B_SPON = {}
for t0, wt in B_WT.items():
    eb = wt[~wt.censored & wt.congress.isin(B_ALL)][["bill_id", "congress", "introduced_date"]].merge(
        bills[["bill_id", "sponsor_bioguide_id"]], on="bill_id")
    eb["n_hist"] = b_count_asof(eb.congress, eb.sponsor_bioguide_id, eb.introduced_date + pd.Timedelta(days=t0)) - 1
    B_SPON[t0] = eb
B_DEG_K = (1, 5, 10)


def _b1b_stats(pe, h, ct, spon_cold):
    """pe: positives; h: in-Congress degree available to the model; ct: cold-in-train mask; spon_cold: bill mask."""
    out = dict(cold_train=float(ct.mean()))
    out.update({f"cold_lt{k}": float((ct & (h < k)).mean()) for k in B_DEG_K})
    out.update({f"new_ge{k}": float((h[ct] >= k).mean()) if ct.any() else np.nan for k in B_DEG_K})
    out.update(all_ge5=float((h >= 5).mean()), spon_cold=float(spon_cold.mean()))
    return out


def _b1b(pool, c):
    seen = b_active_in(pool)
    out = {}
    for t0 in (0, 7):
        pe, eb = B_POS[t0], B_SPON[t0]
        pe, eb = pe[pe.congress == c], eb[eb.congress == c]
        st = ~eb.sponsor_bioguide_id.isin(seen) & (eb.n_hist <= 0)
        out.update({f"{k}_t{t0}": v for k, v in _b1b_stats(pe, pe.n_hist, ~pe.bioguide_id.isin(seen), st).items()})
    return out


b1b = b_eval_pools(_b1b)
_b1b_cols = ["cold_train", *[f"cold_lt{k}" for k in B_DEG_K], *[f"new_ge{k}" for k in B_DEG_K], "all_ge5", "spon_cold"]
_long = pd.concat([b1b[["pool", "train", "eval"]].assign(t0=t0, **{k: b1b[f"{k}_t{t0}"] for k in _b1b_cols})
                   for t0 in (0, 7)]).sort_values(["eval", "pool", "t0"], kind="stable")
b_show(_long[_long.t0 == 0], _b1b_cols)
print("t0 = 7, same pools: cold_lt1 <= {:.2%}, cold_lt5 <= {:.2%}, train-cold with >= 5 edges >= {:.1%}".format(
    *(lambda d: (d.cold_lt1.max(), d.cold_lt5.max(), d.new_ge5.min()))(_long[(_long.t0 == 7) & _long["eval"].isin(B_EVAL)])))

# S4: history = prior Congresses + events of c dated before the unit's cut date (no per-bill streaming).
_s4 = []
for u in B_S4:
    seen = b_active_in(u["prior"])
    pe = B_POS[0][(B_POS[0].congress == u["c"]) & (B_POS[0].introduced_date >= u["lo"]) & (B_POS[0].introduced_date < u["hi"])]
    eb = B_SPON[0][(B_SPON[0].congress == u["c"]) & (B_SPON[0].introduced_date >= u["lo"]) & (B_SPON[0].introduced_date < u["hi"])]
    cut = u["lo"] - pd.Timedelta(days=1)
    h = pd.Series(b_count_asof(pe.congress, pe.bioguide_id, [cut] * len(pe)), index=pe.index)
    hs = b_count_asof(eb.congress, eb.sponsor_bioguide_id, [cut] * len(eb))
    _s4.append(dict(role=u["role"], c=u["c"], **_b1b_stats(pe, h, ~pe.bioguide_id.isin(seen),
                                                           ~eb.sponsor_bioguide_id.isin(seen) & (hs <= 0))))
b1b_s4 = pd.DataFrame(_s4)
s4_lt1, s4_lt5 = (float(b1b_s4.cold_lt1.max()), float(b1b_s4.cold_lt5.max())) if len(b1b_s4) else (np.nan, np.nan)

b1b_lt1 = b_by_split(b1b, "cold_lt1_t0")
b1b_lt5 = b_by_split(b1b, "cold_lt5_t0")
b1b_lt10 = b_by_split(b1b, "cold_lt10_t0")
b1b_warm = b_by_split(b1b, "new_ge1_t0", "min")
b1b_ge5 = b_by_split(b1b, "new_ge5_t0", "min")
b1b_sp = b_by_split(b1b, "spon_cold_t0")
b1b_ct = b_by_split(b1b, "cold_train_t0")
print("per split, max over units at t0=0 (base split | +stream protocol): cold_train " + b_fmt_split(b1b_ct)
      + " | cold_lt1 " + b_fmt_split(b1b_lt1, ".2%") + " | cold_lt5 " + b_fmt_split(b1b_lt5, ".2%"))
print(f"S4 units (history before the cut date): max cold_lt1 {s4_lt1:.2%}, cold_lt5 {s4_lt5:.2%}; "
      f"min share of train-cold positives with >= 5 in-Congress edges by t0=0: {b_fmt_split(b1b_ge5)}")

ch1b = b1b[b1b["eval"].isin(B_EVAL)].assign(pool=lambda d: d.pool + " " + d.train + "->" + d["eval"].astype(str))
ch1b = ch1b.melt(id_vars="pool", value_vars=["cold_train_t0", "cold_lt5_t0", "cold_lt1_t0"],
                 var_name="measure", value_name="share")
ch1b["measure"] = ch1b.measure.map({"cold_train_t0": "no train edge (base split)",
                                    "cold_lt5_t0": "no train edge, < 5 in-Congress edges by t0",
                                    "cold_lt1_t0": "no train edge, no in-Congress edge by t0"})
bar_chart(ch1b, "pool", "share", series="measure", yfmt=".0%", width=800,
          title="Share of Task-1 positives with a cold cosponsor, by how much history is allowed (t0 = 0)",
          subtitle="Pools evaluating 116-118; streaming history up to intro + t0 removes most cold start, but a 5-edge "
                   "neighbourhood is a stricter bar", ylabel="share of positives", name="B_streaming_cold_start")

B_NA_BASE_STREAM = {b: f"base split = protocol without eval-Congress history; its cold share is B1 (INFO); the "
                       f"protocol question is judged on {v}" for v, b in B_STREAM.items()}
B_NA_BASE_STREAM.update({s: "protocol without eval-Congress history; cold share is B1 (INFO)" for s in B_UNIT_SPLITS
                         if s not in B_NA_BASE_STREAM})
w1b = float(b1b_lt1.max())
w1b7 = float(b_by_split(b1b, "cold_lt1_t7").max())
_ok = lambda v, thr=0.02: None if pd.isna(v) else bool(v < thr)
b_record("B", "B1b", "With eval-Congress history up to intro + t0 (+stream), few positives involve a legislator with no edge at all",
             bool(w1b < 0.02), metric=round(w1b, 4), threshold="< 0.02 of positives (t0=0, every unit)",
             quantity="cold_start",
             **b_bs({**{v: _ok(b1b_lt1.get(b)) for v, b in B_STREAM.items() if b in b1b_lt1.index},
                     **({"S4": _ok(s4_lt1)} if len(b1b_s4) else {})}, B_NA_BASE_STREAM),
             detail=(f"+stream protocol: max share of positives whose cosponsor has no train edge and no in-Congress edge by intro + t0: "
                     f"t0=0 {b_fmt_split(b1b_lt1, '.2%')}; t0=7 max {w1b7:.2%}; S4 (events before the cut) {s4_lt1:.2%}. Base "
                     f"splits (no eval-Congress history): {b_fmt_split(b1b_ct)}; these are B1's train-cold shares and are left None in "
                     "by_split: '+stream' is a model-input protocol open to every by-Congress split, so a base split without it is "
                     "a transductive-model choice, not a failure of the split itself. Of the train-cold "
                     f"positives, >= {b1b_warm.min():.1%} have >= 1 in-Congress edge by t0=0. Bills whose sponsor is cold by t0: "
                     f"max {b1b_sp.max():.2%}. 2% threshold: at most a 2% Recall loss if cold nodes were scored at random. "
                     "'Warm' here means one edge; B1c applies a 5-edge bar."))
b_record("B", "B1c", "Positives from train-cold legislators have a usable in-Congress neighbourhood (>= 5 edges) by intro + t0",
             bool(b1b_lt5.max() < 0.02), metric=round(float(b1b_lt5.max()), 4),
             threshold="cold in train AND < 5 in-Congress edges at intro + t0: < 0.02 of positives (t0=0, every unit)",
             **b_bs({**{v: _ok(b1b_lt5.get(b)) for v, b in B_STREAM.items() if b in b1b_lt5.index},
                     **({"S4": _ok(s4_lt5)} if len(b1b_s4) else {})}, B_NA_BASE_STREAM),
             detail=(f"+stream protocol, t0=0: positives cold in train with < 5 in-Congress edges: {b_fmt_split(b1b_lt5, '.2%')} "
                     f"(< 10 edges: {b_fmt_split(b1b_lt10, '.2%')}); S4 {s4_lt5:.2%}. Among train-cold positives only, the share with "
                     f">= 5 in-Congress edges by t0 is >= {b1b_ge5.min():.1%} (>= 1 edge: {b1b_warm.min():.1%}). All positives with "
                     f">= 5 in-Congress edges by t0: >= {b1b.loc[b1b['eval'].isin(B_EVAL), 'all_ge5_t0'].min():.1%}. Degree = "
                     "sponsorships + cosponsorships in the eval Congress dated <= intro + t0. 5 edges = low end of usual "
                     "neighbour-sampling fan-outs; below it the node's representation rests on its input features. Base "
                     "splits forbid in-Congress history (their cold share is B1, INFO), so by_split covers +stream variants and S4 only. If this fails for +stream, keep inductive legislator "
                     "features (party, state, committees, prior-Congress embedding) next to message passing."))

pool   train  eval  cold_train  cold_lt1  cold_lt5  cold_lt10  new_ge1  new_ge5  new_ge10  all_ge5  spon_cold
 P10 108-115   116      0.1800    0.0070    0.0140     0.0190   0.9610   0.9210    0.8970   0.9530     0.0000
 P11 113-115   116      0.1840    0.0070    0.0150     0.0190   0.9610   0.9210    0.8980   0.9530     0.0000
 P12 108-116   117      0.1270    0.0020    0.0060     0.0090   0.9800   0.9490    0.9260   0.9670     0.0010
 P13 114-116   117      0.1270    0.0020    0.0060     0.0090   0.9800   0.9490    0.9260   0.9670     0.0010
 P14 108-117   118      0.1840    0.0040    0.0110     0.0180   0.9800   0.9420    0.9050   0.9680     0.0000
 P15 108-116   118      0.3150    0.0040    0.0140     0.0260   0.9870   0.9550    0.9160   0.9680     0.0000
 P16 115-117   118      0.1840    0.0040    0.0110     0.0180   0.9800   0.9420    0.9050   0.9680     0.0000
t0 = 7, same pools: cold_lt1 <= 0.14%, cold_lt5 <= 0.78%, train-cold with >= 5 edges >= 95.0%
per split, max over units 

[PASS] B1b: With eval-Congress history up to intro + t0 (+stream), few positives involve a legislator with no edge at all | metric=0.0071 (threshold < 0.02 of positives (t0=0, every unit))
[PASS] B1c: Positives from train-cold legislators have a usable in-Congress neighbourhood (>= 5 edges) by intro + t0 | metric=0.0145 (threshold cold in train AND < 5 in-Congress edges at intro + t0: < 0.02 of positives (t0=0, every unit))


### B2. Sponsor overlap

**Why it matters.** The sponsor is the strongest single signal for who cosponsors (sponsor-cosponsor
ties repeat across bills). Bills whose sponsor is new to the train period have no sponsor history,
so the model sees a cold-start *bill neighbourhood*. New sponsors are mostly freshmen, who
sponsor fewer bills, so the bill-level share is usually smaller than the member-level share. Unlike
B1, this depends on the train window: a short sliding window (S2) forgets sponsors who were absent
in its few Congresses. **How to read it.** Share of law-capable bills (and of Task-1 positives) in the
evaluated Congress whose sponsor was not a member during the train Congresses, and the stricter
version, "never sponsored a law-capable bill in the train Congresses" (`fail` = above the noise bound).
The bound is not a fixed band: it is the 95% upper t prediction interval of the **one-step-ahead**
share (train 108..c-1, evaluate c) over the reference Congresses 109-115, i.e. how many new sponsors an
ordinary election brings. As elsewhere, only units evaluating 116-118 decide a split. The streaming
version is in B1b.

In [118]:
# Share of evaluated-Congress bills whose sponsor is new to the train Congresses (as member / as sponsor).
lb = bills[bills.law_capable & bills.congress.isin(B_ALL)][["bill_id", "congress", "sponsor_bioguide_id"]]


def _b2(pool, c):
    seen_m = set(mem.loc[mem.congress.isin(pool), "bioguide_id"])
    seen_s = set(lb.loc[lb.congress.isin(pool), "sponsor_bioguide_id"])
    d = lb[lb.congress == c]
    pe = pos_edges[pos_edges.congress == c]
    return dict(bills=len(d), sponsor_not_member=float((~d.sponsor_bioguide_id.isin(seen_m)).mean()),
                sponsor_never_sponsored=float((~d.sponsor_bioguide_id.isin(seen_s)).mean()),
                pos_edges_new_sponsor=float((~pe.sponsor_bioguide_id.isin(seen_m)).mean()))


b2 = b_eval_pools(_b2)
# Noise calibration: one-step-ahead (expanding 108..c-1 -> c) new-sponsor share, judged against the fixed 108-115 reference.
b2_exp = pd.Series({c: _b2(list(range(B_ALL[0], c)), c)["sponsor_never_sponsored"] for c in B_ALL[1:]})
b2_tab, b2_info = rel_to_train(b2_exp, side="upper")
b2_hi = float(b2_tab.loc[b2_tab.role == "cand", "pi_hi"].iloc[0])
b2["fail"] = b2.sponsor_never_sponsored > b2_hi
b_show(b2, ["bills", "sponsor_not_member", "sponsor_never_sponsored", "pos_edges_new_sponsor", "fail"])
print("one-step-ahead new-sponsor share (expanding pools): " + ", ".join(f"{c}={v:.3f}" for c, v in b2_exp.items())
      + f"; 95% upper t-PI of ref 109-115 = {b2_hi:.3f} (nested-LOO false alarms {b2_info['fa_count']}/{b2_info['n']})")
# Per split: 'cands' rule (only units evaluating 116-118 are judged, like rel_by_split's default).
_u2 = B_UNITS.merge(b2[["pool", "eval", "sponsor_never_sponsored", "fail"]], on="pool")
_u2c = _u2[_u2["eval"].isin(B_EVAL)]
b2_split = _u2c.groupby("split", sort=False)["sponsor_never_sponsored"].max().reindex(B_UNIT_SPLITS)
b2_ok = (~_u2c.groupby("split", sort=False)["fail"].any()).reindex(B_UNIT_SPLITS)
print("max share over units evaluating 116-118 (bills whose sponsor never sponsored in train): " + b_fmt_split(b2_split))
w2 = float(b2_split.max())
_b2s4 = float(b1b_s4.spon_cold.max()) if len(b1b_s4) else np.nan
b_record("B", "B2", "Few val/test bills have a sponsor with no sponsorship history in train, relative to train-to-train noise",
             bool(b2_ok.dropna().all()), metric=round(w2, 4), threshold=f"<= {b2_hi:.3f} ({b2_info['rule']}; one-step-ahead share)",
             quantity="sponsor_overlap",
             by_split=b_split_dict({k: None if pd.isna(v) else bool(v) for k, v in b2_ok.items()},
                                   stream={k: bool(v <= b2_hi) for k, v in b1b_sp.items()},
                                   extra={"S4": bool(_b2s4 <= b2_hi)} if len(b1b_s4) else None),
             detail=(f"Max share of law-capable bills whose sponsor never sponsored in the train Congresses (units evaluating 116-118): "
                     f"{b_fmt_split(b2_split)}. Noise bound: one-step-ahead (expanding) share over 109-115 = "
                     f"{b2_exp.loc[109:115].min():.3f}-{b2_exp.loc[109:115].max():.3f}, 95% upper t-PI {b2_hi:.3f} (nested-LOO false "
                     f"alarms {b2_info['fa_count']}/{b2_info['n']}); candidates one-step-ahead: "
                     + ", ".join(f"{c} {b2_exp[c]:.3f}" for c in B_EVAL) + ". Failing units: "
                     + ("; ".join(f"{r.split} {r.role} {r.train_label}->{r['eval']} {r.sponsor_never_sponsored:.3f}" for _, r in _u2c[_u2c.fail].iterrows()) or "none")
                     + ". S1pool fails only through its protocol: its 118 test is scored by a model refit on 108-116 (two elections "
                     "back); refitting on 108-117 (as S1b / S3 do) or streaming eval-Congress history removes it. With within-Congress "
                     f"history up to t0, bills whose sponsor is cold drop to <= {b1b_sp.max():.2%} (B1b; +stream verdicts use this), "
                     f"S4 (events before the cut date) {_b2s4:.2%}. Share of positive edges on bills with a non-member sponsor <= "
                     f"{b2.pos_edges_new_sponsor.max():.1%}: report metrics separately for new-sponsor bills."))

pool   train  eval  bills  sponsor_not_member  sponsor_never_sponsored  pos_edges_new_sponsor  fail
 P10 108-115   116  14340              0.1300                   0.1320                 0.1480 False
 P11 113-115   116  14340              0.1320                   0.1330                 0.1480 False
 P12 108-116   117  15242              0.0880                   0.0910                 0.1150 False
 P13 114-116   117  15242              0.0880                   0.0920                 0.1150 False
 P14 108-117   118  16565              0.1040                   0.1050                 0.1210 False
 P15 108-116   118  16565              0.2130                   0.2170                 0.2500  True
 P16 115-117   118  16565              0.1040                   0.1060                 0.1210 False
one-step-ahead new-sponsor share (expanding pools): 109=0.051, 110=0.079, 111=0.097, 112=0.126, 113=0.118, 114=0.076, 115=0.089, 116=0.132, 117=0.091, 118=0.105; 95% upper t-PI of ref 109-115 = 0.144 

### B3. Degree distribution stationarity

**Why it matters.** GNN layers aggregate over neighbours. If bills get many more cosponsors, or a
few legislators dominate cosponsoring in later Congresses, the scale of aggregated messages and
the base rate of positives per bill both move. A model trained on old degree regimes is then
miscalibrated at test time. **How to read it.** First table: per Congress, per-bill cosponsor counts
(law-capable bills) and per-legislator cosponsorship counts, KS / PSI against the previous Congress
(KS p-values are tiny at this sample size, so read the *statistic*), Gini and top-10% share (tail
concentration). Second table: PSI of each pool's evaluated Congress against that pool's train
Congresses. The noise reference is **fixed** for every pool: the one-vs-rest PSI of each Congress in
108-115 against the other seven, with a 95% upper t prediction interval on the log scale
(`rel_to_train`). `*` marks a PSI above that bound. The split table compares observed failures with
the number expected by chance (5% per test). Degrees count all active cosponsorships (final
snapshot), a proxy for what is seen at prediction time.

In [119]:
# Per-Congress degree distributions; each pool's eval-vs-train PSI judged against ONE fixed reference (108-115 one-vs-rest PSI).
bill_deg = (bills[bills.law_capable & bills.congress.isin(B_ALL)][["bill_id", "congress"]]
            .assign(deg=lambda d: d.bill_id.map(cosp_c.groupby("bill_id").size()).fillna(0)))
leg_deg = (mem_u.merge(cosp_c.groupby(["congress", "bioguide_id"]).size().rename("deg").reset_index(),
                       on=["congress", "bioguide_id"], how="left").fillna({"deg": 0}))
leg_deg["deg_rel"] = leg_deg.deg / leg_deg.groupby("congress")["deg"].transform("mean")
rows = []
for c in B_ALL:
    bd, ld = bill_deg.loc[bill_deg.congress == c, "deg"], leg_deg.loc[leg_deg.congress == c, "deg"]
    r = dict(congress=c, bill_mean=bd.mean(), bill_zero=float((bd == 0).mean()), bill_gini=b_gini(bd),
             leg_mean=ld.mean(), leg_gini=b_gini(ld), leg_top10=b_top_share(ld))
    if c > B_ALL[0]:
        bp, lp = bill_deg.loc[bill_deg.congress == c - 1, "deg"], leg_deg.loc[leg_deg.congress == c - 1, "deg"]
        r.update(bill_ks=ks(bp, bd)[0], bill_psi=psi(bp, bd), leg_ks=ks(lp, ld)[0], leg_psi=psi(lp, ld))
    rows.append(r)
b3 = pd.DataFrame(rows)
print(b3.round(3).to_string(index=False))

_B3_SER = {"bill": (bill_deg, "deg"), "leg": (leg_deg, "deg"), "leg_rel": (leg_deg, "deg_rel")}
_psi_vs = lambda df, col, pool, c: psi(df.loc[df.congress.isin(pool), col], df.loc[df.congress == c, col])
B3_REF = {}
for k, (df, col) in _B3_SER.items():
    vals = {t: _psi_vs(df, col, [p for p in REL_REF if p != t], t) for t in REL_REF}
    vals.update({c: _psi_vs(df, col, REL_REF, c) for c in B_ALL if c not in REL_REF})
    tab, info = rel_to_train(pd.Series(vals), side="upper", log=True)
    B3_REF[k] = dict(tab=tab, info=info, hi=float(tab.loc[tab.role == "cand", "pi_hi"].iloc[0]))
print("\nFixed noise reference (one-vs-rest PSI inside 108-115; 95% upper t-PI, log scale): "
      + "; ".join(f"{k} {v['tab'].loc[v['tab'].role == 'ref', 'value'].min():.3f}-{v['tab'].loc[v['tab'].role == 'ref', 'value'].max():.3f}"
                  f" -> bound {v['hi']:.3f} (nested-LOO false alarms {v['info']['fa_count']}/{v['info']['n']})"
                  for k, v in B3_REF.items()))


def _b3(pool, c):
    out = {}
    for k, (df, col) in _B3_SER.items():
        out[f"{k}_psi"] = _psi_vs(df, col, pool, c)
        out[f"{k}_fail"] = bool(out[f"{k}_psi"] > B3_REF[k]["hi"])
    return out


b3p = b_eval_pools(_b3)
_b3show = b3p.copy()
for k in _B3_SER:
    _b3show[k] = _b3show[f"{k}_psi"].map("{:.3f}".format) + np.where(_b3show[f"{k}_fail"], "*", " ")
print("\nPSI of the evaluated Congress vs its pool's train Congresses (* = above the fixed noise bound):")
b_show(_b3show, list(_B3_SER))
b3x_raw = b_excess(b3p, ["bill_fail", "leg_fail"])
b3x_rel = b_excess(b3p, ["bill_fail", "leg_rel_fail"])
print("\nPer split, all pools: failures vs chance (raw = bill + raw legislator degree; scale-free = bill + legislator "
      "degree / Congress mean):")
print(pd.concat([b3x_raw[["pools", "tests", "fails", "expected", "p_excess"]],
                 b3x_rel[["fails", "p_excess"]].add_suffix("_scalefree")], axis=1).round(3).to_string())

line_chart(b3.melt(id_vars="congress", value_vars=["bill_gini", "leg_gini", "leg_top10", "bill_zero"],
                   var_name="measure", value_name="value"),
           "congress", "value", series="measure", yfmt=".2f",
           title="Degree concentration per Congress",
           subtitle="Gini of bill / legislator degree, top-10% legislators' share, share of bills with no cosponsor",
           ylabel="value", name="B_degree_concentration")
_bad = b3p[b3p[["bill_fail", "leg_fail"]].any(axis=1)]
b_record("B", "B3", "Val/test bill and legislator degree distributions are no further from the train pool than train Congresses are from each other",
             bool(b3x_raw.ok.all()), metric=int(b3x_raw.fails.max()),
             threshold="per split: failures (PSI > fixed 108-115 95% upper t-PI) not in excess of chance (binomial p >= 0.05)",
             quantity="degree_dist", by_split=b_split_dict(b3x_raw.ok.to_dict()),
             detail=(f"Fixed reference (pool-size independent): bounds bill {B3_REF['bill']['hi']:.3f}, legislator {B3_REF['leg']['hi']:.3f}, "
                     f"scale-free {B3_REF['leg_rel']['hi']:.3f}. Failures / tests (p excess) raw: "
                     + ", ".join(f"{s} {r.fails}/{r.tests} (p={r.p_excess:.2f})" for s, r in b3x_raw.iterrows())
                     + "; scale-free: " + ", ".join(f"{s} {r.fails}/{r.tests}" for s, r in b3x_rel.iterrows())
                     + ". Pools above the bound: " + ("; ".join(f"{r.pool} {r.train}->{r['eval']}" for _, r in _bad.iterrows()) or "none")
                     + ". Raw legislator degree jumps when bill volume jumps (116); if only the raw version fails, normalise "
                     "degree features per Congress / chamber. Final-snapshot degrees (proxy); binomial treats tests as "
                     "independent. +stream variants copy the base verdict (same graph)."))

 congress  bill_mean  bill_zero  bill_gini  leg_mean  leg_gini  leg_top10  bill_ks  bill_psi  leg_ks  leg_psi
      108    13.9150     0.2810     0.7730  267.8930    0.2890     0.2240      NaN       NaN     NaN      NaN
      109    11.3920     0.3630     0.8050  289.3280    0.2860     0.2180   0.0910    0.0360  0.1010   0.0600
      110    12.9060     0.3020     0.7840  356.5880    0.2750     0.2100   0.0660    0.0180  0.1930   0.1910
      111    11.6500     0.2960     0.7850  321.1500    0.2620     0.2020   0.0240    0.0050  0.1090   0.0720
      112    10.5280     0.3800     0.8020  240.2540    0.2360     0.1970   0.0840    0.0350  0.2940   0.4750
      113    13.3430     0.2310     0.7600  256.0670    0.2520     0.2050   0.1490    0.1060  0.0760   0.0440
      114    12.7050     0.2220     0.7570  286.9870    0.2510     0.2010   0.0140    0.0030  0.1250   0.0910
      115    12.1900     0.2000     0.7550  301.1660    0.3070     0.2220   0.0220    0.0060  0.0860   0.1080
      116 

[PASS] B3: Val/test bill and legislator degree distributions are no further from the train pool than train Congresses are from each other | metric=2 (threshold per split: failures (PSI > fixed 108-115 95% upper t-PI) not in excess of chance (binomial p >= 0.05))


### B4. Committee schema stability

**Why it matters.** Bill-committee referral edges are a natural relation for both tasks (the committee
is the gatekeeper for advancement). Committees are renamed, merged, and subcommittees are reshuffled
across Congresses. A committee node that exists only in test has no learned embedding, and a
renamed committee splits one concept into two nodes. **How to read it.** Per Congress: share of
committee codes and of *referral* edges (activity "Referred to") whose code never appeared in an
earlier Congress, split into full committees vs subcommittees. Codes in this data appear to be
normalised to current codes (e.g. 108 rows use today's subcommittee names). That helps stability
but means historical renames are hidden, not absent.

In [120]:
# Committee code novelty per Congress (vs all earlier) and per pool; full committees vs subcommittees.
bc = load("bill_committees", ("bill_id", "committee_code", "committee_type", "activity", "activity_ts", "congress"))
bc = bc[bc.congress.isin(B_ALL)].copy()
bc["is_sub"] = bc.committee_type.eq("Subcommittee")
bc["referral"] = bc.activity.str.lower().str.startswith("referred")
ref = bc[bc.referral]
first_code = ref.groupby("committee_code")["congress"].min()
rows = []
for c in B_ALL[1:]:
    d = ref[ref.congress == c]
    new = d.committee_code.map(first_code) == c
    codes = d.drop_duplicates("committee_code")
    rows.append(dict(congress=c, codes=len(codes), new_codes=float((codes.committee_code.map(first_code) == c).mean()),
                     new_edges_full=float(new[~d.is_sub].mean()), new_edges_sub=float(new[d.is_sub].mean()) if d.is_sub.any() else np.nan,
                     sub_edge_share=float(d.is_sub.mean())))
b4 = pd.DataFrame(rows)
print("Referral edges whose committee code is new vs all earlier Congresses:")
print(b4.set_index("congress").T.to_string(float_format=b_num))


def _b4(pool, c):
    seen = set(ref.loc[ref.congress.isin(pool), "committee_code"])
    d = ref[ref.congress == c]
    return dict(unseen_edges=float((~d.committee_code.isin(seen)).mean()),
                unseen_edges_full=float((~d.loc[~d.is_sub, "committee_code"].isin(seen)).mean()),
                jaccard=len(seen & set(d.committee_code)) / len(seen | set(d.committee_code)))


b4p = b_eval_pools(_b4)
print("By pool (116-118): " + "; ".join(f"{r.pool} unseen {r.unseen_edges:.2%} (full {r.unseen_edges_full:.2%}), Jaccard {r.jaccard:.2f}"
                                      for r in b4p[b4p["eval"].isin(B_EVAL)].itertuples()))
b4_split = b_by_split(b4p, "unseen_edges")
w4 = float(b4_split.max())
b_record("B", "B4", "Val/test committee-referral edges point to committees seen in train (every by-Congress unit)",
             bool(w4 < 0.05), metric=round(w4, 4), threshold="< 0.05",
             by_split=b_split_dict({k: bool(v < 0.05) for k, v in b4_split.items()},
                                   extra={"S4": bool(b_s4_pool(b4p, "unseen_edges") < 0.05)}),
             detail=(f"Max share of referral edges with a committee code unseen in the train Congresses: {b_fmt_split(b4_split, '.2%')} "
                     f"(full committees only: <= {b4p.unseen_edges_full.max():.2%}). Proxy caveat: codes look normalised "
                     "to current codes, so true renames are partly hidden. 5% threshold: unseen relation targets beyond "
                     "this need a fallback (parent-committee node or chamber-level feature)."))

Referral edges whose committee code is new vs all earlier Congresses:
congress         109   110   111   112   113   114   115   116   117   118
codes            108   115   110   114    99    99    99    97    98    88
new_codes      0.120 0.087 0.073 0.053 0.030 0.000 0.000 0.031 0.020 0.068
new_edges_full 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000
new_edges_sub  0.082 0.022 0.013 0.038 0.035 0.000 0.000 0.009 0.002 0.006
sub_edge_share 0.296 0.288 0.255 0.337 0.299 0.297 0.259 0.217 0.255 0.191
By pool (116-118): P10 unseen 0.19% (full 0.00%), Jaccard 0.61; P11 unseen 0.19% (full 0.00%), Jaccard 0.91; P12 unseen 0.05% (full 0.00%), Jaccard 0.62; P13 unseen 0.09% (full 0.00%), Jaccard 0.89; P14 unseen 0.11% (full 0.00%), Jaccard 0.51; P15 unseen 0.12% (full 0.00%), Jaccard 0.50; P16 unseen 0.13% (full 0.00%), Jaccard 0.72
[PASS] B4: Val/test committee-referral edges point to committees seen in train (every by-Congress unit) | metric=0.0128 (threshold < 0.05)


### B5. Subject and policy-area vocabulary stability and coverage

**Why it matters.** Legislative subjects (CRS index terms) and policy areas are candidate bill
features or bill-subject edges. Two failure modes break a temporal split: (a) the **vocabulary
changes** (a thesaurus revision makes old and new subject nodes disjoint), and (b) **coverage lag**:
CRS assigns subjects after introduction, sometimes much later, so recent Congresses have fewer
tagged bills. Coverage lag means test bills look different from train bills purely because of data
latency. It is also a leakage risk (tags are not timestamped, so they may encode later events).
**How to read it.** Per Congress: vocabulary size, share of subject terms / bill-subject links not
seen in any earlier Congress, Jaccard vs previous Congress, and share of law-capable bills with at
least one subject and with a policy area (final snapshot). The last line estimates policy-area
availability **at t0** from the youngest bills of the in-progress Congress 119. Because subjects are
essentially absent at t0, B5b is recorded as a feature-exclusion **requirement** (every split), not as
a split condition.

In [121]:
# Subject vocabulary novelty, coverage per Congress, unseen-subject share per pool, policy area at t0 (119 proxy).
bs = load("bill_subjects", ("bill_id", "subject", "congress"))
bs = bs[bs.congress.isin(B_ALL + [IN_PROGRESS])]
lb_all = bills[bills.law_capable & bills.congress.isin(B_ALL + [IN_PROGRESS])][["bill_id", "congress", "policy_area", "introduced_date"]]
first_subj = bs.groupby("subject")["congress"].min()
vocab = {c: set(bs.loc[bs.congress == c, "subject"]) for c in B_ALL + [IN_PROGRESS]}
rows = []
for c in B_ALL + [IN_PROGRESS]:
    d = bs[bs.congress == c]
    lbc = lb_all[lb_all.congress == c]
    r = dict(congress=c, vocab=len(vocab[c]), subj_coverage=float(lbc.bill_id.isin(d.bill_id).mean()),
             policy_coverage=float(lbc.policy_area.notna().mean()), subj_per_tagged_bill=len(d) / max(1, d.bill_id.nunique()))
    if c > B_ALL[0]:
        r.update(new_terms=float((pd.Series(list(vocab[c])).map(first_subj) == c).mean()),
                 new_links=float((d.subject.map(first_subj) == c).mean()),
                 jaccard_prev=len(vocab[c] & vocab[c - 1]) / len(vocab[c] | vocab[c - 1]))
    rows.append(r)
b5 = pd.DataFrame(rows)
print("Subject vocabulary and coverage per Congress (119 in progress, context only):")
print(b5.set_index("congress").T.to_string(float_format=b_num))


def _b5(pool, c):
    seen = set(bs.loc[bs.congress.isin(pool), "subject"])
    d = bs[bs.congress == c]
    pa_seen = set(lb_all.loc[lb_all.congress.isin(pool), "policy_area"].dropna())
    pa = lb_all.loc[lb_all.congress == c, "policy_area"].dropna()
    return dict(unseen_subject_links=float((~d.subject.isin(seen)).mean()),
                jaccard_vocab=len(seen & vocab[c]) / len(seen | vocab[c]),
                unseen_policy_area_bills=float((~pa.isin(pa_seen)).mean()))


b5p = b_eval_pools(_b5)
print("By pool (116-118): " + "; ".join(f"{r.pool} unseen links {r.unseen_subject_links:.2%}, Jaccard {r.jaccard_vocab:.2f}, "
                                      f"unseen PA bills {r.unseen_policy_area_bills:.2%}" for r in b5p[b5p["eval"].isin(B_EVAL)].itertuples()))
# Annotation at t0: policy area from the shared PA_MISSING_AT_T0 (119 bills aged 0-7 d at the snapshot); subjects locally.
_l119 = lb_all[lb_all.congress == IN_PROGRESS]
_young = _l119[_l119.introduced_date > _l119.introduced_date.max() - pd.Timedelta(days=WINDOW_DAYS)]
pa_t0 = float(1 - PA_MISS_T0)
subj_t0 = float(_young.bill_id.isin(bs.bill_id).mean())
print(f"\nAt-t0 proxy (in-progress 119): policy area present for {pa_t0:.1%} of bills aged 0-7 d (n={PA_MISS_T0_N}, shared "
      f"PA_MISSING_AT_T0); >=1 subject for {subj_t0:.1%} of bills aged 0-{WINDOW_DAYS} d (n={len(_young):,})")

line_chart(b5.melt(id_vars="congress", value_vars=["subj_coverage", "policy_coverage", "jaccard_prev"],
                   var_name="measure", value_name="share").dropna(),
           "congress", "share", series="measure", yfmt=".0%",
           title="Subject coverage and vocabulary continuity per Congress",
           subtitle="Share of law-capable bills with >=1 subject / a policy area; Jaccard of subject vocab vs previous Congress",
           ylabel="share", name="B_subject_coverage")

regime = b5.loc[b5.congress.isin(B_ALL[1:])].sort_values("jaccard_prev").iloc[0]
cov_train = b5.loc[b5.congress.isin(range(108, 116)), "subj_coverage"].min()
cov_test = b5.loc[b5.congress.isin(B_EVAL)].set_index("congress")["subj_coverage"]
b5_split = b_by_split(b5p, "unseen_subject_links")
w5 = float(b5_split.max())
b_record("B", "B5a", "Val/test bill-subject links use subject terms seen in train (every by-Congress unit)",
             bool(w5 < 0.05), metric=round(w5, 4), threshold="< 0.05",
             by_split=b_split_dict({k: bool(v < 0.05) for k, v in b5_split.items()},
                                   extra={"S4": bool(b_s4_pool(b5p, "unseen_subject_links") < 0.05)}),
             detail=(f"Max unseen-subject link share: {b_fmt_split(b5_split, '.2%')}. But the vocabulary has a regime break at "
                     f"Congress {int(regime.congress)} (Jaccard vs previous = {regime.jaccard_prev:.2f}, vocab "
                     f"{int(b5.set_index('congress').vocab[int(regime.congress) - 1])} -> {int(regime.vocab)} terms): pre-"
                     f"{int(regime.congress)} subject nodes are a different thesaurus. Use subjects only from "
                     f"{int(regime.congress)} on, or fall back to policy areas (unseen policy-area bills <= "
                     f"{b5p.unseen_policy_area_bills.max():.2%})."))
b5_tab, b5_info = rel_to_train(b5.set_index("congress").loc[B_ALL, "subj_coverage"], side="lower")
b5b_fail = b5_tab.loc[b5_tab.role == "cand", "fail"]
b_record("B", "B5b", "Requirement: do not use legislative subjects as node / edge features at t0 (not available at prediction time; coverage lags in 117-118)",
             None, metric=round(subj_t0, 4), threshold="info: feature-exclusion requirement (all splits)",
             quantity="subject_coverage", requirement=True,
             detail=(f"At t0 only {subj_t0:.1%} of bills aged 0-{WINDOW_DAYS} d (in-progress 119, n={len(_young):,}; proxy) have any subject "
                     f"(metric), so subject edges cannot exist in an as-of-t0 graph under ANY split; a split condition on them "
                     f"does not discriminate. Final-snapshot coverage also lags: train 108-115 min {cov_train:.1%}; "
                     + ", ".join(f"{c}: {v:.1%}" for c, v in cov_test.items())
                     + f" (lower 95% t-PI {b5_tab.loc[b5_tab.role == 'cand', 'pi_lo'].iloc[0]:.1%}; outside: "
                     + (", ".join(str(c) for c in b5b_fail.index[b5b_fail == True]) or "none") + ")"   # noqa: E712
                     + ", which is CRS indexing lag (missing-not-at-random), so even a 'final snapshot' subject feature would "
                     "differ between train and 117/118. If subjects are ever used (e.g. for an offline analysis), use them from "
                     f"{int(regime.congress)} on with a has_subjects indicator. Policy area: final-snapshot coverage >= "
                     f"{b5.loc[b5.congress.isin(B_ALL), 'policy_coverage'].min():.1%}, but at t0 only about {pa_t0:.0%} of bills have one "
                     f"(shared PA_MISSING_AT_T0, n={PA_MISS_T0_N}, small-sample proxy): mask it when absent, or use title text at t0. "
                     "Section C owns the coverage quantity (subject_coverage)."))

Subject vocabulary and coverage per Congress (119 in progress, context only):
congress                108    109    110   111   112   113   114   115   116    117   118   119
vocab                 4,070  4,050  4,247   957   989   996   980 1,012 1,031  1,053   961   947
subj_coverage         1.000  1.000  1.000 0.990 0.994 0.993 0.962 0.964 0.984  0.618 0.372 0.237
policy_coverage       1.000  1.000  1.000 0.999 0.998 1.000 0.996 0.994 0.995  0.996 0.999 0.991
subj_per_tagged_bill 21.691 18.311 15.434 6.771 6.742 8.385 8.223 8.317 9.664 10.108 8.299 7.552
new_terms               NaN  0.050  0.095 0.099 0.013 0.010 0.000 0.003 0.005  0.001 0.003 0.003
new_links               NaN  0.004  0.012 0.016 0.002 0.002 0.000 0.000 0.000  0.000 0.000 0.007
jaccard_prev            NaN  0.901  0.830 0.194 0.923 0.940 0.933 0.936 0.942  0.937 0.893 0.945
By pool (116-118): P10 unseen links 0.02%, Jaccard 0.21, unseen PA bills 0.00%; P11 unseen links 0.04%, Jaccard 0.95, unseen PA bills 0.00%; P12 u

[PASS] B5a: Val/test bill-subject links use subject terms seen in train (every by-Congress unit) | metric=0.0161 (threshold < 0.05)
[INFO/REQ] B5b: Requirement: do not use legislative subjects as node / edge features at t0 (not available at prediction time; coverage lags in 117-118) | metric=0.0096 (threshold info: feature-exclusion requirement (all splits))


### B6. Cross-split edges: related-bill links and within-Congress future links

**Why it matters.** `bill_related` connects bills. In a temporal split, an edge from a train bill to
a bill in a *later* Congress (or a later-introduced bill in the same Congress) lets the model see
the future when building train representations. Edges that cross the val/test boundary leak
test structure into training. Links to *earlier* Congresses are legitimate context. **How to read
it.** For each source Congress: number of links to earlier / same / later Congresses. Second
table: among same-Congress links, the share pointing to a bill introduced *after* the source bill
(these must be dropped or time-filtered when building a snapshot at t0). Caveat: the source stores
almost only same-Congress links, so the cross-Congress count says little about cross-Congress leakage.

In [122]:
# Direction of bill_related links across Congresses, and within-Congress links to later-introduced bills.
rel = load("bill_related", ("bill_id", "related_bill_id", "relationship", "congress"))
rel = rel[rel.congress.isin(B_ALL + [IN_PROGRESS])].copy()
rel["rel_congress"] = pd.to_numeric(rel.related_bill_id.str.split("-").str[0], errors="coerce")
rel["direction"] = np.select([rel.rel_congress < rel.congress, rel.rel_congress > rel.congress], ["earlier", "later"], "same")
b6 = rel.groupby(["congress", "direction"]).size().unstack(fill_value=0).reindex(columns=["earlier", "same", "later"], fill_value=0)
b6["later_share"] = b6["later"] / b6.sum(axis=1)

intro = bills.set_index("bill_id")["introduced_date"]
same = rel[rel.direction == "same"].copy()
same["src_date"], same["dst_date"] = same.bill_id.map(intro), same.related_bill_id.map(intro)
same = same.dropna(subset=["src_date", "dst_date"])
same["gap"] = (same.dst_date - same.src_date).dt.days
fut = same.groupby("congress").agg(to_later_bill=("gap", lambda g: float((g > 0).mean())),
                                   later_beyond_7d=("gap", lambda g: float((g > 7).mean())),
                                   later_beyond_30d=("gap", lambda g: float((g > 30).mean())))
print("bill_related links by direction of the related bill's Congress, and same-Congress links whose target bill "
      "was introduced after the source (119 = in progress):")
print(b6.join(fut).T.to_string(float_format=b_num))

b6_all = b6.loc[b6.index.isin(B_ALL)]
n_cross = int(b6_all[["earlier", "later"]].sum().sum())
later_eval = b6_all["later"]
share_later = float(later_eval.sum() / b6_all[["earlier", "same", "later"]].sum().sum())
b_record("B", "B6a", "Related-bill links from complete Congresses to later Congresses (cross-split leak)",
             None, metric=round(share_later, 5), threshold="info: uninformative (source stores same-Congress links)",
             detail=(f"{int(later_eval.sum())} links from 108-118 point to a later Congress and only {n_cross} cross Congresses at all "
                     f"(all in {', '.join(str(c) for c in b6_all.index[b6_all[['earlier', 'later']].sum(axis=1) > 0]) or 'none'}), "
                     "against tens of thousands of same-Congress links. bill_related evidently stores almost only "
                     "same-Congress links (a source limitation), so this is NOT evidence that cross-Congress leakage is "
                     "absent. If cross-Congress links are added from another source, drop those whose target is later "
                     "than the source's split."))
w6 = float(fut.loc[fut.index.isin(B_ALL), "to_later_bill"].mean())
b_record("B", "B6b", "Within-Congress related links are time-safe without filtering",
             bool(w6 < 0.05), metric=round(w6, 4), threshold="< 0.05 (requirement: filter if it fails)",
             quantity="leakage_related_links", requirement=True,
             detail=(f"On average {w6:.1%} of same-Congress related links point to a bill introduced later than the "
                     "source (no edge timestamps exist, so target introduction date is the earliest the edge could "
                     "exist). These edges must be filtered at snapshot time t0 (keep only targets introduced <= source "
                     "intro + t0) for every split. A graph-construction requirement, not a split differentiator."))

bill_related links by direction of the related bill's Congress, and same-Congress links whose target bill was introduced after the source (119 = in progress):
congress           108   109    110    111    112    113    114    115    116    117    118    119
earlier          0.000 0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000     36    784
same             5,321 9,000 10,590 10,365 10,000 11,654 13,710 14,844 19,294 17,646 15,593 13,173
later            0.000 0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000     20  0.000
later_share      0.000 0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000  0.001  0.000
to_later_bill    0.390 0.420  0.432  0.439  0.442  0.428  0.435  0.437  0.433  0.429  0.408  0.401
later_beyond_7d  0.307 0.337  0.354  0.358  0.376  0.369  0.379  0.380  0.362  0.354  0.333  0.330
later_beyond_30d 0.214 0.254  0.278  0.284  0.307  0.310  0.320  0.316  0.290  0.289  0.267  0.268
[INFO] B6a: Related-bill links from complete Cong

### B7. Committee edges that encode the advancement label

**Why it matters.** `bill_committees` mixes the referral edge (known on day 0-7) with later activity
edges such as "Reported by", "Markup by", "Discharged from", "Hearings by". "Reported by" *is* the
Task 2 label, and markup / hearings are strong post-introduction signals. If the graph builder keeps
all rows as bill-committee edges, every split (temporal or random) leaks the label. **How to read
it.** Activity mix per Congress, the share of referral edges that occur more than 7 days after
introduction (late referrals are not in a t0=7 snapshot), and how strongly having a non-referral edge
predicts the advancement label.

In [123]:
# Activity mix in bill_committees, late referrals, and label leakage of non-referral committee edges.
bc["act"] = np.select([bc.referral, bc.activity.str.lower().str.startswith("reported"),
                       bc.activity.str.lower().str.startswith(("markup", "hearings", "discharged"))],
                      ["referral", "reported", "markup/hearing/discharge"], "other")
mix = bc.groupby(["congress", "act"]).size().unstack(fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0)
bcd = bc.merge(bills[["bill_id", "introduced_date"]], on="bill_id")
bcd["days"] = (bcd.activity_ts.dt.normalize() - bcd.introduced_date).dt.days
late = bcd[bcd.referral].groupby("congress")["days"].apply(lambda d: float((d > 7).mean())).rename("referral_after_day7")
mix = mix.join(late)
print("Committee-edge activity mix per Congress (row shares) and share of referral edges after day 7:")
print(mix.T.to_string(float_format=b_num))

nonref = set(bc.loc[~bc.referral, "bill_id"])
a = adv[~adv.censored & adv.congress.isin(B_ALL)].copy()
a["has_nonref_edge"] = a.bill_id.isin(nonref)
leak = a.groupby("has_nonref_edge")["label"].agg(["size", "mean"])
print("Advancement label rate by whether the bill has any non-referral committee edge: " + "; ".join(
    f"{k}: n={int(r['size']):,}, rate {r['mean']:.3f}" for k, r in leak.iterrows()))
recall = float(a.loc[a.label == 1, "has_nonref_edge"].mean())
b_record("B", "B7", "Graph builder must keep only referral committee edges known by t0 (label leakage)",
             None, metric=round(recall, 4), threshold="info: must filter",
             quantity="leakage_committee_edges", requirement=True,
             detail=(f"{recall:.1%} of advancement positives have a non-referral committee edge (metric). Bills with such an "
                     f"edge advance at {leak.loc[True, 'mean']:.1%} vs {leak.loc[False, 'mean']:.1%} without: keeping them "
                     "hands the model the label. Non-referral rows are "
                     f"{1 - mix['referral'].mean():.1%} of committee rows. Keep only 'Referred to' edges with activity "
                     f"date <= intro + t0; {mix.loc[mix.index.isin(B_ALL), 'referral_after_day7'].mean():.1%} of referral "
                     "edges arrive after day 7 on average. This is a feature-construction requirement for every split, "
                     "not a reason to prefer one split."))

Committee-edge activity mix per Congress (row shares) and share of referral edges after day 7:
congress                   108   109   110   111   112   113   114   115   116   117   118
markup/hearing/discharge 0.139 0.109 0.132 0.110 0.101 0.122 0.122 0.129 0.104 0.101 0.114
other                    0.004 0.000 0.001 0.004 0.002 0.002 0.001 0.004 0.000 0.000 0.010
referral                 0.782 0.835 0.795 0.830 0.850 0.819 0.811 0.801 0.852 0.861 0.822
reported                 0.075 0.056 0.072 0.056 0.047 0.057 0.065 0.066 0.044 0.038 0.054
referral_after_day7      0.271 0.245 0.192 0.169 0.200 0.206 0.211 0.179 0.133 0.126 0.128
Advancement label rate by whether the bill has any non-referral committee edge: False: n=92,423, rate 0.000; True: n=18,379, rate 0.335
[INFO/REQ] B7: Graph builder must keep only referral committee edges known by t0 (label leakage) | metric=1 (threshold info: must filter)


### B8. Network structure stability per Congress

**Why it matters.** Structural summaries of the bill-legislator cosponsor graph (density, mean
degree) and of homophily (share of cosponsorships within the sponsor's party) describe what a GNN
learns to exploit. Rising polarisation changes how much "same party as sponsor" explains, so a model
trained in a bipartisan era over-predicts cross-party cosponsors in a polarised one. **How to read
it.** Per Congress and chamber: density = edges / (bills x members), mean cosponsors per bill,
same-party share (cosponsor party == sponsor party, D/R only). Pool test: the evaluated value is
compared with the **mean of that pool's train Congresses**, scaled by ONE fixed noise SD (the spread of
the same metric over 108-115): t = (value - pool mean) / (SD_ref x sqrt(1 + 1/k)), fail if |t| exceeds
the 95% t quantile (df = 7). The pool sets the centre (a recent sliding pool is centred near the
present), the fixed reference sets the width, so S2 is not penalised for having 3 Congresses. Last
table: for 116-118, the fixed-reference verdict and the trend residual (`trend_z`), which separates a
steady trend (large t, small trend_z) from a break (large trend_z).

In [124]:
# Density, mean degree, homophily per Congress x chamber; pool-centred t vs a fixed 108-115 noise SD; failures vs chance per split.
lc = cosp_c[cosp_c.law_capable & cosp_c.chamber.notna()]
nb = bills[bills.law_capable & bills.congress.isin(B_ALL)].groupby(["congress", "chamber"]).size().rename("bills")
nm = mem.groupby(["congress", "chamber"])["bioguide_id"].nunique().rename("members_n")
ne = lc.groupby(["congress", "chamber"]).size().rename("edges")
dr = lc[lc.party.isin(["D", "R"]) & lc.sponsor_party.isin(["D", "R"])]
sp = (dr.party == dr.sponsor_party).groupby([dr.congress, dr.chamber]).mean().rename("same_party")
b8 = pd.concat([nb, nm, ne, sp], axis=1).reset_index()
b8["density"] = b8.edges / (b8.bills * b8.members_n)
b8["mean_cosp_per_bill"] = b8.edges / b8.bills
b8["chamber"] = b8.chamber.map({"House of Representatives": "House", "Senate": "Senate"})
print(b8.pivot(index="congress", columns="chamber", values=["density", "mean_cosp_per_bill", "same_party"]).T
      .round(4).to_string())
line_chart(b8, "congress", "same_party", series="chamber", yfmt=".0%", yzero=False,
           title="Party homophily of cosponsorship per Congress",
           subtitle="Share of cosponsorships (law-capable bills, D/R only) where the cosponsor is in the sponsor's party",
           ylabel="same-party share", name="B_same_party_share")
line_chart(b8, "congress", "density", series="chamber", yfmt=".4f",
           title="Density of the bill-legislator cosponsor graph per Congress",
           subtitle="Active cosponsorships / (law-capable bills x members), by chamber of origin",
           ylabel="density", name="B_density")

B8_METRICS = ["density", "mean_cosp_per_bill", "same_party"]
B8_KEYS = [(mt, ch) for mt in B8_METRICS for ch in ("House", "Senate")]
_b8v = b8.set_index(["chamber", "congress"])
B8_SD = {(mt, ch): float(_b8v.loc[ch, mt].reindex(REL_REF).std(ddof=1)) for mt, ch in B8_KEYS}
B8_TC = float(stats.t.ppf(1 - REL_ALPHA / 2, len(REL_REF) - 1))


def _b8(pool, c):
    out = {}
    for mt, ch in B8_KEYS:
        tr = _b8v.loc[ch, mt].reindex(pool)
        t = (float(_b8v.loc[(ch, c), mt]) - tr.mean()) / (B8_SD[(mt, ch)] * np.sqrt(1 + 1 / len(tr)))
        out[f"{mt}|{ch}"] = t
        out[f"{mt}|{ch}_fail"] = bool(abs(t) > B8_TC)
    return out


b8p = b_eval_pools(_b8)
_fcols = [f"{mt}|{ch}_fail" for mt, ch in B8_KEYS]
b8p["fails"] = b8p[_fcols].sum(axis=1)
_b8show = b8p[["pool", "train", "eval", "fails"]].copy()
for mt, ch in B8_KEYS:
    _b8show[f"{mt[:4]}_{ch[0]}"] = b8p[f"{mt}|{ch}"].map("{:+.1f}".format) + np.where(b8p[f"{mt}|{ch}_fail"], "*", " ")
print(f"\nPool-centred t (fixed 108-115 SD; * = |t| > {B8_TC:.2f}); columns dens / mean / same x House (H) / Senate (S):")
b_show(_b8show, ["fails"] + [c for c in _b8show.columns if c[-2:] in ("_H", "_S")])
b8x = b_excess(b8p, _fcols)
print("Per split, all pools, failures vs chance (5% per test, 6 tests per pool) fails/tests (p_excess): " + "; ".join(
    f"{k} {int(r.fails)}/{int(r.tests)} ({r.p_excess:.2f})" for k, r in b8x.iterrows()))

# Fixed-reference verdict per Congress (pool-independent) and trend-vs-break for 116-118.
_rt = []
for mt, ch in B8_KEYS:
    tab, info = rel_to_train(_b8v.loc[ch, mt].reindex(B_ALL), side="two")
    for c in B_EVAL:
        r = tab.loc[c]
        _rt.append(dict(metric=mt, chamber=ch, congress=c, t=r.t_pi, trend_z=r.trend_z, fail=r.fail, trend_fail=r.trend_fail))
b8rt = pd.DataFrame(_rt)
_piv = b8rt.assign(cell=lambda d: d.t.map("{:+.1f}".format) + "/" + d.trend_z.map("{:+.1f}".format)
                   + np.where(d.trend_fail, " B", np.where(d.fail, " T", "")))
print("\nFixed reference 108-115 per Congress: t vs mean / trend_z (T = outside PI but on trend; B = break from trend):")
print(_piv.pivot(index=["metric", "chamber"], columns="congress", values="cell").to_string())
w8 = int(b8x.fails.max())
_brk = b8rt[b8rt.trend_fail == True]   # noqa: E712
b_record("B", "B8", "Val/test graph structure (density, mean degree, homophily per chamber) close to the pool's train mean, relative to fixed train-to-train noise",
             bool(b8x.ok.all()), metric=w8,
             threshold=f"per split: |t| > {B8_TC:.2f} failures (fixed 108-115 SD, pool-centred) not in excess of chance (binomial p >= 0.05)",
             quantity="graph_structure", by_split=b_split_dict(b8x.ok.to_dict()),
             detail=("Failures / tests (p excess): " + ", ".join(f"{s} {r.fails}/{r.tests} (p={r.p_excess:.2f})" for s, r in b8x.iterrows())
                     + ". Pools with failures (116-118): "
                     + ("; ".join(f"{r.pool} {r.train}->{r['eval']}: " + ", ".join(k.replace('_fail', '') for k in _fcols if r[k])
                                  for _, r in b8p[(b8p.fails > 0) & b8p['eval'].isin(B_EVAL)].iterrows()) or "none")
                     + ". Breaks from the 108-115 linear trend: "
                     + ("; ".join(f"{r.metric}/{r.chamber} {r.congress} (trend_z {r.trend_z:+.1f})" for r in _brk.itertuples()) or "none")
                     + ". Reference SD and centre are separated so pool size does not set the width. Remedy for failures: "
                     "per-Congress normalisation of degree features, per-chamber metrics, a same-party-share feature per "
                     "Congress. Binomial treats the 6 correlated tests per pool as independent (proxy)."))

congress                       108     109     110     111     112     113     114     115     116     117     118
                   chamber                                                                                        
density            House    0.0420  0.0358  0.0368  0.0361  0.0308  0.0382  0.0373  0.0340  0.0367  0.0329  0.0286
                   Senate   0.0536  0.0417  0.0545  0.0377  0.0439  0.0555  0.0538  0.0549  0.0584  0.0523  0.0495
mean_cosp_per_bill House   18.6641 15.9702 16.6469 16.2633 13.8567 17.1737 16.6663 15.4905 16.4399 14.9778 12.8776
                   Senate   5.3578  4.2068  5.5609  4.1475  4.4797  5.8285  5.3834  5.7648  5.9568  5.3359  5.1932
same_party         House    0.7192  0.7214  0.7792  0.8068  0.7885  0.7695  0.7743  0.7621  0.8114  0.8486  0.8024
                   Senate   0.6350  0.6497  0.6751  0.7428  0.7754  0.7569  0.7364  0.7218  0.7439  0.7652  0.7747



Pool-centred t (fixed 108-115 SD; * = |t| > 2.36); columns dens / mean / same x House (H) / Senate (S):
pool   train  eval  fails dens_H dens_S mean_H mean_S same_H same_S
 P10 108-115   116      0  +0.1   +1.2   +0.1   +1.2   +1.4   +0.6 
 P11 113-115   116      0  +0.1   +0.4   -0.0   +0.4   +1.2   +0.1 
 P12 108-116   117      1  -1.0   +0.2   -0.9   +0.2   +2.4*  +0.9 
 P13 114-116   117      0  -0.8   -0.4   -0.8   -0.5   +1.9   +0.5 
 P14 108-117   118      0  -2.2   -0.2   -2.3   -0.0   +0.8   +1.0 
 P15 108-116   118      1  -2.3   -0.1   -2.4*  +0.0   +1.0   +1.1 
 P16 115-117   118      0  -1.6   -0.7   -1.7   -0.6   -0.1   +0.5 
Per split, all pools, failures vs chance (5% per test, 6 tests per pool) fails/tests (p_excess): S1a 1/12 (0.46); S1b 1/12 (0.46); S2 0/48 (1.00); S3 1/48 (0.91); S1pool 2/18 (0.23)

Fixed reference 108-115 per Congress: t vs mean / trend_z (T = outside PI but on trend; B = break from trend):
congress                          116          117       

### B9. Dyadic (sponsor-cosponsor pair) novelty: val vs its own test

**Why it matters.** Link-prediction models often succeed by memorising recurring pairs: "X always
cosponsors Y's bills". If most test positives are pairs already seen in train, a pair-memorising
baseline will look strong; if most are new pairs, the model must generalise. Each split's **val**
unit should have a share similar to its **own test** unit, or model selection on val may not
transfer. Consecutive Congresses differ through elections, so some val-test gap is unavoidable. The
gap is therefore judged against the **year-to-year swing among train Congresses**: |new_pair(c) -
new_pair(c-1)| for c in 110-115 with expanding pools (fixed reference, 95% upper t-PI). **How to read
it.** First table: per pool, share of positives whose (cosponsor, sponsor) pair never occurred in the
train Congresses, and the share where both legislators were seen (warm nodes, new pair). Second: per
split, val-test gaps vs the noise bound and failures vs chance. Per-fold gaps are kept in `B_PAIR_GAPS`
so the decision section can relate them to D's Task-1 val-test gap.

In [125]:
# New-pair share per pool; val-vs-own-test gap per split/fold judged against the 110-115 year-to-year swing.
# (cosponsor, sponsor) pair codes and a pair x Congress presence matrix (fast "seen in pool" lookups).
_pair_code = pd.Series(pd.MultiIndex.from_arrays([cosp_c.bioguide_id, cosp_c.sponsor_bioguide_id.fillna("")]).factorize()[0],
                       index=cosp_c.index)
_pair_pres = np.zeros((int(_pair_code.max()) + 1, len(B_ALL)), bool)
_pair_pres[_pair_code.to_numpy(), cosp_c.congress.to_numpy() - B_ALL[0]] = True


def _b9(pool, c):
    seen_m = set(mem.loc[mem.congress.isin(pool), "bioguide_id"])
    pe = pos_edges[pos_edges.congress == c]
    new = ~_pair_pres[_pair_code.loc[pe.index].to_numpy()][:, np.asarray(pool) - B_ALL[0]].any(axis=1)
    warm = (pe.bioguide_id.isin(seen_m) & pe.sponsor_bioguide_id.isin(seen_m)).to_numpy()
    return dict(pos_edges=len(pe), new_pair=float(new.mean()), new_pair_warm_nodes=float((new & warm).mean()))


b9 = b_eval_pools(_b9)
b_show(b9, ["pos_edges", "new_pair", "new_pair_warm_nodes"])
# Reference swing: expanding pools 108..c-1 -> c for every c (one-step-ahead, like S3).
np_exp = pd.Series({c: _b9(list(range(B_ALL[0], c)), c)["new_pair"] for c in B_ALL[1:]})
b9_swing = (np_exp - np_exp.shift(1)).abs().dropna()
b9_tab, b9_info = rel_to_train(b9_swing, side="upper")
b9_hi = float(b9_tab.loc[b9_tab.role == "cand", "pi_hi"].iloc[0])
# Per split / fold gap; S1pool test = positive-weighted pooled share of 117 + 118.
u9 = B_UNITS.merge(b9[["pool", "new_pair", "pos_edges"]], on="pool")
u9["w"] = u9.new_pair * u9.pos_edges
g9 = u9.groupby(["split", "fold", "role"], sort=False)[["w", "pos_edges"]].sum()
g9 = (g9.w / g9.pos_edges).unstack("role").dropna()
g9["gap"] = (g9["val"] - g9["test"]).abs()
g9["fail"] = g9.gap > b9_hi
B_PAIR_GAPS = g9.reset_index()
gap_split = g9.groupby(level="split", sort=False).agg(folds=("gap", "size"), max=("gap", "max"), mean=("gap", "mean"),
                                                       fails=("fail", "sum")).reindex(B_UNIT_SPLITS)
gap_split["expected"] = REL_ALPHA * gap_split.folds
gap_split["p_excess"] = [float(stats.binom.sf(f - 1, n, REL_ALPHA)) for f, n in zip(gap_split.fails, gap_split.folds)]
gap_split["ok"] = gap_split.p_excess >= 0.05
print(f"\nYear-to-year swing |new_pair(c) - new_pair(c-1)| (expanding pools), ref 110-115: "
      + ", ".join(f"{c}={v:.3f}" for c, v in b9_swing.items()) + f"; 95% upper t-PI {b9_hi:.3f} "
      f"(nested-LOO false alarms {b9_info['fa_count']}/{b9_info['n']})")
print("|val - test| new-pair share per split vs that bound, max / mean, fails/folds (p_excess): " + "; ".join(
    f"{k} {r['max']:.3f} / {r['mean']:.3f}, {int(r.fails)}/{int(r.folds)} ({r.p_excess:.2f})" for k, r in gap_split.dropna(subset=["folds"]).iterrows()))
line_chart(pd.concat([np_exp.rename("share").reset_index().rename(columns={"index": "congress"}).assign(pool="expanding 108..c-1"),
                      b9[b9.pool.isin(B_UNITS.loc[B_UNITS.split == "S2", "pool"])][["eval", "new_pair"]].rename(columns={"eval": "congress", "new_pair": "share"})
                      .assign(pool=f"sliding {SLIDE_K} Congresses")]),
           "congress", "share", series="pool", yfmt=".0%", yzero=False,
           title="Share of Task-1 positives that are a new (cosponsor, sponsor) pair, by evaluated Congress",
           subtitle="Pair unseen in the train Congresses; swings track elections, and every by-Congress split inherits them",
           ylabel="new-pair share", name="B_new_pair_share")
b_record("B", "B9", "Each split's val unit has a new-pair share similar to its own test unit (relative to train year-to-year swing)",
             bool(gap_split.ok.all()), metric=round(float(gap_split["max"].max()), 4),
             threshold=f"per split: gaps above the 110-115 swing bound ({b9_hi:.3f}, 95% upper t-PI) not in excess of chance",
             by_split=b_split_dict(gap_split.ok.to_dict()),
             detail=("|val - test| new-pair share, max per split (fails/folds): "
                     + ", ".join(f"{s} {r['max']:.3f} ({int(r.fails)}/{int(r.folds)})" for s, r in gap_split.iterrows())
                     + f". Train year-to-year swing 110-115: {b9_swing.loc[110:115].min():.3f}-{b9_swing.loc[110:115].max():.3f} "
                     f"(bound {b9_hi:.3f}). Under S3, val(c) and test(c-1) are the same Congress, so this gap is the election "
                     "swing every by-Congress split inherits; it does not rank splits unless one exceeds the swing more "
                     "often than chance. Use: report a pair-memorisation baseline (score = past pair count) next to the GNN; "
                     "B_PAIR_GAPS (per split / fold) can be correlated with D's Task-1 val-test gap in the decision section. "
                     "+stream variants copy the base verdict (pairs are counted against train Congresses only)."))

pool   train  eval  pos_edges  new_pair  new_pair_warm_nodes
 P10 108-115   116      36888    0.4130               0.1240
 P11 113-115   116      36888    0.4230               0.1310
 P12 108-116   117      30131    0.3300               0.1110
 P13 114-116   117      30131    0.3360               0.1170
 P14 108-117   118      23691    0.3880               0.1190
 P15 108-116   118      23691    0.5830               0.1200
 P16 115-117   118      23691    0.3940               0.1260

Year-to-year swing |new_pair(c) - new_pair(c-1)| (expanding pools), ref 110-115: 110=0.013, 111=0.090, 112=0.135, 113=0.012, 114=0.079, 115=0.025, 116=0.098, 117=0.083, 118=0.057; 95% upper t-PI 0.168 (nested-LOO false alarms 1/6)
|val - test| new-pair share per split vs that bound, max / mean, fails/folds (p_excess): S1a 0.083 / 0.083, 0/1 (1.00); S1b 0.057 / 0.057, 0/1 (1.00); S2 0.144 / 0.072, 0/7 (1.00); S3 0.135 / 0.070, 0/7 (1.00); S1pool 0.028 / 0.028, 0/1 (1.00)


[PASS] B9: Each split's val unit has a new-pair share similar to its own test unit (relative to train year-to-year swing) | metric=0.1439 (threshold per split: gaps above the 110-115 swing bound (0.168, 95% upper t-PI) not in excess of chance)


### B10. As-of-t0 graph completeness

**Why it matters.** A full-graph GNN trained and evaluated on the final snapshot of a Congress sees
every cosponsor edge of that Congress, including the evaluated bill's own future cosponsors (the
Task 1 labels) and the later activity of its neighbours. At prediction time (introduction + t0), only
part of that graph exists. This is the graph-level version of the leakage in B6b / B7: it applies to
every split, so it is a construction requirement (snapshot or temporal-edge masking), not a split
preference. Its size also shows how different the as-of-t0 graph is from the final one. **How to
read it.** For each eval bill (law-capable, uncensored): `visible` = share of its Congress's final
cosponsor edges dated <= introduction + t0. `nbr_deg_after` = share of the final degree of the bill's
legislator neighbours known at t0 (sponsor + cosponsors by t0) that accrues after t0. `hop1_vis` =
(sponsor + cosponsors by t0) / (sponsor + all cosponsors): share of the bill's final 1-hop neighbourhood
that exists at t0; `hop2_vis` = edges of the as-of 1-hop legislators dated <= t0 / final edges of the final
1-hop legislators (2-hop edge mass). Means over bills. The train Congresses (108-115) matter as much as the
eval ones: training examples must be built the same way (B12).

In [126]:
# Share of the eval Congress's final cosponsor graph visible at intro + t0, and neighbour degree accrued after t0.
b10 = []
for t0, wt in B_WT.items():
    eb = wt[~wt.censored & wt.congress.isin(B_ALL)][["bill_id", "congress", "introduced_date"]].reset_index(drop=True)
    when = (eb.introduced_date + pd.Timedelta(days=t0)).to_numpy().astype("datetime64[ns]")
    vis = np.empty(len(eb))
    for c, idx in eb.groupby("congress").indices.items():
        dts = np.sort(cosp_c.loc[cosp_c.congress == c, "sponsorship_date"].dropna().to_numpy().astype("datetime64[ns]"))
        vis[idx] = np.searchsorted(dts, when[idx], side="right") / len(dts)
    eb["visible"] = vis
    seen_c = cosp_c[cosp_c.bill_id.isin(eb.bill_id) & (cosp_c.lag_days <= t0)][["bill_id", "bioguide_id"]]
    nbr = pd.concat([eb[["bill_id"]].assign(bioguide_id=eb.bill_id.map(bills.set_index("bill_id")["sponsor_bioguide_id"])),
                     seen_c]).merge(eb[["bill_id", "congress", "introduced_date"]], on="bill_id").dropna(subset=["bioguide_id"])
    nbr["asof"] = b_count_asof(nbr.congress, nbr.bioguide_id, nbr.introduced_date + pd.Timedelta(days=t0))
    nbr["final"] = pd.Series(list(zip(nbr.congress, nbr.bioguide_id))).map(B_DEG_FINAL).fillna(0).to_numpy()
    agg = nbr.groupby("bill_id")[["asof", "final"]].sum()
    eb["nbr_deg_after"] = eb.bill_id.map(1 - agg["asof"] / agg["final"].where(agg["final"] > 0))
    # 1-hop / 2-hop neighbourhood of the bill: as-of (sponsor + cosponsors by t0, their edges by t0) vs final
    # (sponsor + all cosponsors, their final edges).
    fin = pd.concat([eb[["bill_id"]].assign(bioguide_id=eb.bill_id.map(bills.set_index("bill_id")["sponsor_bioguide_id"])),
                     cosp_c.loc[cosp_c.bill_id.isin(eb.bill_id), ["bill_id", "bioguide_id"]]]
                    ).merge(eb[["bill_id", "congress"]], on="bill_id").dropna(subset=["bioguide_id"])
    fin["final"] = B_DEG_FINAL.reindex(pd.MultiIndex.from_arrays([fin.congress, fin.bioguide_id])).fillna(0).to_numpy()
    fg = fin.groupby("bill_id")["final"].agg(["size", "sum"])
    eb["hop1_vis"] = eb.bill_id.map(nbr.groupby("bill_id").size() / fg["size"])
    eb["hop2_vis"] = eb.bill_id.map(agg["asof"] / fg["sum"].where(fg["sum"] > 0))
    b10.append(eb.groupby("congress").agg(visible=("visible", "mean"), visible_p90=("visible", lambda v: v.quantile(0.9)),
                                          nbr_deg_after=("nbr_deg_after", "mean"), hop1_vis=("hop1_vis", "mean"),
                                          hop2_vis=("hop2_vis", "mean")).assign(t0=t0).reset_index())
b10 = pd.concat(b10, ignore_index=True)
print(b10.pivot(index="congress", columns="t0", values=["visible", "nbr_deg_after", "hop1_vis", "hop2_vis"]).T
      .to_string(float_format=b_num))
print("90th percentile of visible share over bills (t0=0): " + ", ".join(
    f"{int(r.congress)} {r.visible_p90:.2f}" for r in b10[b10.t0 == 0].itertuples()))
line_chart(b10.melt(id_vars=["congress", "t0"], value_vars=["visible", "nbr_deg_after"], var_name="m", value_name="share")
           .assign(measure=lambda d: d.m.map({"visible": "final edges visible", "nbr_deg_after": "neighbour degree after t0"})
                   + " (t0=" + d.t0.astype(str) + ")"),
           "congress", "share", series="measure", yfmt=".0%",
           title="How much of a Congress's final cosponsor graph exists at prediction time",
           subtitle="Mean over eval bills: share of the Congress's final edges dated <= intro + t0; share of neighbours' degree accrued after t0",
           ylabel="share", name="B_asof_completeness")
ev10 = b10[(b10.t0 == 0) & b10.congress.isin(B_EVAL)]
miss = float(1 - ev10.visible.mean())
nbr_after = float(ev10.nbr_deg_after.mean())
tr10 = b10[(b10.t0 == 0) & b10.congress.isin(range(108, 116))]
B10_TRAIN_VIS = dict(visible=float(tr10.visible.mean()), hop1=float(tr10.hop1_vis.mean()), hop2=float(tr10.hop2_vis.mean()))
B10_EVAL_VIS = dict(visible=float(ev10.visible.mean()), hop1=float(ev10.hop1_vis.mean()), hop2=float(ev10.hop2_vis.mean()))
b_record("B", "B10", "Share of the eval Congress's final cosponsor graph that is NOT available at t0 (full-graph GNN leakage)",
             None, metric=round(miss, 4), threshold="info: must time-mask edges",
             quantity="leakage_full_graph", requirement=True,
             detail=(f"For eval bills of 116-118 at t0=0, on average {miss:.1%} of their Congress's final cosponsor edges are dated after "
                     f"intro + t0 (train 108-115: {1 - tr10.visible.mean():.1%}), and {nbr_after:.1%} of the final degree of their "
                     "known legislator neighbours accrues after t0. 100% of the bill's own Task-1 positives are in the final "
                     "graph. A GNN fed the final snapshot therefore sees mostly future structure; build the eval graph from "
                     "edges dated <= intro + t0 (temporal masking / event-based snapshots) under every split. The same rule applies "
                     "to TRAINING examples: a GNN trained on Congresses c-k..c-1 with their final graphs (and the train targets "
                     "left in the message-passing graph) sees complete neighbourhoods in training but partial ones at test, a "
                     f"self-inflicted covariate shift. On train Congresses 108-115 at t0=0 the visible share is "
                     f"{B10_TRAIN_VIS['visible']:.1%} of the Congress's edges, {B10_TRAIN_VIS['hop1']:.1%} of the bill's final 1-hop "
                     f"neighbourhood and {B10_TRAIN_VIS['hop2']:.1%} of its final 2-hop edge mass (eval 116-118: "
                     f"{B10_EVAL_VIS['visible']:.1%} / {B10_EVAL_VIS['hop1']:.1%} / {B10_EVAL_VIS['hop2']:.1%}). Requirement: generate each "
                     "training example from the as-of-(intro + t0) snapshot of its own Congress and remove its target edges "
                     "from message passing; B12 measures what ignoring this costs a count baseline. Because the missing "
                     "share is similar across Congresses, this does not favour one split."))

congress           108   109   110   111   112   113   114   115   116   117   118
              t0                                                                  
visible       0  0.473 0.471 0.474 0.458 0.507 0.464 0.457 0.475 0.497 0.479 0.465
              7  0.485 0.484 0.486 0.466 0.517 0.473 0.467 0.482 0.505 0.487 0.475
nbr_deg_after 0  0.519 0.525 0.517 0.535 0.488 0.528 0.533 0.524 0.501 0.519 0.510
              7  0.506 0.512 0.504 0.525 0.477 0.518 0.523 0.516 0.493 0.510 0.499
hop1_vis      0  0.703 0.741 0.698 0.711 0.735 0.676 0.692 0.706 0.700 0.736 0.745
              7  0.737 0.773 0.739 0.749 0.769 0.722 0.734 0.745 0.742 0.772 0.778
hop2_vis      0  0.354 0.364 0.357 0.347 0.403 0.331 0.337 0.349 0.362 0.364 0.381
              7  0.381 0.392 0.387 0.373 0.427 0.361 0.364 0.373 0.390 0.388 0.405
90th percentile of visible share over bills (t0=0): 108 0.93, 109 0.88, 110 0.92, 111 0.90, 112 0.85, 113 0.91, 114 0.90, 115 0.89, 116 0.89, 117 0.90, 118 0.89


[INFO/REQ] B10: Share of the eval Congress's final cosponsor graph that is NOT available at t0 (full-graph GNN leakage) | metric=0.5197 (threshold info: must time-mask edges)


### B12. Train / eval input consistency for a temporal GNN

**Why it matters.** B10 shows the eval graph must be the as-of-(intro + t0) snapshot. If the model is
nevertheless *trained* on final graphs of the train Congresses (with each training bill's own target
edges still in the message-passing graph), its inputs in training are complete and partly label-bearing,
while at val/test time they are partial and label-free. The model learns weights for features that look
different at test time (a covariate shift created by the pipeline, not by the data), and the val score
understates what a consistent pipeline would reach. This affects every split equally, so it is a
construction **requirement**; its size tells how strictly the training pipeline must replay time.
**How it is measured.** A count-feature logistic ranker (log1p of: the candidate's pair count with the
sponsor in the 3 previous Congresses, its in-Congress pair count with the sponsor, its in-Congress
degree) is trained on sampled bills of the train Congresses of the S2 pools evaluating 116 and 117, with
in-Congress features taken (a) from the final graph including the bill's own edges, (b) from the final
graph minus the bill's own edges, (c) from the as-of-intro snapshot (t0 = 0). All are evaluated on val
bills with snapshot features (the only inputs available at prediction time). `R20_*` = mean per-bill
share of positives in the top 20 of the point-in-time same-chamber pool (sponsor and cosponsors at t0
excluded); `d_*` = R@20 minus the snapshot-trained model's, with a paired bootstrap 95% CI over val
bills. `hist_only` drops the in-Congress features (reference: what in-Congress history adds at all).
Proxy: a 3-feature linear model, not a GNN; a GNN aggregates the same in-Congress structure.

In [127]:
# Count-feature ranker trained on final-graph vs as-of-t0 in-Congress features, evaluated on as-of-t0 features.
from scipy import optimize as _b12opt

B12_N, B12_HIST_K, B12_SEED = 1000, 3, 12
_f12 = B_POOLS[B_POOLS.used_by.str.contains("S2 ") & B_POOLS.c.isin([116, 117])]
B12_FOLDS = [(list(p.train), int(p.c), p.pool) for p in _f12.itertuples()]
_c12 = sorted(set().union(*[set(t) | {c} for t, c, _ in B12_FOLDS]))
_wt12 = B_WT[0]
_pos12 = B_POS[0][["bill_id", "bioguide_id"]]
_eb12 = _wt12[~_wt12.censored & _wt12.congress.isin(_c12) & _wt12.bill_id.isin(_pos12.bill_id)][["bill_id", "congress", "introduced_date"]]
_eb12 = _eb12.sample(frac=1, random_state=B12_SEED).groupby("congress").head(B12_N)
_eb12 = _eb12.merge(bills[["bill_id", "sponsor_bioguide_id", "chamber"]], on="bill_id")
_eb12["month"] = _eb12.introduced_date.dt.to_period("M").dt.to_timestamp()
_cd = _eb12.merge(ROSTER_ASOF_ALL[["month", "chamber", "bioguide_id"]], on=["month", "chamber"])
_cd = _cd[_cd.bioguide_id != _cd.sponsor_bioguide_id]
_own = cosp_c.loc[cosp_c.bill_id.isin(_eb12.bill_id), ["bill_id", "bioguide_id", "lag_days"]]
_cd = _cd.merge(_own, on=["bill_id", "bioguide_id"], how="left")
_cd = _cd[~(_cd.lag_days <= 0)]                         # cosponsors present at t0 are not candidates
_cd["own"] = _cd.lag_days.notna().astype(int)           # candidate cosponsors this bill at some point (final graph)
_cd = _cd.drop(columns="lag_days").merge(_pos12.assign(y=1), on=["bill_id", "bioguide_id"], how="left")
_cd["y"] = _cd.y.fillna(0).astype(int)
_cd = _cd.reset_index(drop=True)

# Pair counts (sponsor -> cosponsor): final per Congress, as of the bill's intro, and over the previous B12_HIST_K Congresses.
_pc = cosp_c[["congress", "sponsor_bioguide_id", "bioguide_id", "sponsorship_date"]].dropna()
_pf = _pc.groupby(["congress", "sponsor_bioguide_id", "bioguide_id"]).size()
_key = pd.MultiIndex.from_arrays([_cd.congress, _cd.sponsor_bioguide_id, _cd.bioguide_id])
_cd["pair_final"] = _pf.reindex(_key).fillna(0).to_numpy()
_hist = pd.concat([_pf.rename("n").reset_index().assign(congress=lambda d, k=k: d.congress + k) for k in range(1, B12_HIST_K + 1)])
_cd["pair_hist"] = _hist.groupby(["congress", "sponsor_bioguide_id", "bioguide_id"])["n"].sum().reindex(_key).fillna(0).to_numpy()
_ev = _pc[_pc.congress.isin(_c12)].copy()
_ev["d"] = _ev.sponsorship_date.astype("datetime64[ns]")
_ev = _ev.sort_values("d", kind="stable")
_ev["n_cum"] = _ev.groupby(["congress", "sponsor_bioguide_id", "bioguide_id"]).cumcount() + 1
_q = _cd[["congress", "sponsor_bioguide_id", "bioguide_id"]].assign(d=_cd.introduced_date.astype("datetime64[ns]"), _i=np.arange(len(_cd)))
_q = pd.merge_asof(_q.sort_values("d", kind="stable"), _ev[["congress", "sponsor_bioguide_id", "bioguide_id", "d", "n_cum"]],
                   on="d", by=["congress", "sponsor_bioguide_id", "bioguide_id"], direction="backward")
_cd["pair_snap"] = _q.sort_values("_i")["n_cum"].fillna(0).to_numpy()
_cd["deg_final"] = B_DEG_FINAL.reindex(pd.MultiIndex.from_arrays([_cd.congress, _cd.bioguide_id])).fillna(0).to_numpy()
_cd["deg_snap"] = b_count_asof(_cd.congress, _cd.bioguide_id, _cd.introduced_date)
B12_VARIANTS = {"final_with_target": ("pair_final", "deg_final", 0), "final_minus_target": ("pair_final", "deg_final", 1),
                "snapshot": ("pair_snap", "deg_snap", 0), "hist_only": (None, None, 0)}


def _b12_X(d, var):
    pc, dc, minus = B12_VARIANTS[var]
    cols = [np.log1p(d.pair_hist.to_numpy())]
    if pc:
        own = minus * d.own.to_numpy()
        cols += [np.log1p(np.maximum(d[pc].to_numpy() - own, 0)), np.log1p(np.maximum(d[dc].to_numpy() - own, 0))]
    return np.column_stack(cols)


def _b12_fit(X, y, l2=1e-4):
    """L2-regularised logistic regression (intercept + weights) by L-BFGS."""
    Xb = np.column_stack([np.ones(len(X)), X])

    def f(w):
        z = Xb @ w
        p = 1 / (1 + np.exp(-z))
        g = Xb.T @ (p - y) / len(y)
        g[1:] += 2 * l2 * w[1:]
        return float((np.logaddexp(0, z) - y * z).mean() + l2 * (w[1:] @ w[1:])), g
    return _b12opt.minimize(f, np.zeros(Xb.shape[1]), jac=True, method="L-BFGS-B").x


def _b12_recall(d, score, k=20):
    """Per-bill Recall@k (ties broken by a fixed random jitter)."""
    jit = np.random.default_rng(B12_SEED).random(len(d)) * 1e-9
    r = pd.Series(score + jit, index=d.index).groupby(d.bill_id).rank(ascending=False, method="first")
    g = pd.DataFrame({"bill_id": d.bill_id.to_numpy(), "hit": d.y.to_numpy() * (r.to_numpy() <= k), "y": d.y.to_numpy()})
    g = g.groupby("bill_id")[["hit", "y"]].sum()
    return (g.hit / g.y).where(g.y > 0).dropna()


_rng12 = np.random.default_rng(B12_SEED)
_r12, _w12, _d12 = [], [], {v: [] for v in ("final_with_target", "final_minus_target", "hist_only")}
for train, c, pid in B12_FOLDS:
    tr, ev = _cd[_cd.congress.isin(train)], _cd[_cd.congress == c]
    rec = {}
    for v in B12_VARIANTS:
        w = _b12_fit(_b12_X(tr, v), tr.y.to_numpy().astype(float))
        rec[v] = _b12_recall(ev, _b12_X(ev, "hist_only" if v == "hist_only" else "snapshot") @ w[1:])
        _w12.append(dict(pool=pid, variant=v, **{f"w_{n}": x for n, x in zip(["hist", "pair", "deg"], w[1:])}))
    base = rec["snapshot"]
    row = dict(pool=pid, train=_rng_label(train), eval=c, train_bills=int(tr.bill_id.nunique()), val_bills=len(base),
               **{f"R20_{v}": float(r.mean()) for v, r in rec.items()})
    for v in ("final_with_target", "final_minus_target", "hist_only"):
        dlt = (rec[v].reindex(base.index) - base).to_numpy()
        _d12[v].append(dlt)
        bs = dlt[_rng12.integers(0, len(dlt), (1000, len(dlt)))].mean(axis=1)
        row.update({f"d_{v}": float(dlt.mean()), f"d_{v}_lo": float(np.quantile(bs, 0.025)), f"d_{v}_hi": float(np.quantile(bs, 0.975))})
    _r12.append(row)
b12 = pd.DataFrame(_r12)
# Pooled over folds (bill-level paired deltas concatenated; bootstrap over bills).
b12_pool = {}
for v, ds in _d12.items():
    d = np.concatenate(ds)
    bs = d[_rng12.integers(0, len(d), (1000, len(d)))].mean(axis=1)
    b12_pool[v] = (float(d.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975)))
b12_w = pd.DataFrame(_w12)
# Training-input shift: in-Congress pair count of train positives and mean candidate degree, final graph vs snapshot.
_trc = _cd[_cd.congress.isin(set().union(*[set(t) for t, _, _ in B12_FOLDS]))]
b12_shift = dict(pair_pos_final=float(_trc.loc[_trc.y == 1, "pair_final"].mean()), pair_pos_snap=float(_trc.loc[_trc.y == 1, "pair_snap"].mean()),
                 deg_final=float(_trc.deg_final.mean()), deg_snap=float(_trc.deg_snap.mean()))
print(b12[["pool", "train", "eval", "train_bills", "val_bills", "R20_snapshot", "R20_final_minus_target", "R20_final_with_target",
           "R20_hist_only"]].round(3).to_string(index=False))
for v in ("final_minus_target", "final_with_target", "hist_only"):
    print(f"d_{v} (vs snapshot-trained, 95% CI): " + "; ".join(
        f"{r['pool']} {r['d_' + v]:+.3f} [{r['d_' + v + '_lo']:+.3f}, {r['d_' + v + '_hi']:+.3f}]" for _, r in b12.iterrows())
          + "; pooled {:+.3f} [{:+.3f}, {:+.3f}]".format(*b12_pool[v]))
print("weights hist / in-Congress pair / degree (mean over folds): " + "; ".join(
    f"{v} {g.w_hist.mean():+.2f}/{g.w_pair.mean():+.2f}/{g.w_deg.mean():+.2f}" for v, g in b12_w.groupby("variant", sort=False))
      + f". Train positives' in-Congress pair count: final {b12_shift['pair_pos_final']:.2f} vs snapshot {b12_shift['pair_pos_snap']:.2f}")
_ch12 = pd.concat([b12[["pool", "eval"]].assign(variant=v, delta=b12[f"d_{v}"]) for v in ("final_minus_target", "final_with_target", "hist_only")])
_ch12["label"] = _ch12.pool + " val " + _ch12["eval"].astype(str)
_ch12["variant"] = _ch12.variant.map({"final_minus_target": "trained on final graph, own edges removed",
                                      "final_with_target": "trained on final graph incl. own edges",
                                      "hist_only": "no in-Congress features"})
bar_chart(_ch12, "label", "delta", series="variant", yfmt="+.3f",
          title="Val Recall@20 change vs a ranker trained on as-of-t0 snapshots (eval inputs are always snapshots)",
          subtitle="Count-feature logistic ranker, S2 pools evaluating 116 / 117, t0 = 0; below 0 = inconsistent training inputs cost recall",
          ylabel="Recall@20 minus snapshot-trained", name="B_train_eval_consistency")

b12_sig = [bool(r[f"d_{v}_hi"] < 0) for _, r in b12.iterrows() for v in ("final_minus_target", "final_with_target")] + [
    b12_pool[v][2] < 0 for v in ("final_minus_target", "final_with_target")]
b12_worst = float(b12[["d_final_minus_target", "d_final_with_target"]].min().min())
_b12p = b12.set_index("pool")
b_record("B", "B12", "Training inputs match eval inputs: train examples from as-of-(intro + t0) snapshots, target edges excluded from message passing",
         not any(b12_sig), metric=round(b12_worst, 4),
         threshold="FAIL (= the rule is binding) if training on final-graph features lowers val Recall@20 vs snapshot training (paired 95% CI < 0) in any fold or pooled",
         quantity="train_eval_input_shift", requirement=True,
         detail=(f"Val R@20 (snapshot inputs at eval): snapshot-trained {b_fmt_split(_b12p.R20_snapshot, '.3f')}; final graph minus own "
                 f"edges {b_fmt_split(_b12p.R20_final_minus_target, '.3f')}; final incl. own edges {b_fmt_split(_b12p.R20_final_with_target, '.3f')}; "
                 f"history only {b_fmt_split(_b12p.R20_hist_only, '.3f')}. Paired deltas vs snapshot-trained: " + "; ".join(
                     f"{p}: minus-target {r.d_final_minus_target:+.3f} [{r.d_final_minus_target_lo:+.3f}, {r.d_final_minus_target_hi:+.3f}], "
                     f"with-target {r.d_final_with_target:+.3f} [{r.d_final_with_target_lo:+.3f}, {r.d_final_with_target_hi:+.3f}]"
                     for p, r in _b12p.iterrows())
                 + "; pooled minus-target {:+.3f} [{:+.3f}, {:+.3f}]".format(*b12_pool["final_minus_target"])
                 + ", with-target {:+.3f} [{:+.3f}, {:+.3f}]".format(*b12_pool["final_with_target"])
                 + "; in-Congress features at all (snapshot vs history only) {:+.3f} [{:+.3f}, {:+.3f}]".format(
                     -b12_pool["hist_only"][0], -b12_pool["hist_only"][2], -b12_pool["hist_only"][1])
                 + ". Reading: " + ("target edges left in the training graph cost recall (pooled CI < 0)" if b12_pool["final_with_target"][2] < 0
                                    else "target edges in the training graph: no significant cost for this baseline")
                 + "; " + ("final-graph timing alone (own edges removed) also costs recall" if b12_pool["final_minus_target"][2] < 0
                           else "final-graph timing alone (own edges removed) is not significantly worse for this count baseline, so the "
                                "binding part here is target-edge exclusion; snapshot replay is still required because a GNN's "
                                "aggregations see the full 1-/2-hop shift below, which 3 counts do not")
                 + ". Learned weights (hist / pair / degree) move from "
                 + "{:+.2f}/{:+.2f}/{:+.2f} (snapshot) to {:+.2f}/{:+.2f}/{:+.2f} (final incl. own edges)".format(
                     *b12_w[b12_w.variant == "snapshot"][["w_hist", "w_pair", "w_deg"]].mean(),
                     *b12_w[b12_w.variant == "final_with_target"][["w_hist", "w_pair", "w_deg"]].mean())
                 + f". Training-input shift: train positives' in-Congress pair count with the sponsor is {b12_shift['pair_pos_final']:.2f} in "
                 f"the final graph vs {b12_shift['pair_pos_snap']:.2f} at intro; mean candidate degree {b12_shift['deg_final']:.0f} vs "
                 f"{b12_shift['deg_snap']:.0f}. Train Congresses expose only {B10_TRAIN_VIS['hop1']:.1%} of a bill's final 1-hop and "
                 f"{B10_TRAIN_VIS['hop2']:.1%} of its 2-hop edge mass at t0 (B10). Proxy: {B12_N} sampled bills per Congress, 3-feature "
                 "logistic ranker; a GNN aggregates the same in-Congress structure, so the shift carries over. Rule for every split: "
                 "replay time for training examples (as-of snapshot per example, or temporal edge masking) and drop target edges from "
                 "the message-passing graph (standard link-prediction edge split), exactly as at val / test."))

pool   train  eval  train_bills  val_bills  R20_snapshot  R20_final_minus_target  R20_final_with_target  R20_hist_only
 P11 113-115   116         3000       1000        0.2990                  0.3140                 0.2830         0.2610
 P13 114-116   117         3000       1000        0.3100                  0.3080                 0.2900         0.2690
d_final_minus_target (vs snapshot-trained, 95% CI): P11 +0.015 [-0.001, +0.030]; P13 -0.001 [-0.017, +0.014]; pooled +0.007 [-0.005, +0.017]
d_final_with_target (vs snapshot-trained, 95% CI): P11 -0.016 [-0.038, +0.004]; P13 -0.020 [-0.041, +0.000]; pooled -0.018 [-0.031, -0.003]
d_hist_only (vs snapshot-trained, 95% CI): P11 -0.037 [-0.053, -0.022]; P13 -0.041 [-0.055, -0.027]; pooled -0.039 [-0.050, -0.028]
weights hist / in-Congress pair / degree (mean over folds): final_with_target -0.11/+1.54/+0.11; final_minus_target +0.30/+0.61/+0.45; snapshot +0.53/+0.43/-0.06; hist_only +0.67/+nan/+nan. Train positives' in-Congress pair count:

[FAIL/REQ] B12: Training inputs match eval inputs: train examples from as-of-(intro + t0) snapshots, target edges excluded from message passing | metric=-0.0198 (threshold FAIL (= the rule is binding) if training on final-graph features lowers val Recall@20 vs snapshot training (paired 95% CI < 0) in any fold or pooled)


### B11. Node identity across Congresses: chamber and party switchers

**Why it matters.** A legislator keeps one node id across Congresses. When a House member moves to
the Senate, or a member changes party, that node's training neighbourhood comes from the *other*
chamber (whose bills they can no longer cosponsor in the same way) or from the other party's
coalition, while Task 1 ranks only same-chamber legislators and party homophily is strong (B8). A
learned embedding (or sponsor-history feature) then describes the wrong role. If such nodes carry a
visible share of test positives and are ranked much worse, the model needs chamber-specific node
features (or a chamber-switch reset) and the split's test score depends on how many switchers that
Congress happens to have. **How to read it.** Per pool, Task-1 positives (t0 = 0) are classed by
their cosponsor's status in their **most recent train Congress**: `cold` (never a member in train),
`chamber_switch` (served only in the other chamber then), `party_switch` (same chamber, different
`party_code_asof`), `stable`. `other_ch_edges` = for chamber switchers, share of their train
cosponsorships on bills of the other chamber. `R@20` = share of positives of each class that a
`sponsor_hist` ranker (train counts of cosponsoring this sponsor, popularity tie-break, point-in-time
same-chamber pool incl. delegates, sponsor and t0 cosponsors excluded; section D's baseline,
re-implemented per positive) places in the top 20. Computed on the S3 folds and all pools evaluating
116-118. Impact = switcher share x (R@20 stable - R@20 switcher) = the Recall@20 lost overall.

In [128]:
# Classify Task-1 positives by the cosponsor's chamber / party in their latest train Congress; sponsor_hist Recall@20 per class.
_mem1 = mem.sort_values("congress").drop_duplicates(["bioguide_id", "congress", "chamber"])
_party_c = mem.drop_duplicates(["bioguide_id", "congress"], keep="last").set_index(["bioguide_id", "congress"])["party_code_asof"]
_lb11 = bills.set_index("bill_id")[["chamber"]]
_pe11 = B_POS[0][["bill_id", "bioguide_id", "sponsor_bioguide_id", "congress", "introduced_date", "chamber"]].copy()
_pe11["month"] = _pe11.introduced_date.dt.to_period("M").dt.to_timestamp()
_ros11 = ROSTER_ASOF_ALL[["month", "chamber", "bioguide_id"]]
_seen11 = cosp_c.loc[cosp_c.law_capable & (cosp_c.lag_days <= 0), ["bill_id", "bioguide_id", "congress"]]
B11_CLASSES = ["stable", "party_switch", "chamber_switch", "cold"]


def _b11_classify(pool, c):
    """Positives of Congress c with their class relative to the latest train Congress in `pool`."""
    pe = _pe11[_pe11.congress == c].copy()
    mt = _mem1[_mem1.congress.isin(pool)]
    last = mt.groupby("bioguide_id")["congress"].max()
    pe["last"] = pe.bioguide_id.map(last)
    lastch = mt[mt.congress == mt.bioguide_id.map(last)].groupby("bioguide_id")["chamber"].agg(set)
    same_ch = np.array([isinstance(st, set) and ch in st for st, ch in zip(pe.bioguide_id.map(lastch), pe.chamber)])
    p_now = pd.Series(list(zip(pe.bioguide_id, pe.congress))).map(_party_c).to_numpy()
    p_then = pd.Series(list(zip(pe.bioguide_id, pe["last"]))).map(_party_c).to_numpy()
    pe["cls"] = np.select([pe["last"].isna(), ~same_ch, (p_now != p_then) & pd.notna(p_now) & pd.notna(p_then)],
                          ["cold", "chamber_switch", "party_switch"], "stable")
    return pe


def _b11_hits(pool, pe):
    """sponsor_hist rank of every positive in its point-in-time same-chamber pool; hit = rank <= 20 after excluding
    the sponsor and cosponsors already present at t0 (ties: average rank)."""
    tr = cosp_c[cosp_c.congress.isin(pool)]
    pop = tr.groupby("bioguide_id").size().rename("pop")
    hist = tr.groupby(["sponsor_bioguide_id", "bioguide_id"]).size().rename("h").reset_index()
    grp = pe[["sponsor_bioguide_id", "chamber", "month"]].drop_duplicates()
    cand = grp.merge(_ros11, on=["month", "chamber"])
    cand = cand[cand.bioguide_id != cand.sponsor_bioguide_id]
    cand = cand.merge(hist, on=["sponsor_bioguide_id", "bioguide_id"], how="left")
    cand["score"] = cand.h.fillna(0) * 1e6 + cand.bioguide_id.map(pop).fillna(0)
    key = ["sponsor_bioguide_id", "chamber", "month"]
    cand["rank"] = cand.groupby(key)["score"].rank(ascending=False, method="average")
    p = pe.merge(cand[key + ["bioguide_id", "score", "rank"]], on=key + ["bioguide_id"], how="left")
    in_pool = p["rank"].notna()
    # cosponsors present at t0 are excluded from the ranking: subtract those scored above the positive
    sn = _seen11[_seen11.bill_id.isin(p.bill_id)].merge(pe[["bill_id"] + key].drop_duplicates("bill_id"), on="bill_id")
    sn = sn.merge(cand[key + ["bioguide_id", "score"]], on=key + ["bioguide_id"]).rename(columns={"score": "s_seen"})
    m = p.loc[in_pool, ["bill_id", "score"]].reset_index().merge(sn[["bill_id", "s_seen"]], on="bill_id")
    above = m[m.s_seen > m.score].groupby("index").size()
    p["adj_rank"] = p["rank"] - p.index.map(above).fillna(0).to_numpy()
    p["hit"] = (p.adj_rank <= 20).where(in_pool)
    return p


_b11_pools = B_POOLS[B_POOLS.pool.isin(B_UNITS.loc[B_UNITS.split == "S3", "pool"]) | B_POOLS.c.isin(B_EVAL)]
_r11, _hits11 = [], []
for pr in _b11_pools.itertuples():
    pool, c = list(pr.train), pr.c
    pe = _b11_hits(pool, _b11_classify(pool, c))
    tr = cosp_c[cosp_c.congress.isin(pool)]
    sw = pe.loc[pe.cls == "chamber_switch", ["bioguide_id", "chamber"]].drop_duplicates()
    t_sw = tr[tr.bioguide_id.isin(sw.bioguide_id)].merge(sw.rename(columns={"chamber": "eval_ch"}), on="bioguide_id")
    r = dict(pool=pr.pool, train=pr.train_label, eval=c, n_pos=len(pe), in_pool=float(pe.hit.notna().mean()),
             other_ch_edges=float((t_sw.chamber != t_sw.eval_ch).mean()) if len(t_sw) else np.nan,
             n_switch_leg=int(pe.loc[pe.cls.isin(["chamber_switch", "party_switch"]), "bioguide_id"].nunique()))
    for k in B11_CLASSES:
        sel = pe.cls == k
        r[f"share_{k}"] = float(sel.mean())
        r[f"R20_{k}"] = float(pe.loc[sel, "hit"].mean()) if pe.loc[sel, "hit"].notna().any() else np.nan
    r["share_switch"] = r["share_chamber_switch"] + r["share_party_switch"]
    hs = pe.loc[pe.cls.isin(["chamber_switch", "party_switch"]), "hit"].dropna()
    r["R20_switch"] = float(hs.mean()) if len(hs) else np.nan
    r["n_switch_pos"] = int(len(hs))
    r["impact"] = r["share_switch"] * max(0.0, r["R20_stable"] - r["R20_switch"]) if len(hs) else 0.0
    _r11.append(r)
    _hits11.append(pe[["congress", "cls", "hit"]].assign(pool=pr.pool))
b11 = pd.DataFrame(_r11)
b_show(b11, ["share_chamber_switch", "share_party_switch", "share_cold", "other_ch_edges", "R20_stable",
             "R20_switch", "R20_cold", "n_switch_pos", "impact"])
_s3p = b11[b11.pool.isin(B_UNITS.loc[B_UNITS.split == "S3", "pool"])]
print("S3 folds, all eval Congresses: switcher share of positives " + ", ".join(f"{r['eval']} {r.share_switch:.2%}" for _, r in _s3p.iterrows())
      + f"; positives in the point-in-time pool >= {b11.in_pool.min():.1%}")

# Pooled over S3 folds: hit rate per class, reference folds (<= 115) vs candidate folds (116-118).
_h = pd.concat(_hits11)
_h = _h[_h.pool.isin(_s3p.pool)].dropna(subset=["hit"])
_h["period"] = np.where(_h.congress.isin(B_EVAL), "eval 116-118", f"eval {_s3p['eval'].min()}-115")
ch11 = _h.groupby(["cls", "period"]).agg(recall20=("hit", "mean"), n=("hit", "size")).reset_index()
print("Pooled S3 R@20 by class: " + "; ".join(f"{r.cls} {r.period} {r.recall20:.3f} (n={r.n:,})" for r in ch11.itertuples()))
bar_chart(ch11.assign(cls=pd.Categorical(ch11.cls, B11_CLASSES, ordered=True)).sort_values(["cls", "period"])
          .assign(cls=lambda d: d.cls.astype(str)),
          "cls", "recall20", series="period", yfmt=".0%",
          title="sponsor_hist Recall@20 of Task-1 positives by cosponsor's node status vs the latest train Congress",
          subtitle="S3 folds, t0 = 0; chamber / party switchers keep one node id whose train history describes another role",
          ylabel="share of positives in top 20", name="B_switcher_recall")

_c11 = b11[b11["eval"].isin(B_EVAL)]
_u11 = B_UNITS.merge(_c11[["pool", "share_switch", "impact"]], on="pool")
b11_share = _u11.groupby("split", sort=False)["share_switch"].max().reindex(B_UNIT_SPLITS)
b11_imp = _u11.groupby("split", sort=False)["impact"].max().reindex(B_UNIT_SPLITS)
_ok11 = lambda sh, im: None if pd.isna(sh) else bool(sh < 0.02 or im < 0.005)
b11_ok = {k: _ok11(b11_share[k], b11_imp[k]) for k in B_UNIT_SPLITS}
_pool_sw = _h[_h.cls.isin(["chamber_switch", "party_switch"])].hit
_pool_st = _h[_h.cls == "stable"].hit
b_record("B", "B11", "Chamber / party switchers (one node id, train neighbourhood from another chamber or party) are a negligible or harmless slice of val/test positives",
             bool(all(v for v in b11_ok.values() if v is not None)), metric=round(float(b11_share.max()), 4),
             threshold="per unit evaluating 116-118: switcher share of positives < 0.02, OR Recall@20 lost = share x (R@20 stable - switcher) < 0.005",
             quantity="node_identity_switch", by_split=b_split_dict(b11_ok),
             detail=("Max switcher share of Task-1 positives (t0=0) per split (units evaluating 116-118): " + b_fmt_split(b11_share, ".2%")
                     + "; max Recall@20 lost: " + b_fmt_split(b11_imp, ".4f")
                     + f". Pooled S3 folds: sponsor_hist R@20 switchers {_pool_sw.mean():.3f} (n={len(_pool_sw):,}) vs stable "
                     f"{_pool_st.mean():.3f} (n={len(_pool_st):,}); chamber switchers' train cosponsorships are on other-chamber bills "
                     f"in {b11.other_ch_edges.min():.0%}-{b11.other_ch_edges.max():.0%} of cases. 2% share bound as in B1b (at most a 2% "
                     "Recall loss even if every switcher positive were missed); 0.005 = half a Recall@20 point. Per-positive hit "
                     "rate (D reports per-bill Recall@20; same ranker, re-implemented here). Remedy if it fails: chamber-specific "
                     "node features (chamber and party as of the eval Congress, not of the train Congress) or reset the "
                     "learned embedding on a chamber change; also report metrics with switchers excluded. +stream variants copy "
                     "the base verdict (in-Congress history would partly fix it). Cross-reference: D25 measures the same slice "
                     "(quantity node_identity_switch) with a learned SVD embedding and a difference-in-differences against this "
                     "count ranker (D runs later in the notebook, so its numbers are not quoted here). If D25 FAILs while B11 passes, "
                     "both hold: the slice is small (bounded Recall@20 impact, judged here per split), but a transductive "
                     "embedding goes stale on it. The decision section sees one node_identity_switch row with a conflict flag; "
                     "read it as 'split-neutral, model-relevant': for a GNN with learned IDs apply the remedy above."))

pool   train  eval  share_chamber_switch  share_party_switch  share_cold  other_ch_edges  R20_stable  R20_switch  R20_cold  n_switch_pos  impact
 P10 108-115   116                0.0080              0.0000      0.1800          1.0000      0.2350      0.0780    0.0000           293  0.0010
 P11 113-115   116                0.0080              0.0000      0.1840          1.0000      0.2510      0.0780    0.0000           293  0.0010
 P12 108-116   117                0.0030              0.0050      0.1270          1.0000      0.2260      0.0410    0.0000           243  0.0010
 P13 114-116   117                0.0030              0.0050      0.1270          1.0000      0.2370      0.0330    0.0000           243  0.0020
 P14 108-117   118                0.0060              0.0010      0.1810          1.0000      0.2450      0.0710    0.0000           168  0.0010
 P15 108-116   118                0.0100              0.0040      0.3150          1.0000      0.2250      0.0760    0.0000        

[PASS] B11: Chamber / party switchers (one node id, train neighbourhood from another chamber or party) are a negligible or harmless slice of val/test positives | metric=0.014 (threshold per unit evaluating 116-118: switcher share of positives < 0.02, OR Recall@20 lost = share x (R@20 stable - switcher) < 0.005)


### B summary

All graph and node checks recorded in this section. FAIL does not always mean "do not split by
Congress": read the detail column for the remedy (allow within-Congress history up to t0, refit on
val, inductive features, filter edges by time, restrict subjects to a stable vocabulary, normalise
degree features). Checks tagged with a shared `quantity` are deduplicated in the decision section.
Every split-dependent check carries a `by_split` verdict (base splits, `+stream` variants, S1pool and,
where meaningful, S4). Distribution checks (B3, B8, B9) judge each split on failures in excess of chance
against one fixed 108-115 noise reference, so short sliding pools are not penalised for their size.
Every check keys every split in `ALL_SPLITS`; splits a check does not judge get None with a stated reason
(e.g. S5 random, or base splits in B1b / B1c, whose cold share is B1's INFO and is a protocol choice, not a
failure of the split). B11 and section D's D25 share the `node_identity_switch` key, so the decision
section sees one row and a conflict flag if they disagree.
B5b, B6b, B7, B10 and B12 are recorded as **requirements** (`requirement=True`): graph-construction rules every
split must follow (no subject features at t0, time-filter related links, referral-only committee edges,
time-masked cosponsor graph for eval AND training examples, target edges out of message passing). Each leakage path has its own quantity key, so none is hidden behind
another section's check; count split failures from `checks_summary(requirements=False)`.

In [129]:
# Summary of section B checks.
bsum = pd.DataFrame([c for c in CHECKS if c["section"] == "B"])
bsum["result"] = bsum.passed.map({True: "PASS", False: "FAIL"}).fillna("INFO")
with pd.option_context("display.max_colwidth", 80):
    bsum["req"] = bsum.get("requirement", pd.Series(False, index=bsum.index)).fillna(False).map({True: "req", False: ""})
    print(bsum[["id", "result", "req", "metric", "quantity", "condition"]].to_string(index=False))
_sc = bsum[bsum.req == ""]
print(f"\nB split conditions: {int((_sc.result == 'PASS').sum())} pass, {int((_sc.result == 'FAIL').sum())} fail, "
      f"{int((_sc.result == 'INFO').sum())} info; requirements (all splits): {int((bsum.req == 'req').sum())} "
      f"({int(((bsum.req == 'req') & (bsum.result == 'FAIL')).sum())} need filtering)")

 id result req  metric                quantity                                                                                                                                           condition
 B1   INFO      0.3145            node_overlap                                                  Share of val/test Task-1 positive edges from legislators unseen in train (transductive-only model)
B1b   PASS      0.0071              cold_start                                       With eval-Congress history up to intro + t0 (+stream), few positives involve a legislator with no edge at all
B1c   PASS      0.0145                     NaN                                            Positives from train-cold legislators have a usable in-Congress neighbourhood (>= 5 edges) by intro + t0
 B2   FAIL      0.2172         sponsor_overlap                                            Few val/test bills have a sponsor with no sponsorship history in train, relative to train-to-train noise
 B3   PASS      2.0000   

## C. Temporal structure, regime changes, censoring, leakage

A by-Congress split assumes that each Congress is a comparable "draw" of the same process, and that
whatever we feed the model at prediction time (day 0 or day 7 after introduction) was really known at
that time. This section tests the time-related conditions behind that assumption:

1. **Within-Congress dynamics**: are bills introduced on the same calendar rhythm in every Congress?
2. **Cosponsorship timing**: is the 30-day prediction window equally hard in every Congress?
3. **Procedural breakpoints**: does the original-cosponsor share jump on a date inside a Congress, and does
   each split's fit set contain the post-break regime it is evaluated on?
4. **Censoring**: how many bills lose their label because the window runs past the end of the Congress?
5. **Regime changes**: is each chamber's controlling party in every validation/test Congress represented in
   training, and does the joint House x Senate regime matter at all?
6. **Snapshot leakage**: which bill fields describe the bill *after* introduction and must never be features?
7. **Data latency and label completeness**: is the most recent Congress (118) less complete than older ones, and
   are its labels (cosponsor events, committee reports) fully loaded, or only its annotations?
8. **Member turnover / point-in-time pool**: how much does the full-Congress roster distort task-1 densities
   and Recall@20 compared with members in office at t0?
9. **Feature availability and companion bills**: is the policy area known at t0, and do companion /
   reintroduced bills connect evaluation bills to the training portion of each candidate split?
10. **Task-2 prediction time**: which t0 makes which features legitimate, and how much label is spent by waiting?

"Differs from train" verdicts use the shared `rel_to_train()` rule (95% t prediction interval of the 108-115
reference, with nested leave-one-out false-alarm rates). Pool-dependent checks iterate over the shared
evaluation units `iter_eval_units()` (plus the S1pool variant) and record a `by_split` verdict per candidate
split, so the decision section can score splits without parsing text. Quantities owned by another section
(label rates -> A) are shown here only as cross-references (INFO).
Congress 119 is still in session and is used only as context.

In [130]:
# Shared helpers for this section: train pool, evaluation Congresses, law-capable bill frame.
C_TRAIN = [c for c in CONGRESSES if c <= 115]
C_EVAL = [c for c in CONGRESSES if c >= 116]
C_FMT = lambda v: f"{v:,.0f}" if abs(v) >= 1000 else f"{v:.3f}"   # compact float format for transposed tables
C_SECTION = "C"   # section letter: must match QUANTITY_OWNERS so checks_summary() deduplicates correctly
lb = bills[bills.law_capable & bills.congress.isin(CONGRESSES + [IN_PROGRESS])].copy()
lb["c_start"] = lb["congress"].map(congress_start)
lb["c_end"] = lb["congress"].map(congress_end)
wt = {t0: window_targets(t0) for t0 in (0, 7)}   # task-1 windows at t0 = 0 and 7 (shared by C2, C4, C5, C11)
BY_CONGRESS_SPLITS = [k for k, v in SPLIT_DEFS.items() if v["kind"] in ("expanding", "sliding", "rolling")]
EVAL_UNITS = list(iter_eval_units())
C_BY_CONGRESS_ALL = [k for k in ALL_SPLITS if split_kind(k) in BY_CONGRESS_KINDS]   # incl. +stream / S1pool variants


def unit_masks(unit, frame):
    """(train_mask, eval_mask) over `frame` (needs bill_id, congress, introduced_date) for one eval unit.
    S5 uses the shared draw S5_ROLE, S4 the shared cut dates S4_CUT_DATES (same bills as sections A, B, D)."""
    if unit["split"] == "S5":
        role = frame.bill_id.map(S5_ROLE)
        held = ["val", "test"] if unit["role"] == "val" else ["test"]
        return (frame.congress.isin(CONGRESSES) & ~role.isin(held)).to_numpy(), (role == unit["role"]).to_numpy()
    tr = frame.congress.isin(unit["train"]).to_numpy()
    if unit["within"] is None:
        return tr, (frame.congress == unit["eval_congress"]).to_numpy()
    c, q_tr, q_lo, q_hi = unit["within"]
    qd = S4_CUT_DATES[c]
    inc = frame.congress == c
    tr = tr | (inc & (frame.introduced_date < qd[q_tr])).to_numpy()
    ev = inc & (frame.introduced_date >= qd[q_lo]) & (frame.introduced_date < qd[q_hi])
    return tr, ev.to_numpy()


# S1pool (variant, not in iter_eval_units): val 116 fit on 108-115; test 117 and 118 fit once on 108-116.
_S1P = SPLIT_VARIANTS["S1pool"]
S1POOL_UNITS = ([dict(split="S1pool", fold="pool", role="val", train=list(_S1P["train"]), eval_congress=v,
                      train_label=f"{_S1P['train'][0]}-{_S1P['train'][-1]}", within=None) for v in _S1P["val"]]
                + [dict(split="S1pool", fold="pool", role="test", train=list(_S1P["train"]) + list(_S1P["val"]),
                        eval_congress=t, train_label=f"{_S1P['train'][0]}-{_S1P['val'][-1]}", within=None)
                   for t in _S1P["test"]])


def with_variants(d):
    """Copy each base split's per-split verdict to its '+stream' variants (streaming only adds eval-Congress events
    dated before t0, which does not change censoring, regime or twin-leak verdicts). Keys must be in ALL_SPLITS."""
    out = dict(d)
    for k, v in SPLIT_VARIANTS.items():
        if v.get("stream") and v["base"] in d and k not in out:
            out[k] = d[v["base"]]
    return {k: out[k] for k in ALL_SPLITS if k in out}


def combine_by_split(*ds):
    """AND of several by_split dicts (None = not judged; a split fails if any input fails, None if all are None)."""
    out = {}
    for k in ALL_SPLITS:
        vals = [d.get(k) for d in ds if d is not None and k in d]
        if vals:
            judged = [v for v in vals if v is not None]
            out[k] = all(judged) if judged else None
    return out


S4_EVAL = split_eval_congresses("S4")   # candidate Congresses whose within-Congress slices S4 evaluates
S5_NA = ("S5 draws train and eval bills at random from every Congress 108-118, so a per-Congress drift verdict does not "
         "separate its train from its eval (S5's leakage is judged in C20)")
NO_CAND_NA = "evaluates no candidate Congress 116-118 (reference-Congress verdicts are nested-LOO and not held against splits)"


def cand_ok(*tabs):
    """Per candidate Congress: True if it passes every given rel_to_train tab (rows with role 'cand')."""
    oks = [~t.loc[t.role == "cand", "fail"].astype(bool) for t in tabs]
    return pd.concat(oks, axis=1).all(axis=1)


def full_split(bs, ok=None, s4_na=None, s5_na=S5_NA, extra=None, extra_na=None):
    """Complete a by_split dict over ALL_SPLITS and build its na_reason. ok: per-Congress pass Series (cand_ok) used to
    judge S4 on the candidate Congresses its slices evaluate (S4_EVAL); if None, S4 gets None with reason s4_na.
    extra: explicit verdicts for other splits (with reasons extra_na for None values). Returns (by_split, na_reason)."""
    out, na = dict(bs or {}), {}
    if ok is not None:
        out["S4"] = bool(all(ok[c] for c in S4_EVAL if c in ok.index))
    else:
        out["S4"], na["S4"] = None, s4_na
    out["S5"] = None
    na["S5"] = s5_na
    out.update(extra or {})
    na.update(extra_na or {})
    out = {k: out.get(k) for k in ALL_SPLITS}
    for k, v in out.items():
        if v is None and k not in na:
            na[k] = NO_CAND_NA
    return out, {k: v for k, v in na.items() if out[k] is None}


print(f"law-capable bills 108-119: {len(lb):,}; train pool {C_TRAIN[0]}-{C_TRAIN[-1]}, "
      f"evaluation candidates {C_EVAL}")

law-capable bills 108-119: 145,738; train pool 108-115, evaluation candidates [116, 117, 118]


### C1-C2. When are bills introduced inside a Congress?

**Why it matters.** Bills introduced in the first weeks of a Congress (reintroductions, leadership
priorities) and bills dropped in the final months (messaging bills, the end-of-Congress rush) behave
differently: early bills have two years to gather cosponsors, late bills are mostly censored. If the
introduction-time profile differs between train and test Congresses, a model learns a mix that the test
set does not have. It also tells us whether a test set must be a **full Congress**: a partial window
(for example session 1 only, as in S4) has a very different bill mix from a full Congress.

**How to read it.** The heatmap shows the share of each Congress's bills introduced in each quarter of
the Congress (Q1 = first three months). The table gives the session-1 share, the share introduced in the
last 90 days, and the KS distance of `day_of_congress` versus the 108-115 pool (KS is an effect size here:
with ~10k bills every p-value is tiny, so we threshold the statistic, not the p-value).

In [131]:
# C1/C2: distribution of introduction day_of_congress per Congress; session-1 share; last-90-day rush; KS vs train pool.
lb["quarter"] = (lb["day_of_congress"] // 91.3125).clip(0, 7).astype(int) + 1
lb["session1"] = lb["introduced_date"] < lb["c_start"] + pd.DateOffset(years=1)
lb["rush90"] = lb["introduced_date"] >= lb["c_end"] - pd.Timedelta(days=90)
train_days = lb.loc[lb.congress.isin(C_TRAIN), "day_of_congress"].values
rows = []
for c, g in lb[lb.congress.isin(CONGRESSES)].groupby("congress"):
    pool_days = lb.loc[lb.congress.isin(C_TRAIN) & (lb.congress != c), "day_of_congress"].values
    rows.append(dict(congress=c, n_bills=len(g), session1=g.session1.mean(), last90=g.rush90.mean(),
                     median_day=g.day_of_congress.median(), ks_vs_train=ks(pool_days, g.day_of_congress)[0]))
intro = pd.DataFrame(rows).set_index("congress")
print("Introduction timing per Congress (law-capable bills; KS vs 108-115 pool excluding the row's own Congress)")
print(intro.T.to_string(float_format=C_FMT))

qshare = (lb[lb.congress.isin(CONGRESSES)].groupby(["congress", "quarter"]).size()
          / lb[lb.congress.isin(CONGRESSES)].groupby("congress").size()).rename("share").reset_index()
qshare["quarter"] = "Q" + qshare["quarter"].astype(str)
qshare["congress"] = qshare["congress"].astype(str)
heatmap(qshare, "quarter", "congress", "share", fmt=".0%",
        title="Share of each Congress's law-capable bills introduced per quarter of the Congress",
        subtitle="Q1 = first 3 months, Q8 = last 3 months; rows are Congresses", height=420,
        name="C_intro_quarter_heatmap")

# Verdict: shared rel_to_train rule (train rows = LOO KS to the other 108-115 Congresses; upper tail only).
ks_tab, ks_info = rel_to_train(intro["ks_vs_train"], side="upper")
C12_S4_NA = ("S4 evaluates a fixed slice of each candidate Congress (its last 40% of introductions: val 60-80%, test >= 80%), so its introduction-time mix "
             "differs from the full Congress by construction; the whole-Congress timing profile does not describe it (C2's "
             "session-2 contrast and C6b cover S4)")
ks_eval = intro.loc[C_EVAL, "ks_vs_train"]
record_check(C_SECTION, "C1", "Introduction-day distribution (KS vs 108-115) of every eval Congress within train-to-train noise",
             not bool(ks_tab.loc[ks_tab.role == "cand", "fail"].any()), metric=float(ks_eval.max()), threshold=ks_info["rule"],
             detail=rel_detail(ks_tab, ks_info) + ". by_split judges only the candidate Congresses 116-118 each split evaluates "
                    f"(shared 'cands' mode; reference outliers such as 112, KS {intro.loc[112, 'ks_vs_train']:.3f} with a short session 1, are not "
                    "held against S2/S3). Secondary: max eval KS {ks_eval.max():.3f} vs conventional small-effect "
                    "0.1. p-values of KS itself are meaningless at this n, so the statistic is judged against its own LOO spread.",
             **dict(zip(("by_split", "na_reason"), full_split(rel_by_split(ks_tab), s4_na=C12_S4_NA))))
s1_train, r_train = intro.loc[C_TRAIN, "session1"].mean(), intro.loc[C_TRAIN, "last90"].mean()
# Session-1 vs session-2 bills (train pool, uncensored, t0=7): how different is the bill mix a within-Congress split tests on?
w7 = wt[7][~wt[7].censored & wt[7].congress.isin(C_TRAIN) & (wt[7].eligible > 0)].merge(lb[["bill_id", "session1"]], on="bill_id")
w7["rate"] = w7.n_pos / w7.eligible
sess = w7.groupby("session1").agg(bills=("bill_id", "size"), pos_rate=("rate", "mean"), any_pos=("n_pos", lambda v: (v > 0).mean()))
sess.index = sess.index.map({True: "session 1", False: "session 2"})
print("Train pool 108-115, task 1 at t0=7: session-1 vs session-2 bills")
print(sess.round(4).to_string())
sess_rel = float(sess.loc["session 2", "pos_rate"] / sess.loc["session 1", "pos_rate"] - 1)
# Verdict: shared rel_to_train rule on both shares (two-sided: a rush that shrinks matters as much as one that grows).
s1_tab, s1_info = rel_to_train(intro["session1"], side="two")
r9_tab, r9_info = rel_to_train(intro["last90"], side="two")
c2_fail = bool(s1_tab.loc[s1_tab.role == "cand", "fail"].any() or r9_tab.loc[r9_tab.role == "cand", "fail"].any())
c2_t = float(max(s1_tab.loc[s1_tab.role == "cand", "t_pi"].abs().max(), r9_tab.loc[r9_tab.role == "cand", "t_pi"].abs().max()))
record_check(C_SECTION, "C2", "Session-1 share and end-of-Congress (last 90 d) share of eval Congresses within train-to-train noise",
             not c2_fail, metric=c2_t, threshold=f"{s1_info['rule']}, for either share (metric = max |t| over candidates and both shares)",
             detail=f"session-1 share: {rel_detail(s1_tab, s1_info)}. Last-90-day share: {rel_detail(r9_tab, r9_info)}. Train mean session-1 "
                    f"share {s1_train:.3f}, last-90-day share {r_train:.3f}. Session-2 bills have a task-1 positive rate {abs(sess_rel):.0%} "
                    f"{'lower' if sess_rel < 0 else 'higher'} than session-1 bills (t0=7, train pool), so a within-Congress split (S4) that trains "
                    "on session 1 and tests on session 2 tests a structurally different bill mix; use full Congresses as test units.",
             **dict(zip(("by_split", "na_reason"), full_split(combine_by_split(rel_by_split(s1_tab), rel_by_split(r9_tab)),
                                                              s4_na=C12_S4_NA))))

Introduction timing per Congress (law-capable bills; KS vs 108-115 pool excluding the row's own Congress)
congress        108     109     110     111     112     113     114     115     116     117     118
n_bills       8,621  10,697  11,224  10,770  10,612   9,083  10,223  11,414  14,340  15,242  16,565
session1      0.674   0.647   0.673   0.688   0.558   0.641   0.669   0.627   0.615   0.648   0.641
last90        0.044   0.020   0.012   0.029   0.024   0.036   0.033   0.056   0.060   0.058   0.061
median_day  199.000 246.000 251.000 262.500 306.500 211.000 201.000 253.000 266.000 214.000 252.000
ks_vs_train   0.075   0.046   0.057   0.060   0.123   0.030   0.058   0.038   0.062   0.051   0.060


[PASS] C1: Introduction-day distribution (KS vs 108-115) of every eval Congress within train-to-train noise | metric=0.06177 (threshold outside 95% t prediction interval of ref 108-115 (upper tail; n=8))
Train pool 108-115, task 1 at t0=7: session-1 vs session-2 bills
           bills  pos_rate  any_pos
session1                           
session 2  27794    0.0051   0.2431
session 1  53431    0.0074   0.3324
[PASS] C2: Session-1 share and end-of-Congress (last 90 d) share of eval Congresses within train-to-train noise | metric=1.959 (threshold outside 95% t prediction interval of ref 108-115 (two-sided; n=8), for either share (metric = max |t| over candidates and both shares))


### C3-C4. Cosponsorship timing and task difficulty

**Why it matters.** Task 1 predicts cosponsors who join in *(t0, t0+30]*. Beyond *how many* cosponsors a
bill gets (label rate, owned by section A: A3/A4), what matters for the window is *when* the later
cosponsors arrive: if in some Congresses non-original cosponsors trickle in over months while in others
they join in the first weeks, the same 30-day window captures a different slice of the cosponsor list.

**How to read it.** The line chart shows, per Congress, the cumulative share of cosponsorships that are
original (lag 0) and that arrive within 7, 30 and 37 days (37 = end of the t0=7 window). C3 tests the
**timing of non-original cosponsorships** (lag > 0) with a KS distance against the 108-115 pool, and judges
it against the leave-one-out spread among the train Congresses themselves (a FAIL means "more different than
train-to-train noise", not just "different"). C4 (INFO) is a cross-reference to section A's label rate.

In [132]:
# C3: lag distribution per Congress (law-capable, active cosponsorships); KS of non-original lags vs train pool (LOO for train rows).
lc = cosp[cosp.law_capable & cosp.congress.isin(CONGRESSES)][["congress", "lag_days", "is_original"]]
lag = lc.groupby("congress")["lag_days"].agg(
    n="size", original=lambda s: (s == 0).mean(), neg=lambda s: (s < 0).mean(),
    within7=lambda s: (s <= 7).mean(), within30=lambda s: (s <= 30).mean(), within37=lambda s: (s <= 37).mean(),
    median_lag=lambda s: s[s > 0].median())
late = lc[lc.lag_days > 0]
# share of non-original cosponsorships arriving inside the t0=7 window (8..37 d): the part of the timing the task sees
lag["late_in_t7_window"] = late.groupby("congress")["lag_days"].agg(lambda s: ((s > 7) & (s <= 37)).mean())
lag["ks_late_vs_train"] = [ks(late.loc[late.congress.isin(C_TRAIN) & (late.congress != c), "lag_days"].values,
                              late.loc[late.congress == c, "lag_days"].values)[0] for c in lag.index]
print("Cosponsorship lag per Congress (median_lag and KS over non-original cosponsorships, lag > 0; "
      "neg = date before introduction; train rows use leave-one-out pools)")
print(lag.drop(columns=["n", "neg"]).round(3).to_string())
ll = lag[["original", "within7", "within30", "within37"]].reset_index().melt("congress", var_name="share_of", value_name="share")
ll["share_of"] = ll["share_of"].map({"original": "original (lag 0)", "within7": "<= 7 d", "within30": "<= 30 d", "within37": "<= 37 d"})
line_chart(ll, "congress", "share", series="share_of", yfmt=".0%", yzero=False,
           title="Cumulative share of cosponsorships by lag after introduction, per Congress",
           subtitle="Law-capable bills, active (non-withdrawn) cosponsorships", xlabel="Congress",
           name="C_lag_shares")
ks_e, ks_t = lag.loc[C_EVAL, "ks_late_vs_train"], lag.loc[C_TRAIN, "ks_late_vs_train"]
lag_tab, lag_info = rel_to_train(lag["ks_late_vs_train"], side="upper")
band = float(ks_t.max())
record_check(C_SECTION, "C3", "Timing of non-original cosponsorships (lag > 0) in eval Congresses within train-to-train variation",
             not bool(lag_tab.loc[lag_tab.role == "cand", "fail"].any()), metric=float(ks_e.max()),
             threshold=lag_info["rule"], **dict(zip(("by_split", "na_reason"), full_split(rel_by_split(lag_tab), ok=cand_ok(lag_tab)))),
             detail=rel_detail(lag_tab, lag_info) + f". Secondary (old rule): max train LOO KS {band:.3f}; train leave-one-out range "
                    f"{ks_t.min():.3f}-{ks_t.max():.3f} (the noise band; 0.1 is the conventional 'small effect' KS, so a value between "
                    f"the band and 0.1 is a real but modest shift). Worst eval Congress: {int(ks_e.idxmax())} (median non-original lag "
                    f"{lag.loc[int(ks_e.idxmax()), 'median_lag']:.0f} d vs train {lag.loc[C_TRAIN, 'median_lag'].median():.0f} d). Share of late cosponsorships landing in the t0=7 window (8-37 d): train "
                    f"mean {lag.loc[C_TRAIN, 'late_in_t7_window'].mean():.3f}, eval {lag.loc[C_EVAL, 'late_in_t7_window'].round(3).to_dict()}. "
                    f"Original share (lag 0) is owned by A4 and not part of this verdict (train mean {lag.loc[C_TRAIN, 'original'].mean():.3f}, "
                    f"eval {lag.loc[C_EVAL, 'original'].round(3).to_dict()}). Negative-lag rows {lag['neg'].mean():.2%} (data errors, "
                    "excluded from windows). If FAIL: the window catches a different part of the cosponsor arrival curve in eval "
                    "Congresses (118 only; C23 tests whether its lag shift is a within-Congress break); use recency weighting or a post-break indicator and "
                    "report per-Congress results rather than relying on one test Congress. S4 is judged on the same candidate "
                    f"Congresses its slices test ({S4_EVAL}): the lag shift belongs to the Congress's bills, so S4's test slice of a failing "
                    "Congress inherits it (its fit set holds only the earlier part of that Congress, whose late cosponsorships are "
                    "partly censored at the cut).",
             quantity="lag_dist")

Cosponsorship lag per Congress (median_lag and KS over non-original cosponsorships, lag > 0; neg = date before introduction; train rows use leave-one-out pools)
          original  within7  within30  within37  median_lag  late_in_t7_window  ks_late_vs_train
congress                                                                                        
108         0.3890   0.4450    0.5780    0.6090     63.0000             0.2690            0.0360
109         0.3640   0.4300    0.5680    0.5980     63.0000             0.2650            0.0270
110         0.3300   0.4000    0.5440    0.5780     62.0000             0.2650            0.0260
111         0.3410   0.4120    0.5480    0.5780     65.0000             0.2520            0.0140
112         0.3310   0.4030    0.5360    0.5630     70.0000             0.2390            0.0300
113         0.3300   0.4050    0.5500    0.5790     63.0000             0.2600            0.0200
114         0.3590   0.4330    0.5550    0.5790     74.0000    

[FAIL] C3: Timing of non-original cosponsorships (lag > 0) in eval Congresses within train-to-train variation | metric=0.09342 (threshold outside 95% t prediction interval of ref 108-115 (upper tail; n=8)) [fails for: S1b, S2, S3, S4, S1b+stream, S3+stream, S1pool]


In [133]:
# C4 (INFO, cross-reference to A3): task-1 positive rate per eligible legislator per Congress (uncensored law-capable bills).
diff_rows = []
for t0, w in wt.items():
    u = w[~w.censored & w.congress.isin(CONGRESSES) & (w.eligible > 0)].copy()
    u["rate"] = u["n_pos"] / u["eligible"]
    g = u.groupby("congress").agg(bills=("bill_id", "size"), pos_rate=("rate", "mean"),
                                  any_pos=("n_pos", lambda s: (s > 0).mean()), mean_pos=("n_pos", "mean"))
    g["t0"] = t0
    diff_rows.append(g.reset_index())
difficulty = pd.concat(diff_rows)
tab = difficulty.pivot(index="congress", columns="t0", values=["pos_rate", "any_pos", "mean_pos"])
tab.columns = [f"{a}_t{b}" for a, b in tab.columns]
print("Task-1 difficulty per Congress (pos_rate = mean positives / eligible; any_pos = share of bills with >= 1 positive)")
print(tab[["pos_rate_t0", "pos_rate_t7", "any_pos_t7"]].round(4).T.to_string())
rel = {}
for t0 in (0, 7):
    d = difficulty[difficulty.t0 == t0].set_index("congress")["pos_rate"]
    rel[t0] = ((d.loc[C_EVAL] - d.loc[C_TRAIN].mean()) / d.loc[C_TRAIN].mean())
worst = max(rel[0].abs().max(), rel[7].abs().max())
record_check(C_SECTION, "C4", "Task-1 positive rate per eligible legislator, eval vs train (cross-reference; verdict owned by A3)", None,
             metric=float(worst), threshold="INFO (A3 owns the verdict)",
             detail=f"max relative diff vs 108-115 mean; t0=0: {rel[0].round(3).to_dict()}; t0=7: {rel[7].round(3).to_dict()}. "
                    "Shown for the timing context only; not counted separately in the decision.",
             quantity="cosp_density")

Task-1 difficulty per Congress (pos_rate = mean positives / eligible; any_pos = share of bills with >= 1 positive)
congress       108    109    110    111    112    113    114    115    116    117    118
pos_rate_t0 0.0094 0.0079 0.0091 0.0078 0.0074 0.0098 0.0085 0.0074 0.0084 0.0066 0.0047
pos_rate_t7 0.0078 0.0064 0.0074 0.0062 0.0056 0.0075 0.0065 0.0059 0.0066 0.0049 0.0035
any_pos_t7  0.3092 0.2610 0.3086 0.2980 0.2723 0.3348 0.3221 0.3146 0.3291 0.2984 0.2728
[INFO] C4: Task-1 positive rate per eligible legislator, eval vs train (cross-reference; verdict owned by A3) | metric=0.4746 (threshold INFO (A3 owns the verdict))


### C23. Procedural breakpoints inside a Congress (original-cosponsor share and timing)

**Why it matters.** The jump in the original-cosponsor share in 117-118 (table above) may come from a
procedural change (for example electronic cosponsor submission and remote operations in the House in 2020-2021)
rather than from gradual drift. A procedural change is a *break*: it happens on a date, not at a Congress
boundary. If the break falls inside a Congress, that Congress is a mix of two regimes, a validation Congress
can straddle it, and a fit set that ends before it has never seen the new regime. That decides whether a
by-Congress cut is enough or whether a within-Congress cut (or dropping pre-break data) is needed.

**How to read it.** Monthly series by introduction month (law-capable bills, uncensored at t0 = 0): share of
30-day cosponsors that are original, and the median lag of non-original cosponsors joining within 180 days (bills
with a full 180-day window). Each series is de-seasonalised by its month-of-Congress mean (to remove the
start/end-of-Congress cycle). A constant-mean CUSUM ("raw") would call any steady trend a break, because block
permutation destroys trends; the verdict therefore uses the largest level shift BEYOND a linear trend (trend and
step fitted jointly, segments >= 6 months; null = linear trend + 2,000 permutations of 6-month residual blocks,
approximate). C23b then asks whether a split has more test folds on a never-seen post-break regime than
deployment itself would. The chart shows the raw monthly series with Congress boundaries and the trend-adjusted break.

In [134]:
# C23: monthly original share / median lag per chamber, de-seasonalised; largest mean shift with block-permutation p-value.
mc = cosp[cosp.law_capable & cosp.congress.isin(CONGRESSES) & (cosp.lag_days >= 0)][
    ["bill_id", "congress", "chamber", "introduced_date", "lag_days"]].copy()
mc["end"] = mc.congress.map(congress_end)
mc["month"] = mc.introduced_date.dt.to_period("M").dt.to_timestamp()
mc["moc"] = ((mc.month.dt.year - mc.congress.map(congress_start).dt.year) * 12 + mc.month.dt.month - 1).clip(0, 23)
w30 = mc[(mc.lag_days <= 30) & (mc.introduced_date + pd.Timedelta(days=30) <= mc.end)]
l180 = mc[(mc.lag_days > 0) & (mc.lag_days <= 180) & (mc.introduced_date + pd.Timedelta(days=180) <= mc.end)]
MIN_N = 200   # months with fewer cosponsorships are dropped (noisy)


def cusum_break(x, minseg=6, n_perm=2000, block=6, seed=23):
    """Largest standardised mean shift and its block-permutation p-value."""
    def stat(v):
        n = v.shape[-1]
        k = np.arange(minseg, n - minseg + 1)
        cs = np.cumsum(v, axis=-1)
        m1 = cs[..., k - 1] / k
        m2 = (cs[..., -1:] - cs[..., k - 1]) / (n - k)
        z = np.abs(m1 - m2) * np.sqrt(k * (n - k) / n) / v.std(axis=-1, keepdims=True)
        return z.max(-1), k[z.argmax(-1)]
    s_obs, k_obs = stat(x)
    nb = int(np.ceil(len(x) / block))
    blocks = np.array_split(np.arange(len(x)), nb)
    rng = np.random.default_rng(seed)
    perms = np.stack([x[np.concatenate([blocks[i] for i in rng.permutation(nb)])] for _ in range(n_perm)])
    s_null, _ = stat(perms)
    return float(s_obs), int(k_obs), float((s_null >= s_obs).mean())


def trend_break(x, t, minseg=6, n_perm=2000, block=6, seed=23):
    """Largest level shift BEYOND a linear trend: max |t| of the step coefficient in y ~ 1 + t + 1(i >= k) over k
    (trend and step estimated jointly, so a steady rise is not a break and a late break does not bend the trend).
    Null: linear trend over the whole series + block-permuted residuals. Returns (stat, k, p, slope_per_unit)."""
    n = len(x)
    Z = np.column_stack([np.ones(n), t])
    M = np.eye(n) - Z @ np.linalg.pinv(Z)                          # residual maker for [1, t]
    ks_ = np.arange(minseg, n - minseg + 1)
    S = (np.arange(n)[None, :] >= ks_[:, None]).astype(float) @ M  # residualised step regressors (K x n)
    ss = (S ** 2).sum(1)

    def stat(Y):                                                   # Y: (P x n) already residualised on [1, t]
        b = Y @ S.T / ss
        s2 = ((Y ** 2).sum(1, keepdims=True) - b ** 2 * ss) / (n - 3)
        tt = np.abs(b) * np.sqrt(ss) / np.sqrt(np.maximum(s2, 1e-12))
        return tt.max(1), ks_[tt.argmax(1)]
    e = M @ x
    s_obs, k_obs = stat(e[None, :])
    nb = int(np.ceil(n / block))
    blocks = np.array_split(np.arange(n), nb)
    rng = np.random.default_rng(seed)
    perms = np.stack([e[np.concatenate([blocks[i] for i in rng.permutation(nb)])] for _ in range(n_perm)]) @ M.T
    s_null, _ = stat(perms)
    slope = float(np.linalg.lstsq(Z, x, rcond=None)[0][1])
    return float(s_obs[0]), int(k_obs[0]), float((s_null >= s_obs[0]).mean()), slope


series_defs = [("orig_share30", w30, lambda s: (s == 0).mean()), ("median_lag180", l180, "median")]
bp_rows, plot_rows = [], []
for name, df, f in series_defs:
    for ch in ("House of Representatives", "Senate"):
        d = df[df.chamber == ch]
        g = d.groupby(["month", "congress", "moc"]).lag_days.agg(n="size", v=f).reset_index()
        g = g[g.n >= MIN_N].sort_values("month").reset_index(drop=True)
        g["v_ds"] = g.v - g.groupby("moc").v.transform("mean")
        # Detrended: level shift beyond a linear trend (trend and step fitted jointly), so a steady rise is not a break.
        tt = (g.month - g.month.min()).dt.days.to_numpy() / 365.25
        stt, k, pval = cusum_break(g.v_ds.to_numpy())
        stt2, k2, pval2, b1 = trend_break(g.v_ds.to_numpy(), tt)
        Z2 = np.column_stack([np.ones(len(g)), tt, (np.arange(len(g)) >= k2).astype(float)])
        g["v_dt"] = g.v_ds - Z2[:, :2] @ np.linalg.lstsq(Z2, g.v_ds.to_numpy(), rcond=None)[0][:2]
        into2 = int(g.moc[k2])
        bp_rows.append(dict(series=name, chamber=ch[:5], months=len(g), raw_break=g.month[k].strftime("%Y-%m"), raw_p=pval,
                            trend_per_yr=b1, break_month=g.month[k2].strftime("%Y-%m"), congress=int(g.congress[k2]),
                            month_of_congress=into2 + 1, before=g.v[:k2].mean(), after=g.v[k2:].mean(),
                            shift_sd=(g.v_dt[k2:].mean() - g.v_dt[:k2].mean()) / g.v_dt[:k2].std(), stat=stt2, p=pval2,
                            inside=bool(3 <= into2 <= 20)))
        if name == "orig_share30":
            plot_rows += [dict(t=round(m.year + (m.month - 0.5) / 12, 3), v=float(v), chamber=ch[:5], congress=int(c))
                          for m, v, c in zip(g.month, g.v, g.congress)]
bp = pd.DataFrame(bp_rows)
print("Largest level shift per monthly series: raw = constant-mean CUSUM on the de-seasonalised series; trend-adjusted (break_month "
      ".. p) = step beyond a jointly fitted linear trend (verdict); before/after = raw means; trend_per_yr = slope without a step")
print(bp.drop(columns=["months", "stat"]).round(3).to_string(index=False))
alpha23 = 0.05 / len(bp)
breaks = [dict(year=round(pd.Timestamp(r.break_month).year + (pd.Timestamp(r.break_month).month - 1) / 12, 3),
               label=f"{r.chamber} break {r.break_month} (p={r.p:.3f}{'' if abs((pd.Period(r.raw_break, 'M') - pd.Period(r.break_month, 'M')).n) <= 6 else ', location not identified'})") for r in bp[bp.series == "orig_share30"].itertuples()]
bounds = [round(congress_start(c).year + 0.0, 3) for c in CONGRESSES[1:]]
show_d3(r"""
const {width, height, breaks, bounds} = opts;
const m = {top: 48, right: 20, bottom: 56, left: 56};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, 'Monthly share of 30-day cosponsors that are original, by chamber (law-capable bills)',
          'Thin grey lines = Congress boundaries; dashed = largest level shift beyond a linear trend (de-seasonalised)');
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const xs = d3.scaleLinear(d3.extent(data, d => d.t), [0, W]);
const ys = d3.scaleLinear(d3.extent(data, d => d.v), [H, 0]).nice();
g.append('g').call(d3.axisLeft(ys).ticks(5).tickSize(-W).tickFormat(d3.format('.0%')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(12).tickFormat(d3.format('d'))).call(viz.axis);
g.selectAll(null).data(bounds).join('line').attr('x1', d => xs(d)).attr('x2', d => xs(d)).attr('y1', 0).attr('y2', H)
  .attr('stroke', viz.grid).attr('stroke-width', 1);
const keys = ['House', 'Senat'];
const color = d3.scaleOrdinal(keys, viz.palette);
const tip = viz.tooltip(el);
keys.forEach(k => {
  const rows = data.filter(d => d.chamber === k).sort((a, b) => a.t - b.t);
  g.append('path').datum(rows).attr('fill', 'none').attr('stroke', color(k)).attr('stroke-width', 1.5)
    .attr('d', d3.line().x(d => xs(d.t)).y(d => ys(d.v)));
  g.selectAll(null).data(rows).join('circle').attr('cx', d => xs(d.t)).attr('cy', d => ys(d.v)).attr('r', 2.5)
    .attr('fill', color(k)).on('mousemove', (e, d) => tip.show(`<b>${k} ${d.t.toFixed(2)}</b> (Congress ${d.congress})<br>${d3.format('.1%')(d.v)}`, e))
    .on('mouseleave', () => tip.hide());
});
breaks.forEach((b, i) => {
  g.append('line').attr('x1', xs(b.year)).attr('x2', xs(b.year)).attr('y1', 0).attr('y2', H)
    .attr('stroke', viz.ink).attr('stroke-width', 1.5).attr('stroke-dasharray', '5,4');
  g.append('text').attr('x', xs(b.year) - 4).attr('y', 12 + 14 * i).attr('text-anchor', 'end').attr('fill', viz.ink)
    .style('font', '11px sans-serif').text(b.label);
});
viz.legend(svg, [['House', color('House')], ['Senate', color('Senat')]], m.left, height - 10);
""", plot_rows, height=360, width=760, opts=dict(breaks=breaks, bounds=bounds), name="C_orig_share_breaks")


def _post_break_units(bdates):
    """Per eval unit: does it contain bills introduced after a break date while its fit set contains none (the eval
    regime was never seen in training)? bdates = first day of each post-break segment."""
    rows = []
    for un in EVAL_UNITS + S1POOL_UNITS:
        if un["split"] == "S5":
            continue
        fit_end = max(congress_end(c) for c in un["train"])
        ev_end = congress_end(un["eval_congress"])
        if un["within"] is not None:
            c, q_tr, q_lo, q_hi = un["within"]
            fit_end = max(fit_end, S4_CUT_DATES[c][q_tr])
            ev_end = min(S4_CUT_DATES[c][q_hi], congress_end(c))
        rows.append(dict(split=un["split"], fold=un["fold"], role=un["role"], eval_congress=int(un["eval_congress"]),
                         unseen=any((ev_end > d) and (fit_end <= d) for d in bdates)))
    return pd.DataFrame(rows)


bp["break_date"] = [max(pd.Timestamp(m), congress_start(c)) for m, c in zip(bp.break_month, bp.congress)]
# A break location is "identified" when the constant-mean and trend-adjusted fits put it within 6 months of each other;
# otherwise the single-break model is misspecified (e.g. a V-shaped series) and no Congress can be said to straddle it.
bp["identified"] = [abs((pd.Period(a, "M") - pd.Period(b_, "M")).n) <= 6 for a, b_ in zip(bp.raw_break, bp.break_month)]
sig_all = bp[bp.p < alpha23]
sig = sig_all[sig_all.identified]
unid = sig_all[~sig_all.identified]
sig_in = sig[sig.inside]
raw_sig = bp[bp.raw_p < alpha23]
brk_txt = "; ".join(f"{r.series}/{r.chamber}: raw {r.raw_break} (p={r.raw_p:.3f}) -> detrended {r.break_month} (Congress {r.congress}, "
                    f"month {r.month_of_congress}; {r.before:.3f} -> {r.after:.3f}, {r.shift_sd:+.1f} SD, p={r.p:.3f}; linear trend "
                    f"{r.trend_per_yr:+.4f}/yr)" for r in bp.itertuples())
lost_after_dt = [(r.series, r.chamber) for r in bp.itertuples() if r.raw_p < alpha23 and r.p >= alpha23]
record_check(C_SECTION, "C23", "No significant structural break (beyond a linear trend) in original-cosponsor share / cosponsor timing inside a Congress",
             len(sig_in) == 0, metric=int(len(sig_in)),
             threshold=f"metric = number of breaks failing the rule; FAIL if any: trend-adjusted break with block-permutation p < {alpha23:.4f} (0.05 Bonferroni over {len(bp)} series), location "
                       "identified (raw and trend-adjusted fits within 6 months), >= 3 months inside a Congress",
             detail=f"breaks: {brk_txt}. Significant constant-mean breaks: {len(raw_sig)}; significant beyond a linear trend: {len(sig_all)}, "
                    f"of which location-identified: {len(sig)}. Breaks that do not survive the trend adjustment (a steady trend, not a "
                    f"break): {lost_after_dt or 'none'}. Significant but location not identified (the two fits disagree by > 6 months, "
                    f"i.e. the series bends rather than steps; reported, not used): "
                    f"{[(r.series, r.chamber, r.raw_break, r.break_month) for r in unid.itertuples()] or 'none'}. "
                    + (f"Significant inside-Congress breaks: {[(r.series, r.chamber, r.break_month) for r in sig_in.itertuples()]}: that "
                       "Congress mixes two regimes, so a validation unit on it straddles the break and a within-Congress cut (or a "
                       "post-break indicator) matters for it. " if len(sig_in) else "No inside-Congress break survives detrending: Congress "
                       "boundaries are adequate cut points. ")
                    + f"Reconciliation with A16 (A4 original share 117/118 = break vs linear trend at Congress level): identified breaks at a "
                      f"Congress boundary (month 1-3): {[(r.series, r.chamber, r.break_month, int(r.congress)) for r in sig[~sig.inside].itertuples()] or 'none'}"
                      + (" -> agrees with A16 (a level shift at the start of that Congress, not a steady trend). " if len(sig[~sig.inside]) else
                         " -> no trend-adjusted boundary break, which does not support A16's break reading. ")
                    + "A constant-mean CUSUM with block permutation would call any trend a break, which is why the verdict uses the "
                      "trend-adjusted test. Cause (procedural rules vs "
                      "behaviour) is not identifiable from the data; the original share itself is owned by A (A4).")


def _first_post_break_congresses(bdates):
    """Congresses that are the first (whole or partial) Congress after each break date."""
    return sorted({int(next(c for c in CONGRESSES if congress_end(c) > d)) for d in bdates})


pbu = _post_break_units(list(sig.break_date)) if len(sig) else None
if pbu is not None:
    first_pb = _first_post_break_congresses(list(sig.break_date))
    # Chance rate: share of one-step Congress transitions in 108-118 that land on a first post-break Congress, i.e. the
    # probability that a deployment Congress (fit on everything before it) starts a regime its fit set never saw.
    p_dep = len(first_pb) / (len(CONGRESSES) - 1)
    rows23 = []
    for k, g in pbu[pbu.role == "test"].groupby("split"):
        n_t, n_u = len(g), int(g.unseen.sum())
        p_b = float(stats.binom.sf(n_u - 1, n_t, p_dep)) if n_u else 1.0
        rows23.append(dict(split=k, test_units=n_t, post_break_unseen=n_u, share=n_u / n_t, expected=p_dep, binom_p=p_b))
    t23 = pd.DataFrame(rows23).set_index("split")
    print(f"Test units scored on a post-break regime unseen in their fit set vs the deployment rate {p_dep:.2f} "
          f"(first post-break Congresses {first_pb} out of {len(CONGRESSES) - 1} transitions)")
    print(t23.rename_axis(None).round(3).to_string())
    bs23 = with_variants({k: bool(r.binom_p >= 0.05) for k, r in t23.iterrows()} | {"S5": None})
    aff = pbu[pbu.unseen].groupby(["split", "role"]).fold.agg(list)
    val_aff = sorted({k for (k, r) in aff.index if r == "val"})
else:
    t23, bs23, aff, val_aff, p_dep, first_pb = None, None, pd.Series(dtype=object), [], 0.0, []

# Sensitivity of C23b to the definition of "break". C23 uses a narrow one (a within-series level shift in the monthly
# original-share and lag series). Section A's A16 labels a failing candidate part a break when it is off both the
# stationary and the linear-trend interval; under that definition a candidate Congress is post-break if it has any
# Holm-surviving break part. A16 judges only candidates (116-118), so two expected rates bracket the deployment rate:
# reference Congresses assumed unbroken (n_broken / 10 transitions) and candidate transitions only (n_broken / 3).
_az = globals().get("A_ZTAB")
if _az:
    _zt = pd.DataFrame(_az).drop_duplicates(["part", "congress"], keep="last")
    _need = {"level", "fail", "kind", "holm"}
    a16_broken = (sorted({int(c) for c in _zt[_zt.level & _zt.fail & (_zt.kind == "break") & _zt.holm].congress})
                  if _need <= set(_zt.columns) else None)
    a16_src = "section A (A_ZTAB: Holm-surviving parts labelled 'break', as in A16)"
else:
    a16_broken, a16_src = None, None
if a16_broken is None:
    # Section A not in scope (section run alone): use the upper bound, every candidate Congress post-break.
    a16_broken, a16_src = list(C_EVAL), "upper-bound assumption (section A not in scope): every candidate Congress post-break"
rate_lo, rate_c = len(a16_broken) / (len(CONGRESSES) - 1), len(a16_broken) / len(C_EVAL)
pbx = pd.DataFrame([dict(split=un["split"], role=un["role"], post=int(un["eval_congress"]) in a16_broken)
                    for un in EVAL_UNITS + S1POOL_UNITS if un["split"] != "S5" and un["role"] == "test"])
sens = pbx.groupby("split").post.agg(test_units="size", post_break="sum")
sens["share"] = sens.post_break / sens.test_units
for nm, r_ in (("p_rate_lo", rate_lo), ("p_rate_cand", rate_c)):
    sens[nm] = [float(stats.binom.sf(k - 1, n, r_)) if k else 1.0 for n, k in zip(sens.test_units, sens.post_break)]
sens["c23b_narrow"] = [None if t23 is None or k not in t23.index else bool(t23.loc[k, "binom_p"] >= 0.05) for k in sens.index]
sens["a16_lo"], sens["a16_cand"] = sens.p_rate_lo >= 0.05, sens.p_rate_cand >= 0.05
flip = [k for k, r in sens.iterrows() if r.c23b_narrow is not None and r.c23b_narrow != r.a16_cand]
print(f"C23b sensitivity: A16-style break list {a16_broken} ({a16_src}); expected rate {rate_lo:.2f} (reference Congresses "
      f"unbroken) or {rate_c:.2f} (candidate transitions only)")
print(sens.rename_axis(None).round(3).to_string())
record_check(C_SECTION, "C23b", "Share of test units scored on a post-break regime unseen in training is not above the deployment rate",
             None if bs23 is None else all(v for k, v in bs23.items() if v is not None), metric=None if t23 is None else float(t23.share.max()),
             threshold=f"by_split False if binomial P(>= observed post-break test units | n test units, rate {p_dep:.2f}) < 0.05",
             detail=f"significant detrended breaks (p < {alpha23:.4f}): {[(r.series, r.chamber, str(r.break_date.date())) for r in sig.itertuples()]}; "
                    f"first post-break Congresses {first_pb}. Affected units (split, role) -> folds: {aff.to_dict()}. "
                    + ("" if t23 is None else f"Per split (affected/test units, p): "
                       + ", ".join(f"{k} {int(r.post_break_unseen)}/{int(r.test_units)} (p={r.binom_p:.2f})" for k, r in t23.iterrows()) + ". ")
                    + f"Splits whose val unit is affected: {val_aff}. A test unit on the first post-break Congress is the realistic "
                      "deployment case (deploying on 119 faces the same risk if 119 starts a new regime), not a defect: it is only a "
                      "problem if a split has MORE such units than deployment would (judged here), or if all of a split's test evidence "
                      "comes from such units (S1a with one test Congress: read its score as a pessimistic, regime-change estimate). "
                    + f"Definition: 'break' here means a within-series level shift in the monthly original-share and cosponsor-lag series "
                      f"only (C23), so the deployment rate {p_dep:.2f} counts one post-break Congress in {len(CONGRESSES) - 1} transitions. "
                      f"Sensitivity with A16's broader definition (any Holm-surviving failing level part off the trend interval): "
                      f"post-break Congresses {a16_broken} ({a16_src}); expected rate {rate_lo:.2f} (reference Congresses unbroken) to "
                      f"{rate_c:.2f} (candidate transitions only). Per split, post-break test units / test units and p at the two rates: "
                    + ", ".join(f"{k} {int(r.post_break)}/{int(r.test_units)} (p={r.p_rate_lo:.2f} / {r.p_rate_cand:.2f})" for k, r in sens.iterrows())
                    + f". Splits whose verdict flips between the narrow definition and A16's at the candidate rate: {flip or 'none'}"
                    + (" -> their C23b failure depends on the narrow break definition; read it as weak evidence. " if flip else ". ")
                    + "Under A16's definition nearly every candidate test unit is post-break, which is the deployment case, not a split "
                      "defect: no split can avoid it, so the remedy is recalibration on val and recency weighting, not a different split.",
             by_split=bs23, na_reason=("S5 has no time boundary between fit and eval bills; no regime can be unseen" if bs23 else None))

Largest level shift per monthly series: raw = constant-mean CUSUM on the de-seasonalised series; trend-adjusted (break_month .. p) = step beyond a jointly fitted linear trend (verdict); before/after = raw means; trend_per_yr = slope without a step
       series chamber raw_break  raw_p  trend_per_yr break_month  congress  month_of_congress  before   after  shift_sd      p  inside
 orig_share30   House   2021-01 0.0000        0.0030     2021-01       117                  1  0.6200  0.7130    1.2050 0.0000   False
 orig_share30   Senat   2017-04 0.0000        0.0080     2008-01       110                 13  0.7360  0.7820   -1.5450 0.0000    True
median_lag180   House   2024-01 0.0320        0.0430     2024-01       118                 13 42.0450 49.1670    1.6330 0.0380    True
median_lag180   Senat   2017-11 0.2040        0.1030     2017-11       115                 11 40.7950 46.1110    1.0240 0.0230    True


[PASS] C23: No significant structural break (beyond a linear trend) in original-cosponsor share / cosponsor timing inside a Congress | metric=0 (threshold metric = number of breaks failing the rule; FAIL if any: trend-adjusted break with block-permutation p < 0.0125 (0.05 Bonferroni over 4 series), location identified (raw and trend-adjusted fits within 6 months), >= 3 months inside a Congress)
Test units scored on a post-break regime unseen in their fit set vs the deployment rate 0.10 (first post-break Congresses [117] out of 10 transitions)
        test_units  post_break_unseen  share  expected  binom_p
S1a              1                  1 1.0000    0.1000   0.1000
S1b              1                  0 0.0000    0.1000   1.0000
S1pool           2                  2 1.0000    0.1000   0.0100
S2               7                  1 0.1430    0.1000   0.5220
S3               7                  1 0.1430    0.1000   0.5220
S4               3                  0 0.0000    0.1000   1.0000
C23

### C5-C6. Label censoring at the end of each Congress

**Why it matters.** Unpassed bills die when a Congress adjourns, so a bill introduced 20 days before the
end has no full 30-day window (task 1) and one introduced 100 days before the end has no full 180-day
window (task 2). These bills are excluded. If the excluded share differs by Congress, or if the excluded
bills are systematically different (the end-of-Congress rush), the evaluation population differs from
the training population. Note that 119 is censored *much* more because it is in progress.

**How to read it.** The chart shows the censored share per Congress for task 1 (t0=0, t0=7) and task 2.
The table adds the share of evaluation bills each candidate split loses (from `iter_eval_units()`); S4's
test slice is the end of the Congress, so it is hit hardest.

In [135]:
# C5/C6: censored share per Congress for task 1 (t0=0/7, 30 d) and task 2 (180 d); bills lost per split part.
cen = pd.DataFrame({
    "t1_t0=0": wt[0].groupby("congress")["censored"].mean(),
    "t1_t0=7": wt[7].groupby("congress")["censored"].mean(),
    "t2_180d": adv.groupby("congress")["censored"].mean(),
    "t2_bills_lost": adv.groupby("congress")["censored"].sum(),
}).loc[CONGRESSES + [IN_PROGRESS]]
print("Censored share per Congress (119 in progress, context only)")
print(cen.T.to_string(float_format=C_FMT))
cl = cen.loc[CONGRESSES, ["t1_t0=0", "t1_t0=7", "t2_180d"]].reset_index().melt("congress", var_name="task", value_name="share")
line_chart(cl, "congress", "share", series="task", yfmt=".0%",
           title="Share of law-capable bills whose label window runs past the end of the Congress",
           subtitle="Task 1: 30-day window after t0; task 2: 180-day reporting window", xlabel="Congress",
           name="C_censoring")
# Censoring per evaluation unit of every candidate split: share of eval bills lost (task 1 t0=7, task 2).
cfr = lb[lb.congress.isin(CONGRESSES)][["bill_id", "congress", "introduced_date"]].merge(
    wt[7][["bill_id", "censored"]].rename(columns={"censored": "cen_t1"}), on="bill_id").merge(
    adv[["bill_id", "censored"]].rename(columns={"censored": "cen_t2"}), on="bill_id")
lost_rows = []
for un in EVAL_UNITS + S1POOL_UNITS:
    _, ev = unit_masks(un, cfr)
    e = cfr[ev]
    lost_rows.append(dict(split=un["split"], fold=un["fold"], role=un["role"], eval_bills=len(e),
                          t1_lost=e.cen_t1.mean(), t2_lost=e.cen_t2.mean()))
lost_u = pd.DataFrame(lost_rows)
lost = lost_u.groupby(["split", "role"]).agg(units=("fold", "size"), eval_bills=("eval_bills", "mean"),
                                             t1_lost=("t1_lost", "mean"), t2_lost=("t2_lost", "max")).round(3)
print("\nShare of eval bills lost to censoring per split and role (mean over folds; t2_lost = worst fold)")
print(lost.unstack("role").to_string())
lost_split = lost_u.groupby("split")[["t1_lost", "t2_lost"]].max()   # worst eval unit per split
mx1 = float(cen.loc[CONGRESSES, "t1_t0=7"].max())
gap1 = float(cen.loc[C_EVAL, "t1_t0=7"].mean() - cen.loc[C_TRAIN, "t1_t0=7"].mean())
record_check(C_SECTION, "C5", "Task-1 censoring (t0=7) drops < 5% of law-capable bills in every Congress 108-118",
             bool(mx1 < 0.05), metric=mx1, threshold="max censored share < 0.05",
             detail=f"censored share t0=7 ranges {cen.loc[CONGRESSES, 't1_t0=7'].min():.3f}-{mx1:.3f}; eval Congresses lose "
                    f"{gap1:+.3f} more than the train pool on average (end-of-Congress introductions have grown). 5% is the "
                    "level below which excluding bills cannot materially change the evaluation population; censoring is "
                    "structural (last ~37 days): by-Congress splits lose the same calendar slice, while the S4 test slice "
                    f"(last 20% of introductions) loses {lost_u.loc[(lost_u.split == 'S4') & (lost_u.role == 'test'), 't1_lost'].mean():.3f} "
                    f"of its bills (see table). by_split = worst eval unit's task-1 lost share < 0.05. Why 5% is a hard limit: losing a share f "
                    f"widens a metric's CI half-width by 1/sqrt(1-f)-1 (at f=0.05: {1 / np.sqrt(0.95) - 1:.1%}; worst by-Congress unit f="
                    f"{lost_split.loc[[k for k in BY_CONGRESS_SPLITS if k in lost_split.index], 't1_lost'].max():.3f}: "
                    f"{1 / np.sqrt(1 - lost_split.loc[[k for k in BY_CONGRESS_SPLITS if k in lost_split.index], 't1_lost'].max()) - 1:.1%}), "
                    "negligible; the larger risk is selection, because the dropped bills are all end-of-Congress introductions, and "
                    "below 5% they cannot move a population average by more than 5% of the gap between them and the rest.",
             quantity="censoring", by_split=with_variants({k: bool(v < 0.05) for k, v in lost_split.t1_lost.items()}))
# Are the task-2 censored bills different? Compare cosponsor count by day 30 of censored vs uncensored bills.
a2 = adv[adv.congress.isin(CONGRESSES)].merge(wt[0][["bill_id", "n_pos", "n_seen"]], on="bill_id", how="left")
a2["early_cosp"] = a2["n_pos"].fillna(0) + a2["n_seen"].fillna(0)
r2 = cen.loc[CONGRESSES, "t2_180d"].max() - cen.loc[CONGRESSES, "t2_180d"].min()
s2 = smd(a2.loc[~a2.censored, "early_cosp"], a2.loc[a2.censored, "early_cosp"])
s4_t2 = float(lost_u.loc[(lost_u.split == "S4") & (lost_u.role == "test"), "t2_lost"].mean())
oth_t2 = float(lost_u.loc[lost_u.split.isin(BY_CONGRESS_SPLITS), "t2_lost"].mean())
c6_tab, c6_info = rel_to_train(cen.loc[CONGRESSES, "t2_180d"], side="two")
record_check(C_SECTION, "C6", "Task-2 censored share (180 d) of eval Congresses within train-to-train noise",
             not bool(c6_tab.loc[c6_tab.role == "cand", "fail"].any()), metric=float(cen.loc[C_EVAL, "t2_180d"].max()),
             threshold=c6_info["rule"], **dict(zip(("by_split", "na_reason"), full_split(
                 rel_by_split(c6_tab), s4_na="S4's test slice is the end of each Congress, so its censored share is set by position, "
                                             "not by the Congress; judged separately in C6b"))),
             detail=rel_detail(c6_tab, c6_info) + f". Range 108-118 {r2:.3f}. SMD of cosponsors-by-day-30 (censored vs uncensored) = "
                    f"{s2:.3f}: the dropped end-of-Congress bills differ, so task-2 results describe bills introduced in the first ~18 "
                    f"months only. By-Congress eval sets lose {oth_t2:.3f} on average, but the S4 test slice (last 20% of introductions) "
                    f"loses {s4_t2:.3f} (C6b). Task-2 t0 assumed: window starts at introduction (any t0 <= 30 d leaves this unchanged, "
                    "see C25). If FAIL: the eval Congress loses more bills than train did; evaluate on the same uncensored slice in every split.")
record_check(C_SECTION, "C6b", "Within-Congress split S4: its test slice (last 20% of introductions) keeps >= 50% of bills for task 2",
             bool(s4_t2 < 0.5), metric=s4_t2, threshold="task-2 censored share of S4 test < 0.50",
             detail=f"S4 test slices lose {s4_t2:.3f} of their bills to task-2 censoring (task 1 t0=7: "
                    f"{lost_u.loc[(lost_u.split == 'S4') & (lost_u.role == 'test'), 't1_lost'].mean():.3f}); by-Congress eval sets lose "
                    f"{oth_t2:.3f}. The surviving S4 task-2 test bills are the few introduced early in the slice, so S4 cannot "
                    "be used to select or report a task-2 model. by_split = worst eval unit keeps >= 50% of its bills. Why 50%: "
                    f"beyond it the survivors are a minority drawn from the start of the slice (not the slice's population), and the CI "
                    f"half-width grows by 1/sqrt(1-f)-1 = {1 / np.sqrt(max(1 - s4_t2, 1e-9)) - 1:.0%} at the observed loss "
                    f"({lost_u.loc[(lost_u.split == 'S4') & (lost_u.role == 'test'), 'eval_bills'].mean() * (1 - s4_t2):,.0f} task-2 test bills left per fold).",
             by_split=with_variants({k: bool(v < 0.5) for k, v in lost_split.t2_lost.items()}))

Censored share per Congress (119 in progress, context only)
congress        108   109   110   111   112   113   114   115   116   117   118   119
t1_t0=0       0.002 0.014 0.005 0.012 0.015 0.014 0.016 0.025 0.022 0.031 0.030 0.000
t1_t0=7       0.002 0.014 0.005 0.020 0.019 0.020 0.021 0.033 0.029 0.039 0.034 0.000
t2_180d       0.126 0.112 0.128 0.129 0.096 0.149 0.125 0.152 0.173 0.150 0.164 0.119
t2_bills_lost 1,088 1,203 1,440 1,388 1,020 1,356 1,275 1,735 2,474 2,289 2,721 2,025



Share of eval bills lost to censoring per split and role (mean over folds; t2_lost = worst fold)
       units      eval_bills             t1_lost        t2_lost       
role    test val        test         val    test    val    test    val
split                                                                 
S1a        1   1 15,242.0000 14,340.0000  0.0390 0.0290  0.1500 0.1730
S1b        1   1 16,565.0000 15,242.0000  0.0340 0.0390  0.1640 0.1500
S1pool     2   1 15,903.5000 14,340.0000  0.0360 0.0290  0.1640 0.1730
S2         7   7 12,497.0000 11,669.1430  0.0280 0.0260  0.1730 0.1730
S3         7   7 12,497.0000 11,669.1430  0.0280 0.0260  0.1730 0.1730
S4         3   3  3,090.6670  3,093.6670  0.1680 0.0000  0.8580 0.0000
S5         1   1 19,423.0000 19,190.0000  0.0230 0.0250  0.1400 0.1390
[PASS] C5: Task-1 censoring (t0=7) drops < 5% of law-capable bills in every Congress 108-118 | metric=0.03864 (threshold max censored share < 0.05) [fails for: S4]
[PASS] C6: Task-2 censored s

### C7-C10. Party-control regimes and party labels

**Why it matters.** Cosponsorship and committee action depend on who holds the gavel. A model trained only
on Republican-House Congresses may not transfer to a Democratic House. A by-Congress split works best if
each chamber's controlling party in the validation and test Congresses also appears in training (and,
only if it measurably matters, the joint House x Senate regime). If not, the test measures extrapolation
to a new regime, and we should at least report per-regime results.

**How to read it.** The strip shows the majority party per chamber per Congress from the shared
`majority_party()` (point-in-time at mid-Congress, party as of the Congress, independents counted with their
caucus, Senate ties broken by the Vice President's party and cross-checked against the data). Diamonds mark
a control flip versus the previous Congress. C8 checks, for every evaluation unit of the candidate splits, that
each chamber's controlling party appears in training (task 2 depends on the bill's own chamber). C8b asks whether
the *joint* regime (divided vs unified government) changes anything measurable beyond the chamber majority; only
then does joint-regime coverage become a requirement. C9 shows how narrow the margins are and how C8 would change
if a tie went the other way;
C10 checks that per-Congress party labels are used (a time leak otherwise).

In [136]:
# C7: majority per chamber per Congress from majority_party(); unified vs split; control flips.
mp = majority_party()
mp = mp[mp.congress.isin(CONGRESSES + [IN_PROGRESS])]
reg = mp.pivot(index="congress", columns="chamber", values="majority").rename(
    columns={"House of Representatives": "house", "Senate": "senate"})
reg["regime"] = reg["house"] + "H/" + reg["senate"] + "S"
reg["unified_congress"] = reg["house"] == reg["senate"]
reg["house_flip"] = reg["house"] != reg["house"].shift()
reg["senate_flip"] = reg["senate"] != reg["senate"].shift()
reg.loc[reg.index.min(), ["house_flip", "senate_flip"]] = False
print("Majority party per chamber (from majority_party(); point-in-time, caucus-adjusted)")
print(reg.to_string())
flips = [f"{c} {ch}" for c in reg.index for ch in ("house", "senate") if reg.loc[c, f"{ch}_flip"]]
rd = mp.copy()
rd["chamber"] = rd["chamber"].map({"House of Representatives": "House", "Senate": "Senate"})
rd["flip"] = [bool(reg.loc[c, ("house_flip" if ch == "House" else "senate_flip")]) for c, ch in zip(rd.congress, rd.chamber)]
rd["congress"] = rd["congress"].astype(int)
show_d3(r"""
const {width, height} = opts;
const m = {top: 48, right: 20, bottom: 50, left: 70};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, 'Majority party per chamber and Congress (diamond = control flip vs previous Congress)',
          'From majority_party(): mid-Congress, caucus-adjusted, VP breaks Senate ties; 119 in progress');
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const xs = d3.scaleBand(Array.from(new Set(data.map(d => String(d.congress)))), [0, W]).padding(0.08);
const ys = d3.scaleBand(['House', 'Senate'], [0, H]).padding(0.12);
const col = {D: '#2a78d6', R: '#e34948'};
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('rect').attr('x', d => xs(String(d.congress))).attr('y', d => ys(d.chamber))
  .attr('width', xs.bandwidth()).attr('height', ys.bandwidth()).attr('rx', 3).attr('fill', d => col[d.majority] || '#7a7a7a')
  .on('mousemove', (e, d) => tip.show(`<b>${d.congress} ${d.chamber}</b><br>majority ${d.majority}${d.flip ? ' (flip)' : ''}`, e))
  .on('mouseleave', () => tip.hide());
g.selectAll(null).data(data).join('text').attr('x', d => xs(String(d.congress)) + xs.bandwidth() / 2)
  .attr('y', d => ys(d.chamber) + ys.bandwidth() / 2 + 4).attr('text-anchor', 'middle').attr('fill', '#fff')
  .style('font', '600 12px sans-serif').text(d => d.majority);
g.selectAll(null).data(data.filter(d => d.flip)).join('path')
  .attr('transform', d => `translate(${xs(String(d.congress)) + 8},${ys(d.chamber) + 8})`)
  .attr('d', d3.symbol(d3.symbolDiamond, 40)).attr('fill', viz.ink);
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs)).call(viz.axis);
g.append('g').call(d3.axisLeft(ys)).call(viz.axis);
viz.legend(svg, [['Democratic majority', col.D], ['Republican majority', col.R]], m.left, height - 10);
""", rd, height=230, width=720, name="C_regime_strip")
record_check(C_SECTION, "C7", "Party-control regimes and flips across 108-118 (informational)", None,
             metric=f"{reg.loc[CONGRESSES, 'regime'].nunique()} regimes, {len([f for f in flips if int(f.split()[0]) in CONGRESSES])} flips",
             detail=f"regimes: {reg.loc[CONGRESSES, 'regime'].to_dict()}; flips at: {flips}. Unified Congresses: "
                    f"{list(reg.index[reg.unified_congress & reg.index.isin(CONGRESSES)])}.")

Majority party per chamber (from majority_party(); point-in-time, caucus-adjusted)
chamber  house senate regime  unified_congress  house_flip  senate_flip
congress                                                               
108          R      R  RH/RS              True       False        False
109          R      R  RH/RS              True       False        False
110          D      D  DH/DS              True        True         True
111          D      D  DH/DS              True       False        False
112          R      D  RH/DS             False        True        False
113          R      D  RH/DS             False       False        False
114          R      R  RH/RS              True       False         True
115          R      R  RH/RS              True       False        False
116          D      R  DH/RS             False        True        False
117          D      D  DH/DS              True       False         True
118          R      D  RH/DS             False       

[INFO] C7: Party-control regimes and flips across 108-118 (informational) | 4 regimes, 7 flips


In [137]:
# C8: for every evaluation unit (incl. S1pool), does each chamber's controlling party (and the joint regime) appear in training?
cov = []
for un in [u for u in EVAL_UNITS if u["split"] in BY_CONGRESS_SPLITS] + S1POOL_UNITS:
    c, pr = un["eval_congress"], reg.loc[un["train"]]
    cov.append(dict(split=un["split"], fold=un["fold"], role=un["role"], eval_congress=c, train=un["train_label"],
                    eval_regime=reg.loc[c, "regime"], regime_in_train=int((pr.regime == reg.loc[c, "regime"]).sum()),
                    house_in_train=int((pr.house == reg.loc[c, "house"]).sum()),
                    senate_in_train=int((pr.senate == reg.loc[c, "senate"]).sum())))
cov = pd.DataFrame(cov)
cov["chambers_covered"] = (cov.house_in_train > 0) & (cov.senate_in_train > 0)
covs = cov.groupby("split", sort=False).agg(units=("fold", "size"), chambers_covered=("chambers_covered", "mean"),
                                            house_covered=("house_in_train", lambda v: (v > 0).mean()),
                                            senate_covered=("senate_in_train", lambda v: (v > 0).mean()),
                                            joint_regime_covered=("regime_in_train", lambda v: (v > 0).mean()))
print("Share of evaluation units whose chamber majorities (primary) / joint House x Senate regime (INFO) appear in training")
print(covs.round(3).to_string())
ch_split = with_variants(covs.chambers_covered.eq(1.0).to_dict())
uncov = sorted({(r.split, int(r.eval_congress)) for r in cov.itertuples() if not r.chambers_covered})
record_check(C_SECTION, "C8", "Each eval Congress's House majority and Senate majority both appear in its training Congresses",
             all(v for k, v in ch_split.items() if k in ("S1a", "S1b", "S3")), metric=float(covs.loc[["S1a", "S1b", "S3"], "chambers_covered"].min()),
             threshold="per-chamber coverage = 1.0 for every eval unit (judged on S1a, S1b, S3; by_split for all)",
             detail=f"per-chamber coverage per split {covs.chambers_covered.round(2).to_dict()}; uncovered units {uncov}. Task 2 depends on "
                    "the bill's own chamber and A5b shows chamber x controlling party is enough to match the majority-sponsor OR, "
                    "so per-chamber coverage is the requirement. Sliding windows (S2) drop chamber regimes that expanding windows keep. "
                    "S4 and S5 hold by construction: their fit sets contain bills of the eval Congress itself.",
             quantity="party_regime", **dict(zip(("by_split", "na_reason"), full_split(
                 ch_split, extra={"S4": True, "S5": True}))))

# C8b: does the JOINT regime (divided vs unified government) matter beyond the bill chamber's majority?
# Per (Congress, chamber), uncensored task-2 bills: reported rate of majority-sponsored bills and majority-sponsor log OR.
sp_cau = members.drop_duplicates(["bioguide_id", "congress"]).set_index(["bioguide_id", "congress"])["caucus"]
a8 = adv[adv.congress.isin(CONGRESSES) & ~adv.censored].merge(bills[["bill_id", "sponsor_bioguide_id"]], on="bill_id")
a8["cau"] = pd.MultiIndex.from_arrays([a8.sponsor_bioguide_id, a8.congress]).map(sp_cau)
a8["cau"] = a8.cau.where(a8.cau.isin(["D", "R"]), a8.sponsor_party)
a8 = a8.merge(MAJORITY[["congress", "chamber", "majority"]], on=["congress", "chamber"])
a8["maj"] = a8.cau == a8.majority
g8 = a8.groupby(["congress", "chamber", "maj"])["label"].agg(["sum", "size"]).unstack("maj")
lo_ = lambda k, n: np.log((k + 0.5) / (n - k + 0.5))
q8 = pd.DataFrame({"rate_maj": g8[("sum", True)] / g8[("size", True)],
                   "log_or": lo_(g8[("sum", True)], g8[("size", True)]) - lo_(g8[("sum", False)], g8[("size", False)])}).reset_index()
q8["divided"] = q8.congress.map(~reg["unified_congress"])
perm_rows = []
for ch, g in q8.groupby("chamber"):
    for col in ("rate_maj", "log_or"):
        for label, cs_ in (("108-118", CONGRESSES), ("108-115", C_TRAIN)):
            gg = g[g.congress.isin(cs_)]
            x, dv = gg[col].to_numpy(), gg.divided.to_numpy()
            obs = x[dv].mean() - x[~dv].mean()
            combos = np.array(list(__import__("itertools").combinations(range(len(x)), int(dv.sum()))))
            msk = np.zeros((len(combos), len(x)), bool)
            msk[np.arange(len(combos))[:, None], combos] = True
            null = (msk * x).sum(1) / dv.sum() - (~msk * x).sum(1) / (~dv).sum()
            perm_rows.append(dict(chamber=ch[:5], quantity=col, congresses=label, n_divided=int(dv.sum()), diff=obs,
                                  p_exact=float((np.abs(null) >= abs(obs) - 1e-12).mean()), n_perm=len(combos)))
perm = pd.DataFrame(perm_rows)
print("Divided minus unified government, per chamber (exact permutation over Congress labels; 108-115 has only 2 divided)")
print(perm.round(3).to_string(index=False))
p_main = perm[perm.congresses == "108-118"]
alpha8 = 0.05 / len(p_main)
joint_matters = bool((p_main.p_exact < alpha8).any())
jr_split = with_variants(covs.joint_regime_covered.eq(1.0).to_dict())
record_check(C_SECTION, "C8b", "Joint House x Senate regime (divided vs unified) changes task-2 rate or majority OR beyond the chamber majority",
             False if joint_matters else None, metric=float(p_main.p_exact.min()),
             threshold=f"FAIL only if any exact permutation p < {alpha8:.4f} (0.05 Bonferroni over {len(p_main)} tests, 108-118); else INFO",
             detail=f"divided-unified differences (108-118): " + "; ".join(f"{r.chamber} {r.quantity} {r.diff:+.3f} (p={r.p_exact:.3f})" for r in p_main.itertuples())
                    + f". Joint-regime coverage per split (INFO unless the effect is significant): {covs.joint_regime_covered.round(2).to_dict()}; "
                    f"116 ({reg.loc[116, 'regime']}) never occurs in 108-115. Using 108-118 to estimate a process property, not to pick a model; "
                    "with 11 Congresses the test has low power, so a non-significant result means 'no evidence', not 'no effect'.",
             quantity="party_regime_joint", by_split=jr_split if joint_matters else None)

# Val regime = test regime per fold (does tuning transfer?), reported for context.
vt = cov.pivot_table(index=["split", "fold"], columns="role", values="eval_regime", aggfunc="first")
vt_same = (vt["val"] == vt["test"]).groupby("split").mean()
never = [c for c in CONGRESSES if (reg.loc[CONGRESSES, "regime"] == reg.loc[c, "regime"]).sum() == 1]
print(f"val regime = test regime share of folds: {vt_same.round(2).to_dict()}; regimes occurring once in 108-118: "
      f"{[(c, reg.loc[c, 'regime']) for c in never]}")

Share of evaluation units whose chamber majorities (primary) / joint House x Senate regime (INFO) appear in training
        units  chambers_covered  house_covered  senate_covered  joint_regime_covered
split                                                                               
S1a         2            1.0000         1.0000          1.0000                0.5000
S1b         2            1.0000         1.0000          1.0000                1.0000
S2         14            0.5710         0.8570          0.7140                0.3570
S3         14            1.0000         1.0000          1.0000                0.7140
S1pool      3            1.0000         1.0000          1.0000                0.6670
[PASS] C8: Each eval Congress's House majority and Senate majority both appear in its training Congresses | metric=1 (threshold per-chamber coverage = 1.0 for every eval unit (judged on S1a, S1b, S3; by_split for all)) [fails for: S2]
Divided minus unified government, per chamber (exact 

In [138]:
# C9: how robust are the regime labels? Seat margins per chamber and ties (from MAJORITY); C8 sensitivity to ties.
mg = MAJORITY[MAJORITY.congress.isin(CONGRESSES)].copy()
mg["margin"] = (mg.seats_D - mg.seats_R).abs()
mg["margin_share"] = mg.margin / mg.n_voting
print("Seat margin (|D - R| caucus-adjusted, voting members at mid-Congress) per chamber")
print(mg.assign(chamber=mg.chamber.str[:5]).pivot(index="chamber", columns="congress", values="margin").to_string())
ties = mg[mg.tie_broken_by != "none (seat count)"]
alt_cov = []
for r in ties.itertuples():
    key = "house" if r.chamber.startswith("House") else "senate"
    other = "R" if r.majority == "D" else "D"
    rr = reg.loc[r.congress].copy()
    rr[key] = other
    regime = rr["house"] + "H/" + rr["senate"] + "S"
    n_alt = int((reg.loc[C_TRAIN, "regime"] == regime).sum())
    alt_cov.append(f"{r.congress} {key}: label {r.majority} -> regime {reg.loc[r.congress, 'regime']} "
                   f"({int((reg.loc[C_TRAIN, 'regime'] == reg.loc[r.congress, 'regime']).sum())}x in 108-115); "
                   f"if {other}: {regime} ({n_alt}x in 108-115)")
print("ties: " + ("; ".join(f"{r.congress} {r.chamber}: {r.seats_D}-{r.seats_R}, {r.tie_broken_by}" for r in ties.itertuples()) or "none"))
print("C8 sensitivity: " + ("; ".join(alt_cov) or "no ties"))
confirmed = all("agrees" in t for t in ties.tie_broken_by)
narrow = mg[mg.margin_share < 0.03]
record_check(C_SECTION, "C9", "Regime labels are robust: every tie-broken chamber is confirmed by the data (committee-reported sponsors)",
             bool(confirmed), metric=int(len(ties)), threshold="all ties confirmed by data",
             detail=f"ties: {[(int(r.congress), r.chamber[:5], int(r.seats_D), int(r.seats_R), r.majority) for r in ties.itertuples()]}; "
                    f"tie-break evidence: {list(ties.tie_broken_by)}. Sensitivity: {alt_cov}. Chambers with margin < 3% of seats: "
                    f"{[(int(r.congress), r.chamber[:5], int(r.margin)) for r in narrow.itertuples()]}: narrow majorities make a "
                    "regime label a weak proxy for agenda control, so treat regime as a stratification variable for "
                    "reporting, not as a hard requirement.")

Seat margin (|D - R| caucus-adjusted, voting members at mid-Congress) per chamber
congress  108  109  110  111  112  113  114  115  116  117  118
chamber                                                        
House      22   29   32   79   50   32   58   46   37    9    8
Senat       2   10    2   20    6   10    8    2    6    0    2
ties: 117 Senate: 50-50, VP party (reported-bill sponsors D 65% (agrees))
C8 sensitivity: 117 senate: label D -> regime DH/DS (2x in 108-115); if R: DH/RS (0x in 108-115)
[PASS] C9: Regime labels are robust: every tie-broken chamber is confirmed by the data (committee-reported sponsors) | metric=1 (threshold all ties confirmed by data)


In [139]:
# Member roster per Congress (used by C10).
mt = members[members.congress.isin(CONGRESSES + [IN_PROGRESS])].dropna(subset=["chamber"]).copy()

# C10: per-Congress party labels. Does setup's members.party_code_asof match the party recorded on bills that Congress?
mt_c = mt[mt.congress.isin(CONGRESSES)]
mism = mt_c[mt_c.party_code != mt_c.party_code_asof]
print(f"members.party_code (latest) differs from party_code_asof in {len(mism)} member-Congress rows "
      f"({mism.bioguide_id.nunique()} members), e.g.:")
print("; ".join(f"{r.name.split(',')[0]} {r.cmin}-{r.cmax} latest {r.latest} / as-of {r.asof}" for r in
                mism.groupby("name").agg(cmin=("congress", "min"), cmax=("congress", "max"), latest=("party_code", "first"),
                                         asof=("party_code_asof", "first")).reset_index().itertuples()) or "none")
cp = cosp[cosp.congress.isin(CONGRESSES)][["congress", "bioguide_id", "party"]].drop_duplicates()
cp = cp.merge(mt_c[["congress", "bioguide_id", "party_code_asof", "party_code"]], on=["congress", "bioguide_id"], how="inner")
agree_asof, agree_latest = float((cp.party == cp.party_code_asof).mean()), float((cp.party == cp.party_code).mean())
print(f"bill_cosponsors.party agrees with party_code_asof for {agree_asof:.2%} of member-Congress pairs, "
      f"with the latest party_code for {agree_latest:.2%}")
record_check(C_SECTION, "C10", "Per-Congress party labels (members.party_code_asof) match the party recorded on that Congress's bills",
             bool(agree_asof >= 0.995 and agree_asof >= agree_latest), metric=agree_asof,
             threshold=">= 0.995 agreement and >= latest-party agreement",
             detail=f"party_code_asof agrees {agree_asof:.2%} vs latest party_code {agree_latest:.2%}; {len(mism)} member-Congress rows "
                    f"({mism.bioguide_id.nunique()} members, e.g. {', '.join(mism.name.drop_duplicates().head(4))}) would get a "
                    "party adopted later if members.party_code were used. Requirement for every split: node party features "
                    "= party_code_asof (or caucus), never party_code; otherwise future party switches leak into past Congresses.")

members.party_code (latest) differs from party_code_asof in 29 member-Congress rows (10 members), e.g.:
Alexander 108-108 latest R / as-of D; Amash 112-116 latest L / as-of R; Hall 108-108 latest R / as-of D; Kiley 118-118 latest I / as-of R; Lieberman 108-109 latest ID / as-of D; Manchin 111-118 latest I / as-of D; Mitchell 115-116 latest I / as-of R; Sinema 113-117 latest I / as-of D; Specter 108-110 latest D / as-of R; Van Drew 116-116 latest R / as-of D
bill_cosponsors.party agrees with party_code_asof for 99.87% of member-Congress pairs, with the latest party_code for 99.47%
[PASS] C10: Per-Congress party labels (members.party_code_asof) match the party recorded on that Congress's bills | metric=0.9987 (threshold >= 0.995 agreement and >= latest-party agreement)


### C11-C14. Snapshot leakage: which fields are known at t0?

**Why it matters.** The parquet tables are a **snapshot taken years after each Congress ended**. Many
bill-level fields summarize the bill's whole life (latest action, became law, number of actions,
cosponsors, text versions, summaries). Using any of them as a feature at t0 leaks the future, and a
random split (S5) or even a temporal split would look far better than reality. Fields with a timestamp
can be rebuilt "as of t0"; fields without one (policy area, subjects) are only safe if we can show they
are normally assigned before t0.

**How to read it.** C11 is a requirement, not a pass/fail test. The leakage table compares each snapshot field with its value as of t0 = 7 days
(where a timestamped table lets us rebuild it): `share_changed` is the share of law-capable bills whose
snapshot value differs from the t0 value. Then: when are committee referrals and CRS summaries recorded
relative to introduction, and when was each bill last updated relative to the end of its Congress?

In [140]:
# C11: snapshot fields vs their value at t0=7 (rebuilt from timestamped tables), law-capable bills 108-118.
T0 = 7
b = lb[lb.congress.isin(CONGRESSES)][["bill_id", "congress", "introduced_date", "c_end", "n_cosponsors", "n_actions",
                                       "n_text_versions", "n_summaries", "latest_action_date", "law_type", "update_date"]].copy()
cut = b.set_index("bill_id")["introduced_date"] + pd.Timedelta(days=T0)
acts = load("bill_actions", ("bill_id", "action_date"))
acts = acts[acts.bill_id.isin(cut.index)]
n_act_t0 = acts[acts.action_date <= acts.bill_id.map(cut)].groupby("bill_id").size()
tv = load("bill_text_versions", ("bill_id", "version_ts"))
tv = tv[tv.bill_id.isin(cut.index)]
n_tv_t0 = tv[tv.version_ts.dt.normalize() <= tv.bill_id.map(cut)].groupby("bill_id").size()
n_cosp_t0 = wt[7].set_index("bill_id")["n_seen"]
bi = b.set_index("bill_id")
leak = pd.DataFrame([
    ("n_cosponsors", "count incl. later cosponsors", (bi.n_cosponsors != n_cosp_t0.reindex(bi.index).fillna(0)).mean()),
    ("n_actions", "all actions to date", (bi.n_actions != n_act_t0.reindex(bi.index).fillna(0)).mean()),
    ("n_text_versions", "reported/engrossed/enrolled texts", (bi.n_text_versions != n_tv_t0.reindex(bi.index).fillna(0)).mean()),
    ("latest_action_*", "last action of the bill's life", (bi.latest_action_date > cut).mean()),
    ("law_type/law_number", "became law (outcome)", bi.law_type.notna().mean()),
    ("n_summaries", "CRS summaries (no publish date)", (bi.n_summaries > 1).mean()),
    ("update_date", "record last modified", (bi.update_date > bi.c_end).mean()),
], columns=["field", "encodes", "share_changed_after_t0"])
print(f"Snapshot fields vs value as of t0={T0} days (law-capable bills 108-118)")
print(leak.round(3).to_string(index=False))
print("For law_type: share of bills that became law. For n_summaries: share with > 1 summary (later versions). "
      "For update_date: share modified after the Congress ended.")
ncs = float(leak.set_index("field").loc["n_cosponsors", "share_changed_after_t0"])
record_check(C_SECTION, "C11", "Requirement: rebuild bill-level count/outcome fields as of t0 (snapshot values are post-t0)", None,
             metric=ncs, threshold="INFO (requirement for every split)",
             detail=f"headline: the snapshot n_cosponsors differs from the t0={T0} count for {ncs:.1%} of bills. Share differing per field: "
                    + ", ".join(f"{r.field} ({r.share_changed_after_t0:.1%})" for r in leak.itertuples() if r.field != "update_date")
                    + ". Rebuild as of t0 from timestamped tables: n_cosponsors (bill_cosponsors.sponsorship_date), n_actions / "
                      "latest_action_* (bill_actions.action_date), n_text_versions (bill_text_versions.version_ts). Never use: "
                      "law_type/law_number, n_summaries, update_date (record metadata, not a feature). Applies to every split; "
                      "S5 (random) additionally leaks through same-Congress neighbours (C20).",
             quantity="leakage_snapshot_fields", requirement=True)

Snapshot fields vs value as of t0=7 days (law-capable bills 108-118)
              field                           encodes  share_changed_after_t0
       n_cosponsors      count incl. later cosponsors                  0.4850
          n_actions               all actions to date                  0.3580
    n_text_versions reported/engrossed/enrolled texts                  0.1190
    latest_action_*    last action of the bill's life                  0.3580
law_type/law_number              became law (outcome)                  0.0320
        n_summaries   CRS summaries (no publish date)                  0.1130
        update_date              record last modified                  0.9320
For law_type: share of bills that became law. For n_summaries: share with > 1 summary (later versions). For update_date: share modified after the Congress ended.
[INFO/REQ] C11: Requirement: rebuild bill-level count/outcome fields as of t0 (snapshot values are post-t0) | metric=0.4846 (threshold INFO (requ

In [141]:
# C12: committee referral timing relative to introduction (bill_committees.activity_ts).
bc = load("bill_committees", ("bill_id", "activity", "activity_ts", "congress"))
ref = bc[bc.activity.str.lower().str.startswith("referred to", na=False) & bc.bill_id.isin(lb.bill_id)]
# activity_ts is UTC; convert to US Eastern so an evening referral is not counted as the next day
first_ref = (ref.groupby("bill_id")["activity_ts"].min().dt.tz_localize("UTC")
             .dt.tz_convert("America/New_York").dt.tz_localize(None))
r = lb[lb.congress.isin(CONGRESSES)][["bill_id", "congress", "introduced_date"]].copy()
r["ref_lag"] = (r.bill_id.map(first_ref).dt.normalize() - r.introduced_date).dt.days
reft = r.groupby("congress")["ref_lag"].agg(
    has_referral=lambda s: s.notna().mean(), by_day0=lambda s: (s <= 0).mean(), by_day7=lambda s: (s <= 7).mean(),
    median_lag=lambda s: s.median(), p99_lag=lambda s: s.quantile(0.99))
print("Committee referral timing (first 'Referred to' activity, days after introduction)")
print(reft.round(3).T.to_string())
b0 = float(reft.by_day0.min())
record_check(C_SECTION, "C12", "Committee referral is known at t0=0 for >= 95% of law-capable bills in every Congress",
             bool(b0 >= 0.95), metric=b0, threshold="min share referred by day 0 >= 0.95",
             detail=f"min share by day 0 {b0:.3f}, by day 7 {reft.by_day7.min():.3f}; bills without a referral "
                    f"{1 - reft.has_referral.min():.3f} (max). Referral committees are usable as t0 features if referred "
                    "by t0; other bill_committees activities (markup, hearings, reported) are post-introduction and "
                    "reported-by is the task-2 label itself.")

Committee referral timing (first 'Referred to' activity, days after introduction)
congress        108    109    110    111    112    113    114    115    116    117    118
has_referral 0.9800 0.9860 0.9810 0.9860 0.9850 0.9830 0.9870 0.9920 0.9930 0.9900 0.9930
by_day0      0.9760 0.9800 0.9780 0.9840 0.9810 0.9820 0.9850 0.9920 0.9910 0.9900 0.9830
by_day7      0.9790 0.9840 0.9800 0.9850 0.9830 0.9820 0.9860 0.9920 0.9920 0.9900 0.9870
median_lag   0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000
p99_lag      0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.5500
[PASS] C12: Committee referral is known at t0=0 for >= 95% of law-capable bills in every Congress | metric=0.9762 (threshold min share referred by day 0 >= 0.95)


In [142]:
# C13: CRS summary timing. Only update_date exists (last modification), so update_date <= intro+7 is a lower bound
# on availability at t0. Bulk re-loads of old Congresses make this uninformative before ~115.
sm = load("bill_summaries", ("bill_id", "version_code", "update_date", "congress"))
sm = sm[sm.bill_id.isin(lb.bill_id) & (sm.version_code == "00")].groupby("bill_id")["update_date"].min()
r["sum_lag"] = (r.bill_id.map(sm).dt.normalize() - r.introduced_date).dt.days
sumt = r.groupby("congress")["sum_lag"].agg(
    has_summary=lambda s: s.notna().mean(), within7=lambda s: (s <= 7).mean(), within30=lambda s: (s <= 30).mean(),
    median_lag_days=lambda s: s.median())
print("Introduced-version CRS summary: coverage and last-update lag after introduction (days)")
print(sumt.round(3).T.to_string())
w7 = float(sumt.loc[C_EVAL, "within7"].max())
w30 = float(sumt.loc[C_EVAL, "within30"].max())
record_check(C_SECTION, "C13", "Requirement: do not use CRS summary text as a t0 feature (available by t0=7 for too few bills)", None,
             metric=w7, threshold="requirement (a t0 feature must exist for >= 90% of eval bills at t0; observed max share shown)",
             detail=f"share with a summary last updated within 7 days: {sumt.loc[C_EVAL, 'within7'].round(3).to_dict()}; within 30 days max "
                    f"{w30:.3f}; median lag {sumt.loc[C_EVAL, 'median_lag_days'].to_dict()} days. Why 90%: a feature missing for more than "
                    f"10% of eval bills at t0 is mostly a missing-indicator; here {1 - w7:.0%} of eval bills (the best eval Congress) would "
                    "have no summary at t0=7, while every training bill has one in the snapshot (train/test feature mismatch). Proxy: "
                    "update_date is the LAST edit, so availability may be earlier, but nothing shows summaries exist at t0; use the "
                    "introduced title / text instead. Same for every split, so it does not separate splits.",
             requirement=True)

Introduced-version CRS summary: coverage and last-update lag after introduction (days)
congress             108     109     110     111     112     113     114      115      116      117      118
has_summary       1.0000  1.0000  1.0000  1.0000  1.0000  1.0000  1.0000   0.9980   1.0000   0.9990   0.5730
within7           0.0280  0.1040  0.1450  0.1610  0.2410  0.0950  0.0870   0.0470   0.0260   0.0350   0.0120
within30          0.1500  0.4920  0.4100  0.4970  0.4920  0.4770  0.4000   0.2250   0.1650   0.1480   0.1020
median_lag_days 126.0000 31.0000 40.0000 31.0000 32.0000 33.0000 46.0000 155.0000 183.0000 112.0000 136.0000
[INFO/REQ] C13: Requirement: do not use CRS summary text as a t0 feature (available by t0=7 for too few bills) | metric=0.03503 (threshold requirement (a t0 feature must exist for >= 90% of eval bills at t0; observed max share shown))


### C24, C14. Are the labels of the most recent Congresses fully loaded?

**Why it matters.** 118 is the natural test Congress for task 1, and it looks different: lower positive density (A3),
later non-original cosponsors (C3), and many records still being edited (C14). If cosponsor events or committee-report
actions of 117/118 were still missing from the snapshot, those differences would be data latency, not behaviour, and
118 would understate task-1 and task-2 performance. Three tests: (1) internal consistency: the snapshot counts
(bills.n_cosponsors, n_actions, latest_action_date) against the event tables; (2) staleness: events dated after the
record's own update_date; (3) the end-of-Congress tail: if late events were missing, the share of a Congress's
cosponsorships dated in its last 90 days (and of committee reports in its last 180 days) would be too low.

**How to read it.** Mismatch and stale shares should be 0; the tail shares of 117/118 are judged with `rel_to_train`
(lower tail: only a deficit signals missing events). The last line compares, inside 118, how often bills with and
without cosponsor rows were modified after the Congress ended (post-end edits driven by cosponsor loading would differ).

In [143]:
# C24: label completeness per Congress: snapshot counts vs event tables, events after update_date, end-of-Congress tails.
bcs = load("bill_cosponsors", ("bill_id", "sponsorship_date", "withdrawn_date"))
bcs = bcs[bcs.bill_id.isin(cut.index)]
lc24 = lb[lb.congress.isin(CONGRESSES)][["bill_id", "congress", "c_end", "n_cosponsors", "n_actions", "latest_action_date",
                                          "update_date"]].set_index("bill_id")
lc24["ud"] = lc24.update_date.dt.normalize()
rows_all = bcs.groupby("bill_id").size().reindex(lc24.index).fillna(0)
rows_act = bcs[bcs.withdrawn_date.isna()].groupby("bill_id").size().reindex(lc24.index).fillna(0)
act_rows = acts.groupby("bill_id").size().reindex(lc24.index).fillna(0)
act_max = acts.groupby("bill_id").action_date.max().reindex(lc24.index)
ev_max = pd.concat([act_max, bcs.groupby("bill_id").sponsorship_date.max().reindex(lc24.index),
                    bcs.groupby("bill_id").withdrawn_date.max().reindex(lc24.index)], axis=1).max(axis=1)
lc24["cosp_count_mismatch"] = lc24.n_cosponsors != rows_all
lc24["action_count_mismatch"] = lc24.n_actions != act_rows
lc24["latest_action_mismatch"] = lc24.latest_action_date.dt.normalize() != act_max.dt.normalize()
lc24["event_after_update"] = ev_max > lc24.ud
comp = lc24.groupby("congress")[["cosp_count_mismatch", "action_count_mismatch", "latest_action_mismatch", "event_after_update"]].mean()
bcs_c = bcs.merge(lc24[["congress", "c_end"]], left_on="bill_id", right_index=True)
act_c = bcs_c[bcs_c.withdrawn_date.isna()]
comp["withdrawn_row_share"] = bcs_c.groupby("congress").withdrawn_date.agg(lambda v: v.notna().mean())
comp["cosp_last90"] = (act_c.sponsorship_date > act_c.c_end - pd.Timedelta(days=90)).groupby(act_c.congress).mean()
rp = adv[adv.congress.isin(CONGRESSES) & adv.reported_date.notna()].copy()
rp["c_end"] = rp.congress.map(congress_end)
comp["report_last180"] = (rp.reported_date > rp.c_end - pd.Timedelta(days=180)).groupby(rp.congress).mean()
print("Label completeness per Congress (law-capable bills; mismatch / stale shares should be 0)")
print(comp.T.to_string(float_format=lambda v: f"{v:.4f}"))
tail_c_tab, tail_c_info = rel_to_train(comp["cosp_last90"], side="lower")
tail_r_tab, tail_r_info = rel_to_train(comp["report_last180"], side="lower")
cons_bad = float(comp[["cosp_count_mismatch", "action_count_mismatch", "latest_action_mismatch", "event_after_update"]].max().max())
tail_fail = bool(tail_c_tab.loc[tail_c_tab.role == "cand", "fail"].any() or tail_r_tab.loc[tail_r_tab.role == "cand", "fail"].any())
x118 = lc24[lc24.congress == CONGRESSES[-1]].copy()
x118["has_cosp"] = rows_all.reindex(x118.index) > 0
x118["post_end"] = x118.ud > x118.c_end
x118["recent"] = x118.update_date >= SNAPSHOT_DATE - pd.Timedelta(days=365)
pe = x118.groupby("has_cosp")[["post_end", "recent"]].mean()
print(f"{CONGRESSES[-1]}: modified after Congress end / in last 365 d of snapshot, bills with cosponsor rows "
      f"{pe.loc[True, 'post_end']:.3f} / {pe.loc[True, 'recent']:.3f} vs without {pe.loc[False, 'post_end']:.3f} / {pe.loc[False, 'recent']:.3f}")
c24_fail = bool(cons_bad > 0 or tail_fail)
record_check(C_SECTION, "C24", "Labels of the most recent Congresses (cosponsor events, committee reports) are fully loaded",
             not c24_fail, metric=float(min(tail_c_tab.loc[tail_c_tab.role == "cand", "t_pi"].min(),
                                            tail_r_tab.loc[tail_r_tab.role == "cand", "t_pi"].min())),
             threshold=f"consistency/stale shares = 0 in every Congress AND end-of-Congress tail shares not below the PI ({tail_c_info['rule']}); "
                       "metric = min t of the two tails over candidates",
             detail=f"max consistency/stale share {cons_bad:.4f} " + ("(n_cosponsors equals bill_cosponsors rows INCLUDING withdrawn; n_actions "
                    "and latest_action_date match bill_actions; no event is dated after its record's update_date, so no record is older than "
                    "its own events). " if cons_bad == 0 else f"(per-Congress maxima: {comp.iloc[:, :4].max().round(4).to_dict()}). ")
                    + f"Cosponsorships in the last 90 days: {rel_detail(tail_c_tab, tail_c_info)}. Committee reports in the last 180 "
                    f"days: {rel_detail(tail_r_tab, tail_r_info)}. A share ABOVE the reference is not latency but a timing shift (more committee reports late in the "
                    f"Congress in {[int(c) for c in tail_r_tab.index[(tail_r_tab.role == 'cand') & (tail_r_tab.t_pi > tail_r_info['tcrit'])]]}): it "
                    "shortens the effective 180-day window for late bills and belongs to A's task-2 drift checks. Inside {CONGRESSES[-1]}, post-end / last-year edits for bills with vs without "
                    f"cosponsor rows: {pe.loc[True, 'post_end']:.3f}/{pe.loc[True, 'recent']:.3f} vs {pe.loc[False, 'post_end']:.3f}/"
                    f"{pe.loc[False, 'recent']:.3f}. Withdrawn rows: {CONGRESSES[-1]} {comp.loc[CONGRESSES[-1], 'withdrawn_row_share']:.4f} vs 108-117 max "
                    f"{comp.loc[CONGRESSES[:-1], 'withdrawn_row_share'].max():.4f} (more withdrawals recorded; cosp drops them, so labels are "
                    "unaffected). Limits: the counts are parsed from the same BILLSTATUS records as the event tables, so (1) checks parsing, "
                    "not an external source; (3) is the real latency test. If PASS: the 118 density / lag differences (A3, C3) are "
                    "behaviour, and 118 is a valid test Congress. If FAIL: drop or down-weight 118 as a test Congress until reloaded. "
                    f"S4 is judged on the candidate Congresses its slices test ({S4_EVAL}): incomplete labels would hit its end-of-Congress "
                    "test slice hardest.",
             **dict(zip(("by_split", "na_reason"), full_split(combine_by_split(rel_by_split(tail_c_tab), rel_by_split(tail_r_tab)),
                                                              ok=cand_ok(tail_c_tab, tail_r_tab)))))

Label completeness per Congress (law-capable bills; mismatch / stale shares should be 0)
congress                  108    109    110    111    112    113    114    115    116    117    118
cosp_count_mismatch    0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000
action_count_mismatch  0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000
latest_action_mismatch 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000
event_after_update     0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000 0.0000
withdrawn_row_share    0.0012 0.0012 0.0009 0.0008 0.0012 0.0007 0.0007 0.0005 0.0004 0.0004 0.0034
cosp_last90            0.0255 0.0133 0.0045 0.0197 0.0209 0.0293 0.0252 0.0385 0.0442 0.0416 0.0437
report_last180         0.2590 0.2672 0.2447 0.3467 0.2769 0.3423 0.2695 0.3024 0.3377 0.4237 0.4452
118: modified after Congress end / in last 365 d of snapshot, bills with cosponsor rows 0.758 / 0.317 vs withou

In [144]:
# C14: bill record update_date relative to Congress end and to the snapshot date (how "final" each Congress is).
snap = bills["update_date"].max()
u = lb[lb.congress.isin(CONGRESSES + [IN_PROGRESS])].copy()
u["days_after_end"] = (u.update_date - u.c_end).dt.days
u["recent"] = u.update_date >= snap - pd.Timedelta(days=365)
updt = u.groupby("congress").agg(after_end=("days_after_end", lambda s: (s > 0).mean()),
                                 median_days_after_end=("days_after_end", "median"),
                                 updated_last_365d=("recent", "mean"))
print(f"Bill record update_date vs Congress end (snapshot date = max update_date = {snap.date()})")
print(updt.round(3).T.to_string())
rec118, rec_old = float(updt.loc[118, "updated_last_365d"]), float(updt.loc[C_TRAIN, "updated_last_365d"].mean())
record_check(C_SECTION, "C14", f"Recent-edit share of {CONGRESSES[-1]} records (INFO; becomes a split verdict only if C24 finds incomplete labels)",
             (False if c24_fail else None), metric=rec118,
             threshold="INFO unless C24 fails (then: 118 records unsettled = labels may still change)",
             detail=f"share of bills updated within 365 days of the snapshot: 118={rec118:.3f}, 117={updt.loc[117, 'updated_last_365d']:.3f}, "
                    f"108-115 mean={rec_old:.3f}. Every Congress's records post-date its end, confirming bill-level fields reflect "
                    "end-of-life state (C11). Recent edits alone do not mean missing labels: C24 finds "
                    + ("incomplete labels, so 118 records are unsettled and splits testing on 118 are flagged. " if c24_fail else
                       "no missing cosponsor events or committee reports, so the edits concern annotations (summaries, subjects: C15) "
                       "that no t0 model may use; no split verdict. ")
                    + "The old fixed band (train mean + 0.10) compared against a reference that is zero by construction (old records "
                      "are never edited) and was dropped.",
             by_split=(rel_by_split(updt.loc[CONGRESSES, "updated_last_365d"] > rec_old + 0.10) if c24_fail else None))

Bill record update_date vs Congress end (snapshot date = max update_date = 2026-10-17)
congress                     108        109        110        111        112        113        114        115      116    117      118       119
after_end                 1.0000     1.0000     1.0000     1.0000     1.0000     1.0000     1.0000     1.0000   1.0000 0.6950   0.7510    0.0000
median_days_after_end 6,586.0000 5,854.0000 5,121.0000 4,391.0000 3,273.0000 2,930.0000 2,199.0000 1,469.0000 667.0000 8.0000 144.0000 -164.0000
updated_last_365d         0.0000     0.0000     0.0000     0.0000     0.0000     0.0000     0.0000     0.0000   0.0000 0.0000   0.2920    0.8880
[INFO] C14: Recent-edit share of 118 records (INFO; becomes a split verdict only if C24 finds incomplete labels) | metric=0.292 (threshold INFO unless C24 fails (then: 118 records unsettled = labels may still change))


### C15-C16. Data completeness and annotation latency

**Why it matters.** Policy areas, subjects, summaries and committee referrals are added by CRS and
clerks after introduction. If the newest Congress (118, the natural test set) is less annotated than
older ones, a model that uses those features is tested on degraded inputs, and the test score
understates (or misstates) real performance. Congress 119, which is still in progress, lets us measure
how fast annotations arrive: compare coverage for young versus old bills.

**How to read it.** The line chart shows per-Congress coverage of each annotation for law-capable bills.
The second table shows, for Congress 119 only, coverage by bill age at the snapshot date.

In [145]:
# C15: coverage of policy_area, subjects, summaries, committee referral per Congress (law-capable bills).
subj_ids = set(load("bill_subjects", ("bill_id",))["bill_id"].unique())
sum_ids = set(load("bill_summaries", ("bill_id",))["bill_id"].unique())
ref_ids = set(first_ref.index)
cv = lb[["bill_id", "congress", "policy_area", "introduced_date"]].copy()
cv["policy_area"] = cv.policy_area.notna()
cv["subjects"] = cv.bill_id.isin(subj_ids)
cv["summary"] = cv.bill_id.isin(sum_ids)
cv["referral"] = cv.bill_id.isin(ref_ids)
FIELDS = ["policy_area", "subjects", "summary", "referral"]
covt = cv.groupby("congress")[FIELDS].mean()
print("Annotation coverage per Congress (law-capable bills; 119 in progress)")
print(covt.round(3).T.to_string())
cl2 = covt.loc[CONGRESSES].reset_index().melt("congress", var_name="annotation", value_name="coverage")
line_chart(cl2, "congress", "coverage", series="annotation", yfmt=".0%", yzero=False,
           title="Share of law-capable bills with each annotation, per Congress",
           subtitle="Coverage in the snapshot; 119 excluded (in progress)", xlabel="Congress", name="C_coverage")
# Verdicts per field with the shared rel_to_train rule (lower tail: only missing annotations hurt). Only fields that can be
# t0 features count for splits: referral (known at day 0, C12) and policy area IF task 2 predicts after ~30 days (C25).
cov_tabs = {f: rel_to_train(covt.loc[CONGRESSES, f], side="lower") for f in FIELDS}
cov_fail = {f: bool(t.loc[t.role == "cand", "fail"].any()) for f, (t, _) in cov_tabs.items()}
cov_txt = {f: rel_detail(*cov_tabs[f]) for f in FIELDS}
record_check(C_SECTION, "C15a", "Requirement: legislative subjects are not a t0 feature (coverage of recent Congresses still filling in)", None,
             metric=float(covt.loc[C_EVAL, "subjects"].min()), threshold="requirement (INFO; subjects are not available at any t0 <= 90 d, C16/C25)",
             detail=f"subjects coverage: {cov_txt['subjects']}. CRS adds subjects over months ({IN_PROGRESS} bills aged 31-90 d: "
                    f"{cv[(cv.congress == IN_PROGRESS)].pipe(lambda d: d[(SNAPSHOT_DATE - d.introduced_date).dt.days.between(31, 90)].subjects.mean()):.3f}), "
                    "so no t0 model may use them and the 117/118 deficit cannot bias a correctly built model. If subjects were used as "
                    "static bill attributes, the recent test Congresses would be scored on degraded inputs (and train would leak post-t0 "
                    "information): do not.", quantity="subject_coverage", requirement=True)
c15b_fields = ["referral", "policy_area"]
record_check(C_SECTION, "C15b", "Coverage of t0-usable annotations (referral; policy area for task 2 at t0 >= 30 d) in eval Congresses within train noise",
             not any(cov_fail[f] for f in c15b_fields), metric=float(covt.loc[C_EVAL, c15b_fields].min().min()),
             threshold=f"{cov_tabs['referral'][1]['rule']}, per field",
             detail=f"referral: {cov_txt['referral']}. Policy area (counts only for task 2 with t0 >= 30 d, C25; for task 1 and task 2 at t0 <= 7 it "
                    f"must be masked, C18): {cov_txt['policy_area']}. CRS summaries are excluded from this verdict (never a t0 feature, C13; "
                    f"coverage shown in C15c). Task-2 t0 assumed: >= 30 d for the policy-area part, any t0 for referral. S4 judged on "
                    f"the candidate Congresses its slices test ({S4_EVAL}).",
             quantity="annotation_coverage", **dict(zip(("by_split", "na_reason"), full_split(
                 combine_by_split(*(rel_by_split(cov_tabs[f][0]) for f in c15b_fields)),
                 ok=cand_ok(*(cov_tabs[f][0] for f in c15b_fields))))))
record_check(C_SECTION, "C15c", "CRS-summary coverage of eval Congresses (INFO: summaries are not a t0 feature)", None,
             metric=float(covt.loc[C_EVAL, "summary"].min()), threshold="INFO",
             detail=f"summary: {cov_txt['summary']}. The {CONGRESSES[-1]} deficit is annotation latency (C16: CRS writes summaries months "
                    "after introduction) in a field no t0 model may use (C13), so it does not penalise splits that test on 118.")

Annotation coverage per Congress (law-capable bills; 119 in progress)
congress       108    109    110    111    112    113    114    115    116    117    118    119
policy_area 1.0000 1.0000 1.0000 0.9990 0.9980 1.0000 0.9960 0.9940 0.9950 0.9960 0.9990 0.9910
subjects    1.0000 1.0000 1.0000 0.9900 0.9940 0.9930 0.9620 0.9640 0.9840 0.6180 0.3720 0.2370
summary     1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 1.0000 0.9990 0.5780 0.3080
referral    0.9800 0.9860 0.9810 0.9860 0.9850 0.9830 0.9870 0.9920 0.9930 0.9900 0.9930 0.9930


[INFO/REQ] C15a: Requirement: legislative subjects are not a t0 feature (coverage of recent Congresses still filling in) | metric=0.3722 (threshold requirement (INFO; subjects are not available at any t0 <= 90 d, C16/C25))
[PASS] C15b: Coverage of t0-usable annotations (referral; policy area for task 2 at t0 >= 30 d) in eval Congresses within train noise | metric=0.9904 (threshold outside 95% t prediction interval of ref 108-115 (lower tail; n=8), per field)
[INFO] C15c: CRS-summary coverage of eval Congresses (INFO: summaries are not a t0 feature) | metric=0.5776 (threshold INFO)


In [146]:
# C16: annotation latency measured on in-progress Congress 119: coverage by bill age at the snapshot date.
y = cv[cv.congress == IN_PROGRESS].copy()
y["age_days"] = (snap.normalize() - y.introduced_date).dt.days
y["age"] = pd.cut(y.age_days, [-1, 7, 30, 90, 180, 365, 10_000], labels=["0-7", "8-30", "31-90", "91-180", "181-365", ">365"])
lat = y.groupby("age", observed=True).agg(bills=("bill_id", "size"), **{f: (f, "mean") for f in FIELDS})
print(f"Congress {IN_PROGRESS}: coverage by bill age at snapshot ({snap.date()})")
print(lat.round(3).to_string())
young = (y[y.age_days <= 30][FIELDS]).mean()
gap = (lat.iloc[-1][FIELDS] - young)
record_check(C_SECTION, "C16", "Annotations exist within 30 days of introduction (119 bills aged 0-30 d vs > 365 d)", None,
             metric=f"max coverage gap {gap.max():.3f} ({gap.idxmax()})",
             detail=f"coverage for bills aged 0-30 d: {young.round(3).to_dict()}; gap vs oldest bucket {gap.round(3).to_dict()}. "
                    "Informational proxy (snapshot of 119 only, not the historical arrival time): a large gap means the "
                    "annotation is typically assigned after t0 and must not be a t0 feature without a timestamp.")

Congress 119: coverage by bill age at snapshot (2026-10-17)
         bills  policy_area  subjects  summary  referral
age                                                     
0-7         56       0.1430    0.0180   0.0000    0.8210
8-30       585       0.8600    0.0100   0.0460    0.9900
31-90     1123       0.9920    0.0440   0.0850    0.9970
91-180    2160       1.0000    0.1250   0.1100    0.9880
181-365   4046       0.9990    0.1540   0.1630    0.9930
>365      8977       0.9980    0.3420   0.4680    0.9950
[INFO] C16: Annotations exist within 30 days of introduction (119 bills aged 0-30 d vs > 365 d) | max coverage gap 0.426 (summary)


### C18. Is the policy area known at prediction time?

**Why it matters.** Sections B and D use the CRS policy area as a t0 feature (D's empirical-Bayes rate per
policy area, B's bill attributes). In the snapshot almost every bill of 108-118 has one, but CRS assigns it
*after* introduction, and the historical assignment date is not in the data. The in-progress Congress 119 is
the only window onto that lag: if bills a few days old often lack a policy area, then a model trained on
complete snapshots sees a feature at t0 that it would not have in deployment.

**How to read it.** Policy-area missingness of 119 bills by age at the snapshot, from the shared
`PA_MISSING_AT_T0` (so sections B, C and D quote the same numbers; 0-7 and 8-30 days matter for t0 = 0 and 7),
and whether the youngest bills were introduced during a recess (floor actions per day from `bill_actions`).
Proxy: CRS's current speed, not the speed in earlier Congresses.

In [147]:
# C18: policy-area availability at t0 from the shared proxy PA_MISSING_AT_T0 (setup; 119 bills by age at SNAPSHOT_DATE),
# plus how representative the youngest bucket's introduction window is (recess check from bill_actions dates).
pm_t = PA_MISSING_AT_T0
print(f"policy area missing by age at snapshot (shared PA_MISSING_AT_T0, {IN_PROGRESS} law-capable bills): "
      + ", ".join(f"{i} {r.share_missing:.1%} (n={int(r.n)})" for i, r in pm_t.iterrows()))
young_lo = SNAPSHOT_DATE - pd.Timedelta(days=int(pm_t.loc["0-7d", "age_hi"]))
fa = load("bill_actions", ("action_date", "action_type", "congress"))
fa = fa[(fa.congress == IN_PROGRESS) & (fa.action_type == "Floor")]
fl_day = fa.groupby("action_date").size().reindex(pd.date_range(congress_start(IN_PROGRESS), SNAPSHOT_DATE), fill_value=0)
intro_day = lb[lb.congress == IN_PROGRESS].groupby("introduced_date").size().reindex(fl_day.index, fill_value=0)
win = slice(young_lo, SNAPSHOT_DATE)
roll_fl, roll_in = fl_day.rolling(8).mean(), intro_day.rolling(8).mean()
rec = dict(floor_per_day=float(fl_day[win].mean()), floor_median_8d=float(roll_fl.median()),
           intro_per_day=float(intro_day[win].mean()), intro_median_8d=float(roll_in.median()))
print(f"youngest bucket introduced {young_lo.date()}..{SNAPSHOT_DATE.date()}: floor actions/day {rec['floor_per_day']:.1f} "
      f"(119 median 8-day window {rec['floor_median_8d']:.1f}), law-capable introductions/day {rec['intro_per_day']:.1f} "
      f"(median {rec['intro_median_8d']:.1f})")
pa07, pa830 = 1 - PA_MISS_T0, 1 - float(pm_t.loc["8-30d", "share_missing"])
hist_missing = float((~lb.loc[lb.congress.isin(CONGRESSES), "policy_area"].notna()).mean())
recess = rec["floor_per_day"] < 0.5 * rec["floor_median_8d"]
pa_ok = bool(min(pa07, pa830) >= 0.95)
record_check(C_SECTION, "C18", "Requirement: mask policy area at t0 <= 7 d (present for < 95% of 119 bills aged 0-7 and 8-30 days)", None,
             metric=float(min(pa07, pa830)),
             threshold=f"requirement (INFO; the rule binds while availability < 0.95 in either age bucket: {'binds' if not pa_ok else 'does not bind'})",
             detail=f"shared proxy PA_MISSING_AT_T0: present for {pa07:.1%} of 0-7 d bills (n={PA_MISS_T0_N}) and {pa830:.1%} of "
                    f"8-30 d bills (n={int(pm_t.loc['8-30d', 'n'])}); 108-118 snapshot missing share {hist_missing:.3f}. Small sample, "
                    f"and the youngest bills were introduced {young_lo.date()}..{SNAPSHOT_DATE.date()}, when the chambers logged "
                    f"{rec['floor_per_day']:.1f} floor actions/day vs a 119 median of {rec['floor_median_8d']:.1f}"
                    f"{' (a recess / pro-forma period)' if recess else ''}: CRS latency then may not reflect normal weeks. Coverage "
                    "at the same age for non-recess months cannot be measured (policy_area has no timestamp; every older 119 bill "
                    "is already older than 30 d). Even the 8-30 d bucket is below 95%, so the conclusion holds: the snapshot policy "
                    "area is partly a post-t0 field. Train with policy area masked at PA_MISS_T0 plus a missing-indicator, or derive "
                    "the topic from the introduced title/text; applies to D's eb_rate baseline and GNN bill features, in every split. "
                    "Recorded as a requirement: availability at t0 does not depend on the split, so it is not a split failure (same "
                    "rule as C25, B5b and the shared PA_MISS_T0).", requirement=True)

policy area missing by age at snapshot (shared PA_MISSING_AT_T0, 119 law-capable bills): 0-7d 85.7% (n=56), 8-30d 14.0% (n=585), 31-90d 0.8% (n=1123), 91-180d 0.0% (n=2160), >180d 0.2% (n=13023)
youngest bucket introduced 2026-10-10..2026-10-17: floor actions/day 5.6 (119 median 8-day window 15.6), law-capable introductions/day 7.0 (median 26.4)
[INFO/REQ] C18: Requirement: mask policy area at t0 <= 7 d (present for < 95% of 119 bills aged 0-7 and 8-30 days) | metric=0.1429 (threshold requirement (INFO; the rule binds while availability < 0.95 in either age bucket: binds))


### C25. Which prediction time t0 should task 2 use?

**Why it matters.** Task 2 (reported within 180 days) has no fixed prediction time. Whether the policy area, referral
details and early cosponsors are legitimate features depends on it, and that decides whether the latency checks (C15b's
policy-area part, C18, A13 with vs without policy area) constrain the split at all. A later t0 makes more features
available but "spends" part of the label: bills already reported by t0 are no longer a prediction.

**How to read it.** For candidate t0 values: availability at that bill age of each feature. Timestamped features
(referral, cosponsors, actions) use the worst Congress in 108-118; untimestamped ones (policy area, subjects) use 119
bills of about that age at the snapshot (small-sample proxy, n shown); summaries use the update_date lower bound. The
last rows give the share of task-2 positives already reported by t0 (worst Congress), i.e. label lost by waiting.

In [148]:
# C25: feature availability by task-2 prediction time t0 (historical where timestamped, 119 age proxy otherwise) + label spent.
T0S = (0, 7, 30, 60)
AGE_WIN = {0: (0, 2), 7: (5, 9), 30: (25, 35), 60: (50, 70)}   # 119 bill ages used as proxy for "age = t0"
pos2 = adv[adv.congress.isin(CONGRESSES) & ~adv.censored & (adv.label == 1)].copy()
pos2["rep_lag"] = (pos2.reported_date - pos2.introduced_date).dt.days
spent = pd.DataFrame({t: (pos2.rep_lag <= t).groupby(pos2.congress).mean() for t in T0S})
ref_by = pd.DataFrame({t: (r.ref_lag <= t).groupby(r.congress).mean() for t in T0S})
sum_by = pd.DataFrame({t: (r.sum_lag <= t).groupby(r.congress).mean() for t in T0S})
av_rows = {"cosponsors / actions so far (timestamped)": {t: 1.0 for t in T0S},
           "referral committee (min 108-118)": ref_by.min().to_dict(),
           "CRS summary (lower bound, min 116-118)": sum_by.loc[C_EVAL].min().to_dict()}
n_win = {}
for f in ("policy_area", "subjects"):
    av_rows[f"{f} (119 proxy)"] = {}
    for t in T0S:
        m = y.age_days.between(*AGE_WIN[t])
        av_rows[f"{f} (119 proxy)"][t] = float(y.loc[m, f].mean()) if m.any() else np.nan
        n_win[t] = int(m.sum())
av_rows["task-2 positives already reported (max 108-118)"] = spent.max().to_dict()
av = pd.DataFrame(av_rows).T
av.columns = [f"t0={t} (n119={n_win[t]})" for t in T0S]
print("Feature availability at bill age t0 (share of law-capable bills) and task-2 label spent by t0")
print(av.round(3).to_string())
print("Share of task-2 positives reported by t0, per Congress")
print(spent.T.rename(index=lambda t: f"t0={t}").round(3).to_string())
pa_ok = [t for t in T0S if av_rows["policy_area (119 proxy)"][t] >= 0.95]
t0_pa = pa_ok[0] if pa_ok else None
spent_pa = float(spent[t0_pa].max()) if t0_pa is not None else np.nan
record_check(C_SECTION, "C25", "Requirement: fix task 2's prediction time t0 and use only features available at that age", None,
             metric=t0_pa, threshold="INFO (design decision; metric = smallest candidate t0 at which policy area is present for >= 95% of 119 bills)",
             detail=f"availability by t0 {list(T0S)}: " + "; ".join(f"{k}: " + "/".join(f"{v:.2f}" for v in row.values) for k, row in av.iterrows())
                    + f". Policy area reaches 95% at t0={t0_pa} (proxy, n={n_win.get(t0_pa)}), where {spent_pa:.1%} of task-2 positives (worst "
                    f"Congress) are already reported. Options: (a) t0=0 or 7 without policy area (C18 masking; A13 'noPA' and D's noPA "
                    f"variants are the relevant numbers; C15b's policy-area part and C18 then do not constrain the split beyond masking); "
                    f"(b) t0={t0_pa} with policy area, excluding bills reported by t0 and redefining the window as (t0, 180] (C6's censoring "
                    "unchanged if the window stays anchored at introduction). Tags: C6 assumes the window starts at introduction (any t0 "
                    f"<= 30); C15b policy-area part assumes t0 >= 30; C18 assumes t0 in (0, 7) for both tasks; C12 referral is usable at "
                    "any t0. A/D task-2 checks should state which t0 they assume (requested via setup).",
             requirement=True)

Feature availability at bill age t0 (share of law-capable bills) and task-2 label spent by t0
                                                 t0=0 (n119=41)  t0=7 (n119=27)  t0=30 (n119=261)  t0=60 (n119=117)
cosponsors / actions so far (timestamped)                1.0000          1.0000            1.0000            1.0000
referral committee (min 108-118)                         0.9760          0.9790            0.9790            0.9800
CRS summary (lower bound, min 116-118)                   0.0000          0.0120            0.1020            0.1680
policy_area (119 proxy)                                  0.1220          0.2590            0.9690            0.9910
subjects (119 proxy)                                     0.0000          0.1480            0.0270            0.0770
task-2 positives already reported (max 108-118)          0.0020          0.0460            0.2470            0.4410
Share of task-2 positives reported by t0, per Congress
congress    108    109    110    111   

### C17, C22. Member turnover and the point-in-time candidate pool

**Why it matters.** The legacy pool per bill is "everyone who served in the chamber that Congress". Members
who died, resigned or were seated mid-Congress are then candidates for bills introduced while they were out of
office: guaranteed negatives that change the positive density and can take top-k slots from real candidates
(a member who resigned in 118 still has a strong 117 history). This is a construction requirement for every
split, not something that separates splits, but its size tells us whether D's backtest and the labels must
switch to the shared point-in-time pool (`window_targets(point_in_time=True)` / `ROSTER_ASOF`).

**How to read it.** C17: share of the legacy pool not in office in the month of `introduced_date + 7` (per
Congress and chamber), and the share of task-1 positives who are outside the point-in-time pool (delegates and
roster approximation errors: unrankable positives). C22: for S3 test folds, Recall@20 of a simple history
ranker (sponsor-cosponsor affinity in the fit Congresses + recent activity) and the positive density, with the
full-Congress roster vs the point-in-time pool.

In [149]:
# C17: legacy full-Congress pool vs point-in-time pool (ROSTER_ASOF month of introduced_date + 7), t0 = 7, uncensored bills.
wp7 = window_targets(7, point_in_time=True)
pp = wt[7][["bill_id", "congress", "chamber", "censored", "n_pos", "pool", "eligible"]].merge(
    wp7[["bill_id", "pool", "eligible", "pool_month"]], on="bill_id", suffixes=("_full", "_pit"))
pp = pp[~pp.censored & pp.congress.isin(CONGRESSES) & (pp.eligible_full > 0) & (pp.eligible_pit > 0)]
pp["out_share"] = 1 - pp.pool_pit / pp.pool_full
pp["dens_full"], pp["dens_pit"] = pp.n_pos / pp.eligible_full, pp.n_pos / pp.eligible_pit
pp["ch"] = pp.chamber.str[:5]
# Positives outside the point-in-time pool (unrankable there): delegates vs other.
pos7 = cosp[(cosp.lag_days > 7) & (cosp.lag_days <= 37)][["bill_id", "bioguide_id"]].merge(
    pp[["bill_id", "congress", "chamber", "pool_month"]], on="bill_id")
rk = pd.MultiIndex.from_frame(ROSTER_ASOF[["month", "chamber", "bioguide_id"]])
pos7["in_pit"] = pd.MultiIndex.from_arrays([pos7.pool_month, pos7.chamber, pos7.bioguide_id]).isin(rk)
deleg = set(load("member_terms", ("bioguide_id", "member_type")).query("member_type in @NONVOTING_TYPES").bioguide_id)
pos7["delegate"] = pos7.bioguide_id.isin(deleg)
turn = pp.groupby(["congress", "ch"]).out_share.mean().unstack("ch")
turn.columns = [f"out_of_office_{c}" for c in turn.columns]
turn["pos_outside_pit"] = pos7.groupby("congress").in_pit.agg(lambda v: 1 - v.mean())
turn["of_which_delegates"] = pos7[~pos7.in_pit].groupby("congress").delegate.mean()
print("Share of the legacy pool not in office at introduced_date + 7 (mean over bills), and task-1 positives outside the pit pool")
print(turn.round(3).T.to_string())
mx = float(turn[[c for c in turn.columns if c.startswith("out_of_office")]].max().max())
record_check(C_SECTION, "C17", "Requirement: eligible pool = members in office at introduced_date + t0 (not the full-Congress roster)", None,
             metric=mx, threshold="INFO (construction requirement for every split; effect quantified in C22)",
             detail=f"legacy pool share out of office: max {mx:.3f}, House mean {turn.filter(like='House').mean().iloc[0]:.3f}, Senate mean "
                    f"{turn.filter(like='Senat').mean().iloc[0]:.3f}. Positives outside the point-in-time pool: "
                    f"{1 - pos7.in_pit.mean():.2%} overall ({pos7[~pos7.in_pit].delegate.mean():.0%} of them delegates, who are excluded from "
                    "ROSTER_ASOF as non-voting): count them as misses or add delegates to the pool explicitly. Same requirement for "
                    "all splits, so it does not separate them.", requirement=True)

Share of the legacy pool not in office at introduced_date + 7 (mean over bills), and task-1 positives outside the pit pool
congress               108    109    110    111    112    113    114    115    116    117    118
out_of_office_House 0.0090 0.0130 0.0270 0.0220 0.0200 0.0180 0.0140 0.0350 0.0190 0.0320 0.0250
out_of_office_Senat 0.0000 0.0140 0.0200 0.0970 0.0220 0.0480 0.0000 0.0480 0.0200 0.0200 0.0480
pos_outside_pit     0.0160 0.0100 0.0110 0.0170 0.0140 0.0130 0.0120 0.0140 0.0200 0.0190 0.0230
of_which_delegates  0.9970 0.9430 1.0000 0.9330 0.9720 0.9570 1.0000 0.9930 0.9980 0.9950 0.8150
[INFO/REQ] C17: Requirement: eligible pool = members in office at introduced_date + t0 (not the full-Congress roster) | metric=0.09655 (threshold INFO (construction requirement for every split; effect quantified in C22))


In [150]:
# C22: Recall@20 (t0 = 7) of a history ranker with the full-Congress roster vs the point-in-time pool, S3 test folds.
FULL_ROSTER = members.dropna(subset=["chamber"]).groupby(["congress", "chamber"]).bioguide_id.agg(set)
C22_FOLDS = [u for u in iter_eval_units("S3") if u["role"] == "test" and u["eval_congress"] in (115, CONGRESSES[-1])]


def recall20(unit, ch, k=20):
    """Mean Recall@k over uncensored eval bills with >= 1 positive, for the full roster and the point-in-time pool."""
    c, fit = unit["eval_congress"], unit["train"]
    E = pp[(pp.congress == c) & (pp.chamber == ch) & (pp.n_pos > 0)].merge(bills[["bill_id", "sponsor_bioguide_id"]], on="bill_id")
    months = ROSTER_ASOF[(ROSTER_ASOF.congress == c) & (ROSTER_ASOF.chamber == ch)]
    U = pd.Index(sorted(FULL_ROSTER.get((c, ch), set()) | set(months.bioguide_id)))
    hist = cosp[cosp.congress.isin(fit) & cosp.law_capable]
    aff = hist[hist.sponsor_bioguide_id.isin(E.sponsor_bioguide_id)].groupby(["sponsor_bioguide_id", "bioguide_id"]).size()
    spon = pd.Index(E.sponsor_bioguide_id.dropna().unique())
    A = np.zeros((len(spon), len(U)))
    ai, aj = spon.get_indexer(aff.index.get_level_values(0)), U.get_indexer(aff.index.get_level_values(1))
    ok = (ai >= 0) & (aj >= 0)
    A[ai[ok], aj[ok]] = aff.to_numpy()[ok]
    pop = hist[hist.congress == max(fit)].groupby("bioguide_id").size().reindex(U).fillna(0).to_numpy()
    si = spon.get_indexer(E.sponsor_bioguide_id)
    S = np.where(si[:, None] >= 0, A[np.clip(si, 0, None)], 0.0) + 1e-3 * pop[None, :]
    rows = pd.Index(E.bill_id)
    ev = cosp[cosp.bill_id.isin(rows)]
    seen, posv = ev[ev.lag_days <= 7], ev[(ev.lag_days > 7) & (ev.lag_days <= 37)]
    S[np.arange(len(E))[U.get_indexer(E.sponsor_bioguide_id) >= 0], U.get_indexer(E.sponsor_bioguide_id)[U.get_indexer(E.sponsor_bioguide_id) >= 0]] = -np.inf
    r_, c_ = rows.get_indexer(seen.bill_id), U.get_indexer(seen.bioguide_id)
    S[r_[c_ >= 0], c_[c_ >= 0]] = -np.inf
    P = np.zeros(S.shape, bool)
    r_, c_ = rows.get_indexer(posv.bill_id), U.get_indexer(posv.bioguide_id)
    P[r_[c_ >= 0], c_[c_ >= 0]] = True
    npos = E.n_pos.to_numpy()
    full_mask = U.isin(list(FULL_ROSTER.get((c, ch), set())))
    mm = months.groupby("month").bioguide_id.agg(set)
    pit_mask = np.stack([U.isin(list(mm.get(m, set()))) for m in E.pool_month])
    dmask = full_mask & U.isin(list(deleg))   # delegates in the legacy roster (non-voting, excluded from ROSTER_ASOF)
    out = {}
    for name, msk in (("full", np.broadcast_to(full_mask, S.shape)), ("pit", pit_mask), ("pit_deleg", pit_mask | dmask)):
        Sm = np.where(msk, S, -np.inf)
        top = np.argpartition(-Sm, k, axis=1)[:, :k]
        valid = np.take_along_axis(Sm, top, 1) > -np.inf
        out[f"recall_{name}"] = float(((np.take_along_axis(P, top, 1) & valid).sum(1) / npos).mean())
        if name == "full":   # out-of-office (not in pit pool, not a delegate) members occupying full-roster top-k slots
            out[f"oop_in_top{k}"] = float((valid & ~np.take_along_axis(pit_mask | dmask, top, 1)).sum(1).mean())
    return dict(fold=unit["fold"], chamber=ch[:5], bills=len(E), **out)


r22 = pd.DataFrame([recall20(u, ch) for u in C22_FOLDS for ch in ("House of Representatives", "Senate")])
dens = pp[pp.congress.isin([u["eval_congress"] for u in C22_FOLDS])].groupby(["congress", "ch"])[["dens_full", "dens_pit"]].mean()
r22["dens_full"] = [dens.loc[(int(f[2:]), ch), "dens_full"] for f, ch in zip(r22.fold, r22.chamber)]
r22["dens_pit"] = [dens.loc[(int(f[2:]), ch), "dens_pit"] for f, ch in zip(r22.fold, r22.chamber)]
r22["gain_pit"] = r22.recall_pit - r22.recall_full
r22["gain_pit_deleg"] = r22.recall_pit_deleg - r22.recall_full
print("S3 test folds, t0=7: Recall@20 and positive density, full-Congress roster vs point-in-time pool "
      "(oop_in_top20 = out-of-office members in the full-roster top 20, per bill)")
print(r22.round(4).to_string(index=False))
g22 = float(r22.gain_pit_deleg.abs().max())
record_check(C_SECTION, "C22", "Point-in-time pool vs full-Congress roster: effect on task-1 Recall@20 and positive density (S3 test folds)", None,
             metric=g22, threshold="INFO (requirement for every split; metric = max |Recall@20 gain| of pit+delegates vs full roster)",
             detail="Recall@20 gain vs full roster, pit+delegates / pit only (density full -> pit): " + ", ".join(
                    f"{r.fold} {r.chamber} {r.gain_pit_deleg:+.4f} / {r.gain_pit:+.4f} ({r.dens_full:.4f} -> {r.dens_pit:.4f})"
                    for r in r22.itertuples())
                    + f". Out-of-office members take {r22.oop_in_top20.min():.2f}-{r22.oop_in_top20.max():.2f} of the 20 full-roster slots "
                      "per bill. The voting-only ROSTER_ASOF drops delegates, who cast most positives outside it (C17), so a pit pool "
                      "WITHOUT delegates loses House recall; with delegates added back it matches or beats the full roster. Ranker = sponsor-cosponsor counts "
                      "in the fit Congresses + 0.001 x activity in the last fit Congress (proxy for any history-based model). "
                      "Labels and D's ranking pool (_ROSTER) should use the same point-in-time pool including delegates.")

S3 test folds, t0=7: Recall@20 and positive density, full-Congress roster vs point-in-time pool (oop_in_top20 = out-of-office members in the full-roster top 20, per bill)
 fold chamber  bills  recall_full  oop_in_top20  recall_pit  recall_pit_deleg  dens_full  dens_pit  gain_pit  gain_pit_deleg
c=115   House   2649       0.1974        0.3681      0.1930            0.1999     0.0055    0.0057   -0.0043          0.0026
c=115   Senat    822       0.4620        0.4988      0.4717            0.4717     0.0066    0.0070    0.0097          0.0097
c=118   House   3568       0.1857        0.5751      0.1718            0.1852     0.0035    0.0036   -0.0138         -0.0004
c=118   Senat    800       0.4361        0.3262      0.4418            0.4418     0.0034    0.0036    0.0057          0.0057
[INFO] C22: Point-in-time pool vs full-Congress roster: effect on task-1 Recall@20 and positive density (S3 test folds) | metric=0.00971 (threshold INFO (requirement for every split; metric = max |Recall@

### C19-C21. Companion and reintroduced bills across the split boundary

**Why it matters.** Many bills have a twin: an identical House/Senate companion in the same Congress, or
a reintroduction of the same bill from an earlier Congress. Twins share sponsors, cosponsors and fate.
A reintroduction in a *training* Congress is legitimate signal (it was known when the new bill was
introduced). A same-Congress companion in the training portion is legitimate only if it was introduced
earlier; if it was introduced *after* the evaluation bill (possible only under the random split S5), the
model sees the future. Same-Congress companions also show what a by-Congress split forgoes: they are known
at t0 only if the graph includes the evaluation Congress's own history up to t0.

**How to read it.** Twins = same-Congress `bill_related` links (any relationship) plus identical official
titles as introduced (normalized; titles shared by > 30 bills are dropped as generic). For every evaluation
unit: the share of eval bills with a same-Congress twin in the training portion (split into introduced
before vs after the eval bill) and with an identical-title bill in a training Congress (earlier = reintroduction,
later = future leak). Twins are themselves a CRS/clerk annotation assigned after introduction (proxy).

In [151]:
# C19-C21: twins (same-Congress related / identical-title bills) and reintroductions vs each split's train portion.
F = lb[lb.congress.isin(CONGRESSES)][["bill_id", "congress", "introduced_date"]].reset_index(drop=True)
pos_of = pd.Series(F.index, index=F.bill_id)
ti = load("bill_titles", ("bill_id", "title_type", "title"))
ti = ti[(ti.title_type == "Official Title as Introduced") & ti.bill_id.isin(pos_of.index)].drop_duplicates("bill_id")
ti = ti.assign(norm=ti.title.str.lower().str.replace(r"\s+", " ", regex=True).str.strip().str.rstrip("."))
ti = ti[ti.groupby("norm").bill_id.transform("size").between(2, 30)]
tp = ti[["bill_id", "norm"]].merge(ti[["bill_id", "norm"]], on="norm", suffixes=("_a", "_b"))
tp = tp[tp.bill_id_a != tp.bill_id_b]
rl = load("bill_related", ("bill_id", "related_bill_id"))
rl = rl[rl.bill_id.isin(pos_of.index) & rl.related_bill_id.isin(pos_of.index)]
pairs = pd.concat([rl.set_axis(["bill_id_a", "bill_id_b"], axis=1), rl[["related_bill_id", "bill_id"]].set_axis(["bill_id_a", "bill_id_b"], axis=1),
                   tp[["bill_id_a", "bill_id_b"]]]).drop_duplicates()
pa_, pb_ = pos_of[pairs.bill_id_a].to_numpy(), pos_of[pairs.bill_id_b].to_numpy()
ca, cb = F.congress.to_numpy()[pa_], F.congress.to_numpy()[pb_]
da, db = F.introduced_date.to_numpy()[pa_], F.introduced_date.to_numpy()[pb_]
same = ca == cb
title_pair = set(zip(tp.bill_id_a, tp.bill_id_b))
is_title = np.array([(a, b) in title_pair for a, b in zip(pairs.bill_id_a, pairs.bill_id_b)])
n_bills = len(F)


def _any_per_bill(sel):
    """Boolean per bill of F: has at least one pair (as side a) satisfying sel."""
    out = np.zeros(n_bills, bool)
    out[pa_[sel]] = True
    return out


# Signal available within the eval Congress at t0 (needs within-Congress history up to t0): twin introduced on/before day 0.
earlier_twin = _any_per_bill(same & (db <= da))
tw_rows = []
for un in EVAL_UNITS:
    tr, ev = unit_masks(un, F)
    in_tr = tr[pb_]
    sel_ev = ev[pa_]
    def share(cond):
        return float(_any_per_bill(sel_ev & in_tr & cond)[ev].mean()) if ev.any() else np.nan
    tw_rows.append(dict(split=un["split"], fold=un["fold"], role=un["role"], eval_bills=int(ev.sum()),
                        twin_in_train_before=share(same & (db <= da)), twin_in_train_after=share(same & (db > da)),
                        reintro_in_train=share(~same & (cb < ca) & is_title), future_congress_twin=share(~same & (cb > ca)),
                        twin_known_t0_in_eval_congress=float(earlier_twin[ev].mean())))
tw = pd.DataFrame(tw_rows)
tws = tw.groupby("split")[["twin_in_train_before", "twin_in_train_after", "reintro_in_train", "future_congress_twin",
                           "twin_known_t0_in_eval_congress"]].mean()
print(f"pairs: {len(pairs):,} directed twin links ({int(same.sum()):,} same-Congress, {int((~same).sum()):,} cross-Congress)")
print("Share of eval bills linked to the training portion, mean over each split's evaluation units")
print(tws.round(3).to_string())
tw_plot = tws.reset_index().melt("split", value_vars=["twin_in_train_before", "twin_in_train_after", "reintro_in_train", "future_congress_twin"],
                                 var_name="link", value_name="share")
tw_plot["link"] = tw_plot.link.map({"twin_in_train_before": "same-Congress twin, earlier", "twin_in_train_after": "same-Congress twin, LATER (leak)",
                                    "reintro_in_train": "reintroduction (earlier Congress)", "future_congress_twin": "twin in LATER Congress (leak)"})
bar_chart(tw_plot, "split", "share", series="link", yfmt=".0%",
          title="Share of evaluation bills with a twin bill in the training portion, by candidate split",
          subtitle="Twins = same-Congress related / identical-title bills; mean over each split's val and test units",
          ylabel="share of eval bills", height=340, name="C_twins_by_split")
leak_by = tws.twin_in_train_after + tws.future_congress_twin
clean = [k for k in tws.index if leak_by[k] < 0.01]
record_check(C_SECTION, "C19", "Temporal splits (S1-S4) put no twin introduced after the eval bill into training (no future twin leak)",
             bool(leak_by.drop("S5", errors="ignore").max() < 0.01), metric=float(leak_by.drop("S5", errors="ignore").max()),
             threshold="< 1% of eval bills with a later twin in train",
             detail=f"share of eval bills with a later twin in train (same-Congress later + later-Congress): {leak_by.round(3).to_dict()}; "
                    f"clean splits: {clean}. By construction, by-Congress training never contains same-Congress or later bills; "
                    "S4 only contains earlier same-Congress bills. S1pool (train 108-115/116, eval 116-118) is True by the same "
                    "construction (no fit Congress is later than or equal to an eval Congress).",
             by_split=with_variants({k: bool(v < 0.01) for k, v in leak_by.items()}) | {"S1pool": True})
s5_leak = float(leak_by.get("S5", np.nan))
record_check(C_SECTION, "C20", "Random split S5: eval bills with a later twin (same or later Congress) in training < 1%",
             bool(s5_leak < 0.01), metric=s5_leak, threshold="< 0.01",
             detail=f"S5: same-Congress twin introduced later in train {tws.loc['S5', 'twin_in_train_after']:.3f}, twin in a later Congress "
                    f"in train {tws.loc['S5', 'future_congress_twin']:.3f}. These links carry the twin's future cosponsors and "
                    "committee outcome into training, so S5 scores are optimistic by more than the generic random-split bias; "
                    "use S5 only as a leaky upper reference.", by_split={k: (bool(s5_leak < 0.01) if k == "S5" else None) for k in ALL_SPLITS},
             na_reason="C20 judges only the random split S5; temporal splits are judged in C19")
bc_t0 = float(tw.loc[tw.split.isin(BY_CONGRESS_SPLITS), "twin_known_t0_in_eval_congress"].mean())
s4_before = float(tws.loc["S4", "twin_in_train_before"])
reint = tws.reintro_in_train.loc[BY_CONGRESS_SPLITS].round(3).to_dict()
record_check(C_SECTION, "C21", "Same-Congress twin signal a by-Congress split forgoes unless the eval Congress's history up to t0 is in the graph",
             None, metric=bc_t0, threshold="INFO",
             detail=f"{bc_t0:.1%} of by-Congress eval bills have a same-Congress twin introduced on or before them (known at t0 only if "
                    f"within-Congress history is allowed); S4 train contains such a twin for {s4_before:.1%} of its eval bills. "
                    f"Reintroductions from earlier training Congresses (legitimate signal for S1-S3): {reint}. Implication: the "
                    "by-Congress protocol should state whether eval-Congress edges dated before t0 are visible (a streaming / "
                    "inductive setting); if they are, twins become usable features and the S1-vs-S4 gap narrows.")

pairs: 250,731 directed twin links (114,953 same-Congress, 135,778 cross-Congress)
Share of eval bills linked to the training portion, mean over each split's evaluation units
       twin_in_train_before  twin_in_train_after  reintro_in_train  future_congress_twin  twin_known_t0_in_eval_congress
split                                                                                                                   
S1a                  0.0000               0.0000            0.3370                0.0000                          0.3890
S1b                  0.0000               0.0000            0.3640                0.0000                          0.3620
S2                   0.0000               0.0000            0.3250                0.0000                          0.3560
S3                   0.0000               0.0000            0.3270                0.0000                          0.3560
S4                   0.1720               0.0000            0.1500                0.0000           

[PASS] C19: Temporal splits (S1-S4) put no twin introduced after the eval bill into training (no future twin leak) | metric=0 (threshold < 1% of eval bills with a later twin in train) [fails for: S5]
[FAIL] C20: Random split S5: eval bills with a later twin (same or later Congress) in training < 1% | metric=0.4519 (threshold < 0.01) [fails for: S5]
[INFO] C21: Same-Congress twin signal a by-Congress split forgoes unless the eval Congress's history up to t0 is in the graph | metric=0.3585 (threshold INFO)


### Section C summary

The table below lists every check recorded in this section. FAIL does not automatically rule out a
by-Congress split; read the `detail` column for what each failure implies. Three kinds of result appear:
**split-discriminating** checks with a `by_split` verdict (C1, C2, C3, C5, C6, C6b, C8, C15b, C19, C20, C23b, C24;
C14 only if C24 finds incomplete labels; every split in ALL_SPLITS gets a verdict or a stated reason, and S4 is judged on
the candidate Congresses its slices test wherever the verdict is about that Congress's bills: C3, C15b, C24), **drift / timing** checks (C1-C3, C23: how different the recent Congresses
are and whether changes happen inside a Congress), and **requirements** that hold for every split (C11, C13, C15a,
C17, C18, C25 recorded with `requirement=True`; C10, C12, C22 state the same kind of rule: build features as of t0,
use as-of party, mask policy area at t0 <= 7, fix task 2's t0, use the point-in-time pool incl. delegates).
C24 decides whether 118's differences are data latency (it finds no missing cosponsor events or committee reports),
C8b whether the joint House x Senate regime needs to be covered. C4 is a cross-reference to section A.

In [152]:
# Summary of section C checks grouped by result (requirements = construction rules for every split, not split failures).
cs = pd.DataFrame([c for c in CHECKS if c["section"] == C_SECTION])
cs["result"] = cs["passed"].map({True: "PASS", False: "FAIL"}).fillna("INFO")
cs["metric"] = cs["metric"].map(lambda v: f"{v:.4g}" if isinstance(v, (int, float, np.floating)) else str(v))
cs["req"] = cs.get("requirement", pd.Series(False, index=cs.index)).fillna(False).astype(bool)
cs.loc[cs.req, "result"] = "REQUIREMENT"
for res in ("FAIL", "PASS", "INFO", "REQUIREMENT"):
    sub = cs[cs.result == res]
    print(f"{res} ({len(sub)}): " + ", ".join(f"{r.id}={r.metric}" for r in sub.itertuples()))
print(f"Section C: {(cs.result == 'PASS').sum()} pass, {(cs.result == 'FAIL').sum()} fail, {(cs.result == 'INFO').sum()} info, "
      f"{cs.req.sum()} requirements (one line per check with its condition is printed where it is recorded above)")

FAIL (4): C3=0.09342, C23b=1, C6b=0.8078, C20=0.4519
PASS (12): C1=0.06177, C2=1.959, C23=0, C5=0.03864, C6=0.1725, C8=1, C9=1, C10=0.9987, C12=0.9762, C24=1.211, C15b=0.9904, C19=0
INFO (8): C4=0.4746, C7=4 regimes, 7 flips, C8b=0.2455, C14=0.292, C15c=0.5776, C16=max coverage gap 0.426 (summary), C22=0.00971, C21=0.3585
REQUIREMENT (6): C11=0.4846, C13=0.03503, C15a=0.3722, C18=0.1429, C25=30, C17=0.09655
Section C: 12 pass, 4 fail, 8 info, 6 requirements (one line per check with its condition is printed where it is recorded above)


## D. Candidate splits and baseline backtest

Sections A-C test the *preconditions* of a temporal split (similar distributions, node overlap,
no leakage). This section tests the *consequence*: if we actually train simple, transparent
baselines under each candidate split, does the split do its job? A split is useful for model
selection if (1) validation picks the model that is best on test *among close candidates* (regret),
(2) the test set is large enough to resolve **paired** model-vs-model differences, (3) the validation
score level predicts the test level (for early stopping and reporting), and (4) its protocol choices
(expanding vs sliding window, streaming same-Congress history, refit on train + val) do not cost
accuracy. We also quantify how optimistic the leaky random split is and whether a within-Congress
split behaves differently, and (D27) what it costs to keep Congress 118 as a locked final test that
no model choice ever sees. One minimum detectable effect per task (`D_MDE` = 0.01 Recall@20 / 0.01
AUROC) is used by every selection, optimism and sizing check.

**Protocol.** Every split comes from `SPLIT_DEFS` / `iter_eval_units()` and `SPLIT_VARIANTS` (setup).
Each validation unit is scored with baselines fit on its training Congresses; each test unit with
baselines refit on train + validation ("select on val, refit, report test"; D19 checks that refit).
The baselines are counts, smoothed rates, one graph-structural score (`rw2hop`, a 2-step walk on
the time-masked cosponsorship graph) and one **learned, transductive embedding model** (`svd`: a
truncated SVD of the sponsor x cosponsor training matrix, rank and recency chosen on validation), the
closest cheap stand-in for a GNN with per-legislator embeddings (it can go stale and is cold for new
members). The goal is to measure the split, not to win: conclusions that hold for count, graph and
learned models are the robust ones.

**Prediction times.** Task 1 is scored at t0 = 0 and 7 days after introduction (task definition).
Task 2 has no prediction time in the task definition; **this section assumes prediction at
introduction** (same t0 as Task 1). At introduction the CRS policy area is usually missing, so the
deployable Task 2 baseline is `eb_rate_t0safe`; `eb_rate` (policy area always present) is reported as
the "predict about a month after introduction" variant. Task 2 selection (D12b) and test-set sizing
(D9) are judged on the t0-safe variant; the policy-area variant is shown for reference.

In [153]:
# D.0 Protocol variants this section scores on top of SPLIT_DEFS (defined in setup as SPLIT_VARIANTS).
D_VARIANTS = SPLIT_VARIANTS                       # legacy alias; the definitions live in setup
print("Protocol variants (SPLIT_VARIANTS): " + " | ".join(f"{k}: {v['desc']}" for k, v in SPLIT_VARIANTS.items()))

Protocol variants (SPLIT_VARIANTS): S1a+stream: S1a + eval-Congress events dated <= introduced_date + t0 | S1b+stream: S1b + eval-Congress events dated <= introduced_date + t0 | S3+stream: S3 rolling folds + eval-Congress events dated <= introduced_date + t0 | S1pool: train 108-115 | val 116 | test 117+118 pooled (refit on 108-116 once)


### Baselines, metrics and the backtest engine

**Task 1 (cosponsor ranking).** For each evaluable bill (law-capable, window not censored, at least
one new cosponsor in *(t0, t0+30]*) we rank the eligible same-chamber legislators: voting members
**in office in the month of introduced_date + t0** (`ROSTER_ASOF_ALL`, the shared point-in-time pool,
delegates and resident commissioners included as the task definition requires) minus
the sponsor and anyone who already cosponsored by *t0*.
`popularity` = legislator's number of cosponsorships in the training events;
`party+pop` = same caucus as the sponsor first (as-of-Congress caucus, `members.caucus`), then popularity;
`sponsor_hist` = times the legislator cosponsored this sponsor's bills in the training events (ties
broken by popularity); `rw2hop` = a **graph-structural** score closer to what a GNN computes: a two-step
random walk from the sponsor on the time-masked cosponsorship graph, `h1 + 0.3 * h1 @ A`, where `h1` is
the sponsor's row of the sponsor->cosponsor count matrix (row-normalised) and `A` the row-normalised
symmetric graph (H + H^T). It rewards "cosponsors of my cosponsors" (multi-hop structure), not just direct
history; it is a fixed-weight version of the propagation step of LightGCN / SGC-style GNNs. Streaming
variants use the eval Congress's graph as of the first day of the month of *introduced_date + t0*.
`svd_r{rank}[_hl1]` = learned embeddings: rank-r truncated SVD of log(1 + sponsor x cosponsor counts) in
the training events (optionally recency-weighted, half-life 1 Congress), score = reconstructed affinity
sponsor -> candidate. Legislators absent from training have a zero embedding (transductive cold start).
Metrics: Recall@10, Recall@20, NDCG@20 per bill, averaged.
**Task 2 (advancement).** `prior` = constant training rate (AUROC 0.5 by construction);
`chamber_maj` = smoothed rate by chamber x sponsor-in-majority (`majority_party()`, sponsor caucus);
`eb_rate` = empirical-Bayes rate by policy area x chamber x sponsor-in-majority, shrunk toward
`chamber_maj` (20 pseudo-bills); `eb_rate_t0safe` = `eb_rate`, but bills whose policy area would not
yet be assigned at introduction fall back to `chamber_maj` (rate measured on fresh 119th bills).
Metrics: AUROC, average precision (AP) and **AP lift** = AP / positive rate (AP alone is not
comparable across Congresses whose positive rate differs).
**Uncertainty.** 95% percentile intervals from a **cluster bootstrap by sponsor** (500 resamples;
bills of the same sponsor share coalitions, so they are resampled together). `ci_low_bill` /
`ci_high_bill` give the naive bill-level bootstrap for comparison. The same resamples are used for
every baseline of a unit, so **paired differences** (model A - model B on the same bills) get their own,
much tighter, CIs.
**Near-tie families** (for model selection, D11-D12, D21): Task 1 `sponsor_hist` with recency decay
(half-life 1, 2, 4 Congresses or none) and a same-caucus tie-break variant; Task 1 `rw2hop` with
second-hop weight in {0.1, 0.3, 1} and recency half-life 1 (weights 0.1, 0.3); Task 2 `eb_rate` with shrinkage m in
{5, 20, 80} and a variant without sponsor-majority, and the same four members in their t0-safe form
(`*_t0s`, policy area hidden for fresh bills); Task 1 `svd` with rank in {8, ..., 512} (powers of 2) x recency
half-life {none, 1} (D22-D25).

In [154]:
# D.1 Vectorized backtest engine: integer-coded arrays, Task 1 ranking metrics, Task 2 AUROC/AP, cluster bootstrap.
import time
import zlib
from scipy import sparse

_D_B = 500
_D_TOPK = 20
_D_BIG = 10 ** 6                                   # key = group * BIG + day (days since epoch < 1e5)
_D_PRIMARY_T1 = ("cosponsor_t0", "Recall@20")
_D_PRIMARY_T2 = ("advancement", "AUROC")
T1_BASELINES = ["popularity", "party+pop", "sponsor_hist", "rw2hop"]
T2_BASELINES = ["prior", "chamber_maj", "eb_rate", "eb_rate_t0safe"]
T1_FAMILY = {"sh_none": (None, False), "sh_hl4": (4, False), "sh_hl2": (2, False), "sh_hl1": (1, False),
             "sh_none+caucus": (None, True)}            # (recency half-life in Congresses, caucus tie-break)
RW_BETA = 0.3                                         # second-hop weight of the main rw2hop baseline
RW_FAMILY = {"rw_b0.1": (0.1, None), "rw_b0.3": (0.3, None), "rw_b1": (1.0, None),
             "rw_b0.1_hl1": (0.1, 1), "rw_b0.3_hl1": (0.3, 1)}   # (second-hop weight, recency half-life)
T2_FAMILY = {"eb_m5": (5.0, True), "eb_m20": (20.0, True), "eb_m80": (80.0, True), "eb_nomaj_m20": (20.0, False)}
T2S_FAMILY = {f"{k}_t0s": v for k, v in T2_FAMILY.items()}   # same members, policy area hidden for fresh bills
SVD_RANKS = (8, 16, 32, 64, 128, 256, 512)
SVD_FAMILY = {f"svd_r{r}" + ("" if hl is None else f"_hl{hl}"): (r, hl) for hl in (None, 1) for r in SVD_RANKS}
FAMILIES = {"T1 sponsor_hist family": list(T1_FAMILY), "T1 rw2hop family": list(RW_FAMILY),
            "T2 eb_rate family": list(T2_FAMILY), "T2 eb_rate_t0safe family": list(T2S_FAMILY)}
_FAM_OF = {m: f for f, ms in FAMILIES.items() for m in ms}
_INTMAX = np.iinfo(np.int64).max
# ONE minimum detectable effect per task, used by every selection, optimism and test-set sizing check below: the
# smallest test-score difference worth acting on when choosing a model or protocol (a policy choice, not a data
# estimate; see the D7 cell for how it compares with the data). Regret tolerance for selection checks = MDE / 2.
D_MDE = {"cosponsor_t0": 0.01, "cosponsor_t7": 0.01, "advancement": 0.01}
D_TOL = {k: v / 2 for k, v in D_MDE.items()}
_LOCK = max(FOLD_C)                                   # rolling fold reserved as the locked final test (D27)
_DEV = [c for c in FOLD_C if c != _LOCK]              # development folds


def _bs(d):
    """by_split with an explicit key for every split in ALL_SPLITS (None = not judged; reason in na_reason)."""
    return {s: (None if d.get(s) is None else d[s]) for s in ALL_SPLITS}


_NA_FAM = ("near-tie families / learned model are fit only on the S2 / S3 rolling folds (S1a / S1b = S3 folds 117 / 118, "
           "reported as a metric); stream variants, S1pool, S4 and S5 are not run through the selection backtest")

_LEG = pd.Index(pd.unique(pd.concat([members.bioguide_id, cosp.bioguide_id, bills.sponsor_bioguide_id.dropna()])))
_NL = len(_LEG)
_BILLIX = pd.Index(bills.bill_id)
_cdate = cosp["sponsorship_date"]
_CD = dict(leg=_LEG.get_indexer(cosp.bioguide_id), spon=_LEG.get_indexer(cosp.sponsor_bioguide_id),
           cong=cosp.congress.to_numpy(), bill=_BILLIX.get_indexer(cosp.bill_id),
           lag=cosp.lag_days.to_numpy(dtype=float),
           day=np.where(_cdate.notna(), _cdate.values.astype("datetime64[D]").astype(np.int64), _INTMAX))


def _mstart_day(dates):
    """Day number (since epoch) of the first day of each date's month."""
    return pd.to_datetime(dates).dt.to_period("M").dt.to_timestamp().values.astype("datetime64[D]").astype(np.int64)


# Caucus as of each Congress (independents assigned by setup); sponsors without a member row keep their party code.
_CAU = members.drop_duplicates(["bioguide_id", "congress"]).set_index(["bioguide_id", "congress"])["caucus"]
_BL = bills[bills.law_capable & bills.congress.isin(CONGRESSES)][
    ["bill_id", "congress", "chamber", "sponsor_bioguide_id", "sponsor_party", "introduced_date"]].copy()
_BL["sp_caucus"] = pd.MultiIndex.from_arrays([_BL.sponsor_bioguide_id, _BL.congress]).map(_CAU)
_BL["sp_caucus"] = _BL.sp_caucus.where(_BL.sp_caucus.isin(["D", "R"]), _BL.sponsor_party)
_BL["code"] = _BILLIX.get_indexer(_BL.bill_id)
_BL["spon"] = _LEG.get_indexer(_BL.sponsor_bioguide_id.fillna("?"))
_BL["day"] = _BL.introduced_date.values.astype("datetime64[D]").astype(np.int64)
for _t0 in (0, 7):
    _ref = _BL.introduced_date + pd.Timedelta(days=_t0)
    _BL[f"sm{_t0}"] = _mstart_day(_ref)          # streaming graph snapshot: events dated before this month start
    _BL[f"pm{_t0}"] = _mstart_day(_ref.clip(upper=_BL.congress.map(congress_end) - pd.Timedelta(days=ROSTER_REF_DAY + 5)))
_BL = _BL.set_index("code")
_BL["cluster"] = np.where(_BL.spon >= 0, _BL.spon, -1 - np.arange(len(_BL)))   # sponsor-less bills = own cluster
_CENS = {t0: window_targets(t0).assign(code=lambda d: _BILLIX.get_indexer(d.bill_id)).set_index("code")["censored"]
         for t0 in (0, 7)}

# Candidate pools. Legacy: everyone who served in the chamber that Congress. Point-in-time (default): the union
# of the Congress roster and ROSTER_ASOF as candidate columns, masked per bill to members in office in the
# month of introduced_date + t0 (same rule as window_targets(point_in_time=True)).
_mm = members.dropna(subset=["chamber"]).drop_duplicates(["congress", "chamber", "bioguide_id"])
_ROSTER = {k: (_LEG.get_indexer(g.bioguide_id), g.caucus.to_numpy()) for k, g in _mm.groupby(["congress", "chamber"])}
_RA = ROSTER_ASOF_ALL[ROSTER_ASOF_ALL.congress.isin(CONGRESSES)].assign(
    leg=lambda d: _LEG.get_indexer(d.bioguide_id), mday=lambda d: d.month.values.astype("datetime64[D]").astype(np.int64))
assert set(_RA.chamber.unique()) <= set(_mm.chamber.unique()), "ROSTER_ASOF chamber labels differ from members"
_ROSTER_PIT = {}
for (_c, _ch), _g in _RA[_RA.leg >= 0].groupby(["congress", "chamber"]):
    _cand = pd.unique(np.r_[_ROSTER.get((_c, _ch), (np.array([], int),))[0], _g.leg.to_numpy()])
    _cau = pd.MultiIndex.from_arrays([_LEG[_cand], np.full(len(_cand), _c)]).map(_CAU)
    _cau = np.asarray(pd.Series(_cau).fillna("other"))
    _md = np.sort(_g.mday.unique())
    _cp = pd.Series(np.arange(len(_cand)), index=_cand)
    _io = np.zeros((len(_md), len(_cand)), bool)
    _io[np.searchsorted(_md, _g.mday.to_numpy()), _cp.reindex(_g.leg).to_numpy()] = True
    _ROSTER_PIT[(_c, _ch)] = (_cand, _cau, _md, _io)
_D_LOG = {"unmatched_pos": 0, "pos_total": 0, "pos_not_in_office": 0, "pos_pit_total": 0, "pool_pit": [], "pool_legacy": [],
          "stream_s": 0.0}


def _boot_weights(clusters, key, B=_D_B):
    """(B+1, n) resample-count matrix over clusters (row 0 = ones = point estimate). Seeded by key."""
    u, inv = np.unique(np.asarray(clusters), return_inverse=True)
    K = len(u)
    rng = np.random.default_rng(zlib.crc32(str(key).encode()))
    idx = rng.integers(0, K, (B, K))
    cnt = np.bincount((idx + np.arange(B)[:, None] * K).ravel(), minlength=B * K).reshape(B, K)
    return np.vstack([np.ones(len(inv)), cnt[:, inv]]).astype(float)


def _rank_metrics(S, Pos, return_top=False):
    """Per-row Recall@10, Recall@20, NDCG@20 given scores S (ineligible = -inf) and positives Pos
    (and the top-K column indices if asked)."""
    K = _D_TOPK
    top = np.argpartition(-S, K, axis=1)[:, :K]
    top = np.take_along_axis(top, np.argsort(-np.take_along_axis(S, top, 1), axis=1), 1)
    hits = np.take_along_axis(Pos, top, 1).astype(float)
    npos = Pos.sum(1)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    idcg = np.cumsum(disc)[np.minimum(npos, K) - 1]
    res = np.c_[hits[:, :10].sum(1) / npos, hits.sum(1) / npos, (hits @ disc) / idcg]
    return (res, top) if return_top else res


def _counts(train_mask, hl=None, cw=None):
    """Legislator popularity and sponsor x legislator history from training events, optionally recency-decayed
    (half-life hl Congresses) and/or weighted per Congress (cw: array indexed by Congress number)."""
    leg, spon, cg = _CD["leg"][train_mask], _CD["spon"][train_mask], _CD["cong"][train_mask]
    w = None if hl is None else 0.5 ** ((cg.max() - cg) / hl)
    if cw is not None:
        w = cw[cg] if w is None else w * cw[cg]
    pop = np.bincount(leg, weights=w, minlength=_NL).astype(float)
    ok = spon >= 0
    hist = np.bincount(spon[ok] * _NL + leg[ok], weights=None if w is None else w[ok],
                       minlength=_NL * _NL).reshape(_NL, _NL).astype(float)
    return pop, hist


def _rownorm(X):
    return X / np.maximum(X.sum(1, keepdims=True), 1e-12)


def _rw_parts(hist):
    """rw2hop matrices: row-normalised sponsor->cosponsor counts (step 1) and symmetric walk matrix (step 2)."""
    return _rownorm(hist), _rownorm(hist + hist.T)


def _rw_scores(parts, spi, has_sp, cand):
    """Step-1 and step-2 walk mass from each bill's sponsor to every candidate (nb x M each)."""
    Hn, A = parts
    us, inv = np.unique(spi, return_inverse=True)
    R = Hn[us]
    s1, s2 = R[:, cand][inv], (R @ A[:, cand])[inv]
    return s1 * has_sp[:, None], s2 * has_sp[:, None]


@lru_cache(maxsize=None)
def _ev_congress(c):
    """Dated cosponsorship events of Congress c, sorted two ways (by day; by sponsor then day) for streaming."""
    m = (_CD["cong"] == c) & (_CD["day"] < _INTMAX)
    leg, spon, day = _CD["leg"][m], _CD["spon"][m], _CD["day"][m]
    o = np.argsort(day, kind="stable")
    ok = spon >= 0
    key = spon[ok] * _D_BIG + day[ok]
    o2 = np.argsort(key, kind="stable")
    o3 = np.argsort(day[ok], kind="stable")
    return dict(leg_d=leg[o], day_d=day[o], key=key[o2], leg_k=leg[ok][o2],
                day_e=day[ok][o3], pair_e=(spon[ok] * _NL + leg[ok])[o3])


def _stream_hist(c, cut_day):
    """Sponsor x legislator counts of Congress c's events dated before cut_day (streaming graph snapshot)."""
    ev = _ev_congress(c)
    n = np.searchsorted(ev["day_e"], cut_day, "left")
    return np.bincount(ev["pair_e"][:n], minlength=_NL * _NL).reshape(_NL, _NL).astype(float)


def _stream_mats(c, g, cand, t0):
    """Eval-Congress counts known at each bill's prediction time (events dated <= introduced + t0):
    per bill x candidate popularity increment and sponsor-history increment."""
    ts = time.time()
    ev = _ev_congress(c)
    M, nb = len(cand), len(g)
    colpos = np.full(_NL, -1)
    colpos[cand] = np.arange(M)
    thr = g.day.to_numpy() + t0
    dcol = colpos[ev["leg_d"]]
    okc = dcol >= 0
    ud, di = np.unique(ev["day_d"][okc], return_inverse=True)
    cum = np.bincount(di * M + dcol[okc], minlength=len(ud) * M).reshape(len(ud), M).cumsum(0)
    ix = np.searchsorted(ud, thr, "right") - 1
    Pst = np.where((ix >= 0)[:, None], cum[np.maximum(ix, 0)], 0)
    sp = g.spon.to_numpy()
    st = np.searchsorted(ev["key"], sp * _D_BIG, "left")
    en = np.searchsorted(ev["key"], sp * _D_BIG + thr, "right")
    ln = np.where(sp >= 0, en - st, 0)
    rows = np.repeat(np.arange(nb), ln)
    idx = np.repeat(st, ln) + (np.arange(ln.sum()) - np.repeat(np.cumsum(ln) - ln, ln))
    col = colpos[ev["leg_k"][idx]]
    ok = col >= 0
    Hst = np.bincount(rows[ok] * M + col[ok], minlength=nb * M).reshape(nb, M)
    _D_LOG["stream_s"] += time.time() - ts
    return Pst.astype(float), Hst.astype(float)


def _rw_stream(c, g, cand, t0, hist, spi, has_sp):
    """rw2hop parts with the eval Congress's graph as of the first day of the month of introduced + t0
    (monthly snapshots: lags exact streaming by < 1 month, never leaks)."""
    ts = time.time()
    s1, s2 = np.zeros((len(g), len(cand))), np.zeros((len(g), len(cand)))
    sm = g[f"sm{t0}"].to_numpy()
    for d in np.unique(sm):
        r = sm == d
        a, b = _rw_scores(_rw_parts(hist + _stream_hist(c, d)), spi[r], has_sp[r], cand)
        s1[r], s2[r] = a, b
    _D_LOG["stream_s"] += time.time() - ts
    return s1, s2


def _t1_eval(train_mask, eval_codes, t0, stream=False, family=False, pit=True, cnt=None, extra=None, hit_sink=None):
    """Per-bill Task 1 metrics for every baseline (and the near-tie families if asked); returns (dict, codes).
    pit=True ranks the point-in-time pool (in office in the month of introduced + t0); False = legacy roster.
    extra(cand=, spi=, hs=, P=, same=, c=, ch=) -> {name: score matrix} adds models; hit_sink = {model: []} collects,
    for the models named in it, one (bill code, legislator, hit@20) row per positive."""
    cnt = dict(cnt or {})
    cnt.setdefault(None, _counts(train_mask))
    hls = {None} | ({v[0] for v in T1_FAMILY.values()} | {v[1] for v in RW_FAMILY.values()} if family else set())
    for hl in hls - set(cnt):
        cnt[hl] = _counts(train_mask, hl)
    rwp = {hl: _rw_parts(cnt[hl][1]) for hl in ({None} | ({v[1] for v in RW_FAMILY.values()} if family else set()))}
    eb = _BL.loc[np.asarray(eval_codes)]
    eb = eb[~_CENS[t0].reindex(eb.index).fillna(True).to_numpy(bool)]
    rows = np.isin(_CD["bill"], eb.index.to_numpy())
    r_bill, r_leg, r_lag = _CD["bill"][rows], _CD["leg"][rows], _CD["lag"][rows]
    out, code_out = {}, []
    rng = np.random.default_rng(t0 + 17)
    for (c, ch), g in eb.groupby(["congress", "chamber"]):
        if (c, ch) not in (_ROSTER_PIT if pit else _ROSTER):
            continue
        if pit:
            cand, cparty, mdays, inoff = _ROSTER_PIT[(c, ch)]
        else:
            cand, cparty = _ROSTER[(c, ch)]
        M, nb = len(cand), len(g)
        colpos = np.full(_NL, -1)
        colpos[cand] = np.arange(M)
        rowpos = pd.Index(g.index).get_indexer(r_bill)
        sel = rowpos >= 0
        rp, cp, lg = rowpos[sel], colpos[r_leg[sel]], r_lag[sel]
        win = (lg > t0) & (lg <= t0 + WINDOW_DAYS)
        Pos = np.zeros((nb, M), bool)
        Excl = np.zeros((nb, M), bool)
        Pos[rp[win & (cp >= 0)], cp[win & (cp >= 0)]] = True
        seen = (lg <= t0) & (cp >= 0)
        Excl[rp[seen], cp[seen]] = True
        spv = g.spon.to_numpy()
        sp = colpos[np.where(spv >= 0, spv, 0)]
        has_sp = (spv >= 0) & (sp >= 0)
        Excl[np.arange(nb)[has_sp], sp[has_sp]] = True
        if not stream and t0 == 0:
            _D_LOG["pos_total"] += int(win.sum())
            _D_LOG["unmatched_pos"] += int((win & (cp < 0)).sum())
        if pit:                                                   # not in office in the bill's pool month
            mi = np.minimum(np.searchsorted(mdays, g[f"pm{t0}"].to_numpy()), len(mdays) - 1)
            okm = mdays[mi] == g[f"pm{t0}"].to_numpy()
            out_pool = okm[:, None] & ~inoff[mi]
            if not stream and t0 == 0:
                _D_LOG["pos_pit_total"] += int((Pos & ~Excl).sum())
                _D_LOG["pos_not_in_office"] += int((Pos & ~Excl & out_pool).sum())
                _D_LOG["pool_pit"].append((~(Excl | out_pool)).sum(1))
            Excl |= out_pool
        elif not stream and t0 == 0:
            _D_LOG["pool_legacy"].append((~Excl).sum(1))
        Pos &= ~Excl
        keep = Pos.sum(1) > 0
        tieb = rng.random(len(cand)) * 0.5                        # random tie-break, fixed per roster
        if not keep.any():
            continue
        Pos, Excl, g = Pos[keep], Excl[keep], g[keep]
        spv = g.spon.to_numpy()
        hs, spi = spv >= 0, np.where(spv >= 0, spv, 0)
        same = (cparty[None, :] == g.sp_caucus.to_numpy()[:, None])
        Pst, Hst = _stream_mats(c, g, cand, t0) if stream else (0.0, 0.0)

        def ph(hl):
            pop, hist = cnt[hl]
            return pop[cand][None, :] + Pst + tieb[None, :], hist[spi][:, cand] * hs[:, None] + Hst

        P, H = ph(None)
        rw1, rw2 = _rw_stream(c, g, cand, t0, cnt[None][1], spi, hs) if stream else _rw_scores(rwp[None], spi, hs, cand)
        tbr = P * 1e-16                                           # popularity tie-break far below any walk mass
        scores = {"popularity": np.broadcast_to(P, Pos.shape), "party+pop": same * 1e7 + P, "sponsor_hist": H * 1e6 + P,
                  "rw2hop": rw1 + RW_BETA * rw2 + tbr}
        if family:
            for name, (hl, tb) in T1_FAMILY.items():
                Pf, Hf = (P, H) if hl is None else ph(hl)
                scores[name] = Hf * 1e8 + same * 1e6 + Pf if tb else Hf * 1e6 + Pf
            rwc = {None: (rw1, rw2)}
            for name, (beta, hl) in RW_FAMILY.items():
                if hl not in rwc:
                    rwc[hl] = _rw_scores(rwp[hl], spi, hs, cand)
                a, b = rwc[hl]
                scores[name] = a + beta * b + tbr
        if extra is not None:
            scores.update(extra(cand=cand, spi=spi, hs=hs, P=P, same=same, c=c, ch=ch))
        pr, pc = np.nonzero(Pos)
        for b, S in scores.items():
            S = np.array(S, dtype=float)
            S[Excl] = -np.inf
            if hit_sink is None or b not in hit_sink:
                out.setdefault(b, []).append(_rank_metrics(S, Pos))
                continue
            m_, top = _rank_metrics(S, Pos, return_top=True)
            out.setdefault(b, []).append(m_)
            intop = np.zeros(Pos.shape, bool)
            intop[np.arange(len(top))[:, None], top] = True
            hit_sink.setdefault(b, []).append(np.c_[g.index.to_numpy()[pr], cand[pc], intop[pr, pc]])
        code_out.append(g.index.to_numpy())
    if hit_sink is not None:
        for b in list(hit_sink):
            hit_sink[b] = np.vstack(hit_sink[b]) if len(hit_sink[b]) else np.zeros((0, 3), np.int64)
    return {b: np.vstack(v) for b, v in out.items()}, np.concatenate(code_out)


def _t1_summarize(per_bill, codes, key, mask=None, baselines=None):
    """Point estimate + cluster (by sponsor) and bill bootstrap CIs, paired across baselines."""
    if mask is not None:
        codes = codes[mask]
    n = len(codes)
    cl = _BL.cluster.reindex(codes).to_numpy()
    Wc = _boot_weights(cl, key)
    Wb = _boot_weights(np.arange(n), (key, "bill"))
    res = []
    for b in (baselines or per_bill):
        v = per_bill[b] if mask is None else per_bill[b][mask]
        ec = (Wc @ v) / Wc.sum(1)[:, None]
        eb = (Wb @ v) / Wb.sum(1)[:, None]
        lo, hi = np.percentile(ec[1:], [2.5, 97.5], axis=0)
        lob, hib = np.percentile(eb[1:], [2.5, 97.5], axis=0)
        for j, mname in enumerate(["Recall@10", "Recall@20", "NDCG@20"]):
            res.append(dict(baseline=b, metric=mname, value=ec[0, j], ci_low=lo[j], ci_high=hi[j],
                            ci_low_bill=lob[j], ci_high_bill=hib[j], n_eval=n, n_clusters=int(len(np.unique(cl)))))
    return res


# --- Task 2 ---
# Share of fresh bills without a CRS policy area at prediction time: measured on the in-progress 119th Congress
# (the only one observed before CRS catches up). Proxy: bills at most a week old at the data snapshot.
# Shared estimate from setup (PA_MISSING_AT_T0 by age bucket at the snapshot).
D_PA_FRESH_DAYS = int(PA_MISSING_AT_T0.loc["0-7d", "age_hi"])
D_PA_MISS_T0, _D_PA_N = PA_MISS_T0, PA_MISS_T0_N
_D_PA_MISS_30 = float(PA_MISSING_AT_T0.loc["8-30d", "share_missing"])

_ADV = adv[~adv.censored & adv.congress.isin(CONGRESSES)].merge(MAJORITY[["congress", "chamber", "majority"]],
                                                                on=["congress", "chamber"], how="left")
_ADV["code"] = _BILLIX.get_indexer(_ADV.bill_id)
_ADV["sp_caucus"] = _BL.sp_caucus.reindex(_ADV.code).to_numpy()
_ADV["maj"] = (_ADV.sp_caucus == _ADV.majority).astype(int)
_ADV["policy_area"] = _ADV.policy_area.fillna("(none)")
_ADV["day"] = _ADV.introduced_date.values.astype("datetime64[D]").astype(np.int64)
_ADV["y"] = _ADV.label.astype(float)
_ADV["cluster"] = _BL.cluster.reindex(_ADV.code).to_numpy()
_ADV["g_ch"] = pd.factorize(_ADV.chamber)[0]
_ADV["g1"] = pd.factorize(_ADV.chamber + "|" + _ADV.maj.astype(str))[0]
_ADV["g2"] = pd.factorize(_ADV.policy_area + "|" + _ADV.chamber + "|" + _ADV.maj.astype(str))[0]
_ADV["g3"] = pd.factorize(_ADV.policy_area + "|" + _ADV.chamber)[0]
_ADV["pa_hidden"] = np.random.default_rng(1180).random(len(_ADV)) < D_PA_MISS_T0   # fixed random mask (t0-safe)
_ADV = _ADV.reset_index(drop=True)
_NG = {k: int(_ADV[k].max()) + 1 for k in ("g_ch", "g1", "g2", "g3")}


def _grp(tr, te, col, stream, w=None):
    """(sum of labels, count) of te's group in training rows (optionally row-weighted by w), plus eval-Congress
    bills already resolved (introduced >= 180 days before the scored bill) when stream=True."""
    s = np.bincount(tr[col], weights=tr.y if w is None else tr.y * w, minlength=_NG[col])[te[col]]
    n = np.bincount(tr[col], weights=w, minlength=_NG[col])[te[col]].astype(float)
    if stream:
        g, res = te[col].to_numpy(), te.day.to_numpy() + ADV_WINDOW_DAYS
        o = np.lexsort((res, g))
        key = g[o] * _D_BIG + res[o]
        cy = np.r_[0, np.cumsum(te.y.to_numpy()[o])]
        st = np.searchsorted(key, g * _D_BIG, "left")
        en = np.searchsorted(key, g * _D_BIG + te.day.to_numpy(), "right")
        s, n = s + cy[en] - cy[st], n + en - st
    return s, n


def _t2_scores(tr, te, stream=False, family=False, w=None):
    """Fit Task 2 baselines (and the near-tie families) on rows tr (optional row weights w, normalised to mean 1),
    score rows te."""
    if w is not None:
        w = np.asarray(w, float) / np.mean(w)
    p0 = np.average(tr.y, weights=w)
    hid = te.pa_hidden.to_numpy()
    s, n = _grp(tr, te, "g1", stream, w)
    s1 = (s + 20 * p0) / (n + 20)
    s, n = _grp(tr, te, "g2", stream, w)
    eb = (s + 20 * s1) / (n + 20)
    out = {"prior": np.full(len(te), p0), "chamber_maj": s1, "eb_rate": eb, "eb_rate_t0safe": np.where(hid, s1, eb)}
    if family:
        sc, nc = _grp(tr, te, "g_ch", stream, w)
        sch = (sc + 20 * p0) / (nc + 20)
        s3, n3 = _grp(tr, te, "g3", stream, w)
        for name, (m, use_maj) in T2_FAMILY.items():
            out[name] = (s + m * s1) / (n + m) if use_maj else (s3 + m * sch) / (n3 + m)
            out[name + "_t0s"] = np.where(hid, s1 if use_maj else sch, out[name])   # fallback without policy area
    return out


def _auc_ap(s, y, W):
    """Weighted AUROC, AP and positive rate for every row of W (tie-aware, grouped by distinct score)."""
    uniq, inv = np.unique(-s, return_inverse=True)
    A = sparse.csr_matrix((np.ones(len(s)), (np.arange(len(s)), inv)), shape=(len(s), len(uniq)))
    Pg = np.asarray((A.T @ (W * y).T).T)
    Ng = np.asarray((A.T @ (W * (1 - y)).T).T)
    P, N = Pg.sum(1), Ng.sum(1)
    cP, cN = np.cumsum(Pg, 1), np.cumsum(Ng, 1)
    auc = (Pg * (N[:, None] - cN + 0.5 * Ng)).sum(1) / (P * N)
    ap = (Pg * cP / np.maximum(cP + cN, 1e-12)).sum(1) / P
    return auc, ap, P / (P + N)


def _t2_summarize(scores, te, key, baselines=None):
    y = te.y.to_numpy()
    Wc = _boot_weights(te.cluster.to_numpy(), key)
    Wb = _boot_weights(np.arange(len(te)), (key, "bill"))
    res = []
    for b in (baselines or scores):
        auc, ap, prev = _auc_ap(scores[b], y, Wc)
        aucb, apb, prevb = _auc_ap(scores[b], y, Wb)
        for mname, est, estb in (("AUROC", auc, aucb), ("AP", ap, apb), ("AP_lift", ap / prev, apb / prevb)):
            lo, hi = np.percentile(est[1:], [2.5, 97.5])
            lob, hib = np.percentile(estb[1:], [2.5, 97.5])
            res.append(dict(baseline=b, metric=mname, value=est[0], ci_low=lo, ci_high=hi, ci_low_bill=lob,
                            ci_high_bill=hib, n_eval=len(te), n_clusters=int(te.cluster.nunique())))
    return res


print(f"engine ready: {_NL:,} legislators coded, {len(_BL):,} law-capable bills in {CONGRESSES[0]}-{CONGRESSES[-1]}, "
      f"{len(_ADV):,} uncensored advancement labels")
print(f"policy area missing among 119th bills <= {D_PA_FRESH_DAYS} days old at the snapshot (setup PA_MISS_T0): "
      f"{D_PA_MISS_T0:.0%} of {_D_PA_N} bills (8-30 days: {_D_PA_MISS_30:.0%}); eb_rate_t0safe hides policy area for that "
      "share of evaluation bills (Task 2 prediction time = introduction)")

engine ready: 1,452 legislators coded, 128,791 law-capable bills in 108-118, 110,802 uncensored advancement labels
policy area missing among 119th bills <= 7 days old at the snapshot (setup PA_MISS_T0): 86% of 56 bills (8-30 days: 14%); eb_rate_t0safe hides policy area for that share of evaluation bills (Task 2 prediction time = introduction)


In [155]:
# D.2 Run every unit of iter_eval_units() (+ SPLIT_VARIANTS) through both tasks -> SPLIT_RESULTS.
_t_start = time.time()
_rows, _fam_rows, _CACHE = [], [], {}
_S5_assign = pd.Series(S5_ROLE.to_numpy(), index=_BILLIX.get_indexer(S5_ROLE.index))   # shared draw from setup
_S4_CUTDAYS = {c: {q: int(np.datetime64(d.date(), "D").astype(np.int64)) for q, d in v.items()}
               for c, v in S4_CUT_DATES.items()}


def _unit(train, ev, stream=False, pit=True):
    """train: ('congs', tuple) | ('within', c, cutoff_day) | ('random', excluded_roles)
       ev:    ('congs', tuple) | ('window', c, lo_day, hi_day) | ('random', role).
    Returns per-bill Task 1 metrics and Task 2 scores (cached; family members only for by-Congress units)."""
    key = (train, ev, stream, pit)
    if key in _CACHE:
        return _CACHE[key]
    fam = train[0] == "congs" and not stream and pit
    if train[0] == "congs":
        tmask = np.isin(_CD["cong"], train[1])
        tr2 = _ADV[_ADV.congress.isin(train[1])]
    elif train[0] == "within":
        c, cut = train[1], train[2]
        tmask = (_CD["cong"] < c) | ((_CD["cong"] == c) & (_CD["day"] < cut))
        tr2 = _ADV[(_ADV.congress < c) | ((_ADV.congress == c) & (_ADV.day + ADV_WINDOW_DAYS <= cut))]
    else:
        held = _S5_assign[_S5_assign.isin(train[1])].index.to_numpy()
        tmask = ~np.isin(_CD["bill"], held)
        tr2 = _ADV[~_ADV.code.isin(held)]
    if ev[0] == "congs":
        codes = _BL.index[_BL.congress.isin(ev[1])].to_numpy()
    elif ev[0] == "window":
        codes = _BL.index[(_BL.congress == ev[1]) & (_BL.day >= ev[2]) & (_BL.day < ev[3])].to_numpy()
    else:
        codes = _S5_assign.index[_S5_assign == ev[1]].to_numpy()
    out = {"fam": fam}
    cnt = {None: _counts(tmask)}                                  # shared by both t0
    for t0 in (0, 7):
        out[f"cosponsor_t{t0}"] = _t1_eval(tmask, codes, t0, stream=stream, family=fam and t0 == 0, pit=pit, cnt=cnt)
    if pit:
        te2 = _ADV[_ADV.code.isin(codes)]
        ok2 = len(te2) and te2.label.nunique() == 2
        out["advancement"] = (_t2_scores(tr2, te2, stream=stream, family=fam), te2) if ok2 else None
    _CACHE[key] = out
    return out


def _add(split, fold, role, train, ev, train_label, eval_congress, stream=False):
    u = _unit(train, ev, stream)
    key = (train, ev, stream)
    meta = dict(split=split, fold=fold, train_congresses=train_label, eval_congress=eval_congress, eval_role=role)
    for t0 in (0, 7):
        pb, codes = u[f"cosponsor_t{t0}"]
        for r in _t1_summarize(pb, codes, (key, t0), baselines=T1_BASELINES):
            _rows.append(dict(meta, task=f"cosponsor_t{t0}", **r))
        if u["fam"] and t0 == 0 and split in ("S2", "S3"):
            for r in _t1_summarize(pb, codes, (key, t0), baselines=list(T1_FAMILY) + list(RW_FAMILY)):
                _fam_rows.append(dict(meta, task="cosponsor_t0", family=_FAM_OF[r["baseline"]], **r))
    if u["advancement"] is not None:
        sc, te2 = u["advancement"]
        for r in _t2_summarize(sc, te2, key, baselines=T2_BASELINES):
            _rows.append(dict(meta, task="advancement", **r))
        if u["fam"] and split in ("S2", "S3"):
            for r in _t2_summarize(sc, te2, key, baselines=list(T2_FAMILY) + list(T2S_FAMILY)):
                _fam_rows.append(dict(meta, task="advancement", family=_FAM_OF[r["baseline"]], **r))


def _unit_keys(u):
    """Map an iter_eval_units() dict onto the engine's (train, ev) keys."""
    if u["within"] is not None:
        c, tq, lo, hi = u["within"]
        cd = _S4_CUTDAYS[c]
        return ("within", c, cd[tq]), ("window", c, cd[lo], cd[hi])
    if u.get("random") is not None:
        role = u["random"][1]
        return ("random", ("val", "test") if role == "val" else ("test",)), ("random", role)
    return ("congs", tuple(u["train"])), ("congs", (u["eval_congress"],))


_MIDX = {"Recall@10": 0, "Recall@20": 1, "NDCG@20": 2}


def _fold_unit(split, c, role="test", **kw):
    """Engine unit of fold c: rolling 'S3', sliding 'S2', or 'S3norefit' (test Congress c, fit on 108..c-2 = no refit)."""
    ev = c if role == "test" else c - 1
    tr = {"S3": range(CONGRESSES[0], ev), "S2": range(ev - SLIDE_K, ev), "S3norefit": range(CONGRESSES[0], c - 1)}[split]
    return _unit(("congs", tuple(tr)), ("congs", (ev,)), **kw)


def _pdiff(uA, a, uB, b, task, metric, key):
    """Paired cluster-bootstrap difference score(uA, a) - score(uB, b) on the same evaluation bills
    (same resamples for both): (point, ci_low, ci_high, half-width)."""
    if task == "advancement":
        (sa, te), (sb, teb) = uA[task], uB[task]
        assert np.array_equal(te.code.to_numpy(), teb.code.to_numpy())
        y, W = te.y.to_numpy(), _boot_weights(te.cluster.to_numpy(), key)
        ra, rb = _auc_ap(sa[a], y, W), _auc_ap(sb[b], y, W)
        est = ra[0] - rb[0] if metric == "AUROC" else ra[1] / ra[2] - rb[1] / rb[2]
    else:
        (pa, ca), (pb_, cb) = uA[task], uB[task]
        assert np.array_equal(ca, cb)
        W = _boot_weights(_BL.cluster.reindex(ca).to_numpy(), key)
        est = W @ (pa[a][:, _MIDX[metric]] - pb_[b][:, _MIDX[metric]]) / W.sum(1)
    lo, hi = np.percentile(est[1:], [2.5, 97.5])
    return float(est[0]), float(lo), float(hi), float((hi - lo) / 2)


def _n_test_c(s):
    """Number of test Congresses a by-Congress split (or variant) reports; None for S4 / S5."""
    return len(split_eval_congresses(s, roles=("test",))) if split_kind(s) in BY_CONGRESS_KINDS else None


for u in iter_eval_units():
    tr, ev = _unit_keys(u)
    ec = u["eval_congress"] if u["eval_congress"] is not None else pd.NA
    _add(u["split"], u["fold"], u["role"], tr, ev, u["train_label"], ec)
    if u["split"] + "+stream" in SPLIT_VARIANTS:              # streaming variant: same pools, eval-Congress history
        _add(u["split"] + "+stream", u["fold"], u["role"], tr, ev, u["train_label"] + " + stream", ec, stream=True)
_pv = SPLIT_VARIANTS["S1pool"]
_pa, _pval = _pv["train"], _pv["val"][0]
_add("S1pool", "S1pool", "val", ("congs", tuple(_pa)), ("congs", (_pval,)), _rng_label(_pa), _pval)
_add("S1pool", "S1pool", "test", ("congs", tuple(_pa + [_pval])), ("congs", tuple(_pv["test"])), _rng_label(_pa + [_pval]), pd.NA)

SPLIT_RESULTS = pd.DataFrame(_rows)
SPLIT_RESULTS["eval_congress"] = SPLIT_RESULTS["eval_congress"].astype("Int64")
D_FAMILY_RESULTS = pd.DataFrame(_fam_rows)
_d_secs = time.time() - _t_start
_pp, _pl = np.concatenate(_D_LOG["pool_pit"]), None
print(f"backtest: {SPLIT_RESULTS.groupby(['split', 'fold', 'eval_role']).ngroups} split/fold/role units, "
      f"{len(SPLIT_RESULTS):,} result rows (+{len(D_FAMILY_RESULTS):,} near-tie family rows), {_d_secs:.0f}s "
      f"(streaming {_D_LOG['stream_s']:.0f}s); no subsampling (all evaluable bills used)")
print(f"Task 1 (t0=0) positives missing from the Congress roster (dropped): {_D_LOG['unmatched_pos']:,} of "
      f"{_D_LOG['pos_total']:,} ({_D_LOG['unmatched_pos'] / max(_D_LOG['pos_total'], 1):.2%}); positives outside the "
      f"point-in-time pool (dropped): {_D_LOG['pos_not_in_office']:,} of {_D_LOG['pos_pit_total']:,} "
      f"({_D_LOG['pos_not_in_office'] / max(_D_LOG['pos_pit_total'], 1):.2%}); median eligible pool {np.median(_pp):.0f}")


def _sel(task, metric, baseline=None, df=None, **kw):
    d = SPLIT_RESULTS if df is None else df
    d = d[(d.task == task) & (d.metric == metric)]
    if baseline is not None:
        d = d[d.baseline == baseline]
    for k, v in kw.items():
        d = d[d[k] == v]
    return d


_ov = SPLIT_RESULTS[SPLIT_RESULTS.eval_role == "test"]
_ov = _ov[((_ov.task == _D_PRIMARY_T1[0]) & (_ov.metric == _D_PRIMARY_T1[1]))
          | ((_ov.task == _D_PRIMARY_T2[0]) & _ov.metric.isin(["AUROC", "AP_lift"]) & (_ov.baseline != "prior"))]
print("\nMean test score per split (Task 1 Recall@20 at t0=0; Task 2 AUROC and AP lift = AP / positive rate):")
print(_ov.pivot_table(index=["task", "metric", "baseline"], columns="split", values="value").round(3).to_string())

backtest: 60 split/fold/role units, 2,160 result rows (+1,512 near-tie family rows), 26s (streaming 4s); no subsampling (all evaluable bills used)
Task 1 (t0=0) positives missing from the Congress roster (dropped): 0 of 580,002 (0.00%); positives outside the point-in-time pool (dropped): 443 of 580,002 (0.08%); median eligible pool 434

Mean test score per split (Task 1 Recall@20 at t0=0; Task 2 AUROC and AP lift = AP / positive rate):
split                                    S1a  S1a+stream    S1b  S1b+stream  S1pool     S2     S3  S3+stream     S4     S5
task         metric    baseline                                                                                           
advancement  AP_lift   chamber_maj    1.0930      1.0280 1.3100      1.2050  1.2180 1.3300 1.2860     1.3740 1.5180 1.2530
                       eb_rate        2.2150      2.1690 2.1140      2.1170  2.1040 2.1290 2.1590     2.1740 3.8940 2.4280
                       eb_rate_t0safe 1.2750      1.2550 1.3480     

### D1-D2, D20. Does the validation score predict the test score *level*?

The validation Congress stands in for the test Congress when we tune and early-stop. If val and test
scores differ a lot (Congress *c* is systematically easier or harder than *c-1*), loss-based early
stopping and score thresholds are tuned on a different target. We report the gap (test - val) per
split and fold; a gap is *significant* when it exceeds the 95% half-width of the difference under the
cluster bootstrap (sqrt(hw_val^2 + hw_test^2)). **Calibration:** instead of a fixed relative tolerance,
a split FAILS when its gaps are larger than sampling noise explains: chi-square test of
sum (gap / se_diff)^2 over its folds (df = number of folds; a single-fold split reduces to |z| > 1.96).
Val and test share many sponsors, so se_diff is conservative. We also print the noise-expected mean
|relative gap| (sqrt(2/pi) * se_diff / val) next to the observed one. Note that a pure **level shift** does not harm model
*selection* as long as the ranking of candidates holds; rankings and selection regret are measured
directly in D11-D12, D17 and D21, so the Task 2 level gap (D2) is informational. D20 repeats D1 for
the graph-structural `rw2hop` baseline. In the chart each row is a split/fold, the two dots are val and
test with 95% cluster-bootstrap CIs; long segments mean the validation Congress is a poor proxy.

In [156]:
# D1/D2/D20: validation-to-test gaps per split and fold for the primary metrics of both tasks.
def _gaps(task, metric, df=None):
    d = _sel(task, metric, df=df).pivot_table(index=["split", "fold", "baseline"], columns="eval_role",
                                              values=["value", "ci_low", "ci_high"])
    g = pd.DataFrame({"val": d[("value", "val")], "test": d[("value", "test")],
                      "hw_val": (d[("ci_high", "val")] - d[("ci_low", "val")]) / 2,
                      "hw_test": (d[("ci_high", "test")] - d[("ci_low", "test")]) / 2}).reset_index()
    g["gap"] = g.test - g.val
    g["rel_gap"] = g.gap / g.val
    g["hw_diff"] = np.sqrt(g.hw_val ** 2 + g.hw_test ** 2)
    g["signif"] = g.gap.abs() > g.hw_diff
    return g


_g1 = _gaps(*_D_PRIMARY_T1)
_g1b = _gaps("cosponsor_t7", "Recall@20")
_g2 = _gaps(*_D_PRIMARY_T2)
_t1v = _sel(*_D_PRIMARY_T1, split="S1b", eval_role="val")
_best1 = _t1v[_t1v.baseline != "rw2hop"].sort_values("value").baseline.iloc[-1]   # best count baseline; rw2hop gets D20
_best2 = _sel(*_D_PRIMARY_T2, split="S1b", eval_role="val").sort_values("value").baseline.iloc[-1]


def _gap_table(g, best):
    t = g[g.baseline == best].copy()
    return t.groupby("split").agg(mean_abs_gap=("gap", lambda x: x.abs().mean()),
                                  mean_abs_rel_gap=("rel_gap", lambda x: x.abs().mean()),
                                  max_abs_rel_gap=("rel_gap", lambda x: x.abs().max()),
                                  mean_hw_diff=("hw_diff", "mean"), share_signif=("signif", "mean"),
                                  folds=("fold", "nunique"))


_gt1, _gtr, _gt2 = _gap_table(_g1, _best1), _gap_table(_g1, "rw2hop"), _gap_table(_g2, _best2)
print(f"Mean |test - val| per split (Task 1 Recall@20 at t0=0: '{_best1}', 'rw2hop'; Task 2 AUROC: '{_best2}'):")
print(pd.DataFrame({"folds": _gt1.folds, f"T1 {_best1}": _gt1.mean_abs_gap, "T1 max|rel|": _gt1.max_abs_rel_gap,
                    "T1 rw2hop": _gtr.mean_abs_gap, f"T2 {_best2}": _gt2.mean_abs_gap, "T2 hw_diff": _gt2.mean_hw_diff,
                    "T2 share_signif": _gt2.share_signif}).round(3).to_string())

_dd = _sel(*_D_PRIMARY_T1, baseline=_best1)
_dd = _dd[_dd.split.isin(["S1a", "S1b", "S1b+stream", "S4", "S5"]) | ((_dd.split == "S2") & _dd.fold.isin(["c=117", "c=118"]))]
_dd = _dd.assign(row=_dd.split + " " + _dd.fold.where(_dd.split.isin(["S2", "S4"]), ""))[
    ["row", "eval_role", "value", "ci_low", "ci_high"]]
_DUMBBELL_JS = r"""
const {width, height, title, subtitle, xlabel} = opts;
const m = {top: 48, right: 24, bottom: 52, left: 110};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title, subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const rows = Array.from(new Set(data.map(d => d.row)));
const ys = d3.scaleBand(rows, [0, H]).padding(0.3);
const xs = d3.scaleLinear([d3.min(data, d => d.ci_low), d3.max(data, d => d.ci_high)], [0, W]).nice();
const col = {val: viz.palette[0], test: viz.palette[1]};
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(6).tickSize(-H).tickFormat(d3.format('.2f')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').call(d3.axisLeft(ys).tickSize(0)).call(viz.axis).call(a => a.select('.domain').remove());
g.append('text').attr('x', W).attr('y', H + 32).attr('text-anchor', 'end').attr('fill', viz.ink2).style('font', '11px sans-serif').text(xlabel);
rows.forEach(r => {
  const v = data.find(d => d.row === r && d.eval_role === 'val'), t = data.find(d => d.row === r && d.eval_role === 'test');
  if (v && t) g.append('line').attr('x1', xs(v.value)).attr('x2', xs(t.value)).attr('y1', ys(r) + ys.bandwidth() / 2)
    .attr('y2', ys(r) + ys.bandwidth() / 2).attr('stroke', '#b5b4ae').attr('stroke-width', 2);
});
const tip = viz.tooltip(el);
const off = d => ys(d.row) + ys.bandwidth() / 2 + (d.eval_role === 'val' ? -3 : 3);
g.selectAll(null).data(data).join('line').attr('x1', d => xs(d.ci_low)).attr('x2', d => xs(d.ci_high))
  .attr('y1', off).attr('y2', off).attr('stroke', d => col[d.eval_role]).attr('stroke-width', 1.5).attr('opacity', 0.6);
g.selectAll(null).data(data).join('circle').attr('cx', d => xs(d.value)).attr('cy', off).attr('r', 5)
  .attr('fill', d => col[d.eval_role]).attr('stroke', '#fff').attr('stroke-width', 1.5)
  .on('mousemove', (e, d) => tip.show(`<b>${d.row} · ${d.eval_role}</b><br>${d3.format('.3f')(d.value)} [${d3.format('.3f')(d.ci_low)}, ${d3.format('.3f')(d.ci_high)}]`, e))
  .on('mouseleave', () => tip.hide());
viz.legend(svg, [['validation', col.val], ['test', col.test]], m.left, height - 10);
"""
show_d3(_DUMBBELL_JS, _dd, height=60 + 34 * _dd.row.nunique() + 50, width=720,
        opts=dict(title=f"Validation vs test Recall@20 ({_best1}, t0=0) per split",
                  subtitle="dots = point estimate, whiskers = 95% cluster-bootstrap CI (by sponsor); S5 is the leaky random split",
                  xlabel="Recall@20"), name="D_val_test_dumbbell")

# Per-fold gaps of the one-step-ahead protocol (S3 folds; S1a/S1b are the folds c=117 and c=118).
_s3g1 = _g1[(_g1.split == "S3") & (_g1.baseline == _best1)].set_index("fold")
_s3g1b = _g1b[(_g1b.split == "S3") & (_g1b.baseline == _best1)].set_index("fold")
_s3gr = _g1[(_g1.split == "S3") & (_g1.baseline == "rw2hop")].set_index("fold")
_s3g2 = _g2[(_g2.split == "S3") & (_g2.baseline == _best2)].set_index("fold")
_pf = pd.DataFrame({"T1_val": _s3g1.val, "T1_test": _s3g1.test, "T1_gap": _s3g1.gap, "T1_hw_diff": _s3g1.hw_diff,
                    "T1rw_gap": _s3gr.gap, "T2_val": _s3g2.val, "T2_test": _s3g2.test, "T2_gap": _s3g2.gap,
                    "T2_hw_diff": _s3g2.hw_diff})
print(f"\nPer-fold gaps, S3 (test c, val c-1): Task 1 Recall@20 '{_best1}' and rw2hop (t0=0), Task 2 AUROC '{_best2}':")
print(_pf.round(3).to_string())


def _gap_list(g, best, splits=("S1a", "S1b")):
    t = g[(g.baseline == best) & g.split.isin(splits)]
    return ", ".join(f"{r.split} {r.gap:+.3f}{' (signif.)' if r.signif else ''}" for r in t.itertuples())


def _gap_test(g, b):
    """Per split: chi-square test that val-test gaps are sampling noise (sum z^2, df = folds), observed and
    noise-expected mean |relative gap|."""
    t = g[(g.baseline == b) & g.split.isin(ALL_SPLITS)].copy()
    t["z2"] = (t.gap / (t.hw_diff / 1.96)) ** 2
    t["exp_rel"] = np.sqrt(2 / np.pi) * (t.hw_diff / 1.96) / t.val
    r = t.groupby("split").agg(folds=("z2", "size"), chi2=("z2", "sum"), mean_abs_rel=("rel_gap", lambda x: x.abs().mean()),
                               noise_rel=("exp_rel", "mean"))
    r["p"] = stats.chi2.sf(r.chi2, r.folds)
    r["ok"] = r.p >= 0.05
    return r


def _gap_check(cid, cond, g, b, splits=None, quantity=None, extra="", kind="validity", t0_task2=None, na=None):
    r = _gap_test(g, b)
    if splits is not None:
        r = r[r.index.isin(splits)]
    main = "S3+stream" if "S3+stream" in r.index and splits is not None else "S3"
    m = r.loc[main]
    record_check("D", cid, cond, bool(m.ok), metric=float(m.mean_abs_rel), quantity=quantity, kind=kind, t0_task2=t0_task2,
                 threshold="FAIL if gaps exceed cluster-bootstrap noise: chi-square p < 0.05 (sum of z^2 over the split's folds)",
                 by_split=_bs({k: bool(v) for k, v in r.ok.items()}),
                 na_reason=na or "this model / protocol is not scored with a val and a test unit under that split",
                 detail=f"'{b}', {main}: mean |rel gap| {m.mean_abs_rel:.3f} vs {m.noise_rel:.3f} expected from sampling noise alone, "
                        f"chi2 {m.chi2:.1f} on {int(m.folds)} df, p={m.p:.3f}. Per split p: "
                        + ", ".join(f"{k} {v:.2f}" for k, v in r.p.items()) + ". " + extra)
    return r


_gt_d1 = _gap_check("D1", "Task 1: validation Congress predicts the test score level (val-test gaps within sampling noise, S3 folds)",
                    _g1, _best1, quantity="val_test_gap",
                    extra=f"t0=7 mean |rel gap| {_s3g1b.rel_gap.abs().mean():.3f}; S1 gaps: {_gap_list(_g1, _best1)}. A gap beyond "
                          "noise means early stopping / thresholds tune for a different Congress than the one tested.")
_d1 = float(_s3g1.rel_gap.abs().mean())
_d2 = float(_s3g2.gap.abs().mean())
_d2s = float(_s3g2.signif.mean())
record_check("D", "D2", "Task 2: validation AUROC level vs test AUROC level (mean |val-test gap| over one-step-ahead folds; informational)",
             None, metric=_d2, threshold="informational: level shifts do not harm selection if rankings hold (see D12/D17)",
             by_split=_bs({k: round(float(v), 4) for k, v in _gt2.mean_abs_gap.items() if k in ALL_SPLITS}), kind="drift",
             t0_task2=30, na_reason="no val / test pair under that split",
             detail=f"'{_best2}' over S3 folds: {_d2s:.0%} of folds have |gap| beyond the 95% cluster-bootstrap half-width of the "
                    f"difference (mean hw_diff {_s3g2.hw_diff.mean():.3f}); largest gap at {_s3g2.gap.abs().idxmax()} "
                    f"({_s3g2.gap.abs().max():.3f}). S1 gaps: {_gap_list(_g2, _best2)}. Task 2 levels move from Congress to "
                    "Congress (base-rate drift, section A), so a single val Congress mis-states the expected test AUROC; this "
                    "matters for reporting and calibration, not for picking models (D12/D17 test rankings directly). Report "
                    "Task 2 scores per test Congress. t0-safe variant (prediction at introduction): mean |gap| "
                    f"{_g2[(_g2.split == 'S3') & (_g2.baseline == 'eb_rate_t0safe')].gap.abs().mean():.3f}.")
_gap_check("D2b", "Task 2 at introduction (eb_rate_t0safe): val Congress predicts the test AUROC level within noise "
                  "[calibration and reporting only, not model selection]", _g2, "eb_rate_t0safe", kind="drift", t0_task2=0,
           extra="Scope: thresholds, calibration and the expected score level, NOT the choice of model (rankings transfer: "
                 "D12b). A FAIL under a split means Task-2 decision thresholds / probabilities tuned on its val Congress do not "
                 "carry over: recalibrate on the eval Congress's already-resolved bills (streaming, labels known 180 days after "
                 "introduction) and report Task-2 scores per test Congress.")
_gap_check("D20", "Task 1, graph-structural rw2hop: validation Congress predicts the test score level (gaps within noise, S3)",
           _g1, "rw2hop", extra=f"S1 gaps: {_gap_list(_g1, 'rw2hop')}. rw2hop behaves almost exactly like sponsor_hist (D9), so "
                                "the learned-embedding repeat of this check is D22.")

Mean |test - val| per split (Task 1 Recall@20 at t0=0: 'sponsor_hist', 'rw2hop'; Task 2 AUROC: 'eb_rate'):
            folds  T1 sponsor_hist  T1 max|rel|  T1 rw2hop  T2 eb_rate  T2 hw_diff  T2 share_signif
split                                                                                              
S1a             1           0.0070       0.0290     0.0100      0.0050      0.0450           0.0000
S1a+stream      1           0.0190       0.0650     0.0210      0.0000      0.0430           0.0000
S1b             1           0.0180       0.0680     0.0180      0.0050      0.0520           0.0000
S1b+stream      1           0.0100       0.0330     0.0120      0.0080      0.0480           0.0000
S1pool          1           0.0320       0.1280     0.0300      0.0050      0.0380           0.0000
S2              7           0.0120       0.0720     0.0110      0.0370      0.0390           0.5710
S3              7           0.0110       0.0680     0.0100      0.0300      0.0390           


Per-fold gaps, S3 (test c, val c-1): Task 1 Recall@20 'sponsor_hist' and rw2hop (t0=0), Task 2 AUROC 'eb_rate':
       T1_val  T1_test  T1_gap  T1_hw_diff  T1rw_gap  T2_val  T2_test  T2_gap  T2_hw_diff
fold                                                                                     
c=112  0.2750   0.2600 -0.0150      0.0290   -0.0120  0.7280   0.7820  0.0550      0.0370
c=113  0.2600   0.2580 -0.0020      0.0260   -0.0000  0.7820   0.7540 -0.0280      0.0370
c=114  0.2580   0.2650  0.0070      0.0250    0.0050  0.7540   0.7150 -0.0390      0.0380
c=115  0.2650   0.2720  0.0070      0.0240    0.0070  0.7150   0.7420  0.0270      0.0340
c=116  0.2720   0.2540 -0.0180      0.0240   -0.0170  0.7420   0.6880 -0.0550      0.0330
c=117  0.2540   0.2620  0.0070      0.0240    0.0100  0.6880   0.6920  0.0050      0.0450
c=118  0.2620   0.2440 -0.0180      0.0240   -0.0180  0.6920   0.6970  0.0050      0.0520
[PASS] D1: Task 1: validation Congress predicts the test score level (val-tes

,folds,chi2,mean_abs_rel,noise_rel,p,ok
split,,,,,,
S1a,1,0.6438,0.0378,0.0376,0.4223,True
S1a+stream,1,2.7723,0.0740,0.0355,0.0959,True
S1b,1,2.0831,0.0667,0.0369,0.1489,True
S1b+stream,1,0.8694,0.0377,0.0322,0.3511,True
S1pool,1,6.2703,0.1196,0.0381,0.0123,False
S2,7,5.2314,0.0389,0.0386,0.6318,True
S3,7,5.9953,0.0371,0.0381,0.5403,True
S3+stream,7,4.8087,0.0299,0.0343,0.6833,True
S4,3,3.0453,0.0695,0.0584,0.3847,True


### D3-D4. How much does the score *level* move from one Congress to the next?

The rolling-origin folds (S3) repeat the one-step-ahead protocol for every test Congress from
112 to 118. If a fixed baseline's score swings across folds more than sampling noise allows
(between-fold std vs the cluster-bootstrap standard error, hw/1.96), then a single test Congress
(as in S1) gives a Congress-specific number: report scores per Congress and their mean over rolling
folds. This concerns the *reported level*; comparisons between models on the same test bills are
paired and much more stable (D9). Lines are per baseline; shaded bands are 95% cluster-bootstrap CIs.

In [157]:
# D3/D4: fold-to-fold variability of test scores across rolling-origin folds (S3), both tasks.
_BAND_JS = r"""
const {width, height, title, subtitle, ylabel, yfmt} = opts;
const m = {top: 48, right: 120, bottom: 52, left: 56};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title, subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const keys = Array.from(new Set(data.map(d => d.series)));
const color = d3.scaleOrdinal(keys, viz.palette);
const xs = d3.scaleLinear(d3.extent(data, d => d.x), [0, W]);
const ys = d3.scaleLinear([d3.min(data, d => d.lo), d3.max(data, d => d.hi)], [H, 0]).nice();
g.append('g').call(d3.axisLeft(ys).ticks(5).tickSize(-W).tickFormat(d3.format(yfmt)))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(new Set(data.map(d => d.x)).size).tickFormat(d3.format('d'))).call(viz.axis);
g.append('text').attr('x', -m.left + 4).attr('y', -8).attr('fill', viz.ink2).style('font', '11px sans-serif').text(ylabel);
g.append('text').attr('x', W).attr('y', H + 32).attr('text-anchor', 'end').attr('fill', viz.ink2).style('font', '11px sans-serif').text('test Congress');
const tip = viz.tooltip(el);
keys.forEach(k => {
  const rows = data.filter(d => d.series === k).sort((a, b) => a.x - b.x);
  g.append('path').datum(rows).attr('fill', color(k)).attr('opacity', 0.15)
    .attr('d', d3.area().x(d => xs(d.x)).y0(d => ys(d.lo)).y1(d => ys(d.hi)));
  g.append('path').datum(rows).attr('fill', 'none').attr('stroke', color(k)).attr('stroke-width', 2)
    .attr('d', d3.line().x(d => xs(d.x)).y(d => ys(d.y)));
  g.selectAll(null).data(rows).join('circle').attr('cx', d => xs(d.x)).attr('cy', d => ys(d.y)).attr('r', 4)
    .attr('fill', color(k)).attr('stroke', '#fff').attr('stroke-width', 1.5)
    .on('mousemove', (e, d) => tip.show(`<b>${k} · ${d.x}</b><br>${d3.format(yfmt)(d.y)} [${d3.format(yfmt)(d.lo)}, ${d3.format(yfmt)(d.hi)}]<br>n=${d.n}`, e))
    .on('mouseleave', () => tip.hide());
  const last = rows[rows.length - 1];
  g.append('text').attr('x', xs(last.x) + 8).attr('y', ys(last.y) + 4).attr('fill', viz.ink2).style('font', '11px sans-serif').text(k);
});
viz.legend(svg, keys.map(k => [k, color(k)]), m.left, height - 10);
"""


def _band(task, metric, split="S3", baselines=None, label=None):
    d = _sel(task, metric, split=split, eval_role="test")
    if baselines:
        d = d[d.baseline.isin(baselines)]
    return pd.DataFrame({"x": d.eval_congress.astype(int), "y": d.value, "lo": d.ci_low, "hi": d.ci_high,
                         "n": d.n_eval, "series": d.baseline if label is None else label}).reset_index(drop=True)


show_d3(_BAND_JS, _band(*_D_PRIMARY_T1), height=330, width=720,
        opts=dict(title="Task 1 test Recall@20 across rolling-origin folds (S3, t0=0)",
                  subtitle="each point: baselines refit on 108..c-1, evaluated on Congress c; bands = cluster-bootstrap 95% CI",
                  ylabel="Recall@20", yfmt=".2f"), name="D_t1_rolling")
show_d3(_BAND_JS, _band(*_D_PRIMARY_T2, baselines=["chamber_maj", "eb_rate", "eb_rate_t0safe"]), height=330, width=720,
        opts=dict(title="Task 2 test AUROC across rolling-origin folds (S3)",
                  subtitle="prior omitted (AUROC = 0.5 by construction); eb_rate_t0safe hides policy area for fresh bills",
                  ylabel="AUROC", yfmt=".2f"), name="D_t2_rolling")


def _fold_var(task, metric, split="S3"):
    d = _sel(task, metric, split=split, eval_role="test").sort_values("eval_congress")
    out = []
    for b, g in d.groupby("baseline"):
        hw = ((g.ci_high - g.ci_low) / 2).to_numpy()
        v = g.value.to_numpy()
        out.append(dict(baseline=b, mean=v.mean(), std=v.std(ddof=1), cv=v.std(ddof=1) / v.mean(),
                        min=v.min(), max=v.max(), mean_ci_hw=hw.mean(),
                        std_over_se=v.std(ddof=1) / (hw.mean() / 1.96) if hw.mean() > 0 else np.nan))
    return pd.DataFrame(out).set_index("baseline")


_fv1, _fv2 = _fold_var(*_D_PRIMARY_T1), _fold_var(*_D_PRIMARY_T2)
print("Test score across S3 folds (T1 Recall@20 at t0=0, T2 AUROC; std_over_se = between-fold std / cluster-bootstrap SE):")
print(pd.concat({"T1": _fv1, "T2": _fv2.drop(index="prior")}).round(3).to_string())
_r = _fv1.loc[_best1]
_r2 = _fv2.loc[_best2]
record_check("D", "D3", "Task 1: test score level across rolling folds (CV of best baseline across S3 folds; informational)",
             None, metric=float(_r.cv), threshold="informational (CV <= 0.10 = level stable)", quantity="score_stability",
             detail=f"'{_best1}' Recall@20 ranges {_r['min']:.3f}-{_r['max']:.3f} over test Congresses {FOLD_C[0]}-{FOLD_C[-1]}; "
                    f"between-fold std is {_r.std_over_se:.1f}x the cluster-bootstrap SE (>1.5x = real Congress-to-Congress "
                    f"change beyond sampling noise); rw2hop CV {_fv1.loc['rw2hop', 'cv']:.3f}. Affects the reported level only; "
                    "paired model comparisons are sized in D9. Consecutive folds share most training Congresses.")
record_check("D", "D4", "Task 2: test AUROC level across rolling folds (std of best baseline across S3 folds; informational)",
             None, metric=float(_r2["std"]), threshold="informational (std > 0.03 and > 1.5x SE = level moves)", kind="validity",
             t0_task2=30,
             detail=f"'{_best2}' AUROC ranges {_r2['min']:.3f}-{_r2['max']:.3f}; between-fold std = {_r2.std_over_se:.1f}x the "
                    "cluster-bootstrap SE of one fold, so the Task 2 level genuinely changes between Congresses. Implication: report "
                    "Task 2 test scores per Congress and their mean over several rolling folds; a single test Congress (S1) is a "
                    "Congress-specific number. Model comparisons are paired and far more stable (see D9 for the paired sizing).")

Test score across S3 folds (T1 Recall@20 at t0=0, T2 AUROC; std_over_se = between-fold std / cluster-bootstrap SE):
                    mean    std     cv    min    max  mean_ci_hw  std_over_se
   baseline                                                                  
T1 party+pop      0.2010 0.0150 0.0730 0.1770 0.2190      0.0170       1.6750
   popularity     0.1430 0.0160 0.1110 0.1130 0.1620      0.0150       2.1280
   rw2hop         0.2590 0.0080 0.0310 0.2450 0.2700      0.0170       0.9260
   sponsor_hist   0.2590 0.0090 0.0340 0.2440 0.2720      0.0170       0.9960
T2 chamber_maj    0.6030 0.0480 0.0800 0.5350 0.6560      0.0330       2.9100
   eb_rate        0.7240 0.0360 0.0500 0.6880 0.7820      0.0280       2.4730
   eb_rate_t0safe 0.6240 0.0450 0.0710 0.5600 0.6680      0.0290       2.9660
[INFO] D3: Task 1: test score level across rolling folds (CV of best baseline across S3 folds; informational) | metric=0.03396 (threshold informational (CV <= 0.10 = level stable))
[

### D5-D6, D11-D12, D17, D21. Does validation pick the model that is best on test?

The real job of a validation set is choosing among **close** candidates (hyperparameters, decay
rates, regularization). D11, D12 and D21 use near-tie families and measure, per rolling fold, whether
the validation-best candidate is also test-best (hit rate), how much test score is lost by trusting
validation (**regret** = test score of the test-best minus test score of the val-chosen candidate) and
the Kendall tau between the val and test orderings. Family members are often statistically
indistinguishable on one Congress, so a regret only counts as **significant** when the paired
cluster-bootstrap CI of (test-best minus val pick) on that test Congress excludes 0 (this is liberal:
the test-best is itself chosen on test). A check FAILs only if mean regret > D_TOL = MDE / 2 (0.005) **and** at least
2 of 7 folds have significant regret (2+ false alarms out of 7 at the 5% level has probability ~4%).
D17 repeats the selection using the mean val score of the last three rolling folds. D5-D6 compare
the main baselines (informational: most pairs are far apart). D12b repeats D12 for the t0-safe Task 2
family (the deployable one if Task 2 predicts at introduction). **Single-fold splits** (S1a, S1b) cannot
meet a "2 of 7 folds" rule, so their `by_split` entry is the regret measured on their own fold (a
metric, not a verdict), and the detail says where it falls in the S3 distribution of per-fold regret.

In [158]:
# D5/D6 (informational): val vs test ordering of the main baselines; D11/D12/D17/D21: near-tie family selection.
def _order_agree(task, metric, split="S3", baselines=None, df=None):
    d = _sel(task, metric, df=df, split=split)
    if baselines:
        d = d[d.baseline.isin(baselines)]
    piv = d.pivot_table(index=["fold", "eval_role"], columns="baseline", values="value")
    folds = list(piv.index.get_level_values(0).unique())
    rows = []
    for f in folds:
        v, t = piv.loc[(f, "val")], piv.loc[(f, "test")]
        # alternative: pick on the mean val score of this and the 2 previous folds (val Congresses c-3..c-1; no test data)
        prev = [x for x in folds if int(x[2:]) in range(int(f[2:]) - 2, int(f[2:]) + 1)]
        vm = pd.concat([piv.loc[(x, "val")] for x in prev], axis=1).mean(axis=1)
        rows.append(dict(fold=f, same_order=list(v.sort_values().index) == list(t.sort_values().index),
                         tau=stats.kendalltau(v.values, t.values)[0], val_pick=v.idxmax(), pick_3val=vm.idxmax(),
                         test_best=t.idxmax(), regret=t.max() - t[v.idxmax()], regret_3val=t.max() - t[vm.idxmax()],
                         test_spread=t.max() - t.min()))
    return pd.DataFrame(rows).set_index("fold")


def _regret_sig(o, task, metric, split, pick_col="val_pick"):
    """Per fold: half-width and significance of the paired difference (test-best minus pick) on the test Congress."""
    hw, sig = [], []
    for f, r in o.iterrows():
        if r[pick_col] == r.test_best:
            hw.append(0.0), sig.append(False)
            continue
        u = _fold_unit(split, int(f[2:]))
        _, lo, _, h = _pdiff(u, r.test_best, u, r[pick_col], task, metric, ("regret", split, f, task, metric, pick_col))
        hw.append(h), sig.append(lo > 0)
    return np.array(hw), np.array(sig)


_o1 = _order_agree(*_D_PRIMARY_T1)
_o2 = _order_agree(*_D_PRIMARY_T2, baselines=["prior", "chamber_maj", "eb_rate"])
_gapmin1 = _sel(*_D_PRIMARY_T1, split="S3", eval_role="test").pivot_table(index="fold", columns="baseline", values="value")
_close1 = _gapmin1[["rw2hop", "sponsor_hist"]].diff(axis=1).iloc[:, 1]
record_check("D", "D5", "Task 1: main baselines keep their order from validation to test (share of S3 folds; informational)",
             None, metric=float(_o1.same_order.mean()), threshold="informational (see D11/D21 for close candidates)",
             detail=f"Kendall tau(val, test) of the 4 main baselines averages {_o1.tau.mean():.2f}. The closest pair is "
                    f"sponsor_hist vs rw2hop (test difference {_close1.mean():+.3f} on average, sign flips in "
                    f"{(np.sign(_close1) != np.sign(_close1.mean())).mean():.0%} of folds); its paired precision is in D9.")
record_check("D", "D6", "Task 2: main baselines keep their order from validation to test (share of S3 folds; informational)",
             None, metric=float(_o2.same_order.mean()), threshold="informational (candidates far apart)", t0_task2=30,
             detail="prior < chamber_maj < eb_rate by wide margins; see D12 for selection among close candidates.")

_sel_rows, _ORD, _SIGS = [], {}, {}
for fam, task, metric in (("T1 sponsor_hist family", *_D_PRIMARY_T1), ("T1 rw2hop family", *_D_PRIMARY_T1),
                          ("T2 eb_rate family", "advancement", "AUROC"), ("T2 eb_rate family", "advancement", "AP_lift"),
                          ("T2 eb_rate_t0safe family", "advancement", "AUROC")):
    for split in ("S3", "S2"):
        o = _order_agree(task, metric, split=split, df=D_FAMILY_RESULTS[D_FAMILY_RESULTS.family == fam])
        hw = _sel(task, metric, df=D_FAMILY_RESULTS[D_FAMILY_RESULTS.family == fam], split=split, eval_role="test")
        if metric != "AP_lift":
            o["regret_hw"], o["regret_sig"] = _regret_sig(o, task, metric, split)
            o["hw3"], o["sig3"] = _regret_sig(o, task, metric, split, "pick_3val")
        _ORD[(fam, metric, split)] = o
        _sel_rows.append(dict(family=fam, metric=metric, split=split, hit_rate=o.val_pick.eq(o.test_best).mean(),
                              mean_regret=o.regret.mean(), max_regret=o.regret.max(),
                              regret_hw=o.get("regret_hw", pd.Series(np.nan)).mean(),
                              n_sig=int(o.get("regret_sig", pd.Series(0)).sum()), regret_3val=o.regret_3val.mean(),
                              n_sig_3val=int(o.get("sig3", pd.Series(0)).sum()), mean_tau=o.tau.mean(),
                              test_spread=o.test_spread.mean(), test_hw=((hw.ci_high - hw.ci_low) / 2).mean(),
                              picks=", ".join(f"{k}:{v}" for k, v in o.val_pick.value_counts().items())))
_selt = pd.DataFrame(_sel_rows)
print("Near-tie family selection on rolling folds (regret = test-best minus test of val pick; regret_hw = mean paired CI\n"
      "half-width of that difference; n_sig = folds (of 7) where it is significant; *_3val = pick on mean of last 3 val folds):")
print(_selt[_selt.metric != "AP_lift"].drop(columns=["picks"]).round(4).to_string(index=False))
print("Val picks (S3): " + " | ".join(f"{r.family.replace(' family', '')} {r.metric}: {r.picks}"
                                     for r in _selt[_selt.split == "S3"].itertuples()))
_regc = pd.concat([pd.DataFrame({"fold": o.index, "regret": o.regret.values, "family": f"{k[0]} ({k[1]})"})
                   for k, o in _ORD.items() if k[2] == "S3" and k[1] != "AP_lift"])
bar_chart(_regc, x="fold", y="regret", series="family", yfmt=".4f", ylabel="regret",
          title="Model-selection regret per rolling fold (S3, near-tie families)",
          subtitle="test score lost by picking the validation-best candidate instead of the test-best (0 = val picked right)",
          height=300, name="D_selection_regret")


def _sel_verdict(fam, metric, split, col="regret", sig="regret_sig", folds=None):
    o = _ORD[(fam, metric, split)]
    if folds is not None:
        o = o[o.index.isin([f"c={c}" for c in folds])]
    tol = D_TOL["advancement" if fam.startswith("T2") else "cosponsor_t0"]
    return bool(not ((o[col].mean() > tol) and (o[sig].sum() >= 2)))


def _fold_regret(fam, metric, c):
    """Single-Congress splits (S1a = S3 fold 117, S1b = fold 118): the regret on that fold (a metric, not a verdict)."""
    return round(float(_ORD[(fam, metric, "S3")].loc[f"c={c}", "regret"]), 4)


def _fold_note(fam, metric):
    o = _ORD[(fam, metric, "S3")]
    parts = []
    for s_ in ("S1a", "S1b"):
        f = f"c={SPLIT_DEFS[s_]['test'][0]}"
        pct = (o.regret <= o.loc[f, "regret"]).mean()
        parts.append(f"{s_} fold regret {o.loc[f, 'regret']:.4f} ({'significant' if o.loc[f, 'regret_sig'] else 'not significant'}, "
                     f"at the {pct:.0%} point of the 7 S3 fold regrets)")
    return "; ".join(parts)


def _sel_check(cid, fam, metric, cond, unit, t0_task2=None):
    r = _selt[(_selt.family == fam) & (_selt.metric == metric) & (_selt.split == "S3")].iloc[0]
    r2 = _selt[(_selt.family == fam) & (_selt.metric == metric) & (_selt.split == "S2")].iloc[0]
    tol = D_TOL["advancement" if fam.startswith("T2") else "cosponsor_t0"]
    record_check("D", cid, cond, _sel_verdict(fam, metric, "S3"), metric=float(r.mean_regret), kind="validity", t0_task2=t0_task2,
                 threshold=f"FAIL if mean regret > D_TOL = MDE/2 = {tol} {unit} AND regret significant (paired cluster CI) in >= 2 of 7 folds",
                 by_split=_bs({"S3": _sel_verdict(fam, metric, "S3"), "S2": _sel_verdict(fam, metric, "S2"),
                               "S1a": _fold_regret(fam, metric, SPLIT_DEFS["S1a"]["test"][0]),
                               "S1b": _fold_regret(fam, metric, SPLIT_DEFS["S1b"]["test"][0])}), na_reason=_NA_FAM,
                 detail=f"Family {FAMILIES[fam]}: hit rate {r.hit_rate:.0%}, max regret {r.max_regret:.4f}, mean regret "
                        f"{r.mean_regret:.4f} vs mean paired half-width {r.regret_hw:.4f}; significant in {r.n_sig}/7 folds; mean "
                        f"tau(val,test) {r.mean_tau:.2f}; family spread on test {r.test_spread:.3f} vs single-score CI hw "
                        f"{r.test_hw:.3f}. S2 folds: regret {r2.mean_regret:.4f} ({r2.n_sig}/7 significant). Picking on the mean of "
                        f"the last 3 val folds: regret {r.regret_3val:.4f} ({r.n_sig_3val}/7 significant). {_fold_note(fam, metric)} "
                        "(by_split S1a/S1b = that regret, a metric). Tolerance = half the section-wide MDE (smallest improvement worth "
                        "acting on); the significance rule keeps churn among statistically tied members from causing a FAIL.")


_sel_check("D11", "T1 sponsor_hist family", "Recall@20",
           "Task 1: validation picks the test-best among close count-history candidates (regret vs paired noise, S3 folds)", "Recall@20")
_sel_check("D12", "T2 eb_rate family", "AUROC",
           "Task 2: validation picks the test-best among close candidates (regret vs paired noise, S3 folds, AUROC)", "AUROC", t0_task2=30)
_sel_check("D12b", "T2 eb_rate_t0safe family", "AUROC",
           "Task 2 at introduction (t0-safe family, policy area hidden for fresh bills): validation picks the test-best "
           "(regret vs paired noise, S3)", "AUROC", t0_task2=0)
_sel_check("D21", "T1 rw2hop family", "Recall@20",
           "Task 1, graph-structural rw2hop family: validation picks the test-best (regret vs paired noise, S3 folds)", "Recall@20")
_v3 = {(f, m, sp): _sel_verdict(f, m, sp, "regret_3val", "sig3") for f, m in (("T1 sponsor_hist family", "Recall@20"),
       ("T1 rw2hop family", "Recall@20"), ("T2 eb_rate family", "AUROC"), ("T2 eb_rate_t0safe family", "AUROC"))
       for sp in ("S3", "S2")}
_r3 = _selt[(_selt.metric != "AP_lift") & (_selt.split == "S3")].set_index("family")
record_check("D", "D17", "Selecting on the mean of the last 3 rolling val folds picks near-test-best candidates (all families, S3)",
             all(v for k, v in _v3.items() if k[2] == "S3"), metric=float(_r3.regret_3val.max()),
             threshold="per family: FAIL if mean regret > D_TOL (MDE/2 = 0.005) AND significant in >= 2 of 7 folds", kind="validity",
             by_split=_bs({sp: all(v for k, v in _v3.items() if k[2] == sp) for sp in ("S3", "S2")}), na_reason=_NA_FAM,
             detail="Regret with 3-fold val selection (single val in brackets): " + "; ".join(
                 f"{f}: {r.regret_3val:.4f} ({r.n_sig_3val}/7 signif.) [{r.mean_regret:.4f}, {r.n_sig}/7]" for f, r in _r3.iterrows())
             + ". Val Congresses c-3..c-1, each scored with its own one-step-ahead training set, so no test data is used. "
               "Only rolling/sliding splits can do this (S1a/S1b have one val Congress).")

[INFO] D5: Task 1: main baselines keep their order from validation to test (share of S3 folds; informational) | metric=0.5714 (threshold informational (see D11/D21 for close candidates))
[INFO] D6: Task 2: main baselines keep their order from validation to test (share of S3 folds; informational) | metric=1 (threshold informational (candidates far apart))
Near-tie family selection on rolling folds (regret = test-best minus test of val pick; regret_hw = mean paired CI
half-width of that difference; n_sig = folds (of 7) where it is significant; *_3val = pick on mean of last 3 val folds):
                  family    metric split  hit_rate  mean_regret  max_regret  regret_hw  n_sig  regret_3val  n_sig_3val  mean_tau  test_spread  test_hw
  T1 sponsor_hist family Recall@20    S3    0.7143       0.0028      0.0143     0.0025      1       0.0007           0    0.8571       0.0238   0.0177
  T1 sponsor_hist family Recall@20    S2    0.4286       0.0026      0.0080     0.0042      1       0.0028

[PASS] D11: Task 1: validation picks the test-best among close count-history candidates (regret vs paired noise, S3 folds) | metric=0.002763 (threshold FAIL if mean regret > D_TOL = MDE/2 = 0.005 Recall@20 AND regret significant (paired cluster CI) in >= 2 of 7 folds)
[PASS] D12: Task 2: validation picks the test-best among close candidates (regret vs paired noise, S3 folds, AUROC) | metric=0.007766 (threshold FAIL if mean regret > D_TOL = MDE/2 = 0.005 AUROC AND regret significant (paired cluster CI) in >= 2 of 7 folds)
[PASS] D12b: Task 2 at introduction (t0-safe family, policy area hidden for fresh bills): validation picks the test-best (regret vs paired noise, S3) | metric=0.004518 (threshold FAIL if mean regret > D_TOL = MDE/2 = 0.005 AUROC AND regret significant (paired cluster CI) in >= 2 of 7 folds)
[PASS] D21: Task 1, graph-structural rw2hop family: validation picks the test-best (regret vs paired noise, S3 folds) | metric=0.0002027 (threshold FAIL if mean regret > D_TOL = MDE

### D22-D25. A learned, transductive embedding model: selection, staleness and cold start

Count heuristics cannot overfit and have no per-legislator parameters, so they cannot show the two
failure modes that matter most for a GNN under a by-Congress split: (1) a **high-capacity model's
validation choice may not carry over to test** (it can fit Congress-specific structure), and (2)
**learned per-legislator embeddings go stale** across an election (a member's role changes: chamber or
party switch, majority status flip, new committee portfolio), which a count heuristic hides. We fit a
truncated SVD of log(1 + sponsor x cosponsor counts) on each fold's training Congresses (ranks 8-128,
typical GNN sizes, plus 256 / 512, close to memorised history; each with or without recency weighting:
14 candidates; rank = capacity), choose the candidate on the validation Congress and refit on train + val for test, exactly as the S3 protocol prescribes. D22 is
the val-test level check (as D1), D23 the selection regret (as D11), D24 reports Recall@20 separately
for positives from **new** legislators (zero embedding) and **returning** ones, and D25 tests
staleness: among returning legislators, is the learned model's recall loss on members whose role
changed larger than the count heuristic's loss on the same positives (difference in differences,
cluster-bootstrap CI)? The member-committee roster is not in the data, so "committee shift" is a proxy:
cosine similarity < 0.5 between the committee-referral profiles of the member's sponsored bills in their
last training Congress and in the eval Congress (both with >= 5 referred bills); it is used only to
stratify the evaluation, never as a feature.

In [159]:
# D22-D25: truncated-SVD embeddings per S3 fold (val selection, refit, level gap), per-positive hits by legislator status.
_SVD_CACHE, _LU = {}, {}
_TR = lambda lo, hi: tuple(range(lo, hi))
_CH1 = {"House of Representatives": "H", "Senate": "S"}
_REG = MAJORITY.assign(ch=MAJORITY.chamber.map(_CH1)).pivot(index="congress", columns="ch", values="majority")
_REG = {int(c): f"{r.H}H/{r.S}S" for c, r in _REG.iterrows()}
_MAJ = MAJORITY.set_index(["congress", "chamber"]).majority


def _svd_factors(train, hl):
    """Rank-max(SVD_RANKS) factors (U*s, Vt) of log(1 + sponsor x cosponsor counts) of the training Congresses."""
    if (train, hl) not in _SVD_CACHE:
        _, hist = _counts(np.isin(_CD["cong"], train), hl)
        U, sv, Vt = np.linalg.svd(np.log1p(hist), full_matrices=False)
        r = max(SVD_RANKS)
        _SVD_CACHE[(train, hl)] = (U[:, :r] * sv[:r], Vt[:r])
    return _SVD_CACHE[(train, hl)]


def _regime_cw(train, ec, factor=2.0):
    """Per-Congress training weights: x factor for Congresses with the eval Congress's House/Senate majorities."""
    cw = np.ones(max(CONGRESSES) + 2)
    for c in train:
        if _REG.get(c) == _REG.get(ec):
            cw[c] = factor
    return cw


def _learn_unit(train, ec, svd_names=tuple(SVD_FAMILY), weighted=False, hit_names=()):
    """Task 1 per-bill metrics of the baselines + SVD members (+ regime / recency weighted sponsor_hist) fit on
    `train`, evaluated on Congress ec at t0=0; weighted=True also fits Task 2 eb_rate(_t0safe) under the weights."""
    key = (train, ec, tuple(svd_names), weighted, tuple(hit_names))
    if key in _LU:
        return _LU[key]
    tmask = np.isin(_CD["cong"], train)
    facs = {hl: _svd_factors(train, hl) for hl in {SVD_FAMILY[n][1] for n in svd_names}}
    cw = _regime_cw(train, ec)
    wc = ({"sh_recency_hl1": _counts(tmask, 1), "sh_regime_x2": _counts(tmask, None, cw),
           "sh_regime_x2_hl1": _counts(tmask, 1, cw)} if weighted else {})

    def extra(cand, spi, hs, P, **_):
        out = {}
        us, inv = np.unique(spi, return_inverse=True)
        for n in svd_names:
            r, hl = SVD_FAMILY[n]
            US, Vt = facs[hl]
            out[n] = (US[us, :r] @ Vt[:r][:, cand])[inv] * hs[:, None] + P * 1e-12   # popularity breaks exact ties
        for n, (_pop, hist) in wc.items():
            out[n] = hist[spi][:, cand] * hs[:, None] * 1e9 + P
        return out

    sink = {n: [] for n in hit_names} if hit_names else None
    u = {"cosponsor_t0": _t1_eval(tmask, _BL.index[_BL.congress == ec].to_numpy(), 0, extra=extra, hit_sink=sink), "hits": sink}
    if weighted:
        tr2, te2 = _ADV[_ADV.congress.isin(train)], _ADV[_ADV.congress == ec]
        rec, reg = 0.5 ** (max(train) - tr2.congress.to_numpy()), cw[tr2.congress.to_numpy()]
        sc = {}
        for n, w in (("unweighted", None), ("recency_hl1", rec), ("regime_x2", reg), ("regime_x2_hl1", rec * reg)):
            o = _t2_scores(tr2, te2, w=w)
            sc[f"eb_rate|{n}"], sc[f"eb_rate_t0safe|{n}"] = o["eb_rate"], o["eb_rate_t0safe"]
        u["advancement"] = (sc, te2)
    _LU[key] = u
    return u


_ts_l = time.time()
_SVDN = list(SVD_FAMILY)
_SVD_FIX = "svd_r32_hl1"                                # fixed GNN-sized embedding (rank 32) for the staleness sensitivity
_LV, _SVD_PICK, _lrows = {}, {}, []
for c in [FOLD_C[0] - 1] + FOLD_C:                      # T(c): fit on 108..c-1, evaluate c (= val unit of fold c+1)
    hn = ()
    if c in FOLD_C:
        vpb = _LV[c - 1]["cosponsor_t0"][0]
        _SVD_PICK[c] = pd.Series({n: vpb[n][:, 1].mean() for n in _SVDN}).idxmax()
        hn = tuple(dict.fromkeys((_SVD_PICK[c], _SVD_FIX, "sponsor_hist")))
    _LV[c] = _learn_unit(_TR(CONGRESSES[0], c), c, weighted=c in FOLD_C, hit_names=hn)
for c in FOLD_C:
    vpb, (tpb, tcodes) = _LV[c - 1]["cosponsor_t0"][0], _LV[c]["cosponsor_t0"]
    v = pd.Series({n: vpb[n][:, 1].mean() for n in _SVDN})
    t = pd.Series({n: tpb[n][:, 1].mean() for n in _SVDN})
    pick, best = v.idxmax(), t.idxmax()
    _, lo, _, hw = _pdiff(_LV[c], best, _LV[c], pick, "cosponsor_t0", "Recall@20", ("svd_regret", c)) if pick != best else (0, 0, 0, 0)
    dsh = _pdiff(_LV[c], pick, _LV[c], "sponsor_hist", "cosponsor_t0", "Recall@20", ("svd_vs_sh", c))
    lev = {role: next(r for r in _t1_summarize(pb, cd, ("svd_lvl", c, role), baselines=[pick]) if r["metric"] == "Recall@20")
           for role, (pb, cd) in (("val", _LV[c - 1]["cosponsor_t0"]), ("test", (tpb, tcodes)))}
    _lrows.append(dict(fold=f"c={c}", val_pick=pick, test_best=best, regret=t[best] - t[pick], regret_hw=hw,
                       regret_sig=pick != best and lo > 0, tau=stats.kendalltau(v.values, t.values)[0],
                       val=lev["val"]["value"], test=lev["test"]["value"],
                       hw_val=(lev["val"]["ci_high"] - lev["val"]["ci_low"]) / 2,
                       hw_test=(lev["test"]["ci_high"] - lev["test"]["ci_low"]) / 2,
                       svd_minus_sh=dsh[0], svd_minus_sh_lo=dsh[1], svd_minus_sh_hi=dsh[2]))
_LSEL = pd.DataFrame(_lrows).set_index("fold")
_cap = pd.DataFrame([dict(congress=c, member=n, rank=SVD_FAMILY[n][0], recency="hl1" if SVD_FAMILY[n][1] else "none",
                          val=_LV[c - 1]["cosponsor_t0"][0][n][:, 1].mean(), test=_LV[c]["cosponsor_t0"][0][n][:, 1].mean())
                     for c in FOLD_C for n in _SVDN])
print(f"SVD embeddings: {len(_SVD_CACHE)} factorizations, {time.time() - _ts_l:.0f}s. Per S3 fold (val = Congress c-1 with "
      "fit on 108..c-2; test = c with refit on 108..c-1), Recall@20 at t0=0:")
print(_LSEL[["val_pick", "test_best", "regret", "regret_hw", "regret_sig", "tau", "val", "test", "svd_minus_sh"]].round(4).to_string())
line_chart(_cap.groupby(["rank", "recency"])[["test"]].mean().reset_index().assign(series=lambda d: "test, recency " + d.recency),
           x="rank", y="test", series="series", yfmt=".3f", ylabel="mean test Recall@20", xlabel="SVD rank (capacity)",
           title="Learned embeddings: test Recall@20 by capacity (mean over S3 folds 112-118)",
           subtitle=f"sponsor_hist reference {_fv1.loc['sponsor_hist', 'mean']:.3f}; the val-selected member per fold is in the table",
           height=300, name="D_svd_capacity")

# D22 (level gap) and D23 (selection regret) for the learned model.
_gsvd = _LSEL.reset_index()[["fold", "val", "test", "hw_val", "hw_test"]].assign(baseline="svd_pick")
_gsvd = pd.concat([_gsvd.assign(split="S3")] + [_gsvd[_gsvd.fold == f"c={SPLIT_DEFS[s_]['test'][0]}"].assign(split=s_)
                                                 for s_ in ("S1a", "S1b")])
_gsvd = _gsvd.assign(gap=_gsvd.test - _gsvd.val, rel_gap=(_gsvd.test - _gsvd.val) / _gsvd.val,
                     hw_diff=np.sqrt(_gsvd.hw_val ** 2 + _gsvd.hw_test ** 2))
_gap_check("D22", "Task 1, learned SVD embeddings (val-selected): validation Congress predicts the test score level (S3)",
           _gsvd, "svd_pick", na=_NA_FAM, extra=f"Repeats D1 for a model with per-legislator parameters; val picks: "
                                    f"{', '.join(f'{k}:{v}' for k, v in _LSEL.val_pick.value_counts().items())}.")
_l_ok = bool(not ((_LSEL.regret.mean() > D_TOL["cosponsor_t0"]) and (_LSEL.regret_sig.sum() >= 2)))
record_check("D", "D23", "Task 1, learned SVD embeddings: validation picks the test-best rank / recency (regret vs paired noise, S3)",
             _l_ok, metric=float(_LSEL.regret.mean()), quantity="learned_model_selection", kind="validity",
             threshold="FAIL if mean regret > D_TOL (MDE/2 = 0.005) Recall@20 AND regret significant (paired cluster CI) in >= 2 of 7 folds (as D11)",
             by_split=_bs({"S3": _l_ok, **{s_: round(float(_LSEL.loc[f"c={SPLIT_DEFS[s_]['test'][0]}", "regret"]), 4) for s_ in ("S1a", "S1b")}}),
             na_reason=_NA_FAM,
             detail=f"Hit rate {(_LSEL.val_pick == _LSEL.test_best).mean():.0%}, max regret {_LSEL.regret.max():.4f}, significant in "
                    f"{int(_LSEL.regret_sig.sum())}/7 folds, mean tau(val, test) over the {len(_SVDN)} members {_LSEL.tau.mean():.2f}. Picks: "
                    f"{', '.join(f'{k}:{v}' for k, v in _LSEL.val_pick.value_counts().items())}. Val-picked SVD minus sponsor_hist on "
                    f"test: mean {_LSEL.svd_minus_sh.mean():+.4f}, significant in {int(((_LSEL.svd_minus_sh_lo > 0) | (_LSEL.svd_minus_sh_hi < 0)).sum())}/7 "
                    "folds. by_split S1a/S1b = regret on their single fold (metric). A learned model whose capacity is chosen on "
                    "the previous Congress keeps working on the next one if this passes.")

SVD embeddings: 16 factorizations, 8s. Per S3 fold (val = Congress c-1 with fit on 108..c-2; test = c with refit on 108..c-1), Recall@20 at t0=0:
           val_pick     test_best  regret  regret_hw  regret_sig    tau    val   test  svd_minus_sh
fold                                                                                               
c=112  svd_r256_hl1  svd_r256_hl1  0.0000     0.0000       False 0.9121 0.2860 0.2722        0.0123
c=113  svd_r256_hl1  svd_r512_hl1  0.0032     0.0062       False 0.7363 0.2722 0.2782        0.0205
c=114  svd_r512_hl1  svd_r256_hl1  0.0004     0.0053       False 0.9341 0.2814 0.2902        0.0255
c=115  svd_r256_hl1  svd_r512_hl1  0.0011     0.0059       False 0.8901 0.2906 0.2938        0.0217
c=116  svd_r512_hl1  svd_r512_hl1  0.0000     0.0000       False 0.9560 0.2948 0.2757        0.0215
c=117  svd_r512_hl1  svd_r512_hl1  0.0000     0.0000       False 0.9560 0.2757 0.2832        0.0217
c=118  svd_r512_hl1  svd_r512_hl1  0.0000     0.0000  

[PASS] D22: Task 1, learned SVD embeddings (val-selected): validation Congress predicts the test score level (S3) | metric=0.03721 (threshold FAIL if gaps exceed cluster-bootstrap noise: chi-square p < 0.05 (sum of z^2 over the split's folds))
[PASS] D23: Task 1, learned SVD embeddings: validation picks the test-best rank / recency (regret vs paired noise, S3) | metric=0.0006684 (threshold FAIL if mean regret > D_TOL (MDE/2 = 0.005) Recall@20 AND regret significant (paired cluster CI) in >= 2 of 7 folds (as D11))


In [160]:
# D24/D25: Recall@20 per positive by legislator status (new / returning with or without a role change), SVD vs sponsor_hist.
_first_c = pd.Series(_CD["cong"]).groupby(_CD["leg"]).min()                 # first Congress with a cosponsorship
_mem = members[members.congress.isin(CONGRESSES)].assign(leg=lambda d: _LEG.get_indexer(d.bioguide_id))
_mem = _mem.sort_values("congress")[["leg", "congress", "chamber", "party_code_asof", "caucus"]]
_bc = load("bill_committees", ("bill_id", "committee_code", "parent_committee_code"))
_bc = _bc[_bc.parent_committee_code.isna()].merge(_BL.reset_index()[["bill_id", "spon", "congress"]], on="bill_id")
_bc = _bc[_bc.spon >= 0].drop_duplicates(["bill_id", "committee_code"])
_cprof = _bc.groupby(["spon", "congress", "committee_code"]).size()        # committee-referral profile per sponsor-Congress
_nref = _bc.drop_duplicates("bill_id").groupby(["spon", "congress"]).size()
_COS_MIN, _NREF_MIN = 0.5, 5


def _cos_profile(pairs):
    """Cosine similarity of committee-referral profiles for rows (leg, c_prev, c_now); NaN if < _NREF_MIN bills."""
    out = np.full(len(pairs), np.nan)
    for i, (l, c0, c1) in enumerate(pairs):
        if _nref.get((l, c0), 0) < _NREF_MIN or _nref.get((l, c1), 0) < _NREF_MIN:
            continue
        a, b = _cprof.loc[(l, c0)], _cprof.loc[(l, c1)]
        a, b = a.align(b, fill_value=0)
        out[i] = float(a @ b / np.sqrt((a @ a) * (b @ b)))
    return out


_STRATA = ["new (no training history)", "returning: chamber or party switch", "returning: majority status flipped",
           "returning: committee shift (proxy)", "returning: unchanged"]
_H = []
for c in FOLD_C:
    h, pk = _LV[c]["hits"], _SVD_PICK[c]
    a, b, f = h[pk], h["sponsor_hist"], h[_SVD_FIX]
    assert np.array_equal(a[:, :2], b[:, :2]) and np.array_equal(f[:, :2], b[:, :2])
    d = pd.DataFrame({"code": a[:, 0], "leg": a[:, 1], "hit_svd": a[:, 2], "hit_sh": b[:, 2], "hit_r32": f[:, 2], "congress": c})
    d["chamber"] = _BL.chamber.reindex(d.code).to_numpy()
    prev = _mem[_mem.congress < c].groupby("leg").last()
    cur = _mem[_mem.congress == c].groupby("leg").last()
    pv, cu = prev.reindex(d.leg), cur.reindex(d.leg)
    new = ~(_first_c.reindex(d.leg).to_numpy() < c)
    sw = (pv.chamber.to_numpy() != d.chamber.to_numpy()) | (pv.party_code_asof.to_numpy() != cu.party_code_asof.to_numpy())
    in_maj_prev = pv.caucus.to_numpy() == _MAJ.reindex(pd.MultiIndex.from_arrays([pv.congress.fillna(0).astype(int), pv.chamber])).to_numpy()
    in_maj_now = cu.caucus.to_numpy() == _MAJ.reindex(pd.MultiIndex.from_arrays([np.full(len(d), c), d.chamber])).to_numpy()
    ul = d[["leg"]].assign(c0=pv.congress.fillna(0).astype(int).to_numpy()).drop_duplicates()
    ul["cos"] = _cos_profile(list(zip(ul.leg, ul.c0, np.full(len(ul), c))))
    cs_ = d[["leg"]].merge(ul.drop(columns="c0"), on="leg", how="left").cos.to_numpy()
    d["stratum"] = np.select([new, pv.chamber.isna().to_numpy() | sw, in_maj_prev != in_maj_now, cs_ < _COS_MIN],
                             _STRATA[:4], _STRATA[4])
    _H.append(d)
_H = pd.concat(_H, ignore_index=True)
_H["changed"] = _H.stratum.isin(_STRATA[1:4])
_ubill, _H["bpos"] = np.unique(_H.code.to_numpy(), return_inverse=True)
_Wh = _boot_weights(_BL.cluster.reindex(_ubill).to_numpy(), "D_stale")


def _rec(mask, col, W=_Wh, bpos=None):
    """Micro Recall@20 over positives in mask, per bootstrap row of W (cluster bootstrap by sponsor)."""
    bp = _H.bpos.to_numpy() if bpos is None else bpos
    nb = W.shape[1]
    num = W @ np.bincount(bp[mask], weights=_H[col].to_numpy()[mask].astype(float), minlength=nb)
    den = W @ np.bincount(bp[mask], minlength=nb).astype(float)
    return num / np.maximum(den, 1e-12)


def _ci(x):
    return x[0], *np.percentile(x[1:], [2.5, 97.5])


_strt = []
for st in _STRATA:
    m = (_H.stratum == st).to_numpy()
    rs, rh = _rec(m, "hit_svd"), _rec(m, "hit_sh")
    _strt.append(dict(stratum=st, n_pos=int(m.sum()), share=m.mean(), svd=rs[0], svd_r32=_rec(m, "hit_r32")[0], sponsor_hist=rh[0],
                      svd_minus_sh=_ci(rs - rh)[0], ci_lo=_ci(rs - rh)[1], ci_hi=_ci(rs - rh)[2]))
_strt = pd.DataFrame(_strt).set_index("stratum")
print(f"Recall@20 per positive by legislator status, S3 test folds {FOLD_C[0]}-{FOLD_C[-1]} pooled (svd = val-picked member per "
      "fold; CI = cluster bootstrap by sponsor):")
print(_strt.round(4).to_string())
bar_chart(_strt.reset_index().melt(id_vars="stratum", value_vars=["svd", "sponsor_hist"], var_name="model", value_name="recall"),
          x="stratum", y="recall", series="model", yfmt=".2f", ylabel="Recall@20 (per positive)",
          title="Recall@20 by legislator status: learned embeddings vs count history (S3 test folds pooled)",
          subtitle="new = no cosponsorship in the training Congresses (zero embedding); changes measured vs the member's last training Congress",
          height=320, name="D_staleness")


def _did(mask_ch, mask_un, W=_Wh, bpos=None, col="hit_svd"):
    """(unchanged - changed) recall gap of the svd model minus the same gap of sponsor_hist, per bootstrap row."""
    g_s = _rec(mask_un, col, W, bpos) - _rec(mask_ch, col, W, bpos)
    g_h = _rec(mask_un, "hit_sh", W, bpos) - _rec(mask_ch, "hit_sh", W, bpos)
    return g_s - g_h


_un = (_H.stratum == _STRATA[4]).to_numpy()
_did_all = _ci(_did(_H.changed.to_numpy(), _un))
_did_r32 = _ci(_did(_H.changed.to_numpy(), _un, col="hit_r32"))
_did_type = {st: _ci(_did((_H.stratum == st).to_numpy(), _un)) for st in _STRATA[1:4]}
_did_fold = []
for c in FOLD_C:
    fm = (_H.congress == c).to_numpy()
    ub, bp = np.unique(_H.code.to_numpy()[fm], return_inverse=True)
    bpos = np.zeros(len(_H), int)
    bpos[fm] = bp
    Wf = _boot_weights(_BL.cluster.reindex(ub).to_numpy(), ("D_stale", c))
    _did_fold.append((c, *_ci(_did(fm & _H.changed.to_numpy(), fm & _un, Wf, bpos)),
                      _ci(_did(fm & _H.changed.to_numpy(), fm & _un, Wf, bpos, "hit_r32"))[1]))
_did_fold = pd.DataFrame(_did_fold, columns=["congress", "did", "lo", "hi", "lo_r32"])
_n_sig_st, _n_sig_r32 = int((_did_fold.lo > 0).sum()), int((_did_fold.lo_r32 > 0).sum())
_new = _strt.loc[_STRATA[0]]
_ret = _H.stratum != _STRATA[0]
record_check("D", "D24", "Task 1: Recall@20 for new vs returning legislators, learned embeddings vs count history (S3 test, informational)",
             None, metric=float(_new.svd), threshold="informational (cold start for a transductive model; section B sizes it)",
             detail=f"New legislators: {_new.share:.1%} of test positives; Recall@20 svd {_new.svd:.3f} vs sponsor_hist "
                    f"{_new.sponsor_hist:.3f} (zero embedding and zero training popularity: they are ranked below every member "
                    "with history, so they are never in the top 20). Returning: svd "
                    f"{_rec(_ret.to_numpy(), 'hit_svd')[0]:.3f}, sponsor_hist {_rec(_ret.to_numpy(), 'hit_sh')[0]:.3f}. A transductive GNN "
                    "has no embedding for new members: give it inductive node features (party, state, chamber, committee) or score "
                    "new members separately; S3 / S1 test Congresses contain the realistic share of new members, a random split does not.")
_d25_ok = not (((_did_all[1] > 0) and (_n_sig_st >= 2)) or ((_did_r32[1] > 0) and (_n_sig_r32 >= 2)))
record_check("D", "D25", "Task 1: learned embeddings do not go stale for returning legislators whose role changed (DiD vs count history, S3)",
             bool(_d25_ok), metric=float(_did_all[0]), quantity="embedding_staleness", kind="drift",
             threshold="FAIL if, for the val-picked or the fixed rank-32 SVD, the pooled difference in differences (svd recall loss "
                       "on changed vs unchanged returning members, minus the same loss for sponsor_hist) has CI > 0 AND is "
                       "significant in >= 2 of 7 folds",
             by_split=_bs({"S3": bool(_d25_ok)}), na_reason="measured on the pooled S3 test folds 112-118 (S1a / S1b folds are part of that pool)",
             detail=f"DiD pooled {_did_all[0]:+.4f} [{_did_all[1]:+.4f}, {_did_all[2]:+.4f}]; by change type: " + "; ".join(
                 f"{k.split(': ')[1]} {v[0]:+.4f} [{v[1]:+.4f}, {v[2]:+.4f}] (n={int((_H.stratum == k).sum()):,})" for k, v in _did_type.items())
             + f"; significant in {_n_sig_st}/7 folds. Fixed rank-32 embedding ({_SVD_FIX}): DiD {_did_r32[0]:+.4f} "
               f"[{_did_r32[1]:+.4f}, {_did_r32[2]:+.4f}], significant in {_n_sig_r32}/7 folds. Positive = the learned model loses more on members whose role changed than "
               "the count heuristic does on the same positives. Committee shift is a proxy (referral profile of sponsored bills, "
               "eval Congress used for stratification only). If this fails, the GNN needs Congress-specific node features or "
               "embeddings refreshed with the val Congress (refit, D19), and only forward-in-time splits (S1/S3) can reveal it.")

Recall@20 per positive by legislator status, S3 test folds 112-118 pooled (svd = val-picked member per fold; CI = cluster bootstrap by sponsor):
                                    n_pos  share    svd  svd_r32  sponsor_hist  svd_minus_sh   ci_lo  ci_hi
stratum                                                                                                    
new (no training history)           30032 0.1574 0.0000   0.0000        0.0000        0.0000  0.0000 0.0001
returning: chamber or party switch   1440 0.0075 0.0681   0.0688        0.0694       -0.0014 -0.0152 0.0112
returning: majority status flipped  64778 0.3396 0.2444   0.2398        0.2276        0.0167  0.0141 0.0194
returning: committee shift (proxy)   3870 0.0203 0.1765   0.1840        0.1532        0.0233  0.0124 0.0330
returning: unchanged                90650 0.4752 0.2584   0.2529        0.2387        0.0197  0.0172 0.0223


[INFO] D24: Task 1: Recall@20 for new vs returning legislators, learned embeddings vs count history (S3 test, informational) | metric=3.33e-05 (threshold informational (cold start for a transductive model; section B sizes it))
[PASS] D25: Task 1: learned embeddings do not go stale for returning legislators whose role changed (DiD vs count history, S3) | metric=0.002947 (threshold FAIL if, for the val-picked or the fixed rank-32 SVD, the pooled difference in differences (svd recall loss on changed vs unchanged returning members, minus the same loss for sponsor_hist) has CI > 0 AND is significant in >= 2 of 7 folds)


### D7. How optimistic is the random bill split?

A random split mixes bills of the same Congress (and of future Congresses) across train and test,
so the baselines see the very legislators, coalitions and sponsor relationships they are tested on.
We compare the random split's test bills of each Congress with the temporal (S3) test score of
that same Congress, **on the same bills** (the S3 model scores the S5 test bills of Congress c too), so
the optimism has a paired cluster-bootstrap CI. Calibration: optimism counts when its mean exceeds the
section-wide minimum detectable effect `D_MDE` (0.01 Recall@20 / 0.01 AUROC, the smallest difference worth
acting on; the cell first compares it with the data) and it is significant in at least 2 of 7 Congresses.

**Why one fixed MDE.** Earlier rounds used the median validation margin between the two best near-tie
candidates as the MDE. That margin measures the *spacing of our candidate grid* (e.g. shrinkage 5 vs
20 pseudo-bills), not what matters in practice, and made almost every Task-2 pair "relevant". The cell
prints that margin next to the MDE: when the grid margin is far below the MDE, validation is choosing
among ties, which is harmless and is exactly what the regret tolerance (MDE / 2) allows. A positive gap confirms
the random split overstates performance and must only be used as a leaky upper-bound reference.

In [161]:
# Diagnostic: median validation margin (val pick minus runner-up) over S3 folds and near-tie families = candidate-grid
# spacing; compared with the fixed section-wide D_MDE (defined with the engine constants).
def _val_margins(df, metric):
    v = df[(df.split == "S3") & (df.eval_role == "val") & (df.metric == metric)]
    return v.groupby(["family", "fold"]).value.apply(lambda x: float(np.diff(np.sort(x.to_numpy())[-2:])[0]))


_mg1 = pd.concat([_val_margins(D_FAMILY_RESULTS[D_FAMILY_RESULTS.task == "cosponsor_t0"], "Recall@20"),
                  _cap.groupby("congress").val.apply(lambda x: float(np.diff(np.sort(x.to_numpy())[-2:])[0]))])
_mg2 = _val_margins(D_FAMILY_RESULTS[D_FAMILY_RESULTS.task == "advancement"], "AUROC")
D_VAL_MARGIN = {"cosponsor_t0": float(_mg1.median()), "advancement": float(_mg2.median())}
_hwpair = _sel(*_D_PRIMARY_T1, split="S3", eval_role="test").assign(hw=lambda d: (d.ci_high - d.ci_low) / 2).hw.median()
print(f"MDE used by every check: Task 1 {D_MDE['cosponsor_t0']} Recall@20 ({D_MDE['cosponsor_t0'] / _fv1.loc[_best1, 'mean']:.0%} of "
      f"'{_best1}' mean test level), Task 2 {D_MDE['advancement']} AUROC; regret tolerance MDE/2. Median val margin pick vs "
      f"runner-up (grid spacing, not practical importance): Task 1 {D_VAL_MARGIN['cosponsor_t0']:.4f} (IQR {_mg1.quantile(.25):.4f}-"
      f"{_mg1.quantile(.75):.4f}), Task 2 {D_VAL_MARGIN['advancement']:.4f} (IQR {_mg2.quantile(.25):.4f}-{_mg2.quantile(.75):.4f}) "
      "-> most family choices are ties below the MDE")

# D7: random-split optimism = S5 test score minus S3 (temporal) test score on the same bills of Congress c.
_opt = []
_s5key = _unit_keys(next(u for u in iter_eval_units("S5") if u["role"] == "test"))
for t0, task in ((0, "cosponsor_t0"), (7, "cosponsor_t7")):
    pb, codes = _unit(*_s5key)[task]
    cong = _BL.congress.reindex(codes).to_numpy()
    for c in FOLD_C:
        msk = cong == c
        if msk.sum() < 30:
            continue
        p3, c3 = _fold_unit("S3", c)[task]
        ix = pd.Index(c3).get_indexer(codes[msk])
        assert (ix >= 0).all()
        W = _boot_weights(_BL.cluster.reindex(codes[msk]).to_numpy(), ("D7", c, t0))
        for b in T1_BASELINES:
            r5, r3 = pb[b][msk, 1], p3[b][ix, 1]
            est = W @ (r5 - r3) / W.sum(1)
            lo, hi = np.percentile(est[1:], [2.5, 97.5])
            _opt.append(dict(task=task, baseline=b, congress=c, random=r5.mean(), temporal=r3.mean(), optimism=est[0],
                             lo=lo, hi=hi, n_random=int(msk.sum())))
_opt = pd.DataFrame(_opt)
print(f"Random (S5) minus temporal (S3) Recall@20 on the same S5 test bills, mean over Congresses {FOLD_C[0]}-{FOLD_C[-1]}:")
print(_opt[_opt.task == "cosponsor_t0"].groupby(["task", "baseline"]).agg(mean_random=("random", "mean"), mean_temporal=("temporal", "mean"),
                                             mean_optimism=("optimism", "mean"), n_sig=("lo", lambda x: int((x > 0).sum())))
      .round(3).to_string())
_s5t2 = _sel(*_D_PRIMARY_T2, split="S5", eval_role="test").set_index("baseline").value
_s3t2 = _sel(*_D_PRIMARY_T2, split="S3", eval_role="test").groupby("baseline").value.mean()
print(f"Task 2 AUROC eb_rate: random split test {_s5t2['eb_rate']:.3f} vs mean S3 test {_s3t2['eb_rate']:.3f}")
_oc = _opt[(_opt.task == "cosponsor_t0") & (_opt.baseline == _best1)]
line_chart(pd.concat([_oc.assign(split="temporal (S3)", score=_oc.temporal),
                      _oc.assign(split="random (S5)", score=_oc.random)]),
           x="congress", y="score", series="split", yfmt=".2f", ylabel="Recall@20", xlabel="test Congress",
           title=f"Random vs temporal test Recall@20 per Congress ({_best1}, t0=0)",
           subtitle="random split = same Congress's bills seen in training; temporal = only earlier Congresses",
           height=320, name="D_random_optimism")
_d7 = float(_oc.optimism.mean())
_d7_ok = not ((_d7 > D_MDE["cosponsor_t0"]) and ((_oc.lo > 0).sum() >= 2))
record_check("D", "D7", "Random bill split does not inflate scores relative to the temporal split (paired optimism on the same bills, Task 1)",
             bool(_d7_ok), metric=_d7, threshold=f"FAIL if mean optimism > D_MDE ({D_MDE['cosponsor_t0']:.4f} Recall@20) AND "
             "significant (paired cluster CI) in >= 2 of 7 Congresses", by_split=_bs({"S5": bool(_d7_ok)}), kind="validity",
             na_reason="D7 judges only the random split S5; the temporal S3 test score of the same bills is the reference",
             detail=f"Mean S5-minus-S3 Recall@20 for '{_best1}' over Congresses {FOLD_C[0]}-{FOLD_C[-1]}: {_d7:+.3f} "
                    f"({_d7 / _oc.temporal.mean():+.0%} relative, {_d7 / D_MDE['cosponsor_t0']:.0f}x the MDE); significant in "
                    f"{int((_oc.lo > 0).sum())}/7 Congresses. Task 2 eb_rate AUROC random "
                    f"{_s5t2.get('eb_rate', np.nan):.3f} vs temporal {_s3t2.get('eb_rate', np.nan):.3f}. FAIL means the random split "
                    "leaks same-Congress information and overstates performance: use it only as a leaky upper bound. Part of the "
                    "gap is legitimate same-Congress history (see D13: the streaming protocol recovers it without leakage).")

MDE used by every check: Task 1 0.01 Recall@20 (4% of 'sponsor_hist' mean test level), Task 2 0.01 AUROC; regret tolerance MDE/2. Median val margin pick vs runner-up (grid spacing, not practical importance): Task 1 0.0021 (IQR 0.0008-0.0051), Task 2 0.0005 (IQR 0.0002-0.0019) -> most family choices are ties below the MDE
Random (S5) minus temporal (S3) Recall@20 on the same S5 test bills, mean over Congresses 112-118:
                           mean_random  mean_temporal  mean_optimism  n_sig
task         baseline                                                      
cosponsor_t0 party+pop          0.2240         0.2030         0.0210      5
             popularity         0.1680         0.1510         0.0160      4
             rw2hop             0.3320         0.2620         0.0700      7
             sponsor_hist       0.3340         0.2590         0.0750      7
Task 2 AUROC eb_rate: random split test 0.750 vs mean S3 test 0.724


[FAIL] D7: Random bill split does not inflate scores relative to the temporal split (paired optimism on the same bills, Task 1) | metric=0.07453 (threshold FAIL if mean optimism > D_MDE (0.0100 Recall@20) AND significant (paired cluster CI) in >= 2 of 7 Congresses) [fails for: S5]


### D8, D13-D14. Same-Congress history: within-Congress split (S4) and the streaming protocol

A temporal GNN deployed during a Congress can see that Congress's cosponsorships up to the
prediction time. S4 tests this on the last 20% of each Congress's bills. The **streaming** variants
(`S1a+stream`, `S1b+stream`, `S3+stream`) keep the by-Congress pools but let the baselines also
count every eval-Congress cosponsorship dated on or before *introduced_date + t0* of the bill being
scored (Task 2: eval-Congress bills whose 180-day label had resolved by the scored bill's
introduction). This is leakage-free by construction and is how a temporal model would be used. If
streaming adds a lot, the by-Congress split should be evaluated in streaming mode (and the cold-start
problem from section B is largely an artefact of forbidding same-Congress history).

In [162]:
# D8: S4 vs S3 on the same Congress; D13/D14: gain and val-test gaps of the streaming by-Congress protocol.
_w = []
for c in SPLIT_DEFS["S4"]["folds"]:
    for task, metric, b in ((*_D_PRIMARY_T1, _best1), ("cosponsor_t7", "Recall@20", _best1), (*_D_PRIMARY_T2, _best2)):
        a = _sel(task, metric, b, split="S4", fold=f"c={c}", eval_role="test")
        z = _sel(task, metric, b, split="S3", eval_role="test", eval_congress=c)
        if len(a) and len(z):
            _w.append(dict(congress=c, task=task, baseline=b, S4_test=float(a.value.iloc[0]),
                           S3_test=float(z.value.iloc[0]), n_S4=int(a.n_eval.iloc[0]), n_S3=int(z.n_eval.iloc[0])))
_w = pd.DataFrame(_w)
_w["diff"] = _w.S4_test - _w.S3_test
print("S4 (last 20% of bills, same-Congress history) minus S3 test score on the same Congress, best baselines:")
print(_w.pivot_table(index="congress", columns="task", values=["S4_test", "S3_test", "diff"]).round(3).to_string())
_w1 = _w[_w.task == _D_PRIMARY_T1[0]]
_w2 = _w[_w.task == _D_PRIMARY_T2[0]]
record_check("D", "D8", "Within-Congress split (S4) gives scores comparable to by-Congress split (mean S4-S3 test diff, Task 1)",
             None, metric=float(_w1["diff"].mean()), threshold="informational", by_split=_bs({"S4": round(float(_w1["diff"].mean()), 4)}),
             na_reason="S4-only comparison (S4 test score minus the S3 test score of the same Congress)",
             detail=f"Task 1 '{_best1}' S4-S3 Recall@20 diff by Congress: " + ", ".join(f"{r.congress}: {r.diff:+.3f}" for r in _w1.itertuples())
                    + f". Task 2 S4 test sizes {_w2.n_S4.tolist()} vs S3 {_w2.n_S3.tolist()} uncensored bills "
                    f"(diff {_w2['diff'].round(3).tolist()}). S4 mixes two effects (same-Congress history and a late-bill population); "
                    "the streaming variant (D13) isolates the history effect on the full Congress.")

_st = []
for task, metric, b in ((*_D_PRIMARY_T1, _best1), ("cosponsor_t7", "Recall@20", _best1), (*_D_PRIMARY_T1, "popularity"),
                        (*_D_PRIMARY_T1, "rw2hop"), (*_D_PRIMARY_T2, _best2)):
    for role in ("val", "test"):
        p = _sel(task, metric, b, split="S3", eval_role=role).set_index("fold")
        s = _sel(task, metric, b, split="S3+stream", eval_role=role).set_index("fold")
        for f in p.index:
            _st.append(dict(series=f"{task} {metric} ({b})", fold=f, role=role, congress=int(p.eval_congress[f]),
                            plain=p.value[f], stream=s.value[f], gain=s.value[f] - p.value[f]))
_st = pd.DataFrame(_st)
print("\nStreaming (S3+stream) vs plain S3, test folds:")
print(_st[_st.role == "test"].groupby("series").agg(mean_plain=("plain", "mean"), mean_stream=("stream", "mean"),
                                                    mean_gain=("gain", "mean"), min_gain=("gain", "min"),
                                                    max_gain=("gain", "max")).round(4).to_string())
_gs1 = _gaps(*_D_PRIMARY_T1)
_gs1 = _gs1[(_gs1.split == "S3+stream") & (_gs1.baseline == _best1)]
_gs2 = _g2[(_g2.split == "S3+stream") & (_g2.baseline == _best2)]
print("Val-test gaps: " + "; ".join(f"{n}: mean |rel gap| {g.rel_gap.abs().mean():.3f}, signif {g.signif.mean():.0%}"
                                     for n, g in (("T1 S3", _s3g1), ("T1 S3+stream", _gs1), ("T2 S3", _s3g2), ("T2 S3+stream", _gs2))))
_stc = _st[(_st.role == "test") & (_st.series == f"{_D_PRIMARY_T1[0]} {_D_PRIMARY_T1[1]} ({_best1})")]
line_chart(pd.concat([_stc.assign(protocol="S3 (train Congresses only)", score=_stc.plain),
                      _stc.assign(protocol="S3+stream (+ eval-Congress history to t0)", score=_stc.stream)]),
           x="congress", y="score", series="protocol", yfmt=".2f", ylabel="Recall@20", xlabel="test Congress",
           title=f"Streaming same-Congress history: test Recall@20 per Congress ({_best1}, t0=0)",
           subtitle="streaming counts eval-Congress cosponsorships dated on or before each bill's prediction time",
           height=320, name="D_stream_gain")
_sg1 = _stc.gain
_sg2 = _st[(_st.role == "test") & (_st.series == f"{_D_PRIMARY_T2[0]} {_D_PRIMARY_T2[1]} ({_best2})")].gain
_sg1b = _st[(_st.role == "test") & (_st.series == f"cosponsor_t7 Recall@20 ({_best1})")].gain
_sgr = _st[(_st.role == "test") & (_st.series == f"{_D_PRIMARY_T1[0]} {_D_PRIMARY_T1[1]} (rw2hop)")].gain
_sgS1 = {v: float(_sel(*_D_PRIMARY_T1, _best1, split=v, eval_role="test").value.iloc[0]
                  - _sel(*_D_PRIMARY_T1, _best1, split=v[:-7], eval_role="test").value.iloc[0]) for v in ("S1a+stream", "S1b+stream")}
record_check("D", "D13", "Streaming eval-Congress history changes test scores (mean gain S3+stream - S3, Task 1 Recall@20)",
             None, metric=float(_sg1.mean()), threshold="informational",
             by_split=_bs({"S3+stream": round(float(_sg1.mean()), 4), **{k: round(v, 4) for k, v in _sgS1.items()}}),
             na_reason="gain is defined only for the streaming variants (vs their plain split)",
             detail=f"Task 1 '{_best1}' gain t0=0 {_sg1.mean():+.3f} (range {_sg1.min():+.3f} to {_sg1.max():+.3f}), t0=7 "
                    f"{_sg1b.mean():+.3f}; rw2hop gain {_sgr.mean():+.3f} (monthly graph snapshots, so slightly conservative); "
                    f"Task 2 '{_best2}' AUROC gain {_sg2.mean():+.3f}. A positive Task 1 gain means the model must "
                    "be evaluated with within-Congress history up to t0 (streaming), otherwise the by-Congress split understates "
                    "a temporal GNN and overstates cold start. The baselines simply add counts; a GNN can weight recent edges more.")
_gap_check("D14", "Task 1: under the streaming protocol the val Congress still predicts the test score (gaps within noise, S3+stream)",
           _g1, _best1, splits=("S1a+stream", "S1b+stream", "S3+stream"), na="plain splits are judged in D1",
           extra=f"Plain S3 mean |rel gap| {_d1:.3f}. Task 2 S3+stream mean |gap| {_gs2.gap.abs().mean():.3f} AUROC (plain "
                 f"{_d2:.3f}). PASS means the streaming protocol is as usable for model selection as the plain by-Congress split.")

S4 (last 20% of bills, same-Congress history) minus S3 test score on the same Congress, best baselines:
             S3_test                               S4_test                                  diff                          
task     advancement cosponsor_t0 cosponsor_t7 advancement cosponsor_t0 cosponsor_t7 advancement cosponsor_t0 cosponsor_t7
congress                                                                                                                  
116           0.6880       0.2540       0.2360      0.8100       0.2960       0.2890      0.1220       0.0420       0.0530
117           0.6920       0.2620       0.2350      0.6860       0.3160       0.2790     -0.0070       0.0550       0.0440
118           0.6970       0.2440       0.2230      0.7320       0.3190       0.2870      0.0350       0.0750       0.0630
[INFO] D8: Within-Congress split (S4) gives scores comparable to by-Congress split (mean S4-S3 test diff, Task 1) | metric=0.05721 (threshold informational)



[INFO] D13: Streaming eval-Congress history changes test scores (mean gain S3+stream - S3, Task 1 Recall@20) | metric=0.03697 (threshold informational)
[PASS] D14: Task 1: under the streaming protocol the val Congress still predicts the test score (gaps within noise, S3+stream) | metric=0.02937 (threshold FAIL if gaps exceed cluster-bootstrap noise: chi-square p < 0.05 (sum of z^2 over the split's folds))


,folds,chi2,mean_abs_rel,noise_rel,p,ok
split,,,,,,
S1a+stream,1,2.1646,0.0646,0.0350,0.1412,True
S1b+stream,1,0.6562,0.0326,0.0321,0.4179,True
S3+stream,7,4.3074,0.0294,0.0339,0.7438,True


### D9, D15. Are the test sets large enough for what the split is used for?

A split is used to **compare** models and protocols on the same test bills, so the relevant noise is
that of the *paired difference*: Congress-level difficulty shifts move both models together and
cancel. For each decision-relevant pair we compute, on every rolling test Congress, the paired
difference and its cluster-bootstrap CI. Two different questions follow:
(a) **single-Congress risk**: how often would ONE test Congress (S1a / S1b) fail to confirm the
direction of a real difference? `p_miss_1C` = share of the 7 test Congresses whose paired CI does not
exclude 0 in the direction of the mean (wrong sign or not significant); `p_flip_1C` = share with the
opposite sign. More test Congresses do not reduce the Congress-to-Congress heterogeneity of an effect,
so the deployment Congress can still differ: `pi_next` = 95% prediction half-width of the effect in a
new Congress (t, 6 df). (b) **average effect**: the CI half-width of the mean effect over k = 1, 2, 7
test Congresses, 1.96 * max(sd_between, se_1C) / sqrt(k).
A pair is *relevant* if |mean difference| >= `D_MDE` (0.01, the same MDE as every other check here);
smaller differences are not worth acting on. The `by_split` verdict: a split holds if, for every relevant
model / near-tie pair, its miss risk is <= 20% (80% power): empirical `p_miss_1C` for one test Congress,
the normal approximation Phi(|mean| / (sd / sqrt(k)) - 1.96) for k = 2 (S1pool) and k = 7 (S2 / S3).
**Caveat: these are different quantities.** The one-Congress risk includes Congress-to-Congress
heterogeneity of the effect; the k-Congress risk is about the *average* effect, so multi-fold splits
pass more easily by construction. The deployment-relevant risk, `pi_next` relative to the MDE (how far
the effect can move in the next Congress), does not shrink with k; D9b records it for every by-Congress
split, also from the recent folds 115-118 only (the early folds 112-115 are least like 119). Task 2
pairs are judged on the **t0-safe** family; policy-area pairs (kind `ref@d30`) are shown for reference.

In [163]:
# D9: paired-difference sizing per pair; D15 (info): absolute-level spread and pooled 117+118 test sets.
_S3u = lambda c: _fold_unit("S3", c)
_LVu = lambda c: _LV[c]
_PAIRS = [  # (kind, task, metric, label, unit A, a, unit B, b); a / b may be callables c -> member name
    ("model", *_D_PRIMARY_T1, "rw2hop - sponsor_hist", _S3u, "rw2hop", _S3u, "sponsor_hist"),
    ("model", *_D_PRIMARY_T1, "sponsor_hist - party+pop", _S3u, "sponsor_hist", _S3u, "party+pop"),
    ("model", *_D_PRIMARY_T1, "svd(val pick) - sponsor_hist", _LVu, _SVD_PICK.get, _LVu, "sponsor_hist"),
    ("near-tie", *_D_PRIMARY_T1, "sh_hl1 - sh_hl2", _S3u, "sh_hl1", _S3u, "sh_hl2"),
    ("near-tie", *_D_PRIMARY_T1, "sh_hl1 - sh_none+caucus", _S3u, "sh_hl1", _S3u, "sh_none+caucus"),
    ("near-tie", *_D_PRIMARY_T1, "rw_b0.1 - rw_b0.3", _S3u, "rw_b0.1", _S3u, "rw_b0.3"),
    ("near-tie", *_D_PRIMARY_T1, "svd_r32 - svd_r128", _LVu, "svd_r32", _LVu, "svd_r128"),
    ("protocol", *_D_PRIMARY_T1, "expanding - sliding (sponsor_hist)", _S3u, "sponsor_hist",
     lambda c: _fold_unit("S2", c), "sponsor_hist"),
    ("protocol", *_D_PRIMARY_T1, "stream - plain (sponsor_hist)", lambda c: _fold_unit("S3", c, stream=True),
     "sponsor_hist", _S3u, "sponsor_hist"),
    ("model", *_D_PRIMARY_T2, "eb_rate_t0safe - chamber_maj", _S3u, "eb_rate_t0safe", _S3u, "chamber_maj"),
    ("near-tie", *_D_PRIMARY_T2, "eb_m5_t0s - eb_m20_t0s", _S3u, "eb_m5_t0s", _S3u, "eb_m20_t0s"),
    ("near-tie", *_D_PRIMARY_T2, "eb_m20_t0s - eb_m80_t0s", _S3u, "eb_m20_t0s", _S3u, "eb_m80_t0s"),
    ("near-tie", *_D_PRIMARY_T2, "eb_m20_t0s - eb_nomaj_m20_t0s", _S3u, "eb_m20_t0s", _S3u, "eb_nomaj_m20_t0s"),
    ("ref@d30", *_D_PRIMARY_T2, "eb_rate - chamber_maj", _S3u, "eb_rate", _S3u, "chamber_maj"),
    ("ref@d30", *_D_PRIMARY_T2, "eb_m20 - eb_nomaj_m20", _S3u, "eb_m20", _S3u, "eb_nomaj_m20"),
    ("protocol", *_D_PRIMARY_T2, "expanding - sliding (eb_rate_t0safe)", _S3u, "eb_rate_t0safe",
     lambda c: _fold_unit("S2", c), "eb_rate_t0safe"),
]
_nm = lambda x, c: x if isinstance(x, str) else x(c)
_K_SPLIT = {s_: _n_test_c(s_) for s_ in ALL_SPLITS if _n_test_c(s_) is not None}
_tq = stats.t.ppf(0.975, len(FOLD_C) - 1) * np.sqrt(1 + 1 / len(FOLD_C))
_tq6 = stats.t.ppf(0.975, len(_DEV) - 1) * np.sqrt(1 + 1 / len(_DEV))          # development folds only (D27)
_RECENT = [c for c in FOLD_C if c >= _LOCK - 3]                                  # 115-118: closest to the deployment Congress
_tqr = stats.t.ppf(0.975, len(_RECENT) - 1) * np.sqrt(1 + 1 / len(_RECENT))
_prs, _pd_rows = [], []
for kind, task, metric, lab, uA, a, uB, b in _PAIRS:
    d = pd.DataFrame([dict(zip(("diff", "lo", "hi", "hw"), _pdiff(uA(c), _nm(a, c), uB(c), _nm(b, c), task, metric,
                                                                  ("pair", lab, c))), congress=c) for c in FOLD_C])
    _pd_rows.append(d.assign(pair=lab))
    mu, sd, hw1 = d["diff"].mean(), d["diff"].std(ddof=1), d.hw.mean()
    sd_eff = max(sd, hw1 / 1.96)
    sgn = np.sign(mu)
    n_dir = int(((d.lo > 0) if sgn > 0 else (d.hi < 0)).sum())
    row = dict(kind=kind, task="T1" if task.startswith("cos") else "T2", pair=lab, mean_diff=mu, sd_between=sd, hw_1C=hw1,
               n_sig_dir=n_dir, p_miss_1C=1 - n_dir / len(FOLD_C), p_flip_1C=float((np.sign(d["diff"]) == -sgn).mean()),
               pi_next=_tq * sd, relevant=bool(abs(mu) >= D_MDE[task]))
    for k in (1, 2, 7):
        row[f"hw_mean_k{k}"] = 1.96 * sd_eff / np.sqrt(k)
        row[f"miss_k{k}"] = float(1 - stats.norm.cdf(abs(mu) / (sd_eff / np.sqrt(k)) - 1.96))
    row["pi_rel"] = row["pi_next"] / D_MDE[task]                                  # deployment risk in MDE units
    row["pi_rel_recent"] = _tqr * d[d.congress.isin(_RECENT)]["diff"].std(ddof=1) / D_MDE[task]
    dv, lk = d[d.congress.isin(_DEV)], d[d.congress == _LOCK].iloc[0]           # D27: development folds vs locked fold
    mu6, sd6 = dv["diff"].mean(), dv["diff"].std(ddof=1)
    sd6e, k6 = max(sd6, dv.hw.mean() / 1.96), len(_DEV)
    row.update(mean_dev=mu6, relevant_dev=bool(abs(mu6) >= D_MDE[task]), hw_mean_k6=1.96 * sd6e / np.sqrt(k6),
               miss_k6=float(1 - stats.norm.cdf(abs(mu6) / (sd6e / np.sqrt(k6)) - 1.96)), pi_dev=_tq6 * sd6e,
               lock_diff=float(lk["diff"]), lock_lo=float(lk.lo), lock_hi=float(lk.hi),
               lock_in_pi=bool(abs(lk["diff"] - mu6) <= _tq6 * sd6e))
    _prs.append(row)
_prs = pd.DataFrame(_prs)
print(f"Paired differences on S3 test Congresses {FOLD_C[0]}-{FOLD_C[-1]} (MDE: T1 {D_MDE['cosponsor_t0']:.4f}, T2 "
      f"{D_MDE['advancement']:.4f}; p_miss_1C = share of single test Congresses not confirming the direction):")
print(_prs[["kind", "task", "pair", "mean_diff", "sd_between", "hw_1C", "relevant", "p_miss_1C", "p_flip_1C", "pi_next",
            "pi_rel", "hw_mean_k2", "hw_mean_k7", "miss_k2", "miss_k7"]].round(4).to_string(index=False))
_dec_pairs = _prs[_prs.kind.isin(["model", "near-tie"]) & _prs.relevant]


def _split_risk(k):
    """Max miss risk over relevant decision pairs for a split with k test Congresses ((a): empirical for k = 1)."""
    if not len(_dec_pairs):
        return 0.0, "none"
    r = _dec_pairs.p_miss_1C if k == 1 else _dec_pairs[f"miss_k{min(k, 7) if k in (1, 2, 7) else 7}"]
    return float(r.max()), _dec_pairs.pair.iloc[int(np.argmax(r.to_numpy()))]


_risk = {s_: _split_risk(1 if s_ in ("S1a", "S1b", "S1a+stream", "S1b+stream") else (2 if s_ == "S1pool" else 7))
         for s_ in _K_SPLIT}
_d9_split = {s_: bool(v[0] <= 0.2) for s_, v in _risk.items()}
_PREC_JS = r"""
const {width, height, title, subtitle, mde} = opts;
const m = {top: 48, right: 24, bottom: 56, left: 230};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title, subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const rows = Array.from(new Set(data.map(d => d.pair)));
const keys = Array.from(new Set(data.map(d => d.measure)));
const color = d3.scaleOrdinal(keys, viz.palette);
const y0 = d3.scaleBand(rows, [0, H]).padding(0.25), y1 = d3.scaleBand(keys, [0, y0.bandwidth()]).padding(0.1);
const xs = d3.scaleLinear([0, Math.max(mde * 1.2, d3.max(data, d => d.value))], [0, W]).nice();
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(6).tickSize(-H).tickFormat(d3.format('.2f')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').call(d3.axisLeft(y0).tickSize(0)).call(viz.axis).call(a => a.select('.domain').remove());
g.append('line').attr('x1', xs(mde)).attr('x2', xs(mde)).attr('y1', 0).attr('y2', H).attr('stroke', viz.ink)
  .attr('stroke-width', 1.5).attr('stroke-dasharray', '4,3');
g.append('text').attr('x', xs(mde) + 4).attr('y', -6).attr('fill', viz.ink2).style('font', '11px sans-serif').text(opts.mlabel);
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('rect').attr('x', 0).attr('y', d => y0(d.pair) + y1(d.measure))
  .attr('height', y1.bandwidth()).attr('width', d => Math.max(xs(d.value), 1)).attr('rx', 2).attr('fill', d => color(d.measure))
  .on('mousemove', (e, d) => tip.show(`<b>${d.pair}</b><br>${d.measure}: ${d3.format('.2f')(d.value)}<br>mean paired difference: ${d3.format('+.4f')(d.mean_diff)}`, e))
  .on('mouseleave', () => tip.hide());
viz.legend(svg, keys.map(k => [k, color(k)]), m.left, height - 10);
"""
_szc = pd.concat([_dec_pairs.assign(measure="one test Congress (empirical, S1a / S1b)", value=_dec_pairs.p_miss_1C),
                  _dec_pairs.assign(measure="mean of 2 test Congresses (normal approx.)", value=_dec_pairs.miss_k2),
                  _dec_pairs.assign(measure="mean of 7 rolling test Congresses (normal approx.)", value=_dec_pairs.miss_k7)])
_szc = _szc.assign(pair=_szc.task + " " + _szc.pair)[["pair", "measure", "value", "mean_diff"]]
show_d3(_PREC_JS, _szc, height=110 + 42 * max(len(_dec_pairs), 1), width=760,
        opts=dict(title="Risk that the test set fails to confirm a real model difference (relevant pairs, S3 test Congresses)",
                  subtitle="share of evaluations whose paired CI misses the true direction; dashed line = 20% (80% power)",
                  mde=0.2, mlabel="20% miss risk"), name="D_paired_precision")
_dp_all = _prs[_prs.kind.isin(["model", "near-tie"])]
record_check("D", "D9", "Test-set size: a single test Congress confirms the direction of every relevant model / near-tie difference (paired, S3)",
             _d9_split.get("S1b"), metric=round(_risk["S1b"][0], 3),
             threshold="per split: max over relevant pairs (|mean diff| >= D_MDE) of the miss risk <= 0.20 (empirical share of "
                       "single test Congresses for 1-Congress splits; normal approximation for k = 2 / 7)",
             by_split=_bs(_d9_split), kind="validity", t0_task2=0,
             na_reason="S4 / S5 test sets are within-Congress samples, not whole Congresses: sized in the D15 table",
             detail=f"Relevant decision pairs: {len(_dec_pairs)} of {len(_dp_all)} (MDE T1 {D_MDE['cosponsor_t0']:.4f}, T2 "
                    f"{D_MDE['advancement']:.4f}). Miss risk by split: " + ", ".join(f"{k} {v[0]:.2f} ({v[1]})" for k, v in _risk.items())
                    + ". Worst single-Congress pairs: " + "; ".join(f"{r.task} {r.pair}: mean {r.mean_diff:+.4f}, p_miss_1C "
                    f"{r.p_miss_1C:.2f}, p_flip {r.p_flip_1C:.2f}, next-Congress PI +/-{r.pi_next:.4f}"
                    for r in _dec_pairs.sort_values("p_miss_1C").tail(3).itertuples())
                    + f". Average-effect CI half-width (median over relevant pairs) k=1 {_dec_pairs.hw_mean_k1.median():.4f}, k=2 "
                    f"{_dec_pairs.hw_mean_k2.median():.4f}, k=7 {_dec_pairs.hw_mean_k7.median():.4f}. More test Congresses narrow the "
                    "CI of the average effect but not the next-Congress prediction interval (Congress x model interaction; D9b: "
                    f"up to {_dec_pairs.pi_rel.max():.1f}x the MDE). The k = 1 and k = 7 risks are different quantities (single-Congress "
                    "direction vs average effect), so S2 / S3 pass partly by construction. Task 2 "
                    "pairs use the t0-safe family (prediction at introduction); policy-area pairs ('ref@d30') are in the table only.")
_pr_dec = _dec_pairs.set_index(_dec_pairs.task + " " + _dec_pairs.pair)
_pimax, _pimax_r = float(_pr_dec.pi_rel.max()), float(_pr_dec.pi_rel_recent.max())
record_check("D", "D9b", "Deployment risk: how far a relevant model difference can move in the next Congress (95% PI half-width / MDE; "
             "informational)", None, metric=round(_pimax, 2), kind="drift", t0_task2=0,
             threshold="informational: > 1 means the next Congress can reverse or erase an MDE-sized advantage seen on test",
             by_split=_bs({s_: round(_pimax, 2) for s_ in _K_SPLIT}),
             na_reason="S4 / S5 do not evaluate whole future Congresses",
             detail="pi_next / MDE per relevant pair (all 7 folds; recent folds 115-118 in brackets): " + "; ".join(
                 f"{k} {r.pi_rel:.1f} [{r.pi_rel_recent:.1f}]" for k, r in _pr_dec.iterrows())
             + f". Same value for every by-Congress split: the prediction interval for a NEW Congress does not shrink with the "
               f"number of test Congresses (it is Congress x model heterogeneity, t with {len(FOLD_C) - 1} df). The 7 S3 test "
               "Congresses include 112-115, which differ most from 119 (sections A / C); the recent-fold value uses only "
               f"{_RECENT[0]}-{_RECENT[-1]} (t with {len(_RECENT) - 1} df, hence wide). Read with D9: a split can pin down the average "
               "difference and still leave the 119 outcome uncertain.")

# Absolute level (informational): CI half-widths of the score itself, between-fold std, pooled 117+118 tests.
_hw = SPLIT_RESULTS[(SPLIT_RESULTS.eval_role == "test") & SPLIT_RESULTS.split.isin(["S1b", "S3", "S1pool", "S4"])
                    & (((SPLIT_RESULTS.task == _D_PRIMARY_T1[0]) & (SPLIT_RESULTS.metric == _D_PRIMARY_T1[1]) & (SPLIT_RESULTS.baseline == _best1))
                       | ((SPLIT_RESULTS.task == _D_PRIMARY_T2[0]) & (SPLIT_RESULTS.metric == "AUROC")
                          & (SPLIT_RESULTS.baseline == _best2)))].copy()
_hw["hw_cluster"] = (_hw.ci_high - _hw.ci_low) / 2
_hw["hw_bill"] = (_hw.ci_high_bill - _hw.ci_low_bill) / 2
_hwt = _hw.groupby(["task", "split"]).agg(n_eval=("n_eval", "mean"), n_clusters=("n_clusters", "mean"),
                                          value=("value", "mean"), hw_bill=("hw_bill", "mean"), hw_cluster=("hw_cluster", "mean"))
_hwt["cluster/bill"] = _hwt.hw_cluster / _hwt.hw_bill
print(f"\nAbsolute-score CI half-widths ('{_best1}' Recall@20, '{_best2}' AUROC; bill vs sponsor-cluster bootstrap; S3/S4 = mean over folds):")
print(_hwt.round(4).to_string())
_bstd1, _bstd2 = float(_fv1.loc[_best1, "std"]), float(_fv2.loc[_best2, "std"])
_k_abs = {k: int(np.ceil((1.96 * sd / thr) ** 2)) for k, sd, thr in (("T1", _bstd1, 2 * D_MDE["cosponsor_t0"]),
                                                                       ("T2", _bstd2, 2 * D_MDE["advancement"]))}
_sp2 = _hwt.loc[("advancement", "S1pool")]
record_check("D", "D15", "Absolute score level: test Congresses needed for a precise reported score (informational)",
             None, metric=f"T1 {_k_abs['T1']}, T2 {_k_abs['T2']} Congresses",
             threshold="informational: report test scores per Congress plus the mean over rolling folds",
             detail=f"Between-fold std of the absolute score (S3): T1 {_bstd1:.4f}, T2 {_bstd2:.4f}; Congresses for 1.96*std/sqrt(k) "
                    f"<= 2 x MDE ({2 * D_MDE['cosponsor_t0']:.2f} R@20 / {2 * D_MDE['advancement']:.2f} AUROC; a reported level is coarser than a paired comparison): T1 {_k_abs['T1']}, T2 {_k_abs['T2']}. Single-Congress cluster hw: T1 "
                    f"{_hwt.loc[(_D_PRIMARY_T1[0], 'S1b'), 'hw_cluster']:.3f}, T2 {_hwt.loc[('advancement', 'S1b'), 'hw_cluster']:.3f} "
                    f"(cluster/bill {_hwt.loc[('advancement', 'S1b'), 'cluster/bill']:.1f}x); pooled 117+118 (S1pool) T2 hw "
                    f"{_sp2.hw_cluster:.3f}. This is about what score to *expect* on a new Congress, not about choosing between "
                    "models (D9): report per-Congress scores and their spread.")

Paired differences on S3 test Congresses 112-118 (MDE: T1 0.0100, T2 0.0100; p_miss_1C = share of single test Congresses not confirming the direction):
    kind task                                 pair  mean_diff  sd_between  hw_1C  relevant  p_miss_1C  p_flip_1C  pi_next  pi_rel  hw_mean_k2  hw_mean_k7  miss_k2  miss_k7
   model   T1                rw2hop - sponsor_hist    -0.0002      0.0013 0.0035     False     1.0000     0.4286   0.0035  0.3451      0.0025      0.0013   0.9610   0.9440
   model   T1             sponsor_hist - party+pop     0.0581      0.0114 0.0127      True     0.0000     0.0000   0.0298  2.9848      0.0158      0.0085   0.0000   0.0000
   model   T1         svd(val pick) - sponsor_hist     0.0210      0.0041 0.0070      True     0.0000     0.0000   0.0109  1.0850      0.0057      0.0031   0.0000   0.0000
near-tie   T1                      sh_hl1 - sh_hl2     0.0065      0.0018 0.0039     False     0.1429     0.0000   0.0048  0.4818      0.0028      0.0015   0.00

[FAIL] D9: Test-set size: a single test Congress confirms the direction of every relevant model / near-tie difference (paired, S3) | metric=0.429 (threshold per split: max over relevant pairs (|mean diff| >= D_MDE) of the miss risk <= 0.20 (empirical share of single test Congresses for 1-Congress splits; normal approximation for k = 2 / 7)) [fails for: S1a, S1b, S1a+stream, S1b+stream, S1pool]
[INFO] D9b: Deployment risk: how far a relevant model difference can move in the next Congress (95% PI half-width / MDE; informational) | metric=15.07 (threshold informational: > 1 means the next Congress can reverse or erase an MDE-sized advantage seen on test)

Absolute-score CI half-widths ('sponsor_hist' Recall@20, 'eb_rate' AUROC; bill vs sponsor-cluster bootstrap; S3/S4 = mean over folds):
                         n_eval  n_clusters  value  hw_bill  hw_cluster  cluster/bill
task         split                                                                   
advancement  S1b    13,844.0000 

### D10. Do older Congresses help (expanding) or hurt (sliding window)?

If the process drifts, old Congresses can teach stale patterns and a sliding window (only the last 3
Congresses) can match or beat the expanding window. Both windows are scored on the same test bills,
so the per-fold difference has a **paired** cluster-bootstrap CI. We run it for plain count
baselines, for their recency-weighted versions (half-life 1 Congress), and for the graph-structural
`rw2hop` (with and without recency weighting), and for the learned SVD embeddings (val-picked member,
same hyperparameters in both windows). If only the unweighted models prefer the sliding
window, the right conclusion is "weight recent edges", not "drop old Congresses". Folds share most
training Congresses, so the sign test is descriptive. Positive values = expanding window is better.

In [164]:
# D10: expanding (S3) minus sliding k=3 (S2) test scores per fold, paired cluster bootstrap, several models.
_ex = []
_D10_SERIES = [("T1 sponsor_hist", *_D_PRIMARY_T1, "sponsor_hist"), ("T1 sh_hl1 (recency)", *_D_PRIMARY_T1, "sh_hl1"),
               ("T1 rw2hop", *_D_PRIMARY_T1, "rw2hop"), ("T1 rw_b0.3_hl1 (recency)", *_D_PRIMARY_T1, "rw_b0.3_hl1"),
               ("T1 popularity", *_D_PRIMARY_T1, "popularity"), ("T1 sponsor_hist t0=7", "cosponsor_t7", "Recall@20", "sponsor_hist"),
               (f"T2 {_best2} AUROC", *_D_PRIMARY_T2, _best2), (f"T2 {_best2} AP_lift", "advancement", "AP_lift", _best2)]
for c in FOLD_C:
    ue, us_ = _fold_unit("S3", c), _fold_unit("S2", c)
    for lab, task, metric, b in _D10_SERIES:
        d, lo, hi, _ = _pdiff(ue, b, us_, b, task, metric, ("D10", c, lab))
        _ex.append(dict(series=lab, congress=c, diff=d, lo=lo, hi=hi))
    _pk = _SVD_PICK[c]                                    # learned model: same member refit on the sliding window
    d, lo, hi, _ = _pdiff(_LV[c], _pk, _learn_unit(_TR(c - SLIDE_K, c), c, svd_names=(_pk,)), _pk, *_D_PRIMARY_T1, ("D10svd", c))
    _ex.append(dict(series="T1 svd (val pick)", congress=c, diff=d, lo=lo, hi=hi))
_ex = pd.DataFrame(_ex)
_ex["sig_neg"], _ex["sig_pos"] = _ex.hi < 0, _ex.lo > 0
_ext = _ex.groupby("series", sort=False).agg(mean_diff=("diff", "mean"), fold_std=("diff", "std"),
                                             folds_expanding_better=("diff", lambda x: (x > 0).mean()),
                                             folds_sliding_signif=("sig_neg", "mean"), folds_expanding_signif=("sig_pos", "mean"))
print("Expanding minus sliding (k=3) on the same test bills, S3/S2 folds:")
print(_ext.round(4).to_string())
_exc = _ex[_ex.series.isin(["T1 sponsor_hist", "T1 sh_hl1 (recency)", "T1 svd (val pick)", f"T2 {_best2} AUROC"])]
line_chart(_exc, x="congress", y="diff", series="series", yfmt="+.3f", ylabel="expanding − sliding", xlabel="test Congress",
           title="Expanding window minus sliding window (k=3) test score per fold",
           subtitle="above 0 = using all earlier Congresses beats using only the last 3 (paired on the same test bills)",
           height=320, name="D_expanding_vs_sliding")
_hurt = {k: bool((_ext.loc[k, "mean_diff"] < 0) and (_ext.loc[k, "folds_sliding_signif"] >= 0.5)) for k in _ext.index}
_helps = {k: bool((_ext.loc[k, "mean_diff"] > 0) and (_ext.loc[k, "folds_expanding_signif"] >= 0.5)) for k in _ext.index}
_dec = ["T1 sh_hl1 (recency)", "T1 rw_b0.3_hl1 (recency)", "T1 svd (val pick)", f"T2 {_best2} AUROC"]
_d10 = not any(_hurt[k] for k in _dec)
_sign = {k: stats.binomtest(int((_ex[_ex.series == k]["diff"] < 0).sum()), len(FOLD_C), 0.5).pvalue for k in _ext.index}
_conc = ("recency weighting is needed for count/graph features (unweighted models prefer the sliding window, weighted ones do not)"
         if (_d10 and any(_hurt[k] for k in ("T1 sponsor_hist", "T1 rw2hop"))) else
         "old Congresses hurt even with recency weighting: use a sliding window" if not _d10 else "older Congresses do not hurt")
record_check("D", "D10", "Older Congresses do not hurt a recency-weighted model: expanding >= sliding window (k=3) on test",
             _d10, metric=f"T1 sh_hl1 diff={_ext.loc['T1 sh_hl1 (recency)', 'mean_diff']:+.4f}; T1 svd "
                          f"{_ext.loc['T1 svd (val pick)', 'mean_diff']:+.4f}; T2 {_ext.loc[f'T2 {_best2} AUROC', 'mean_diff']:+.4f}",
             threshold="FAIL if, for a recency-weighted or val-tuned model (sh_hl1, rw_b0.3_hl1, val-picked svd, T2), mean diff < 0 "
                       "and sliding is significantly better (paired cluster CI) in >= 50% of folds",
             by_split=_bs({**{s: _d10 for s in ("S1a", "S1b", "S3", "S1a+stream", "S1b+stream", "S3+stream", "S1pool")},
                           "S2": not any(_helps[k] for k in _dec)}), kind="validity",
             na_reason="S4 / S5 train on all earlier Congresses plus part of the eval Congress: the window choice does not apply",
             detail="Mean diff (folds sliding signif. better / expanding signif. better): " + "; ".join(
                 f"{k} {r.mean_diff:+.4f} ({r.folds_sliding_signif:.0%}/{r.folds_expanding_signif:.0%})" for k, r in _ext.iterrows())
             + f". Descriptive sign tests (dependent folds): " + ", ".join(f"{k} p={v:.2f}" for k, v in _sign.items() if k in _dec)
             + f". Conclusion: {_conc}. by_split: expanding splits pass only if a recency-weighted model is used; S2 fails "
               "if expanding is significantly better.")

Expanding minus sliding (k=3) on the same test bills, S3/S2 folds:
                          mean_diff  fold_std  folds_expanding_better  folds_sliding_signif  folds_expanding_signif
series                                                                                                             
T1 sponsor_hist             -0.0124    0.0056                  0.0000                0.8571                  0.0000
T1 sh_hl1 (recency)         -0.0008    0.0011                  0.2857                0.1429                  0.0000
T1 rw2hop                   -0.0124    0.0053                  0.0000                0.8571                  0.0000
T1 rw_b0.3_hl1 (recency)    -0.0007    0.0015                  0.4286                0.1429                  0.0000
T1 popularity               -0.0202    0.0104                  0.0000                0.8571                  0.0000
T1 sponsor_hist t0=7        -0.0117    0.0055                  0.0000                0.8571                  0.0000
T2 eb

[PASS] D10: Older Congresses do not hurt a recency-weighted model: expanding >= sliding window (k=3) on test | T1 sh_hl1 diff=-0.0008; T1 svd -0.0003; T2 +0.0027 (threshold FAIL if, for a recency-weighted or val-tuned model (sh_hl1, rw_b0.3_hl1, val-picked svd, T2), mean diff < 0 and sliding is significantly better (paired cluster CI) in >= 50% of folds)


### D26. Regime-matched vs recency training weights

Section A finds the deployment Congress (119, unified Republican control) closest to the last
Congresses with the same House / Senate majorities, and the majority-sponsor effect depends on which
party controls the chamber. If that matters for prediction, "train on the most recent Congresses"
(recency weighting, sliding windows) is the wrong remedy and training should upweight Congresses with
the same regime as the target. The regime of a Congress is known when it starts (election results), so
this is leakage-free. On every S3 test fold we refit on 108..c-1 with four weightings and score the same
test bills: unweighted, recency (half-life 1 Congress), regime-matched (x2 for training Congresses with
Congress c's House and Senate majorities, from `majority_party()`), and regime x recency. Task 1 uses
`sponsor_hist`; Task 2 uses `eb_rate_t0safe` (prediction at introduction) and `eb_rate`. Positive
paired differences = regime weighting helps. Folds with no regime-matched training Congress give 0.

In [165]:
# D26: paired differences between training weightings on S3 test folds (same bills), both tasks.
_rg = []
_W_PAIRS = [("T1 sponsor_hist", *_D_PRIMARY_T1, "regime x2 - unweighted", "sh_regime_x2", "sponsor_hist"),
            ("T1 sponsor_hist", *_D_PRIMARY_T1, "regime x2 - recency", "sh_regime_x2", "sh_recency_hl1"),
            ("T1 sponsor_hist", *_D_PRIMARY_T1, "regime x2 + recency - recency", "sh_regime_x2_hl1", "sh_recency_hl1"),
            ("T1 sponsor_hist", *_D_PRIMARY_T1, "recency - unweighted", "sh_recency_hl1", "sponsor_hist")]
for b in ("eb_rate_t0safe", "eb_rate"):
    _W_PAIRS += [(f"T2 {b}", *_D_PRIMARY_T2, "regime x2 - unweighted", f"{b}|regime_x2", f"{b}|unweighted"),
                 (f"T2 {b}", *_D_PRIMARY_T2, "regime x2 - recency", f"{b}|regime_x2", f"{b}|recency_hl1"),
                 (f"T2 {b}", *_D_PRIMARY_T2, "regime x2 + recency - recency", f"{b}|regime_x2_hl1", f"{b}|recency_hl1"),
                 (f"T2 {b}", *_D_PRIMARY_T2, "recency - unweighted", f"{b}|recency_hl1", f"{b}|unweighted")]
for c in FOLD_C:
    nmatch = sum(_REG.get(x) == _REG.get(c) for x in range(CONGRESSES[0], c))
    for model, task, metric, lab, a, b in _W_PAIRS:
        d, lo, hi, hw = _pdiff(_LV[c], a, _LV[c], b, task, metric, ("D26", c, model, lab)) if (nmatch or "regime" not in lab) \
            else (0.0, 0.0, 0.0, 0.0)
        _rg.append(dict(model=model, comparison=lab, congress=c, regime=_REG.get(c), n_match=nmatch, diff=d, lo=lo, hi=hi))
_rg = pd.DataFrame(_rg)
_rgt = _rg.groupby(["model", "comparison"], sort=False).agg(mean_diff=("diff", "mean"), folds_better=("lo", lambda x: int((x > 0).sum())),
                                                          folds_worse=("hi", lambda x: int((x < 0).sum())))
print("Regime per Congress (House/Senate majority): " + ", ".join(f"{c}: {r}" for c, r in _REG.items() if c >= CONGRESSES[0]))
print(f"S3 folds with >= 1 regime-matched training Congress: {int((_rg.drop_duplicates('congress').n_match > 0).sum())}/{len(FOLD_C)}. "
      "Paired test differences (folds better / worse = paired cluster CI excludes 0):")
print(_rgt.drop(index="T2 eb_rate", level="model").round(4).to_string())
bar_chart(_rg[_rg.comparison == "regime x2 + recency - recency"].assign(fold="c=" + _rg.congress.astype(str) + " " + _rg.regime)[
              ["fold", "diff", "model"]], x="fold", y="diff", series="model", yfmt="+.4f", ylabel="regime+recency − recency",
          title="Does upweighting same-regime Congresses beat recency weighting alone? (S3 test folds)",
          subtitle="paired difference on the same test bills; Task 1 Recall@20, Task 2 AUROC; 0 = no regime-matched training Congress",
          height=300, name="D_regime_weighting")
_key = _rgt.xs("regime x2 + recency - recency", level="comparison")
_reg_helps = {m: bool((r.mean_diff > 0) and (r.folds_better >= 2) and (r.folds_worse == 0)) for m, r in _key.iterrows()}
_d26 = not any(_reg_helps[m] for m in ("T1 sponsor_hist", "T2 eb_rate_t0safe"))
_r119 = _REG.get(IN_PROGRESS)
record_check("D", "D26", "Recency-only training is not beaten by regime-matched weighting (regime x2 + recency vs recency, S3 test folds)",
             bool(_d26), metric="; ".join(f"{m} {r.mean_diff:+.4f}" for m, r in _key.iterrows()),
             threshold="FAIL if, for T1 sponsor_hist or T2 eb_rate_t0safe, regime + recency beats recency alone on average AND is "
                       "significantly better (paired cluster CI) in >= 2 folds and worse in none",
             by_split=_bs({"S2": bool(_d26)}), quantity="regime_weighting", kind="drift", t0_task2=0,
             na_reason="only a sliding window drops older same-regime Congresses; expanding splits keep them (weighting is a remedy, not a split)",
             detail="Mean paired diff (folds better/worse): " + "; ".join(f"{m} {cmp}: {r.mean_diff:+.4f} ({int(r.folds_better)}/{int(r.folds_worse)})"
                                                                          for (m, cmp), r in _rgt.iterrows() if "regime" in cmp)
             + f". Deployment Congress {IN_PROGRESS} regime {_r119}; same-regime Congresses in {CONGRESSES[0]}-{CONGRESSES[-1]}: "
               f"{[c for c in CONGRESSES if _REG.get(c) == _r119]}. A FAIL means training should upweight same-regime Congresses "
               "(which a sliding window drops, hence by_split S2) rather than only recent ones; a PASS means recency weighting "
               "(or the expanding window) is enough for these baselines. Regime weights are known at the start of a Congress.")

Regime per Congress (House/Senate majority): 108: RH/RS, 109: RH/RS, 110: DH/DS, 111: DH/DS, 112: RH/DS, 113: RH/DS, 114: RH/RS, 115: RH/RS, 116: DH/RS, 117: DH/DS, 118: RH/DS, 119: RH/RS
S3 folds with >= 1 regime-matched training Congress: 5/7. Paired test differences (folds better / worse = paired cluster CI excludes 0):
                                                 mean_diff  folds_better  folds_worse
model             comparison                                                         
T1 sponsor_hist   regime x2 - unweighted            0.0010             2            3
                  regime x2 - recency              -0.0156             0            5
                  regime x2 + recency - recency     0.0011             1            0
                  recency - unweighted              0.0209             7            0
T2 eb_rate_t0safe regime x2 - unweighted           -0.0001             0            1
                  regime x2 - recency              -0.0020             0 

[PASS] D26: Recency-only training is not beaten by regime-matched weighting (regime x2 + recency vs recency, S3 test folds) | T1 sponsor_hist +0.0011; T2 eb_rate_t0safe -0.0001; T2 eb_rate -0.0000 (threshold FAIL if, for T1 sponsor_hist or T2 eb_rate_t0safe, regime + recency beats recency alone on average AND is significantly better (paired cluster CI) in >= 2 folds and worse in none)


### D16. Task 2 sensitivity to information not available at prediction time

CRS assigns the policy area some days after introduction, so a Task 2 model scored at introduction
often does not have it. `eb_rate_t0safe` hides the policy area for the share of bills that lack it
when fresh (measured on the in-progress 119th Congress). If the score drops noticeably, the Task 2
feature set must be built "as of" the prediction time, otherwise every split overstates performance.

In [166]:
# D16: eb_rate vs eb_rate_t0safe on S3 test folds (policy area hidden for fresh bills).
_ts = _sel(*_D_PRIMARY_T2, split="S3", eval_role="test").pivot_table(index="eval_congress", columns="baseline", values="value")
_ts["drop"] = _ts.eb_rate - _ts.eb_rate_t0safe
print(_ts[["eb_rate", "eb_rate_t0safe", "drop"]].round(4).T.to_string())
record_check("D", "D16", "Task 2: AUROC drop when policy area is unavailable at introduction (eb_rate - eb_rate_t0safe, S3 test)",
             None, metric=float(_ts["drop"].mean()), threshold="informational",
             detail=f"Policy area hidden for {D_PA_MISS_T0:.0%} of evaluation bills (share missing among the {_D_PA_N} 119th bills "
                    f"<= {D_PA_FRESH_DAYS} days old at the snapshot, a small-sample proxy; {_D_PA_MISS_30:.0%} at <= 30 days). Mean drop {_ts['drop'].mean():.3f} AUROC "
                    f"(max {_ts['drop'].max():.3f}). The drop is roughly uniform across folds, so it shifts all splits alike "
                    "rather than favouring one; build Task 2 features as of the prediction time.")

eval_congress     112    113    114    115    116    117    118
baseline                                                       
eb_rate        0.7822 0.7539 0.7151 0.7421 0.6875 0.6924 0.6971
eb_rate_t0safe 0.6276 0.6684 0.6636 0.6585 0.5688 0.5598 0.6188
drop           0.1546 0.0855 0.0516 0.0837 0.1187 0.1325 0.0782
[INFO] D16: Task 2: AUROC drop when policy area is unavailable at introduction (eb_rate - eb_rate_t0safe, S3 test) | metric=0.1007 (threshold informational)


### D18. Does the candidate pool definition matter?

Task 1 ranks the point-in-time pool (members in office in the month of introduction + t0). The
legacy pool (everyone who served in the chamber at any time that Congress) also contains departed and
not-yet-seated members, which inflates the negatives. We rerun the S3 test folds with the legacy pool
to see whether scores or the ordering of baselines depend on that choice.

In [167]:
# D18: point-in-time vs legacy candidate pool on S3 test folds (t0=0), Task 1.
_rs = []
for c in FOLD_C:
    (pp, cp_), (pl, cl) = _fold_unit("S3", c)["cosponsor_t0"], _fold_unit("S3", c, pit=False)["cosponsor_t0"]
    for b in ("sponsor_hist", "rw2hop", "popularity"):
        _rs.append(dict(congress=c, baseline=b, point_in_time=pp[b][:, 1].mean(), legacy=pl[b][:, 1].mean(),
                        n_pit=len(cp_), n_legacy=len(cl)))
_rs = pd.DataFrame(_rs)
_rs["diff"] = _rs.point_in_time - _rs.legacy
_pl = np.concatenate(_D_LOG["pool_legacy"])
print("Point-in-time minus legacy pool, mean Recall@20 over S3 test folds: " + ", ".join(
    f"{b} {g['diff'].mean():+.4f}" for b, g in _rs.groupby("baseline")))
_ord_same = (_rs.pivot_table(index="congress", columns="baseline", values="point_in_time").rank(axis=1)
             == _rs.pivot_table(index="congress", columns="baseline", values="legacy").rank(axis=1)).all(axis=1).mean()
record_check("D", "D18", "Task 1: scores and baseline ordering are insensitive to the candidate pool (point-in-time vs legacy roster, S3)",
             None, metric=float(_rs[_rs.baseline == "sponsor_hist"]["diff"].mean()), threshold="informational",
             detail=f"Mean Recall@20 point-in-time minus legacy: " + ", ".join(
                 f"{b} {g['diff'].mean():+.4f}" for b, g in _rs.groupby("baseline")) + f"; baseline ordering identical in "
                    f"{_ord_same:.0%} of folds (any swap is between the tied sponsor_hist and rw2hop). Median eligible pool {np.median(_pp):.0f} (point-in-time) vs {np.median(_pl):.0f} "
                    f"(legacy); {_D_LOG['pos_not_in_office'] / max(_D_LOG['pos_pit_total'], 1):.1%} of positives fall outside the "
                    "point-in-time pool (members joining mid-window or roster approximation) and are dropped. The pool choice "
                    "shifts all splits alike; the notebook uses the point-in-time pool everywhere.")

Point-in-time minus legacy pool, mean Recall@20 over S3 test folds: popularity +0.0021, rw2hop +0.0021, sponsor_hist +0.0024
[INFO] D18: Task 1: scores and baseline ordering are insensitive to the candidate pool (point-in-time vs legacy roster, S3) | metric=0.002352 (threshold informational)


### D19. Refit on train + validation, or keep the validation-selected model?

Test units "select on val, refit on train + val". A GNN chooses its epoch count by early stopping
on val, and the refit has no val left to stop on, so the refit must reuse a fixed budget (the epoch
or hyperparameter value chosen on val). For each S3 fold we compare the test score of the
val-selected family member **without refit** (fit on 108..c-2) and **with refit** (same member, fit
on 108..c-1). If refitting does not help, keeping the val-selected checkpoint (and its early-stopping
signal) loses nothing; if it helps, refit with the val-chosen budget.

In [168]:
# D19: refit (train+val) vs no refit (train only) for the val-selected member of each family, S3 test folds.
_rf = []
for c in FOLD_C:
    ur, un = _fold_unit("S3", c), _fold_unit("S3norefit", c)
    for fam, task, metric in (("T1 sponsor_hist family", *_D_PRIMARY_T1), ("T1 rw2hop family", *_D_PRIMARY_T1),
                              ("T2 eb_rate family", "advancement", "AUROC")):
        pick = _ORD[(fam, metric, "S3")].loc[f"c={c}", "val_pick"]
        d, lo, hi, hw = _pdiff(ur, pick, un, pick, task, metric, ("refit", c, fam))
        _rf.append(dict(family=fam, congress=c, pick=pick, gain=d, hw=hw, sig_pos=lo > 0, sig_neg=hi < 0))
    _pk = _SVD_PICK[c]                                    # learned model: val-picked member fit on 108..c-2 vs refit on 108..c-1
    d, lo, hi, hw = _pdiff(_LV[c], _pk, _learn_unit(_TR(CONGRESSES[0], c - 1), c, svd_names=(_pk,)), _pk, *_D_PRIMARY_T1, ("refit_svd", c))
    _rf.append(dict(family="T1 svd (val pick)", congress=c, pick=_pk, gain=d, hw=hw, sig_pos=lo > 0, sig_neg=hi < 0))
    urs, uns = _fold_unit("S3", c, stream=True), _fold_unit("S3norefit", c, stream=True)
    for b in ("sponsor_hist", "rw2hop"):                      # same comparison when eval-Congress history is streamed
        d, lo, hi, hw = _pdiff(urs, b, uns, b, *_D_PRIMARY_T1, ("refit_stream", c, b))
        _rf.append(dict(family=f"T1 {b} (+stream)", congress=c, pick=b, gain=d, hw=hw, sig_pos=lo > 0, sig_neg=hi < 0))
_rf = pd.DataFrame(_rf)
_rft = _rf.groupby("family").agg(mean_gain=("gain", "mean"), min_gain=("gain", "min"), max_gain=("gain", "max"),
                                 mean_hw=("hw", "mean"), folds_refit_better=("sig_pos", "sum"), folds_refit_worse=("sig_neg", "sum"))
print("Refit (fit on 108..c-1) minus no refit (fit on 108..c-2), same val-selected member, S3 test Congress c:")
print(_rft.round(4).to_string())
bar_chart(_rf[~_rf.family.str.contains("stream")].assign(fold="c=" + _rf.congress.astype(str))[["fold", "gain", "family"]],
          x="fold", y="gain", series="family",
          yfmt="+.4f", ylabel="refit − no refit", title="Test-score gain from refitting on train + validation (S3 folds)",
          subtitle="same hyperparameters (chosen on val); above 0 = refitting with the val Congress helps", height=300,
          name="D_refit_gain")
_worse = _rft[_rft.folds_refit_worse >= 2].index.tolist()
_better = _rft[_rft.folds_refit_better >= len(FOLD_C) / 2].index.tolist()
record_check("D", "D19", "Refitting on train+val with hyperparameters fixed from val does not hurt the test score (S3 folds)",
             not _worse, metric=float(_rft.mean_gain.min()), kind="validity",
             threshold="FAIL if refit is significantly worse (paired cluster CI) in >= 2 of 7 folds for any family",
             detail="Mean gain (folds refit signif. better / worse): " + "; ".join(
                 f"{k} {r.mean_gain:+.4f} ({int(r.folds_refit_better)}/{int(r.folds_refit_worse)})" for k, r in _rft.iterrows())
             + f". Refit clearly helps for: {', '.join(_better) or 'none'}. With streaming, refit still adds "
               f"{_rft.loc['T1 sponsor_hist (+stream)', 'mean_gain']:+.4f} (sponsor_hist) / {_rft.loc['T1 rw2hop (+stream)', 'mean_gain']:+.4f} "
               "(rw2hop), so the eval Congress's own history does not replace the val Congress. For count baselines 'refit' = "
               "adding the val Congress's edges to the graph; for a GNN that is two separate things: (1) the val Congress's edges "
               "must always be in the message-passing graph at test time (no retraining needed), (2) retraining weights on "
               "train+val should reuse the val-chosen epoch count; whether (2) adds beyond (1) can only be measured with the GNN "
               "(compare on the S3 folds as here).")

Refit (fit on 108..c-1) minus no refit (fit on 108..c-2), same val-selected member, S3 test Congress c:
                           mean_gain  min_gain  max_gain  mean_hw  folds_refit_better  folds_refit_worse
family                                                                                                  
T1 rw2hop (+stream)           0.0257    0.0220    0.0308   0.0067                   7                  0
T1 rw2hop family              0.0648    0.0491    0.0762   0.0109                   7                  0
T1 sponsor_hist (+stream)     0.0244    0.0184    0.0281   0.0064                   7                  0
T1 sponsor_hist family        0.0589    0.0447    0.0731   0.0107                   7                  0
T1 svd (val pick)             0.0644    0.0503    0.0763   0.0115                   7                  0
T2 eb_rate family             0.0050   -0.0004    0.0122   0.0045                   3                  0


[PASS] D19: Refitting on train+val with hyperparameters fixed from val does not hurt the test score (S3 folds) | metric=0.005004 (threshold FAIL if refit is significantly worse (paired cluster CI) in >= 2 of 7 folds for any family)


### D27. A locked final test: develop on folds 112-117, report once on 118

In S3 / S2 every test Congress 112-117 is also the validation Congress of the next fold, and D9
recommends comparing models on the mean over rolling test folds. If the team compares GNN
architectures or feature sets on that mean and then reports it, the reported number was chosen on
test data. The fix is a **locked test**: develop, tune and compare on the rolling folds 112-117 only
(hyperparameters on each fold's val Congress, architectures on the mean over the 6 development folds
or on the mean of their val Congresses, D17), then freeze the pipeline (features, model, val-chosen
epochs), refit on 108-117 and score Congress 118 **once**. Congress 119 is the prospective test once
its windows close. This cell measures what reserving 118 costs: whether the selection checks still
pass on 6 folds, how much the miss risk (D9) and the level CI (D15) grow from k = 7 to k = 6, and
whether the development folds predict the locked fold (118 inside the development prediction interval).
In the chart each row is a relevant model pair: grey dots = development folds, band = 95% prediction
interval for a new Congress from those folds, orange dot = the locked fold 118 with its paired CI.

In [169]:
# D27: final-test isolation; selection, sizing and level precision on dev folds (k=6) vs all folds (k=7); 118 as locked test.
_kd, _ka = len(_DEV), len(FOLD_C)
_dev_f = [f"c={c}" for c in _DEV]
_lk = []
for fam, metric in (("T1 sponsor_hist family", "Recall@20"), ("T1 rw2hop family", "Recall@20"),
                    ("T2 eb_rate family", "AUROC"), ("T2 eb_rate_t0safe family", "AUROC")):
    o = _ORD[(fam, metric, "S3")]
    od = o[o.index.isin(_dev_f)]
    _lk.append(dict(item=fam.replace(" family", ""), regret_k7=o.regret.mean(), sig_k7=int(o.regret_sig.sum()),
                    ok_k7=_sel_verdict(fam, metric, "S3"), regret_k6=od.regret.mean(), sig_k6=int(od.regret_sig.sum()),
                    ok_k6=_sel_verdict(fam, metric, "S3", folds=_DEV),
                    ok_k6_3val=_sel_verdict(fam, metric, "S3", "regret_3val", "sig3", folds=_DEV),
                    ok_k6_S2=_sel_verdict(fam, metric, "S2", folds=_DEV)))
_ld = _LSEL[_LSEL.index.isin(_dev_f)]
_lk.append(dict(item="T1 svd (learned)", regret_k7=_LSEL.regret.mean(), sig_k7=int(_LSEL.regret_sig.sum()), ok_k7=_l_ok,
                regret_k6=_ld.regret.mean(), sig_k6=int(_ld.regret_sig.sum()),
                ok_k6=bool(not ((_ld.regret.mean() > D_TOL["cosponsor_t0"]) and (_ld.regret_sig.sum() >= 2))),
                ok_k6_3val=True, ok_k6_S2=True))
_lk = pd.DataFrame(_lk).set_index("item")
print(f"Selection on development folds {_DEV[0]}-{_DEV[-1]} (k={_kd}) vs all folds (k={_ka}): mean regret, significant folds, verdict")
print(_lk[["regret_k7", "sig_k7", "ok_k7", "regret_k6", "sig_k6", "ok_k6"]].round(4).to_string())

# Level precision (D15) and locked-fold coverage: per model, mean over dev folds, CI of the mean, and 118 vs the dev PI.
_lvl_src = {f"T1 {_best1}": _sel(*_D_PRIMARY_T1, _best1, split="S3", eval_role="test").set_index("eval_congress").value,
            "T1 svd (val pick)": pd.Series(_LSEL.test.to_numpy(), index=[int(f[2:]) for f in _LSEL.index]),
            "T2 eb_rate_t0safe": _sel(*_D_PRIMARY_T2, "eb_rate_t0safe", split="S3", eval_role="test").set_index("eval_congress").value,
            "T2 eb_rate (policy area)": _sel(*_D_PRIMARY_T2, "eb_rate", split="S3", eval_role="test").set_index("eval_congress").value}
_lv = []
for k, v in _lvl_src.items():
    v.index = v.index.astype(int)
    vd = v.reindex(_DEV)
    _lv.append(dict(model=k, mean_k7=v.mean(), hw_mean_k7=1.96 * v.std(ddof=1) / np.sqrt(_ka), mean_k6=vd.mean(),
                    hw_mean_k6=1.96 * vd.std(ddof=1) / np.sqrt(_kd), pi_dev=_tq6 * vd.std(ddof=1), locked_118=v[_LOCK],
                    in_pi=bool(abs(v[_LOCK] - vd.mean()) <= _tq6 * vd.std(ddof=1))))
_lv = pd.DataFrame(_lv).set_index("model")
print(f"Reported level: mean over test folds and its 95% CI half-width (k={_ka} vs k={_kd}); {_LOCK} vs the dev prediction interval:")
print(_lv.round(4).to_string())

# Paired sizing (D9) without the locked fold.
_rel6 = _prs[_prs.kind.isin(["model", "near-tie"]) & _prs.relevant_dev]
_m6, _m7 = float(_rel6.miss_k6.max()) if len(_rel6) else 0.0, float(_dec_pairs.miss_k7.max()) if len(_dec_pairs) else 0.0
_cov = _prs[_prs.kind.isin(["model", "near-tie", "protocol"]) & (_prs.relevant | _prs.relevant_dev)]
print(f"Paired sizing, relevant pairs: max miss risk k={_ka} {_m7:.3f} -> k={_kd} {_m6:.3f}; median CI half-width of the mean "
      f"difference {_dec_pairs.hw_mean_k7.median():.4f} -> {_rel6.hw_mean_k6.median():.4f}; {_LOCK} paired difference inside the "
      f"dev prediction interval for {int(_cov.lock_in_pi.sum())}/{len(_cov)} relevant model / protocol pairs")

_LOCK_JS = r"""
const {width, height, title, subtitle, xlabel} = opts;
const m = {top: 48, right: 24, bottom: 56, left: 250};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, title, subtitle);
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const rows = Array.from(new Set(data.map(d => d.pair)));
const ys = d3.scaleBand(rows, [0, H]).padding(0.3);
const xs = d3.scaleLinear([Math.min(0, d3.min(data, d => d.lo)), d3.max(data, d => d.hi)], [0, W]).nice();
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).ticks(6).tickSize(-H).tickFormat(d3.format('+.2f')))
  .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
  .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').call(d3.axisLeft(ys).tickSize(0)).call(viz.axis).call(a => a.select('.domain').remove());
g.append('line').attr('x1', xs(0)).attr('x2', xs(0)).attr('y1', 0).attr('y2', H).attr('stroke', viz.ink).attr('stroke-width', 1);
g.append('text').attr('x', W).attr('y', H + 34).attr('text-anchor', 'end').attr('fill', viz.ink2).style('font', '11px sans-serif').text(xlabel);
const cy = d => ys(d.pair) + ys.bandwidth() / 2;
const tip = viz.tooltip(el);
g.selectAll(null).data(data.filter(d => d.kind === 'band')).join('rect').attr('x', d => xs(d.lo)).attr('width', d => Math.max(xs(d.hi) - xs(d.lo), 1))
  .attr('y', d => ys(d.pair)).attr('height', ys.bandwidth()).attr('fill', viz.palette[0]).attr('opacity', 0.18).attr('rx', 2)
  .on('mousemove', (e, d) => tip.show(`<b>${d.pair}</b><br>dev mean ${d3.format('+.4f')(d.value)}<br>95% PI [${d3.format('+.4f')(d.lo)}, ${d3.format('+.4f')(d.hi)}]`, e))
  .on('mouseleave', () => tip.hide());
g.selectAll(null).data(data.filter(d => d.kind === 'dev')).join('circle').attr('cx', d => xs(d.value)).attr('cy', cy).attr('r', 3.5)
  .attr('fill', '#8a8984').attr('opacity', 0.8)
  .on('mousemove', (e, d) => tip.show(`<b>${d.pair}</b><br>dev fold ${d.congress}: ${d3.format('+.4f')(d.value)}`, e))
  .on('mouseleave', () => tip.hide());
const lk = data.filter(d => d.kind === 'lock');
g.selectAll(null).data(lk).join('line').attr('x1', d => xs(d.lo)).attr('x2', d => xs(d.hi)).attr('y1', d => cy(d) + 6).attr('y2', d => cy(d) + 6)
  .attr('stroke', viz.palette[1]).attr('stroke-width', 2);
g.selectAll(null).data(lk).join('circle').attr('cx', d => xs(d.value)).attr('cy', d => cy(d) + 6).attr('r', 5)
  .attr('fill', viz.palette[1]).attr('stroke', '#fff').attr('stroke-width', 1.5)
  .on('mousemove', (e, d) => tip.show(`<b>${d.pair}</b><br>locked fold ${d.congress}: ${d3.format('+.4f')(d.value)} [${d3.format('+.4f')(d.lo)}, ${d3.format('+.4f')(d.hi)}]`, e))
  .on('mouseleave', () => tip.hide());
viz.legend(svg, [['development folds', '#8a8984'], ['95% PI for a new Congress (dev folds)', viz.palette[0]], ['locked fold (paired CI)', viz.palette[1]]], m.left, height - 10);
"""
_pdl = pd.concat(_pd_rows, ignore_index=True)
_ch = []
for r in _cov.itertuples():
    lab = f"{r.task} {r.pair}"
    dd = _pdl[(_pdl.pair == r.pair) & _pdl.congress.isin(_DEV)]
    _ch.append(dict(pair=lab, kind="band", value=r.mean_dev, lo=r.mean_dev - r.pi_dev, hi=r.mean_dev + r.pi_dev, congress=0))
    _ch += [dict(pair=lab, kind="dev", value=x, lo=x, hi=x, congress=int(c)) for x, c in zip(dd["diff"], dd.congress)]
    _ch.append(dict(pair=lab, kind="lock", value=r.lock_diff, lo=r.lock_lo, hi=r.lock_hi, congress=_LOCK))
show_d3(_LOCK_JS, pd.DataFrame(_ch), height=110 + 40 * max(len(_cov), 1), width=760,
        opts=dict(title=f"Do the development folds {_DEV[0]}-{_DEV[-1]} predict the locked test Congress {_LOCK}?",
                  subtitle="paired test difference per relevant model / protocol pair (Task 1 Recall@20, Task 2 AUROC at introduction)",
                  xlabel=f"paired difference (MDE = {D_MDE['cosponsor_t0']})"), name="D_locked_test")

_sel_ok6 = bool(_lk.ok_k6.all())
_d27 = _sel_ok6 and (_m6 <= 0.2)
_d27_s2 = bool(_lk.drop(index="T1 svd (learned)").ok_k6_S2.all()) and (_m6 <= 0.2)
_dhw = _lv.hw_mean_k6 - _lv.hw_mean_k7
record_check("D", "D27", f"Final-test isolation: developing on rolling folds {_DEV[0]}-{_DEV[-1]} and reserving {_LOCK} as a locked test "
             "keeps selection and paired sizing adequate", bool(_d27), metric=round(_m6, 3), kind="validity",
             threshold=f"PASS if every selection check (D11/D12/D12b/D21/D23 rule, regret tol MDE/2) still passes on the {_kd} dev "
                       f"folds AND the max miss risk over relevant pairs at k={_kd} is <= 0.20",
             by_split=_bs({"S3": bool(_d27), "S3+stream": bool(_d27), "S2": bool(_d27_s2)}),
             na_reason={**{s_: "one test Congress, reported once: isolated by construction, nothing to reserve (its size is judged in D9)"
                           for s_ in ("S1a", "S1b", "S1a+stream", "S1b+stream", "S1pool")},
                        "S4": "within-Congress split: no rolling reuse of test Congresses", "S5": "random split: leaky reference only"},
             detail=f"Selection with k={_kd} (k={_ka}): " + "; ".join(f"{i} regret {r.regret_k6:.4f} ({r.sig_k6}/{_kd} signif., "
                    f"{'pass' if r.ok_k6 else 'FAIL'}) [{r.regret_k7:.4f}, {r.sig_k7}/{_ka}]" for i, r in _lk.iterrows())
                    + f". 3-val-fold selection (D17) on dev folds passes for all families: {bool(_lk.ok_k6_3val.all())}. Paired sizing: max "
                    f"miss risk {_m7:.3f} (k={_ka}) -> {_m6:.3f} (k={_kd}); median CI half-width of a mean difference "
                    f"{_dec_pairs.hw_mean_k7.median():.4f} -> {_rel6.hw_mean_k6.median():.4f}. Reported-level CI half-width (D15) changes by (k={_kd} minus k={_ka}) "
                    + ", ".join(f"{k} {v:+.4f}" for k, v in _dhw.items())
                    + f". Locked fold {_LOCK}: level inside the dev prediction interval for "
                    + ", ".join(f"{k} {'yes' if r.in_pi else 'NO'} ({r.locked_118:.3f} vs dev {r.mean_k6:.3f} +/- {r.pi_dev:.3f})" for k, r in _lv.iterrows())
                    + f"; paired differences inside the dev PI for {int(_cov.lock_in_pi.sum())}/{len(_cov)} relevant pairs (outside: "
                    + (", ".join(f"{r.task} {r.pair} {r.lock_diff:+.4f} vs dev {r.mean_dev:+.4f} +/- {r.pi_dev:.4f}"
                                 for r in _cov[~_cov.lock_in_pi].itertuples()) or "none")
                    + f"). A locked level outside the dev interval is drift in {_LOCK} (sections A / C), not a protocol error: report "
                      "it as measured and do not re-tune on it. Protocol: "
                      f"select on folds {_DEV[0]}-{_DEV[-1]} (val c-1, test c; architectures on the dev-fold mean or the D17 val mean), "
                      f"freeze, refit on {CONGRESSES[0]}-{_LOCK - 1}, report {_LOCK} once, then {IN_PROGRESS} prospectively. A FAIL "
                      "would mean the reserved Congress is needed for reliable selection (then report the 7-fold mean and say it was "
                      "used for selection).")
D_FINAL_PROTOCOL = dict(
    develop=f"S3(+stream) rolling folds {_DEV[0]}-{_DEV[-1]}: train {CONGRESSES[0]}..c-2 -> val c-1 -> refit {CONGRESSES[0]}..c-1 -> test c",
    select_hyperparameters="per fold on the val Congress c-1 (or the mean of the last 3 val Congresses, D17)",
    compare_architectures=f"mean over the {_kd} development test folds (paired, same bills); never on {_LOCK}",
    locked_test=_LOCK, locked_train=list(range(CONGRESSES[0], _LOCK)),
    prospective_test=IN_PROGRESS, mde=D_MDE, reserve_cost=dict(max_miss_k7=_m7, max_miss_k6=_m6, selection_ok_k6=_sel_ok6))
print("D_FINAL_PROTOCOL: " + " | ".join(f"{k}: {v}" for k, v in D_FINAL_PROTOCOL.items() if k in ("develop", "locked_test", "prospective_test")))

Selection on development folds 112-117 (k=6) vs all folds (k=7): mean regret, significant folds, verdict
                   regret_k7  sig_k7  ok_k7  regret_k6  sig_k6  ok_k6
item                                                                 
T1 sponsor_hist       0.0028       1   True     0.0032       1   True
T1 rw2hop             0.0002       0   True     0.0002       0   True
T2 eb_rate            0.0078       1   True     0.0017       0   True
T2 eb_rate_t0safe     0.0045       0   True     0.0053       0   True
T1 svd (learned)      0.0007       0   True     0.0008       0   True
Reported level: mean over test folds and its 95% CI half-width (k=7 vs k=6); 118 vs the dev prediction interval:
                          mean_k7  hw_mean_k7  mean_k6  hw_mean_k6  pi_dev  locked_118  in_pi
model                                                                                        
T1 sponsor_hist            0.2591      0.0065   0.2617      0.0050  0.0172      0.2439  False
T1 svd (va

[PASS] D27: Final-test isolation: developing on rolling folds 112-117 and reserving 118 as a locked test keeps selection and paired sizing adequate | metric=0.115 (threshold PASS if every selection check (D11/D12/D12b/D21/D23 rule, regret tol MDE/2) still passes on the 6 dev folds AND the max miss risk over relevant pairs at k=6 is <= 0.20)
D_FINAL_PROTOCOL: develop: S3(+stream) rolling folds 112-117: train 108..c-2 -> val c-1 -> refit 108..c-1 -> test c | locked_test: 118 | prospective_test: 119


### Section D summary

One line per check. FAIL does not automatically rule out a split: read the detail for the
implication (e.g. report the mean over rolling folds instead of a single test Congress).

In [170]:
# Summary of section D checks (with the splits each check fails for) and the objects exposed to the decision section.
_dc = pd.DataFrame([c for c in CHECKS if c["section"] == "D"])
_dc["result"] = _dc.passed.map({True: "PASS", False: "FAIL"}).fillna("INFO")
_dc["metric"] = _dc.metric.map(lambda v: f"{v:.4g}" if isinstance(v, (int, float, np.floating)) else str(v))
_dc["fails_for"] = _dc.by_split.map(lambda d: ",".join(k for k, v in (d or {}).items() if v is False))
for _res in ("FAIL", "PASS", "INFO"):                     # compact: the full condition lines are printed above
    _x = _dc[_dc.result == _res]
    print(f"{_res} ({len(_x)}): " + ", ".join(r.id + (f" [fails for {r.fails_for}]" if r.fails_for else "") for r in _x.itertuples()))
print(f"\nSPLIT_DEFS + SPLIT_VARIANTS (setup): {ALL_SPLITS} | SPLIT_RESULTS: {len(SPLIT_RESULTS):,} rows "
      f"({SPLIT_RESULTS.task.nunique()} tasks x {SPLIT_RESULTS.baseline.nunique()} baselines; CIs = cluster bootstrap by sponsor, "
      f"ci_*_bill = bill bootstrap) | D_FAMILY_RESULTS: {len(D_FAMILY_RESULTS):,} rows | D_PAIRED: paired-difference sizing | "
      "D_LEARNED: SVD selection per fold | D_STALENESS: recall by legislator status | D_REGIME: weighting comparison | "
      "D_LOCKED: selection / level on dev folds vs all folds | D_FINAL_PROTOCOL: locked-test protocol")
D_PAIRED, D_LEARNED, D_STALENESS, D_REGIME, D_LOCKED = _prs, _LSEL, _strt, _rgt, dict(selection=_lk, level=_lv)

FAIL (3): D2b [fails for S1b,S2,S3,S4,S1b+stream,S3+stream], D7 [fails for S5], D9 [fails for S1a,S1b,S1a+stream,S1b+stream,S1pool]
PASS (15): D1 [fails for S1pool], D20 [fails for S1pool], D11, D12, D12b, D21, D17, D22, D23, D25, D14, D10, D26, D19, D27
INFO (12): D2, D3, D4, D5, D6, D24, D8, D13, D9b, D15, D16, D18

SPLIT_DEFS + SPLIT_VARIANTS (setup): ['S1a', 'S1b', 'S2', 'S3', 'S4', 'S5', 'S1a+stream', 'S1b+stream', 'S3+stream', 'S1pool'] | SPLIT_RESULTS: 2,160 rows (3 tasks x 8 baselines; CIs = cluster bootstrap by sponsor, ci_*_bill = bill bootstrap) | D_FAMILY_RESULTS: 1,512 rows | D_PAIRED: paired-difference sizing | D_LEARNED: SVD selection per fold | D_STALENESS: recall by legislator status | D_REGIME: weighting comparison | D_LOCKED: selection / level on dev folds vs all folds | D_FINAL_PROTOCOL: locked-test protocol


## Decision: which split to use

Sections A-D each tested one family of conditions a split relies on and recorded every result in the
shared check registry (`CHECKS`) and, for D, in the backtest table `SPLIT_RESULTS`. This section reads
only those two objects: nothing below is hardcoded, so if you change a threshold or rerun on new data,
the tables, scores and the recommendation update with it.

**How checks are used.** Every check has a *kind*. **Validity** checks ask whether a split gives an
honest, usable estimate (leakage, val predicts test, enough test Congresses, censoring): a FAIL is a
reason to prefer a different split, and only these rank the splits. **Drift** checks ask whether later
Congresses differ from earlier ones: every forward split fails them by design (that is the split doing
its job), so a drift FAIL selects a *remedy* (recalibration, recency weighting, per-Congress reporting),
not a different split. **Requirements** are construction rules every split must follow (time-filter
edges, rebuild features as of t0); they never count as split failures. A few split-dependent checks
have no kind ("unclassified"); they are not used for the main ranking, but the recommendation is
re-run with them included as a robustness test.

In [171]:
# All recorded checks, grouped by section: status, kind, metric, threshold and a compact detail line.
pd.set_option("display.width", 250)


def _dec_status(passed, req):
    s = {True: "PASS", False: "FAIL"}.get(passed if isinstance(passed, (bool, np.bool_)) else None, "INFO")
    return s + ("/REQ" if req else "")


def _dec_fmt(v, n=28):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool):
        return f"{v:.4g}"
    s = str(v)
    return s if len(s) <= n else s[: n - 1] + "~"


def _dec_short(s, n):
    s = " ".join(str(s or "").split())
    return s if len(s) <= n else s[: n - 1] + "~"


DEC_ALL = checks_summary(dedup=False).reset_index(drop=True)
DEC_ALL["status"] = [_dec_status(p, r) for p, r in zip(DEC_ALL.passed, DEC_ALL.requirement)]
DEC_ALL["kind_lbl"] = DEC_ALL.kind.map(lambda k: k if isinstance(k, str) else "unclass.")
_dec_kept = checks_summary(dedup=True)
_dec_kept_ids = set(_dec_kept.id)
DEC_ALL["counted"] = DEC_ALL.id.isin(_dec_kept_ids)
DEC_SECTIONS = {"A": "A. targets & covariates", "B": "B. graph & nodes", "C": "C. temporal & leakage",
                "D": "D. backtest"}
print(f"{len(DEC_ALL)} checks recorded; {len(_dec_kept_ids)} remain after de-duplicating shared quantities "
      "('dup' = verdict owned by another section's check of the same quantity; not counted twice)")
for sec, g in DEC_ALL.groupby("section", sort=True):
    t = pd.DataFrame({
        "id": g.id, "status": g.status, "kind": g.kind_lbl, "dup": np.where(g.counted, "", "dup"),
        "metric": g.metric.map(_dec_fmt), "threshold": g.threshold.map(lambda v: _dec_short(v, 34)),
        "condition / detail": [_dec_short(c, 58) + " | " + _dec_short(d, 50) for c, d in zip(g.condition, g.detail)],
    })
    print(f"\n--- {DEC_SECTIONS.get(sec, sec)} ({len(g)} checks) ---")
    print(t.to_string(index=False, max_colwidth=115))

# Counts used for decisions: de-duplicated, construction requirements excluded.
_dec_nr = checks_summary(requirements=False)
_dec_cnt = (_dec_nr.assign(status=[_dec_status(p, False) for p in _dec_nr.passed],
                           kind=_dec_nr.kind.map(lambda k: k if isinstance(k, str) else "unclassified"))
            .pivot_table(index="section", columns=["kind", "status"], values="id", aggfunc="count", fill_value=0))
print("\nDe-duplicated, non-requirement checks by section x kind x status (these are the counts the decision uses):")
print(_dec_cnt.to_string())
print(f"Construction requirements (apply to every split, not counted as failures): "
      f"{', '.join(checks_requirements().id)}")
_dec_conf = _dec_nr[_dec_nr.conflict.fillna(False).astype(bool)]
if len(_dec_conf):
    print("Quantities where sections disagree (kept verdict | hidden FAILs):")
    for r in _dec_conf.itertuples():
        print(f"  {r.quantity}: kept {r.id} = {_dec_status(r.passed, False)} | hidden fails: {r.hidden_fail_ids or '-'}"
              f" | other checks: {r.dup_ids}")

102 checks recorded; 100 remain after de-duplicating shared quantities ('dup' = verdict owned by another section's check of the same quantity; not counted twice)

--- A. targets & covariates (26 checks) ---
  id status  kind dup    metric                          threshold                                                                                              condition / detail
  A1   PASS drift        -2.933 outside 95% t prediction interval~ Advancement base rate of each candidate within the spread~ | VERDICT RULE: FAIL iff >= 1 test survives a Holm ~
 A1d   PASS drift        -1.594 outside 95% t prediction interval~ Number of reported bills (numerator of the Task-2 rate) w~ | VERDICT RULE: FAIL iff >= 1 test survives a Holm ~
 A1b   FAIL drift          2.74 outside 95% t prediction interval~ Share of chamber-passed bills that the Task-2 label misse~ | VERDICT RULE: FAIL iff >= 1 test survives a Holm ~
 A1c   PASS drift        -2.122 outside 95% t prediction interval~ Alternativ

In [172]:
# Chart: check status by section (stacked bars, one bar per section x kind group, all recorded checks).
_dec_c = DEC_ALL.assign(grp=lambda d: np.where(d.requirement, "requirement", d.kind_lbl.replace({"unclass.": "unclassified"})))
_dec_c["st"] = _dec_c.status.str.replace("/REQ", "", regex=False)
_dec_chart1 = (_dec_c.groupby(["section", "grp", "st"]).size().rename("n").reset_index()
               .assign(sec=lambda d: d.section.map(DEC_SECTIONS)))
show_d3(r"""
const {width, height} = opts;
const m = {top: 48, right: 20, bottom: 40, left: 210};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, 'Check status by section and kind (all recorded checks)',
          'Validity FAILs rank splits; drift FAILs pick remedies; requirements are build rules for every split');
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const grpOrder = ['validity', 'drift', 'unclassified', 'requirement'];
const rows = [];
Array.from(new Set(data.map(d => d.sec))).sort().forEach(s => grpOrder.forEach(k => {
  if (data.some(d => d.sec === s && d.grp === k)) rows.push(s + ' | ' + k); }));
const sts = ['FAIL', 'PASS', 'INFO'];
const color = {PASS: viz.palette[2], FAIL: viz.palette[7], INFO: '#b9b8b2'};
const byRow = rows.map(r => { const [s, k] = r.split(' | '); const o = {row: r};
  sts.forEach(t => { const f = data.find(d => d.sec === s && d.grp === k && d.st === t); o[t] = f ? f.n : 0; }); return o; });
const stack = d3.stack().keys(sts)(byRow);
const xs = d3.scaleLinear([0, d3.max(byRow, d => d.FAIL + d.PASS + d.INFO) || 1], [0, W]).nice();
const ys = d3.scaleBand(rows, [0, H]).padding(0.18);
g.append('g').call(d3.axisTop(xs).ticks(6).tickSize(-H)).call(a => a.selectAll('line').attr('stroke', viz.grid))
  .call(a => a.select('.domain').remove()).call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
g.append('g').call(d3.axisLeft(ys).tickSize(0)).call(viz.axis).call(a => a.select('.domain').remove());
const tip = viz.tooltip(el);
stack.forEach(layer => g.selectAll(null).data(layer).join('rect')
  .attr('y', d => ys(d.data.row)).attr('height', ys.bandwidth())
  .attr('x', d => xs(d[0])).attr('width', d => Math.max(0, xs(d[1]) - xs(d[0])))
  .attr('fill', color[layer.key]).attr('stroke', '#fff').attr('stroke-width', 1)
  .on('mousemove', (e, d) => tip.show(`<b>${d.data.row}</b><br>${layer.key}: ${d.data[layer.key]}`, e))
  .on('mouseleave', () => tip.hide()));
viz.legend(svg, sts.map(s => [s, color[s]]), m.left, height - 12);
svg.append('text').attr('x', m.left + W).attr('y', height - 3).attr('text-anchor', 'end').attr('fill', viz.ink2)
  .style('font', '11px sans-serif').text('number of checks');
""", _dec_chart1[["sec", "grp", "st", "n"]], height=470, width=760, name="X_check_status")

### Split scorecard: what supports and what undermines each candidate

For every candidate split (the five base splits plus D's protocol variants) the scorecard combines
(1) the **validity** checks that judged it (passes support the split, fails undermine it), (2) the
unclassified split-dependent checks (shown separately), (3) how many drift checks it exposes, and (4) the
backtest in `SPLIT_RESULTS`: for each task the baseline that **validation** would pick (best mean val
score, as a modeler would choose), its mean val and test score, the mean |val - test| gap per fold
relative to the test score's own 95% CI half-width (ratio < 1 = the gap is within sampling noise),
the fold-to-fold standard deviation of the test score, and the number of test bills. Task 2 uses only
baselines available at introduction (no policy area, per requirement C18 and the D16 drop).
The `SPLIT_RESULTS` baselines are all **unweighted** (every training Congress counts the same). Section D
also scored recency-weighted family members (e.g. `sh_hl1`, half-life 1 Congress) in `D_FAMILY_RESULTS`,
but only for S2 and S3. The `*_rw` columns repeat the val pick with those members added, so S2 and S3
can be compared on a model that is allowed to down-weight old Congresses; `-` means "not evaluated".
**How to read it:** a usable split has 0 validity fails with many checks judged; a split judged by few
checks has not "passed", it was simply not tested. Splits with 1 test fold have no fold variability.

In [173]:
# Per-split scorecard from checks_split_scores() (validity / unclassified / drift), SPLIT_RESULTS and D_FAMILY_RESULTS.
DEC_PRIMARY = {"cosponsor_t0": globals().get("_D_PRIMARY_T1", ("cosponsor_t0", "Recall@20"))[1],
               "cosponsor_t7": globals().get("_D_PRIMARY_T1", ("cosponsor_t0", "Recall@20"))[1],
               "advancement": globals().get("_D_PRIMARY_T2", ("advancement", "AUROC"))[1]}
# Baselines that need the policy area (missing for most bills at t0 <= 7 d: C18) are not a Task-2 option.
DEC_T2_EXCLUDE = [b for b in SPLIT_RESULTS.baseline.unique() if b == "eb_rate"]
DEC_TASK_LABEL = {"cosponsor_t0": "T1 t0=0", "cosponsor_t7": "T1 t0=7", "advancement": "T2"}
# Recency-weighted / near-tie family rows (S2 and S3 only). Task 2 keeps only the t0-safe members (policy area hidden).
_dec_fam = globals().get("D_FAMILY_RESULTS", pd.DataFrame(columns=SPLIT_RESULTS.columns))
_dec_fam = _dec_fam[(_dec_fam.task != "advancement") | _dec_fam.baseline.astype(str).str.endswith("_t0s")]
DEC_FAM_SPLITS = sorted(set(_dec_fam.split))
DEC_RES_EXT = pd.concat([SPLIT_RESULTS, _dec_fam[SPLIT_RESULTS.columns.intersection(_dec_fam.columns)]],
                        ignore_index=True)


def _dec_backtest(split, task, res=None):
    """Val-selected baseline of one split/task and its val/test level, gap, fold sd and size."""
    res = SPLIT_RESULTS if res is None else res
    m = DEC_PRIMARY[task]
    d = res[(res.split == split) & (res.task == task) & (res.metric == m)]
    if task == "advancement":
        d = d[~d.baseline.isin(DEC_T2_EXCLUDE)]
    if d.empty:
        return None
    best = d[d.eval_role == "val"].groupby("baseline").value.mean().idxmax()
    b = d[d.baseline == best]
    v = b[b.eval_role == "val"].set_index("fold")
    t = b[b.eval_role == "test"].set_index("fold")
    gap = (v.value - t.value).dropna()
    hw = ((t.ci_high - t.ci_low) / 2).mean()
    return dict(split=split, task=task, metric=m, baseline=best, val=v.value.mean(), test=t.value.mean(),
                gap_abs=gap.abs().mean(), gap_signed=gap.mean(), test_hw=hw,
                gap_ratio=gap.abs().mean() / hw if hw > 0 else np.nan,
                fold_sd=t.value.std(ddof=1) if len(t) > 1 else np.nan, n_folds=len(t), n_eval=int(t.n_eval.sum()))


DEC_BT = pd.DataFrame([r for s in ALL_SPLITS for k in DEC_PRIMARY for r in [_dec_backtest(s, k)] if r])
DEC_BT_RW = pd.DataFrame([r for s in DEC_FAM_SPLITS for k in DEC_PRIMARY for r in [_dec_backtest(s, k, DEC_RES_EXT)] if r])

import contextlib as _dec_ctx
import io as _dec_io
with _dec_ctx.redirect_stdout(_dec_io.StringIO()):   # the kind=None warning is reported once, below
    DEC_SC_V = checks_split_scores(kinds=("validity",))
DEC_SC_U = checks_split_scores(kinds=(None,))
DEC_SC_D = checks_split_scores(kinds=("drift",))
DEC_UNCLASS = sorted(set(checks_by_split(kinds=(None,)).id))
print(f"Unclassified split-dependent checks (not ranked; used in the robustness re-run): {', '.join(DEC_UNCLASS) or '-'}")

_bt1 = DEC_BT[DEC_BT.task == "cosponsor_t0"].set_index("split")
_bt2 = DEC_BT[DEC_BT.task == "advancement"].set_index("split")
_bt1r = DEC_BT_RW[DEC_BT_RW.task == "cosponsor_t0"].set_index("split") if len(DEC_BT_RW) else pd.DataFrame(columns=["baseline", "test"])
_bt2r = DEC_BT_RW[DEC_BT_RW.task == "advancement"].set_index("split") if len(DEC_BT_RW) else pd.DataFrame(columns=["baseline", "test"])
DEC_SCORE = pd.DataFrame({
    "kind": [split_kind(s) for s in ALL_SPLITS],
    "val_judged": DEC_SC_V.n_judged, "val_fail": DEC_SC_V.n_fail,
    "unc_judged": DEC_SC_U.n_judged, "unc_fail": DEC_SC_U.n_fail,
    "drift_fail": DEC_SC_D.n_fail, "drift_judged": DEC_SC_D.n_judged,
    "folds": _bt1.n_folds, "T1_base": _bt1.baseline, "T1_test": _bt1.test, "T1_gap/hw": _bt1.gap_ratio,
    "T1_fold_sd": _bt1.fold_sd, "T1_n_eval": _bt1.n_eval, "T1_base_rw": _bt1r.baseline, "T1_test_rw": _bt1r.test,
    "T2_base": _bt2.baseline, "T2_test": _bt2.test, "T2_gap/hw": _bt2.gap_ratio, "T2_fold_sd": _bt2.fold_sd,
    "T2_base_rw": _bt2r.baseline, "T2_test_rw": _bt2r.test,
}, index=pd.Index(ALL_SPLITS, name="split"))
print(f"\nScorecard (T1 = cosponsor t0=0 {DEC_PRIMARY['cosponsor_t0']}, T2 = advancement {DEC_PRIMARY['advancement']}; "
      "val-selected baseline; gap/hw = mean |val-test| / test 95% CI half-width; *_rw = val pick incl. recency-weighted "
      f"members, evaluated for {', '.join(DEC_FAM_SPLITS) or 'no split'} only):")
print(DEC_SCORE.round(3).astype(object).where(DEC_SCORE.notna(), "-").to_string())
_dec_nostream = [s for s in SPLIT_DEFS if split_kind(s) in BY_CONGRESS_KINDS and s + "+stream" not in SPLIT_VARIANTS]
print(f"Splits with no +stream variant evaluated (their streaming behaviour is unknown, not a protocol mismatch): "
      f"{', '.join(_dec_nostream) or 'none'}")

print("\nWhat undermines (-) / supports (+) each split (validity checks; unclassified fails in [ ]):")
_dec_lb = checks_by_split(kinds=None)
for s in ALL_SPLITS:
    g = _dec_lb[_dec_lb.split == s]
    gv = g[g.kind == "validity"]
    gu = g[~g.kind.map(lambda k: isinstance(k, str))]
    sup = ", ".join(gv.id[gv.split_passed == True])        # noqa: E712
    und = ", ".join(gv.id[gv.split_passed == False])       # noqa: E712
    uf = ", ".join(gu.id[gu.split_passed == False])        # noqa: E712
    print(f"  {s:10} - {und or 'none':12} + {sup or '-'}" + (f"  [unclassified fails: {uf}]" if uf else ""))

  WARNING checks_split_scores: 15 split-dependent check(s) have kind=None and are NOT scored: B1, B1c, B4, B5a, B9, C1, C2, C23b, C6, C6b, C24, C19, C20, D8, D13. Classify them (record_check(kind=...) or QUANTITY_KIND) or pass kinds including None to score them.
Unclassified split-dependent checks (not ranked; used in the robustness re-run): B1, B1c, B4, B5a, B9, C1, C19, C2, C20, C23b, C24, C6, C6b, D13, D8

Scorecard (T1 = cosponsor t0=0 Recall@20, T2 = advancement AUROC; val-selected baseline; gap/hw = mean |val-test| / test 95% CI half-width; *_rw = val pick incl. recency-weighted members, evaluated for S2, S3 only):
                 kind val_judged val_fail unc_judged unc_fail drift_fail drift_judged folds       T1_base T1_test T1_gap/hw T1_fold_sd T1_n_eval T1_base_rw T1_test_rw         T2_base T2_test T2_gap/hw T2_fold_sd  T2_base_rw T2_test_rw
split                                                                                                                                   

In [174]:
# Chart: split x check matrix of per-split verdicts (validity + unclassified split-dependent checks).
_dec_m = checks_by_split(kinds=("validity", None), include_omitted=True)
_dec_m = _dec_m.assign(v=np.select([_dec_m.split_passed == True, _dec_m.split_passed == False],   # noqa: E712
                                   ["PASS", "FAIL"], "not judged"),
                       grp=_dec_m.kind.map(lambda k: k if isinstance(k, str) else "unclassified"),
                       why=_dec_m.na_reason.fillna("").map(lambda s: _dec_short(s, 90)))
_dec_m = (_dec_m.assign(_o=(_dec_m.grp != "validity").astype(int))
          .sort_values(["_o", "section", "id"], kind="stable").drop(columns="_o"))
show_d3(r"""
const {width, height, splits} = opts;
const m = {top: 52, right: 16, bottom: 34, left: 92};
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, 'Per-split verdicts of split-dependent checks',
          'Left block: validity checks (rank splits). Right block: unclassified (robustness only). Hover for detail.');
const g = svg.append('g').attr('transform', `translate(${m.left},${m.top})`);
const W = width - m.left - m.right, H = height - m.top - m.bottom;
const ids = Array.from(new Set(data.map(d => d.id)));
const xs = d3.scaleBand(ids, [0, W]).padding(0.06), ys = d3.scaleBand(splits, [0, H]).padding(0.08);
const color = {PASS: viz.palette[2], FAIL: viz.palette[7], 'not judged': '#ecebe7'};
const tip = viz.tooltip(el);
g.selectAll(null).data(data).join('rect').attr('x', d => xs(d.id)).attr('y', d => ys(d.split))
  .attr('width', xs.bandwidth()).attr('height', ys.bandwidth()).attr('rx', 2).attr('fill', d => color[d.v])
  .on('mousemove', (e, d) => tip.show(`<b>${d.id} · ${d.split}</b> (${d.grp})<br>${d.v}${d.why ? '<br>' + d.why : ''}`, e))
  .on('mouseleave', () => tip.hide());
const nVal = new Set(data.filter(d => d.grp === 'validity').map(d => d.id)).size;
if (nVal > 0 && nVal < ids.length) g.append('line').attr('x1', xs(ids[nVal]) - 1).attr('x2', xs(ids[nVal]) - 1)
  .attr('y1', -4).attr('y2', H).attr('stroke', viz.ink2).attr('stroke-dasharray', '3,2');
g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).tickSize(0)).call(viz.axis)
  .call(a => a.select('.domain').remove()).selectAll('text').attr('transform', 'rotate(-45)').attr('text-anchor', 'end')
  .attr('dx', '-0.3em').attr('dy', '0.4em');
g.append('g').call(d3.axisLeft(ys).tickSize(0)).call(viz.axis).call(a => a.select('.domain').remove());
viz.legend(svg, Object.keys(color).map(k => [k, color[k]]), m.left, 38);
""", _dec_m[["id", "split", "v", "grp", "why"]], height=400, width=860,
     opts=dict(splits=list(ALL_SPLITS)), name="X_split_check_matrix")

In [175]:
# Chart: val (hollow) and test (filled) scores of the val-selected baseline per fold, by split and task.
_dec_pts = []
for r in DEC_BT[DEC_BT.task.isin(["cosponsor_t0", "advancement"])].itertuples():
    d = SPLIT_RESULTS[(SPLIT_RESULTS.split == r.split) & (SPLIT_RESULTS.task == r.task)
                      & (SPLIT_RESULTS.metric == r.metric) & (SPLIT_RESULTS.baseline == r.baseline)]
    for q in d.itertuples():
        _dec_pts.append(dict(task=f"{DEC_TASK_LABEL[r.task]} {r.metric}",
                             split=r.split, role=q.eval_role, fold=q.fold, value=float(q.value), base=r.baseline,
                             ev="" if pd.isna(q.eval_congress) else str(int(q.eval_congress))))
_dec_pts = pd.DataFrame(_dec_pts)
show_d3(r"""
const {width, height, splits} = opts;
const tasks = Array.from(new Set(data.map(d => d.task)));
const m = {top: 52, right: 14, bottom: 58, left: 50}, gapX = 46;
const svg = d3.select(el).append('svg').attr('width', width).attr('height', height);
viz.title(svg, 'Validation vs test score of the val-selected baseline, every fold of every split',
          'Hollow = validation, filled = test; vertical spread = fold-to-fold variability (one panel per task)');
const PW = (width - m.left - m.right - gapX * (tasks.length - 1)) / tasks.length, H = height - m.top - m.bottom;
const color = {val: viz.palette[0], test: viz.palette[1]};
const tip = viz.tooltip(el);
tasks.forEach((t, i) => {
  const dd = data.filter(d => d.task === t);
  const g = svg.append('g').attr('transform', `translate(${m.left + i * (PW + gapX)},${m.top})`);
  const xs = d3.scaleBand(splits, [0, PW]).padding(0.2);
  const ys = d3.scaleLinear(d3.extent(dd, d => d.value), [H, 0]).nice();
  g.append('g').call(d3.axisLeft(ys).ticks(5).tickSize(-PW).tickFormat(d3.format('.2f')))
    .call(a => a.selectAll('line').attr('stroke', viz.grid)).call(a => a.select('.domain').remove())
    .call(a => a.selectAll('text').attr('fill', viz.ink2).style('font', '11px sans-serif'));
  g.append('g').attr('transform', `translate(0,${H})`).call(d3.axisBottom(xs).tickSize(0)).call(viz.axis)
    .call(a => a.select('.domain').remove()).selectAll('text').attr('transform', 'rotate(-40)').attr('text-anchor', 'end');
  g.append('text').attr('x', 0).attr('y', -8).attr('fill', viz.ink).style('font', '600 11px sans-serif').text(t);
  g.selectAll(null).data(dd).join('circle')
    .attr('cx', d => xs(d.split) + xs.bandwidth() * (d.role === 'val' ? 0.3 : 0.7)).attr('cy', d => ys(d.value)).attr('r', 3.5)
    .attr('fill', d => d.role === 'test' ? color.test : '#fff').attr('stroke', d => color[d.role]).attr('stroke-width', 1.5)
    .on('mousemove', (e, d) => tip.show(`<b>${d.split} · ${d.fold}</b> (${d.role}${d.ev ? ', Congress ' + d.ev : ''})<br>${d.base}: ${d3.format('.3f')(d.value)}`, e))
    .on('mouseleave', () => tip.hide());
});
viz.legend(svg, [['validation', color.val], ['test', color.test]], m.left, height - 8);
""", _dec_pts, height=380, width=880, opts=dict(splits=list(ALL_SPLITS)), name="X_val_test_by_split")

### Recommendation: explicit rules applied to the recorded checks

The recommendation is a short chain of rules. Each rule names the checks it reads, prints the values
it used and its verdict, and says what it implies. A conclusion is labelled **robust** when at least two
independent checks support it and none contradicts it (and no section disagrees on that quantity);
otherwise it is **borderline** and the printout says what to watch. The split ranking is computed per
task (checks are assigned to Task 1, Task 2 or both from their wording) and is re-run with the
unclassified checks included. A pick is labelled ROBUST only if (a) both runs pick the same split, (b) the
criterion that separates it from the runner-up comes from rules that are all robust, and (c) its
model-selection checks were run on that split itself rather than inherited from its base split.

| Rule | Question | If it fails |
|---|---|---|
| R1 | Any validity FAIL for the split (leakage, val does not predict test, too few test Congresses, censoring)? | drop the split for model selection |
| R2 | Cold start: share of eval positives from legislators unseen in train (B1) vs with in-Congress history (B1b/B1c) | inductive features and/or the `+stream` protocol |
| R3 | Does streaming eval-Congress history up to t0 change scores (D13, SPLIT_RESULTS) and keep val honest (D14)? | evaluate a temporal GNN with `+stream` |
| R4 | Party regimes: do they change the effects (A5b, D26, C23b)? | evaluate per regime |
| R4b | Does the split drop the eval Congress's regime from training (C8), and does that matter given R4? | prefer a window that keeps every regime |
| R5 | Expanding vs sliding window, with and without recency weighting (D10, SPLIT_RESULTS, D_FAMILY_RESULTS, A14) | sliding window, or expanding + recency weights |
| R6 | Val predicts test level and model choice (D1, D20, D14, D2b, D11/D12/D12b/D21/D23/D17), per split actually tested | report per-Congress scores; flag inferred evidence |
| R7 | Fold stability and test size (D3, D4, D9, D27) | use several rolling test folds; lock the last Congress |
| R8 | Drift FAILs exposed by the chosen split | remedies (below), not a different split |

In [176]:
# Rule engine: read CHECKS / SPLIT_RESULTS, apply R1-R8, rank splits per task and print the recommendation.
_DEC_C = {c["id"]: c for c in CHECKS}
_DEC_KEPT = checks_summary(requirements=False).set_index("id")


def _dc(cid):
    return _DEC_C.get(cid)


def _dpass(cid):
    """Overall verdict of a check (True / False / None) or 'missing'."""
    c = _dc(cid)
    return "missing" if c is None else c["passed"]


def _dsplit(cid, split):
    c = _dc(cid)
    if c is None or not c.get("by_split"):
        return None
    return c["by_split"].get(split)


def _dconflict(cid):
    return bool(cid in _DEC_KEPT.index and bool(_DEC_KEPT.loc[cid, "conflict"]))


def _dtask(row):
    """Task a check is about, from the wording of its condition: 'T1', 'T2' or 'both'. (The t0_task2 tag is not
    used: checks such as D9 / D26 carry it for their Task-2 part but also cover Task 1.)"""
    t = str(row["condition"]).lower()
    t1 = any(k in t for k in ("task 1", "task-1", "cosponsor"))
    t2 = any(k in t for k in ("task 2", "task-2", "advancement"))
    return "T1" if t1 and not t2 else "T2" if t2 and not t1 else "both"


DEC_RULES = []


def _drule(rid, question, evidence, verdict, implication, support, against=()):
    """Record one rule outcome. support / against = check ids whose result backs / contradicts the verdict."""
    support = [s for s in support if _dc(s) is not None]
    against = [a for a in against if _dc(a) is not None]
    conf = [s for s in list(support) + list(against) if _dconflict(s)]
    robust = len(support) >= 2 and not against and not conf
    DEC_RULES.append(dict(rule=rid, question=question, evidence=evidence, verdict=verdict, implication=implication,
                          support=", ".join(support), against=", ".join(against),
                          strength="robust" if robust else "borderline"))


# --- R1: validity filter (per task) ---------------------------------------------------------------
_dec_lbv = checks_by_split(kinds=("validity",), include_omitted=True)
_dec_lbu = checks_by_split(kinds=(None,), include_omitted=True)
_dec_task_of = {r["id"]: _dtask(r) for r in CHECKS}


def _dec_task_scores(lb, task):
    g = lb[lb.id.map(lambda i: _dec_task_of.get(i, "both") in (task, "both"))]
    out = {}
    for s in ALL_SPLITS:
        h = g[g.split == s]
        out[s] = dict(judged=int(h.judged.sum()), fail=int((h.split_passed == False).sum()),           # noqa: E712
                      fail_ids=", ".join(h.id[h.split_passed == False]))                                # noqa: E712
    return pd.DataFrame(out).T


DEC_TASKS = {"T1": "cosponsor_t0", "T2": "advancement"}
DEC_TS = {t: (_dec_task_scores(_dec_lbv, t), _dec_task_scores(_dec_lbu, t)) for t in DEC_TASKS}
for t, (sv, su) in DEC_TS.items():
    bad = sv[sv.fail > 0]
    _drule(f"R1-{t}", f"{t}: which splits have a validity FAIL?",
           "; ".join(f"{s}: {r.fail_ids}" for s, r in bad.iterrows()) or "none",
           f"drop {', '.join(bad.index) or 'none'}",
           "not usable for model selection / honest reporting of this task",
           support=sorted({i for r in bad.itertuples() for i in r.fail_ids.split(", ") if i}))

# --- R2: cold start -------------------------------------------------------------------------------
DEC_COLD_MAX = 0.02   # same bound as B1b / B1c (share of positives); B1 itself has no threshold (INFO)
_b1 = _dc("B1")
_b1_by = {k: v for k, v in ((_b1 or {}).get("by_split") or {}).items() if isinstance(v, (int, float))}
_b1_max = max(_b1_by.values()) if _b1_by else np.nan
_cold_big = bool(_b1_by) and _b1_max > DEC_COLD_MAX
_stream_ok = _dpass("B1b") is True and _dpass("B1c") in (True, "missing")
_drule("R2", "Cold start: are eval positives from legislators unseen in training?",
       f"B1 train-cold share of positives by split {', '.join(f'{k} {v:.1%}' for k, v in _b1_by.items())} "
       f"(bound {DEC_COLD_MAX:.0%}); B1b +stream = {_dc('B1b')['metric'] if _dc('B1b') else 'n/a'}, "
       f"B1c = {_dc('B1c')['metric'] if _dc('B1c') else 'n/a'}",
       ("cold start is LARGE for a transductive model without in-Congress history" if _cold_big else
        "cold start is small") + ("; resolved once eval-Congress history up to t0 is in the graph" if _stream_ok else ""),
       ("use inductive node features (party, chamber, state, committee, tenure) AND time-filtered in-Congress "
        "history (+stream); never rely on per-legislator embeddings learned only from training Congresses")
       if _cold_big else "a transductive model is acceptable",
       support=["B1", "B1b", "B1c"] if _stream_ok else ["B1"],
       against=[] if _stream_ok or not _cold_big else ["B1b"])

# --- R3: streaming protocol -----------------------------------------------------------------------
DEC_MDE = dict(globals().get("D_MDE", {"cosponsor_t0": 0.01, "cosponsor_t7": 0.01, "advancement": 0.01}))
_dec_bt = DEC_BT.set_index(["split", "task"])


def _dec_gain(task, base="S3"):
    try:
        return float(_dec_bt.loc[(base + "+stream", task), "test"] - _dec_bt.loc[(base, task), "test"])
    except KeyError:
        return np.nan


DEC_STREAM_GAIN = {t: _dec_gain(k) for t, k in DEC_TASKS.items()}
DEC_PREFER_STREAM = {t: bool(DEC_STREAM_GAIN[t] >= DEC_MDE[k]) and _dpass("D14") is not False
                     for t, k in DEC_TASKS.items()}
for t, k in DEC_TASKS.items():
    # Built outside the f-string: nested same-quote f-strings need Python 3.12+.
    _d13_note = f"; D13 = {_dec_fmt(_dc('D13')['metric'])}" if t == "T1" and _dc('D13') else ""
    _drule(f"R3-{t}", f"{t}: does eval-Congress history up to t0 (+stream) change the test score?",
           f"S3+stream - S3 test {DEC_PRIMARY[k]} (val-selected baseline) = {DEC_STREAM_GAIN[t]:+.4f} vs MDE "
           f"{DEC_MDE[k]}{_d13_note}; D14 (val honest under stream) = "
           f"{_dec_status(_dpass('D14'), False) if _dc('D14') else 'n/a'}",
           "yes: evaluate with +stream" if DEC_PREFER_STREAM[t] else "no material gain: plain or +stream are equivalent",
           "the GNN must see the eval Congress's own edges dated <= intro + t0 (time-filtered neighbor sampling)"
           if DEC_PREFER_STREAM[t] else "streaming is optional for this task; keep it for consistency with Task 1",
           support=(["D13", "D14"] + (["B1b"] if t == "T1" else [])) if DEC_PREFER_STREAM[t] else [])

# --- R4: party regimes ----------------------------------------------------------------------------
_c8_bad = [s for s in ALL_SPLITS if _dsplit("C8", s) is False]
_c23_bad = [s for s in ALL_SPLITS if _dsplit("C23b", s) is False]
_regime_stable = _dpass("A5b") is True and _dpass("D26") is True
_drule("R4", "Party control: do regime changes alter the effects a model learns?",
       f"A5b (majority effect matches same regime) = {_dec_status(_dpass('A5b'), False)}; D26 (regime weighting not "
       f"needed) = {_dec_status(_dpass('D26'), False)}; C23b (post-break test units) fails for {', '.join(_c23_bad) or 'none'}",
       "regime effects are stable given chamber-majority features" if _regime_stable else "regime changes effects: evaluate per regime",
       "include chamber-majority / sponsor-in-majority features; report test folds per regime",
       support=[i for i in ("A5b", "D26") if _dpass(i) is True],
       against=[i for i in ("A5b", "D26") if _dpass(i) is False])
# R4b: C8 says a split drops the eval Congress's regime from training. If R4 finds regime effects stable, losing a
# regime matters less, so A5b / D26 PASS count AGAINST "dropping a regime hurts" (the penalty is then borderline).
_drule("R4b", "Does the split drop the eval Congress's party regime from training, and does that matter?",
       f"C8 regime coverage fails for {', '.join(_c8_bad) or 'none'}; but R4 finds regime effects "
       f"{'stable' if _regime_stable else 'NOT stable'} (A5b {_dec_status(_dpass('A5b'), False)}, "
       f"D26 {_dec_status(_dpass('D26'), False)})",
       (f"mild penalty for {', '.join(_c8_bad)}" if _regime_stable else f"avoid {', '.join(_c8_bad)}") if _c8_bad
       else "every split keeps the eval regime",
       ("a missing regime is a coverage gap, not a demonstrated score loss; use it only as a tie-break"
        if _regime_stable else "keep every regime in training (expanding window)") if _c8_bad else "-",
       support=["C8"] if _c8_bad else [],
       against=[i for i in ("A5b", "D26") if _dpass(i) is True] if _c8_bad else [])

# --- R5: expanding vs sliding, with and without recency weighting ---------------------------------
DEC_TOL = {k: v / 2 for k, v in DEC_MDE.items()}


def _dec_win_diff(task, base, res):
    """Mean over folds of test(S3) - test(S2) for one baseline (same test bills per fold); NaN if missing."""
    m = DEC_PRIMARY[task]
    d = res[(res.task == task) & (res.metric == m) & (res.baseline == base) & (res.eval_role == "test")
            & res.split.isin(["S2", "S3"])].pivot_table(index="eval_congress", columns="split", values="value")
    return float((d["S3"] - d["S2"]).mean()) if {"S2", "S3"} <= set(d.columns) and len(d) else np.nan


_rw_base = {"cosponsor_t0": [b for b in ("sh_hl1", "rw_b0.3_hl1") if (DEC_RES_EXT.baseline == b).any()]}
_unw_base = {"cosponsor_t0": [b for b in ("sponsor_hist", "rw2hop") if (SPLIT_RESULTS.baseline == b).any()]}
DEC_WIN = {"T1 unweighted " + b: _dec_win_diff("cosponsor_t0", b, SPLIT_RESULTS) for b in _unw_base["cosponsor_t0"]}
DEC_WIN.update({"T1 recency " + b: _dec_win_diff("cosponsor_t0", b, DEC_RES_EXT) for b in _rw_base["cosponsor_t0"]})
_t2b = _bt2.baseline.get("S3") if "S3" in _bt2.index else None
if _t2b is not None:
    DEC_WIN[f"T2 {_t2b}"] = _dec_win_diff("advancement", _t2b, SPLIT_RESULTS)
for k in ("cosponsor_t0", "advancement"):
    if {"S2", "S3"} <= set(DEC_BT_RW[DEC_BT_RW.task == k].split):
        _r = DEC_BT_RW[DEC_BT_RW.task == k].set_index("split")
        DEC_WIN[f"{DEC_TASK_LABEL[k].split()[0]} val-picked incl. " + ("recency" if k == "cosponsor_t0" else "near-tie family")] = \
            float(_r.loc["S3", "test"] - _r.loc["S2", "test"])
_tol1, _tol2 = DEC_TOL["cosponsor_t0"], DEC_TOL["advancement"]
_unw = [v for k, v in DEC_WIN.items() if k.startswith("T1 unweighted") and np.isfinite(v)]
_rw = [v for k, v in DEC_WIN.items() if (k.startswith("T1 recency") or k == "T1 val-picked incl. recency") and np.isfinite(v)]
_t2w = [v for k, v in DEC_WIN.items() if k.startswith("T2") and np.isfinite(v)]
DEC_SLIDE_BETTER_UNW = bool(_unw) and max(_unw) < -_tol1          # unweighted: sliding wins by more than MDE/2
DEC_EXP_BETTER_RW = bool(_rw) and min(_rw) > _tol1                 # recency-weighted: expanding wins by more than MDE/2
DEC_SLIDE_BETTER_RW = bool(_rw) and max(_rw) < -_tol1
_t2_eq = bool(_t2w) and all(abs(v) <= DEC_MDE["advancement"] for v in _t2w)
_d10 = _dpass("D10")
_a14 = _dc("A14")
if DEC_EXP_BETTER_RW:
    _r5v = "expanding window + recency weighting beats sliding"
elif DEC_SLIDE_BETTER_RW:
    _r5v = "sliding window beats expanding even with recency weighting"
else:
    _r5v = ("window length is a wash ONLY with recency weighting; without it the sliding window is better"
            if DEC_SLIDE_BETTER_UNW else "window length does not matter")
_drule("R5", "Do older Congresses hurt (expanding S3 vs sliding S2), with and without recency weighting?",
       "test S3 - S2: " + ", ".join(f"{k} {v:+.4f}" for k, v in DEC_WIN.items() if np.isfinite(v))
       + f" (tolerance MDE/2 = {_tol1} T1, MDE = {DEC_MDE['advancement']} T2); D10 (recency-weighted not worse) = "
       f"{_dec_status(_d10, False)}; A14 distribution closeness favouring sliding = "
       f"{_dec_fmt(_a14['metric']) if _a14 else 'n/a'} (> 0.5 favours sliding, proxy only)",
       _r5v + ("; Task 2 is equivalent within the MDE" if _t2_eq else ""),
       ("CONDITIONAL: use an expanding window only together with recency weighting (decay chosen on val); with "
        "unweighted count / graph features S2 is as good or better. The window choice gives no score reason to "
        "prefer S3 over S2; only R4b (regime coverage) and fold evidence do") if DEC_SLIDE_BETTER_UNW and not DEC_EXP_BETTER_RW
       else "keep all Congresses 108..c-1 with recency weights" if DEC_EXP_BETTER_RW else "train on the last k Congresses",
       support=(["D10"] if _d10 is True else []),
       against=(["A14"] if _a14 is not None and isinstance(_a14["metric"], (int, float)) and _a14["metric"] > 0.5 else []))
DEC_R5_TEXT = _r5v

# --- R6: val predicts test ------------------------------------------------------------------------
# Selection checks (does val pick the test-best model?) and level checks, read PER SPLIT from by_split: evidence
# from a base split is not transferred to its +stream variant.
DEC_SEL_IDS = [i for i in ("D11", "D12", "D12b", "D21", "D23", "D17") if _dc(i)]
_lvl1 = [i for i in ("D1", "D20", "D14", "D22") if _dc(i)]


def _dec_judged(ids, split, task):
    """Checks among ids that judged this split (by_split True/False; numeric = metric only) and are about task."""
    out = {}
    for i in ids:
        if _dec_task_of.get(i, "both") not in (task, "both"):
            continue
        v = _dsplit(i, split)
        if isinstance(v, (bool, np.bool_)):
            out[i] = bool(v)
    return out


def _dec_sel_status(split, task):
    """(own selection checks {id: pass}, inherited-from-base {id: pass} or {}) for one split."""
    own = _dec_judged(DEC_SEL_IDS, split, task)
    base = SPLIT_VARIANTS[split]["base"] if split in SPLIT_VARIANTS else None
    inh = _dec_judged(DEC_SEL_IDS, base, task) if (base and not own) else {}
    return own, inh


_r6_splits = [s for s in ALL_SPLITS if split_kind(s) in BY_CONGRESS_KINDS and len(split_eval_congresses(s, ("test",))) > 1]
_r6_txt, _r6_inf = [], []
for _s in _r6_splits:
    own, inh = _dec_sel_status(_s, "T1")
    lv = _dec_judged(_lvl1, _s, "T1")
    _r6_txt.append(f"{_s}: level {','.join(f'{i}={_dec_status(v, False)}' for i, v in lv.items()) or 'none'}; selection "
                   + (",".join(f"{i}={_dec_status(v, False)}" for i, v in own.items()) if own else
                      f"NOT RUN (inferred from {SPLIT_VARIANTS[_s]['base']}: {','.join(inh) or 'none'})" if inh else "none"))
    if not own and inh:
        _r6_inf.append(_s)
_sel_ok = all(_dpass(i) is True for i in DEC_SEL_IDS if _dec_task_of.get(i) != "T2")
_tested = [x for x in _r6_splits if _dec_sel_status(x, "T1")[0]]
_drule("R6-T1", "T1: on the splits where selection was tested, does val predict the test level and pick the test-best model?",
       "; ".join(t for x, t in zip(_r6_splits, _r6_txt) if x in _tested),
       (f"yes for {', '.join(_tested)} (by-Congress val is a faithful proxy there)"
        if all(_dpass(i) is True for i in _lvl1) and _sel_ok else "partly"),
       "tune / early-stop on val Congress c-1, refit on 108..c-1 (D19), test on c",
       support=[i for i in _lvl1 + DEC_SEL_IDS if _dpass(i) is True and _dec_task_of.get(i) != "T2"],
       against=[i for i in _lvl1 + DEC_SEL_IDS if _dpass(i) is False and _dec_task_of.get(i) != "T2"])
if _r6_inf:
    # Variants judged on the level only: their selection evidence is inherited, so this conclusion is borderline by
    # construction (support counts only checks that judged the variant itself; the inherited ones are 'against').
    _inf_lvl = sorted({i for x in _r6_inf for i, v in _dec_judged(_lvl1, x, "T1").items() if v})
    _inf_sel = sorted({i for x in _r6_inf for i in _dec_sel_status(x, "T1")[1]})
    _drule("R6-T1s", f"T1: is val a faithful proxy under {', '.join(_r6_inf)} (streaming)?",
           "; ".join(t for x, t in zip(_r6_splits, _r6_txt) if x in _r6_inf),
           "level carries over (tested); model selection NOT tested, inferred from the base split",
           f"before trusting model choices made under {', '.join(_r6_inf)}, rerun D11 / D17 (sponsor_hist and rw2hop "
           "families) on its folds; until then report that selection reliability is inferred from plain S3",
           support=[i for i in _inf_lvl if i in ("D14",)], against=_inf_sel)
_d2b_bad = [s for s in ALL_SPLITS if _dsplit("D2b", s) is False]
_drule("R6-T2", "T2: does the val Congress predict the test AUROC level and the model choice?",
       f"level: D2b = {_dec_status(_dpass('D2b'), False)} (fails for {', '.join(_d2b_bad) or 'none'}); D4 std = "
       f"{_dec_fmt(_dc('D4')['metric']) if _dc('D4') else 'n/a'}; selection D12/D12b = "
       f"{_dec_status(_dpass('D12'), False)}/{_dec_status(_dpass('D12b'), False)}",
       ("model ranking transfers but the AUROC LEVEL does not transfer between Congresses" if _dpass("D2b") is False
        else "level and ranking transfer"),
       ("select Task-2 models on val (rankings transfer), but report per-Congress AUROC and the fold mean and never quote "
        "a val AUROC as the expected test level (D2b). AUROC is invariant to monotone recalibration, so no recalibration "
        "fixes a D2b failure; if calibrated probabilities are needed, recalibrate on the eval Congress's already-resolved "
        "bills (D2b), not on Congress c-1") if _dpass("D2b") is False else
       "select on val and report per-Congress AUROC",
       support=[i for i in ("D12", "D12b", "D6") if _dpass(i) is True] + (["D2b", "D4"] if _dpass("D2b") is False else []),
       against=[])

# --- R7: fold stability, test size, locked test ---------------------------------------------------
_d9_bad = [s for s in ALL_SPLITS if _dsplit("D9", s) is False]
_fp = globals().get("D_FINAL_PROTOCOL", {})
_drule("R7", "Is one test Congress enough, and can the last Congress be locked away?",
       f"D9 single-test-Congress miss risk fails for {', '.join(_d9_bad) or 'none'}; D3 T1 fold CV = "
       f"{_dec_fmt(_dc('D3')['metric']) if _dc('D3') else 'n/a'}; D27 locked test = {_dec_status(_dpass('D27'), False)} "
       f"(locked {_fp.get('locked_test', 'n/a')})",
       "use rolling one-step-ahead test folds" + (f" and lock {_fp.get('locked_test')} as the final test"
                                                  if _dpass("D27") is True else ""),
       "compare architectures on the mean over development folds (paired, same bills); touch the locked Congress once",
       support=[i for i in ("D9", "D27", "D17") if _dc(i) and _dpass(i) in (True, False)],
       against=[])

# --- R8: drift remedies (FAILs of kind 'drift') ----------------------------------------------------
DEC_REMEDY = {
    "label_validity_t2": "Task-2 label misses more chamber-passed bills lately: also report label_alt (reported OR passed)",
    "label_rate_t2_chamber": "per-chamber advancement rate moves: chamber-specific intercepts; if probabilities are needed, "
                             "recalibrate on the eval Congress's already-resolved bills (as D2b), not on c-1",
    "label_rate_t2": "advancement base rate moves: use AUROC and AP-lift, not raw AP or accuracy; recalibrate (if needed) on "
                     "the eval Congress's resolved bills",
    "cosp_density": "positive density moves: compare ranking metrics per Congress; do not compare raw recall across Congresses",
    "orig_share": "original-cosponsor share moves: t0=0 vs t0=7 positives differ by Congress; report both t0",
    "bill_volume": "bill volume moves: normalise counts per Congress (degree / activity features as rates)",
    "model_auroc_transfer_t2": "Task-2 model degrades forward: retrain every Congress, recency-weight, report per Congress",
    "cand_fail_count": "several drifts at once: rely on rolling folds rather than one test Congress",
    "lag_dist": "cosponsor timing moves: window labels at fixed t0 (already) and monitor lag per Congress",
    "sponsor_overlap": "new sponsors: inductive sponsor features plus in-Congress history (+stream)",
}
# Remedies keyed by check id, for drift checks without a quantity tag.
DEC_REMEDY_ID = {
    "D2b": "Task-2 AUROC level does not transfer between Congresses: report per-Congress AUROC and the fold mean, never a "
           "val AUROC as the expected test level (AUROC ignores recalibration, so recalibrating cannot fix this); for "
           "calibrated probabilities recalibrate on the eval Congress's already-resolved bills, not on c-1",
}
DEC_REMEDY_DEFAULT = ("level shift between Congresses: report per Congress; if calibrated scores are needed, recalibrate on "
                      "the eval Congress's already-resolved data, not on the previous Congress")
_dec_drift = checks_summary(requirements=False, kinds=("drift",))
_dec_drift = _dec_drift[_dec_drift.passed.map(lambda v: v is False or v is np.False_)]
DEC_DRIFT_FAILS = [(r.id, r.quantity if isinstance(r.quantity, str) else None) for r in _dec_drift.itertuples()]
_drule("R8", "Which drift checks fail (remedies, not a different split)?",
       ", ".join(f"{i}" + (f" ({q})" if q else "") for i, q in DEC_DRIFT_FAILS) or "none",
       f"{len(DEC_DRIFT_FAILS)} drift FAILs", "see the remedy list below",
       support=[i for i, _ in DEC_DRIFT_FAILS])

DEC_RULES = pd.DataFrame(DEC_RULES)
for r in DEC_RULES.itertuples():
    print(f"[{r.rule}] {r.question} -> {r.verdict}  <{r.strength.upper()}; support: {r.support or '-'}"
          f"{'; against: ' + r.against if r.against else ''}>\n    evidence: {_dec_short(r.evidence, 460)}\n"
          f"    implies : {r.implication}")

[R1-T1] T1: which splits have a validity FAIL? -> drop S1a, S1b, S4, S5, S1a+stream, S1b+stream, S1pool  <ROBUST; support: C5, D1, D20, D7, D9>
    evidence: S1a: D9; S1b: D9; S4: C5; S5: D7; S1a+stream: D9; S1b+stream: D9; S1pool: D1, D20, D9
    implies : not usable for model selection / honest reporting of this task
[R1-T2] T2: which splits have a validity FAIL? -> drop S1a, S1b, S1a+stream, S1b+stream, S1pool  <BORDERLINE; support: D9>
    evidence: S1a: D9; S1b: D9; S1a+stream: D9; S1b+stream: D9; S1pool: D9
    implies : not usable for model selection / honest reporting of this task
[R2] Cold start: are eval positives from legislators unseen in training? -> cold start is LARGE for a transductive model without in-Congress history; resolved once eval-Congress history up to t0 is in the graph  <ROBUST; support: B1, B1b, B1c>
    evidence: B1 train-cold share of positives by split S1a 18.0%, S1b 18.1%, S2 20.5%, S3 20.4%, S1pool 31.4% (bound 2%); B1b +stream = 0.0071, B1c = 0.0145
  

In [177]:
# Rank the surviving splits per task (validity only, then with unclassified checks) and print the decision.


DEC_MIN_JUDGED_FRAC = 0.5   # a split judged by < half as many checks as the best-covered split has too little evidence


def _dec_penalties(s):
    """Rule penalties of a split: {rule: 0/1}. R4b = C8 says the eval regime is missing from training. R5 = the
    window comparison favours the other window by more than MDE/2 for a recency-weighted model (else no penalty)."""
    k = split_kind(s)
    return {"R4b": int(_dsplit("C8", s) is False),
            "R5": int((k == "sliding" and DEC_EXP_BETTER_RW) or (k in ("rolling", "expanding") and DEC_SLIDE_BETTER_RW))}


def _dec_protocol(s, task):
    """'match' / 'plain' (a +stream variant exists and R3 asks for it) / 'no +stream variant evaluated' / 'stream'."""
    want = DEC_PREFER_STREAM[task]
    if s.endswith("+stream"):
        return "match" if want else "stream (not needed)"
    if not want:
        return "match"
    return "plain (stream variant exists)" if s + "+stream" in SPLIT_VARIANTS else "no +stream variant evaluated"


def _dec_rank(task, with_unclassified=False):
    sv, su = DEC_TS[task]
    k = DEC_TASKS[task]
    nj = sv.judged.astype(int) + (su.judged.astype(int) if with_unclassified else 0)
    min_j = int(np.ceil(DEC_MIN_JUDGED_FRAC * nj.max()))
    rows = []
    for s in ALL_SPLITS:
        bt = _dec_bt.loc[(s, k)] if (s, k) in _dec_bt.index else None
        fail = int(sv.loc[s, "fail"]) + (int(su.loc[s, "fail"]) if with_unclassified else 0)
        judged = int(sv.loc[s, "judged"]) + (int(su.loc[s, "judged"]) if with_unclassified else 0)
        proto = _dec_protocol(s, task)
        pen = _dec_penalties(s)
        own, inh = _dec_sel_status(s, task)
        rows.append(dict(split=s, fail=fail, judged=judged, low_evidence=int(judged < min_j),
                         protocol_mismatch=int(proto not in ("match",)), protocol=proto,
                         rule_penalty=sum(pen.values()), pen_rules=",".join(r for r, v in pen.items() if v) or "-",
                         sel_checks=len(own), sel_inferred=int(bool(inh)),
                         folds=int(bt["n_folds"]) if bt is not None else 0,
                         test=float(bt["test"]) if bt is not None else np.nan,
                         drift_fail=int(DEC_SCORE.loc[s, "drift_fail"])))
    r = pd.DataFrame(rows)
    # order: no validity fails, enough evidence, the protocol R3 asks for (a split with no +stream variant is treated
    # like a plain split: its streaming behaviour is unknown), R4b/R5 penalties, more checks judged, more test folds,
    # fewer drift fails. The test score is shown but not used: it differs by protocol and is not a validity criterion.
    r = r.sort_values(["fail", "low_evidence", "protocol_mismatch", "rule_penalty", "judged", "folds", "drift_fail"],
                      ascending=[True, True, True, True, False, False, True], kind="stable").reset_index(drop=True)
    return r


DEC_RECO = {}
_crit = ["fail", "low_evidence", "protocol_mismatch", "rule_penalty", "judged", "folds"]
for t, k in DEC_TASKS.items():
    r0, r1 = _dec_rank(t), _dec_rank(t, with_unclassified=True)
    top0, top1 = r0.split.iloc[0], r1.split.iloc[0]
    clean = r0[(r0.fail == 0) & (r0.low_evidence == 0)]
    runner = r0.split.iloc[1] if len(r0) > 1 else None
    tie = runner is not None and r0.iloc[0][_crit].tolist() == r0.iloc[1][_crit].tolist()
    DEC_RECO[t] = dict(split=top0, robust_to_unclassified=top0 == top1, alt=top1, runner_up=runner, tie=tie,
                       clean=list(clean.split), rank=r0)
    print(f"\n=== {t} ({k}) ranking (validity checks for this task + shared ones) ===")
    print(r0.head(5).round(3).to_string(index=False))
    print(f"With unclassified checks included the top split is {top1} "
          f"({'same' if top0 == top1 else 'DIFFERENT -> borderline'}); usable (0 validity fails, enough evidence): "
          f"{', '.join(clean.split)}; too little evidence: {', '.join(r0.split[(r0.fail == 0) & (r0.low_evidence == 1)]) or '-'}")

print("\n" + "=" * 100 + "\nRECOMMENDATION\n" + "=" * 100)
_lock = _fp.get("locked_test") if _dpass("D27") is True else None
_rs = DEC_RULES.set_index("rule").strength
for t, k in DEC_TASKS.items():
    rc = DEC_RECO[t]
    s = rc["split"]
    base = SPLIT_VARIANTS[s]["base"] if s in SPLIT_VARIANTS else s
    d = SPLIT_DEFS[base]
    bt = _dec_bt.loc[(s, k)]
    risks = [i for i in (DEC_SC_D.loc[s, "fail_ids"] or "").split(", ") if i and _dec_task_of.get(i, "both") in (t, "both")]
    unc = [i for i in (DEC_TS[t][1].loc[s, "fail_ids"] or "").split(", ") if i]
    # which ranking criterion separates the pick from the runner-up, and which rule(s) produced that criterion
    _r = rc["rank"].set_index("split")
    ru = rc["runner_up"]
    _why = next((c for c in _crit if _r.loc[s, c] != _r.loc[ru, c]), "drift_fail")
    if _why == "rule_penalty":
        _ps, _pr = _dec_penalties(s), _dec_penalties(ru)
        _why_rules = [x for x in _ps if _ps[x] != _pr[x]]
    else:
        _why_rules = {"fail": [f"R1-{t}"], "protocol_mismatch": [f"R3-{t}"]}.get(_why, [])
    _weak = [x for x in _why_rules if _rs.get(x) != "robust"]
    own, inh = _dec_sel_status(s, t)
    reasons = []
    if not rc["robust_to_unclassified"]:
        reasons.append(f"including unclassified checks would pick {rc['alt']}")
    if rc["tie"]:
        reasons.append(f"tied with {ru} on every ranking criterion except the drift count")
    if _weak:
        reasons.append(f"separated from {ru} only by borderline rule(s) {', '.join(_weak)}")
    if _why in ("judged", "folds", "drift_fail"):
        reasons.append(f"separated from {ru} only by '{_why}' (amount of evidence, not a rule)")
    if not own:
        reasons.append("selection reliability (does val pick the test-best model?) was not tested on this split"
                       + (f"; inferred from plain {base} ({', '.join(inh)} pass there)" if inh else ""))
    strength = "BORDERLINE" if reasons else "ROBUST"
    print(f"\n{t} ({k}): use {s} = {d['name']}{' + streaming eval-Congress history' if s.endswith('+stream') else ''}  [{strength}]")
    print(f"  definition : {SPLIT_VARIANTS[s]['desc'] if s in SPLIT_VARIANTS else d['desc']}")
    if d["kind"] in BY_CONGRESS_KINDS and "folds" in d:
        dev = [c for c in d["folds"] if c != _lock]
        print(f"  folds      : development test Congresses {dev[0]}-{dev[-1]} (val = c-1, train = "
              f"{'108..c-2' if d['kind'] == 'rolling' else f'last {SLIDE_K} before val'})"
              + (f"; LOCKED final test {_lock} (train {_rng_label(_fp.get('locked_train', []))}, touched once); "
                 f"prospective {_fp.get('prospective_test', IN_PROGRESS)}" if _lock else ""))
    print(f"  backtest   : val-picked '{bt['baseline']}' (unweighted SPLIT_RESULTS baselines) test {DEC_PRIMARY[k]} "
          f"{bt['test']:.3f} over {int(bt['n_folds'])} fold(s), fold sd {bt['fold_sd']:.3f}, mean |val-test| = "
          f"{bt['gap_ratio']:.2f}x the test CI half-width, {int(bt['n_eval']):,} test bills")
    _ok = _dec_lbv[(_dec_lbv.split == s) & (_dec_lbv.split_passed == True)].id                    # noqa: E712
    print(f"  relies on  : validity checks passed for {s} ({t} + shared): "
          f"{', '.join(i for i in _ok if _dec_task_of.get(i, 'both') in (t, 'both')) or '-'}")
    print(f"  selection  : " + (f"tested on {s}: {', '.join(f'{i}={_dec_status(v, False)}' for i, v in own.items())}" if own else
                                f"NOT tested on {s}; selection reliability under {s} is INFERRED from plain {base} "
                                f"({', '.join(f'{i}={_dec_status(v, False)}' for i, v in inh.items()) or 'nothing'}). "
                                f"Rerun D11 / D17 (sponsor_hist and rw2hop families) on {s} folds to confirm"))
    print(f"  risks      : drift FAILs on this split for {t} -> {', '.join(risks) or 'none'}"
          + (f"; unclassified fails -> {', '.join(unc)}" if unc else ""))
    _ru_note = (f"{ru} (loses on '{_why}': {_r.loc[ru, _why]} vs {_r.loc[s, _why]}"
                + (f" [{_r.loc[ru, 'protocol']}]" if _why == "protocol_mismatch" else "")
                + (f"; decided by {', '.join(f'{x} ({_rs.get(x)})' for x in _why_rules)}" if _why_rules else "")
                + f"; runner-up test {_r.loc[ru, 'test']:.3f} vs pick {_r.loc[s, 'test']:.3f})")
    print(f"  runner-up  : {_ru_note}")
    for x in reasons:
        print(f"  borderline : {x}")
    # Window condition (R5): is the pick an expanding window whose advantage depends on recency weighting?
    _wt = {kk: vv for kk, vv in DEC_WIN.items() if kk.startswith(t) and np.isfinite(vv)}
    if split_kind(s) in ("rolling", "expanding") and "S2" in rc["clean"] and _wt:
        _wtxt = ", ".join(f"{kk} {vv:+.4f}" for kk, vv in _wt.items())
        if t == "T1" and DEC_SLIDE_BETTER_UNW and not DEC_EXP_BETTER_RW:
            print(f"  condition  : expanding only WITH recency weighting; otherwise S2 is as good or better (R5, S3 minus S2: "
                  f"{_wtxt}). S2 is a legitimate alternative"
                  + (" (no +stream variant of S2 evaluated yet)" if DEC_PREFER_STREAM[t] and "S2" in _dec_nostream else ""))
        elif t != "T1":
            _mx = max(abs(v) for v in _wt.values())
            print(f"  condition  : S3 vs S2 test (S3 minus S2): {_wtxt}; "
                  + (f"|diff| <= MDE {DEC_MDE[k]}, so the windows are equivalent on score and S2 is a legitimate alternative"
                     if _mx <= DEC_MDE[k] else "the window matters on score: prefer the better one")
                  + "; no recency-weighted Task-2 baseline was evaluated")
    if t == "T2" and _dpass("D2b") is False:
        print("  reporting  : AUROC level does not transfer between Congresses (D2b): report per-Congress AUROC and the fold "
              "mean; never quote a val AUROC as the expected test level")
    _fids, _uids = DEC_TS[t][0]["fail_ids"], DEC_TS[t][1]["fail_ids"]
    print("  not for selection: " + ", ".join(
        f"{x} ({_fids[x] or 'too little evidence' + (', unclassified fail ' + _uids[x] if _uids[x] else '')})"
        for x in ALL_SPLITS if x not in rc["clean"]))
    DEC_RECO[t].update(strength=strength, reasons=reasons)

# One pipeline for both tasks? Use Task 1's protocol for Task 2 if it is usable there and costs < 1 MDE.
_s1, _s2 = DEC_RECO["T1"]["split"], DEC_RECO["T2"]["split"]
if _s1 != _s2:
    _g = float(_dec_bt.loc[(_s1, "advancement"), "test"] - _dec_bt.loc[(_s2, "advancement"), "test"])
    _share = _s1 in DEC_RECO["T2"]["clean"] and _g > -DEC_MDE["advancement"]
    print(f"\nOne pipeline for both tasks: {_s1} for Task 2 is {'usable' if _s1 in DEC_RECO['T2']['clean'] else 'NOT usable'} "
          f"and changes the Task-2 test {DEC_PRIMARY['advancement']} by {_g:+.4f} (MDE {DEC_MDE['advancement']}) -> "
          + (f"run both tasks on {_s1} (same folds; Task 2 simply gains little from streaming)" if _share
             else f"keep {_s2} for Task 2")
          + (f". Note: Task-2 selection checks were not run on {_s1} either (inferred from plain "
             f"{SPLIT_VARIANTS[_s1]['base'] if _s1 in SPLIT_VARIANTS else _s1})" if _share and not _dec_sel_status(_s1, 'T2')[0] else ""))

print("\nRemedies for the drift the chosen splits expose (R8):")
for i, q in DEC_DRIFT_FAILS:
    print(f"  {i:5} {q or '(no quantity)':24} -> {DEC_REMEDY_ID.get(i) or DEC_REMEDY.get(q, DEC_REMEDY_DEFAULT)}")
print("\nRobust vs borderline conclusions:")
print("  picks     : " + "; ".join(f"{t} {DEC_RECO[t]['split']} [{DEC_RECO[t]['strength']}]" for t in DEC_TASKS))
for lab in ("robust", "borderline"):
    rr = DEC_RULES[DEC_RULES.strength == lab]
    print(f"  {lab:10}: " + "; ".join(f"{r.rule} ({r.verdict})" for r in rr.itertuples()))
# Evidence gaps the decision depends on (computed from what was and was not evaluated).
_gaps = []
_rw_missing = [x for x in ALL_SPLITS if split_kind(x) in BY_CONGRESS_KINDS and x not in DEC_FAM_SPLITS]
if _rw_missing:
    _gaps.append(f"recency-weighted baselines (e.g. sh_hl1) exist only for {', '.join(DEC_FAM_SPLITS)}; not evaluated for "
                 f"{', '.join(_rw_missing)} (add them to SPLIT_RESULTS so a recency-weighted val pick is possible there)")
if DEC_PREFER_STREAM["T1"] and _dec_nostream:
    _gaps.append(f"no +stream variant for {', '.join(_dec_nostream)}: Task-1 standing under streaming is unknown")
for t in DEC_TASKS:
    _own, _inh = _dec_sel_status(DEC_RECO[t]["split"], t)
    if not _own:
        _gaps.append(f"{t} pick {DEC_RECO[t]['split']}: no model-selection check (D11 / D17 / D21 / D23) run on its folds")
print("\nEvidence gaps (close these before treating the borderline picks as settled):")
for x in _gaps or ["none"]:
    print(f"  - {x}")
print("\nMonitor on every new fold / Congress: " + ", ".join(sorted({q for _, q in DEC_DRIFT_FAILS if q}))
      + "; plus the validity checks the choice relies on (rerun this notebook when a Congress completes).")


=== T1 (cosponsor_t0) ranking (validity checks for this task + shared ones) ===
     split  fail  judged  low_evidence  protocol_mismatch                      protocol  rule_penalty pen_rules  sel_checks  sel_inferred  folds   test  drift_fail
 S3+stream     0       8             0                  0                         match             0         -           0             1      7 0.2960           9
        S3     0      11             0                  1 plain (stream variant exists)             0         -           4             0      7 0.2590           9
        S2     0       9             0                  1  no +stream variant evaluated             1       R4b           3             0      7 0.2720          10
S1a+stream     1       7             0                  0                         match             0         -           0             0      1 0.3070           6
S1b+stream     1       7             0                  0                         match            

### Practical guidance for the GNN

Read this together with the RECOMMENDATION printout above; the guidance assumes its rules came out the
way they do in the current run (re-check if a rerun flips a verdict).

- **Folds.** Develop on rolling-origin folds: for each development test Congress *c*, train on 108..c-2,
  tune and early-stop on *c-1*, refit on 108..c-1 with the tuned hyperparameters, and score *c*. Compare
  architectures on the mean over the development folds (paired, same bills), never on one Congress. Keep
  the last complete Congress (`D_FINAL_PROTOCOL['locked_test']`) as a locked test you score once, and treat
  the in-progress 119th as a prospective check only. A single by-Congress split (train 108-115 / val 116 /
  test 117 or 118) is fine for quick debugging, but R7/D9 show one test Congress cannot settle close model
  comparisons.
- **Inductive, not purely transductive.** Many eval positives come from legislators absent from the
  training Congresses (R2). Give legislator nodes inductive features (party as of the Congress, chamber,
  state, committee seats, tenure, majority status) and do not depend on per-node ID embeddings learned only
  from the training Congresses. GraphSAGE-style aggregation or a heterogeneous GNN with feature encoders
  fits; a lookup-table embedding model does not.
- **Time-filtered neighbor sampling (the `+stream` protocol).** For a bill scored at t0, the graph may
  contain every edge dated <= introduced_date + t0: earlier Congresses plus the eval Congress's own
  sponsorships and cosponsorships so far. Never use the eval Congress's final graph (B10), committee edges
  recorded after t0 (B7), related-bill links not yet known (B6b), or snapshot count fields (C11). Build
  training examples the same way, from as-of-t0 snapshots (B12), so train and eval inputs match.
- **Window and weighting (conditional).** The expanding window is only as good as the sliding one *if*
  the model down-weights old Congresses (R5/D10). With unweighted count or graph features, the last-3-
  Congress sliding window (S2) scores as well or better. So: if you use an expanding window, add a recency
  weight or edge-time decay and choose its half-life on validation. If you do not, S2 is a legitimate
  choice. The remaining arguments for the expanding window are weak (R4b: S2 leaves the eval Congress's
  party regime out of training, but regime effects look stable, A5b/D26) or about evidence (more checks
  were run on S3). Add chamber-majority / sponsor-in-majority features rather than regime-specific weighting (R4/D26).
- **Model selection under streaming.** The selection checks (D11/D17/D21/D23) were run on plain S3
  folds, not on S3+stream. Until they are rerun under streaming, treat "val picks the test-best model" for
  the streaming protocol as inferred, not tested. D14 only shows that the score *level* carries over.
- **Task 2.** Pick models on validation (rankings transfer, D12/D12b). The AUROC *level* does not carry
  over from one Congress to the next (D2b), so report per-Congress AUROC plus the fold mean, and never quote
  a val AUROC as the expected test level. AUROC does not change under monotone recalibration, so
  recalibrating cannot fix this. If you need calibrated probabilities, recalibrate on the eval Congress's
  already-resolved bills (labels are known 180 days after introduction), not on Congress c-1. Hide the
  policy area for bills younger than its availability lag (C18/C25). Task 2 can share Task 1's folds.
- **Monitor.** Every quantity in the "Monitor" line drifts between Congresses. Rerun this notebook when a
  Congress completes. If a validity check the choice relies on flips, the ranking changes automatically.